# Safe Procure
### Multi-agent vendor risk intelligence built on CrewAI

**CrewAI final assignment · FORE School of Management · Group No. 16**  
Prateek Kumar Singh · Shardul Anil Awate · Sanatan Jain

**Business problem.** Procurement teams score suppliers on internal performance data, but the risks that stop a
production line are usually a mix of *internal* signals (slipping on-time delivery, rising defects, compliance incidents)
and *external* ones (commodity shortages, port congestion, regional disruption). Spreadsheet scorecards miss the second
half and never say *what to do*.

**Safe Procure** answers a plain-English sourcing question (for example *"Which vendors put our Q4 production at risk, and
what should we do about each one?"*) with a crew of **10 specialist agents** that audit the data, analyse performance and
spend in parallel, read **live news**, forecast next quarter, decide **Keep / Watch / Develop / Replace** for every vendor,
plan mitigation, **self-review**, and write a board-ready brief, plus a **Procurement Copilot** for follow-up questions.

| Assignment requirement | How Safe Procure meets it |
|---|---|
| Minimum 2 sub-agents | **10 agents + 1 copilot**, orchestrated with a CrewAI **Flow**: parallel fan-out, router, QA revision loop |
| Minimum 1 tool | **8 custom tools**: data audit, scorecard, vendor profile, spend analytics, forecasting, **live web news search**, alternative sourcing, scenario re-scoring |
| Dashboard / report | Inline dashboard in this notebook, a **9-tab web app** with live agent mission control, a downloadable **HTML executive report** and CSV |
| Dynamic agent input | Free-text question + form (weights, scope, thresholds, dataset size/seed) **or your own CSV/Excel upload**; the Intake agent re-plans the work for each question |
| Colab notebook | This notebook runs top to bottom and shows the workflow, sub-agents, tools, inputs and final output |

> All vendor names in the sample data are fictional. The market news is real and fetched live on every run.

## Architecture

```
                         ┌──────────────────────┐
  your question + form ─▶│ 1  Intake Router      │  turns the request into a plan
                         └──────────┬───────────┘
                         ┌──────────▼───────────┐
                         │ 2  Data Quality Audit │  tool: audit_dataset
                         └──────────┬───────────┘
          ┌─────────────────────────┼─────────────────────────┐   (run in parallel)
┌─────────▼─────────┐   ┌───────────▼──────────┐   ┌───────────▼──────────┐
│ 3 Performance     │   │ 4 Cost & Spend       │   │ 5 Market Risk        │
│   Analyst         │   │   Analyst            │   │   Researcher         │
│ scorecard,profile │   │ spend_analytics      │   │ live news search     │
└─────────┬─────────┘   └───────────┬──────────┘   └───────────┬──────────┘
          └─────────────────────────┼─────────────────────────┘   (join: and_)
                         ┌──────────▼───────────┐
                         │ 6  Forecast Agent     │  tool: forecast_performance
                         └──────────┬───────────┘
                         ┌──────────▼───────────┐◀─────────────┐
                         │ 7  Strategist         │  scenario    │ revise (max 1)
                         └──────────┬───────────┘  re-scoring   │
                              ◇ risk gate ◇                     │
                  high risk ┌───────┴────────┐ low risk         │
                 ┌──────────▼──────────┐     │                  │
                 │ 8 Mitigation Planner │    │                  │
                 └──────────┬──────────┘     │                  │
                         ┌──▼────────────────▼──┐               │
                         │ 9  QA Reviewer (router)│──────────────┘
                         └──────────┬───────────┘  approve
                         ┌──────────▼───────────┐
                         │ 10 Executive Writer   │  board-ready brief
                         └──────────────────────┘
```

**Engineering highlights**

* **Deterministic numbers, agentic reasoning.** Every KPI, score, forecast and savings figure is computed in Python
  (pandas/numpy). Agents reason over tool outputs, so the numbers are auditable.
* **Grounding guard.** After every agent, code checks that each vendor it names exists, that savings figures stay
  within what the data supports, and that key numbers trace back to computed facts. Anything else is removed and logged.
* **Resilient multi-provider router.** Free-tier limits (measured: Groq 8,000 tokens/min per model; Gemini Flash 20
  requests/day) would stall a 10-agent crew. A custom `ResilientLLM` gives each agent a home model, paces a
  sliding-window token budget per model, skips models whose per-request ceiling is too small, and fails over across
  **5 models on 2 providers** (Groq GPT-OSS 120B/20B, Qwen 3.8, Gemini Flash-Lite, Gemma) without losing the
  conversation.
* **Self-correction.** Guardrails plus an independent QA agent on a different model family can send the strategy
  back for one revision.

## 1 · Setup
Installs the pinned libraries (about one minute on Colab).

In [1]:
import sys, os, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "crewai==1.15.23", "ddgs", "fastapi",
                    "uvicorn[standard]", "python-multipart", "plotly", "openpyxl"], check=True)
os.environ["CREWAI_TRACING_ENABLED"] = "false"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.makedirs("safeprocure/web", exist_ok=True)
print("Python", sys.version.split()[0], "| Colab" if IN_COLAB else "| local")

Python 3.12.10 | local


## 2 · API keys
**For the reviewer:** keys for this submission are already filled in below, so *Runtime › Run all* works with no setup.
They are dedicated free-tier keys that will be revoked after grading.

Otherwise keys are read from **Colab Secrets** (key icon in the left sidebar: add `GROQ_API_KEY` and `GEMINI_API_KEY`),
or you are prompted with a hidden input. They are never printed or written to disk. Both providers have free tiers:
[console.groq.com](https://console.groq.com) and [aistudio.google.com](https://aistudio.google.com). One key is enough;
two give the router more capacity.

In [2]:
# Submission keys (dedicated free-tier keys, revoked after grading). Empty = use Colab Secrets or a prompt.
GROQ_KEY = ""
GEMINI_KEY = ""

from getpass import getpass
for name, pasted in (("GROQ_API_KEY", GROQ_KEY), ("GEMINI_API_KEY", GEMINI_KEY)):
    if os.environ.get(name):
        continue
    val = pasted.strip() or None
    if not val and IN_COLAB:
        try:
            from google.colab import userdata
            val = userdata.get(name)
        except Exception:
            val = None
    if not val:
        val = getpass(f"{name} (press Enter to skip): ").strip()
    if val:
        os.environ[name] = val
print({n: ("set" if os.environ.get(n) else "missing") for n in ("GROQ_API_KEY", "GEMINI_API_KEY")})

{'GROQ_API_KEY': 'set', 'GEMINI_API_KEY': 'set'}


## 3 · The Safe Procure package
Each cell below writes one module. Expand a cell to read the code.

**`safeprocure/config.py`** · Model roster and measured free-tier limits. Each agent gets a *home* model; agents that run in parallel get different homes.

In [3]:
%%writefile safeprocure/config.py
"""Model roster and runtime configuration for Safe Procure.

API keys are read from environment variables only (GROQ_API_KEY, GEMINI_API_KEY).
They are never written to disk by this package.
"""
from __future__ import annotations

import os
from dataclasses import dataclass, field

GROQ_BASE = "https://api.groq.com/openai/v1"
GEMINI_BASE = "https://generativelanguage.googleapis.com/v1beta/openai/"


@dataclass(frozen=True)
class Endpoint:
    """One concrete model on one provider."""

    provider: str          # "groq" | "gemini"
    model: str
    tpm: int               # tokens-per-minute budget we pace against
    label: str             # short display name for the UI
    rpm: int = 25          # requests-per-minute budget we pace against
    latency: float = 3.0   # typical seconds per call (measured), used for latency-aware routing

    @property
    def base_url(self) -> str:
        return GROQ_BASE if self.provider == "groq" else GEMINI_BASE

    @property
    def api_key(self) -> str | None:
        env = "GROQ_API_KEY" if self.provider == "groq" else "GEMINI_API_KEY"
        return os.environ.get(env) or None

    @property
    def key(self) -> str:
        return f"{self.provider}:{self.model}"


# Measured on this account (2026-10-02):
#   Groq: every model returns x-ratelimit-limit-tokens 8000/min and 1000 requests/day.
#   Gemini free tier: gemini-3.x-flash allows only 20 requests/day/model (error detail
#   GenerateRequestsPerDayPerProjectPerModel-FreeTier), so Flash is a last resort.
#   Flash-Lite and Gemma 4 accept tool calls and have larger free quotas
#   (exact RPM/TPM for these is UNVERIFIED - paced conservatively below).
# Gemma 4 measured 25-90 s per call (and empty replies), so it is kept only as a last resort.
# Each model has its own bucket, so giving parallel agents different primary models
# multiplies throughput instead of making them queue behind one another.
GROQ_120B = Endpoint("groq", "openai/gpt-oss-120b", 7000, "GPT-OSS 120B", 28, 2.5)
GROQ_QWEN = Endpoint("groq", "qwen/qwen3.8-27b", 7000, "Qwen 3.8 27B", 28, 1.5)
GROQ_20B = Endpoint("groq", "openai/gpt-oss-20b", 7000, "GPT-OSS 20B", 28, 1.5)
GEMMA_26B = Endpoint("gemini", "gemma-4-26b-a4b-it", 14_000, "Gemma 4 26B", 24, 25.0)
GEMMA_31B = Endpoint("gemini", "gemma-4-31b-it", 14_000, "Gemma 4 31B", 24, 35.0)
GEMINI_LITE = Endpoint("gemini", "gemini-3.1-flash-lite", 200_000, "Gemini 3.1 Flash-Lite", 12, 6.0)
GEMINI_FLASH = Endpoint("gemini", "gemini-3.8-flash", 200_000, "Gemini 3.8 Flash", 4, 15.0)

ENDPOINTS = (GROQ_120B, GROQ_QWEN, GROQ_20B, GEMMA_26B, GEMMA_31B, GEMINI_LITE, GEMINI_FLASH)


@dataclass(frozen=True)
class Tier:
    """An ordered failover chain. The first healthy endpoint with budget wins."""

    name: str
    chain: tuple[Endpoint, ...] = field(default_factory=tuple)


TIERS: dict[str, Tier] = {
    # Each agent has a "home" model (first entry). Agents that run at the same time get
    # different homes so they never queue behind each other.
    "intake": Tier("intake", (GROQ_20B, GEMINI_LITE, GROQ_QWEN)),
    # Flash-Lite hallucinated vendor names in tests, so it only leads tasks that do not name vendors.
    "lite": Tier("lite", (GEMINI_LITE, GROQ_20B, GROQ_QWEN, GROQ_120B, GEMMA_26B)),  # Gemma: last resort only
    "analysis_b": Tier("analysis_b", (GROQ_120B, GROQ_QWEN, GROQ_20B, GEMINI_LITE)),
    "analysis": Tier("analysis", (GROQ_QWEN, GROQ_120B, GROQ_20B, GEMINI_LITE)),
    "research": Tier("research", (GROQ_20B, GEMINI_LITE, GROQ_QWEN, GROQ_120B)),
    "reasoning": Tier("reasoning", (GROQ_120B, GROQ_QWEN, GEMINI_LITE, GEMINI_FLASH)),
    # Independent reviewer: a different model family from the strategist
    "review": Tier("review", (GROQ_QWEN, GROQ_120B, GROQ_20B, GEMINI_LITE)),
}


def keys_status() -> dict[str, bool]:
    return {
        "groq": bool(os.environ.get("GROQ_API_KEY")),
        "gemini": bool(os.environ.get("GEMINI_API_KEY")),
    }


DEFAULT_WEIGHTS = {
    "delivery": 30,
    "quality": 25,
    "cost": 20,
    "responsiveness": 10,
    "compliance": 15,
}

Writing safeprocure/config.py


**`safeprocure/telemetry.py`** · Thread-safe event log. Every stage, tool call, model call and failover is recorded; the web app streams it live.

In [4]:
%%writefile safeprocure/telemetry.py
"""Thread-safe event log for one analysis run.

Every agent stage, tool call, LLM call and failover is recorded here. The web app
streams these events live (Server-Sent Events) to draw the mission-control view, and
the notebook prints them as a readable trace.
"""
from __future__ import annotations

import threading
import time
from collections import defaultdict
from typing import Any, Callable


class RunTelemetry:
    def __init__(self, run_id: str, printer: Callable[[dict], None] | None = None):
        self.run_id = run_id
        self.t0 = time.time()
        self._events: list[dict[str, Any]] = []
        self._cond = threading.Condition()
        self._printer = printer
        self.usage: dict[str, dict[str, float]] = defaultdict(
            lambda: {"calls": 0, "prompt_tokens": 0, "completion_tokens": 0, "seconds": 0.0, "failovers": 0}
        )
        self.closed = False

    # ------------------------------------------------------------------ write
    def emit(self, type_: str, **data: Any) -> dict[str, Any]:
        with self._cond:
            ev = {"seq": len(self._events), "t": round(time.time() - self.t0, 2), "type": type_, **data}
            self._events.append(ev)
            self._cond.notify_all()
        if self._printer:
            try:
                self._printer(ev)
            except Exception:
                pass
        return ev

    def record_usage(self, endpoint_key: str, prompt: int, completion: int, seconds: float) -> None:
        with self._cond:
            u = self.usage[endpoint_key]
            u["calls"] += 1
            u["prompt_tokens"] += prompt
            u["completion_tokens"] += completion
            u["seconds"] += seconds

    def record_failover(self, endpoint_key: str) -> None:
        with self._cond:
            self.usage[endpoint_key]["failovers"] += 1

    def close(self) -> None:
        with self._cond:
            self.closed = True
            self._cond.notify_all()

    # ------------------------------------------------------------------- read
    def since(self, seq: int, timeout: float = 15.0) -> list[dict[str, Any]]:
        """Block until events after `seq` exist (or timeout / closed)."""
        with self._cond:
            if len(self._events) <= seq and not self.closed:
                self._cond.wait(timeout)
            return self._events[seq:]

    @property
    def events(self) -> list[dict[str, Any]]:
        with self._cond:
            return list(self._events)

    def usage_summary(self) -> dict[str, Any]:
        with self._cond:
            per = {k: dict(v) for k, v in self.usage.items()}
        total_in = sum(v["prompt_tokens"] for v in per.values())
        total_out = sum(v["completion_tokens"] for v in per.values())
        return {
            "per_model": per,
            "total_tokens": int(total_in + total_out),
            "prompt_tokens": int(total_in),
            "completion_tokens": int(total_out),
            "llm_calls": int(sum(v["calls"] for v in per.values())),
            "failovers": int(sum(v["failovers"] for v in per.values())),
            "elapsed": round(time.time() - self.t0, 1),
        }

Writing safeprocure/telemetry.py


**`safeprocure/llm.py`** · **ResilientLLM**: a CrewAI `BaseLLM` that paces token budgets, routes by latency and capacity, and fails over across providers without losing tool-calling history.

In [5]:
%%writefile safeprocure/llm.py
"""ResilientLLM: a CrewAI-compatible LLM that routes across providers.

Why: free-tier Groq allows ~8,000 tokens/minute per model. A 10-agent run needs far more.
This class gives every agent an ordered *failover chain* of (provider, model) endpoints and

  1. paces requests with a process-wide sliding-window token budget per endpoint,
  2. proactively skips an endpoint whose budget is exhausted (no wasted 429s),
  3. on a real error (429 / 5xx / timeout / bad tool format) puts that endpoint on
     cooldown and transparently retries on the next endpoint,
  4. reports every call, token count and failover to the run's telemetry.

Native tool calling is delegated to CrewAI's own OpenAI-compatible client, so agents keep
full function-calling support on Groq and Gemini.
"""
from __future__ import annotations

import asyncio
import logging
import re
import threading
import time
from collections import deque
from typing import Any

from pydantic import ConfigDict, PrivateAttr

from crewai.llms.base_llm import BaseLLM
from crewai.llms.providers.openai.completion import OpenAICompletion

from .config import TIERS, Endpoint


class _QuietProviderErrors(logging.Filter):
    """CrewAI logs every provider error at ERROR level; the router already records and handles them."""

    def filter(self, record: logging.LogRecord) -> bool:
        return not str(record.getMessage()).startswith(("OpenAI API call failed", "Error during"))


for _h in logging.getLogger().handlers or [logging.lastResort]:
    if _h is not None:
        _h.addFilter(_QuietProviderErrors())
logging.getLogger().addFilter(_QuietProviderErrors())
for _name in ("ddgs", "primp", "httpx"):
    logging.getLogger(_name).setLevel(logging.CRITICAL)
try:   # Safe Procure renders its own trace; switch off CrewAI's decorative console panels
    from crewai.events.event_listener import event_listener as _crewai_listener
    _crewai_listener.formatter.verbose = False
except Exception:  # pragma: no cover - internal API may move between CrewAI versions
    pass


class _Pacer:
    """Process-wide sliding 60s token window + cooldowns, one per endpoint."""

    def __init__(self) -> None:
        self._lock = threading.Lock()
        self._window: dict[str, deque[tuple[float, int]]] = {}
        self._cooldown: dict[str, float] = {}

    def _used(self, key: str, now: float) -> int:
        q = self._window.setdefault(key, deque())
        while q and now - q[0][0] > 60:
            q.popleft()
        return sum(t for _, t in q)

    def wait_needed(self, ep: Endpoint, est: int) -> float:
        """Seconds to wait before `est` tokens fit in this endpoint's budget (inf if cooling)."""
        now = time.time()
        with self._lock:
            cool = self._cooldown.get(ep.key, 0) - now
            if cool > 0:
                return cool
            used = self._used(ep.key, now)
            q = self._window[ep.key]
            if len(q) + 1 > ep.rpm:                       # request-rate budget
                return max(0.0, 60 - (now - q[0][0])) + 0.25
            if used + est <= ep.tpm:
                return 0.0
            # time until enough old entries expire
            freed, need = 0, used + est - ep.tpm
            for ts, toks in q:
                freed += toks
                if freed >= need:
                    return max(0.0, 60 - (now - ts)) + 0.25
            return 60.0

    def reserve(self, ep: Endpoint, tokens: int) -> None:
        with self._lock:
            self._window.setdefault(ep.key, deque()).append((time.time(), tokens))

    def settle(self, ep: Endpoint, est: int, actual: int) -> None:
        """Replace the reservation estimate with the real token count."""
        with self._lock:
            q = self._window.setdefault(ep.key, deque())
            for i in range(len(q) - 1, -1, -1):
                if q[i][1] == est:
                    q[i] = (q[i][0], actual)
                    return

    def cool(self, ep: Endpoint, seconds: float) -> None:
        with self._lock:
            self._cooldown[ep.key] = max(self._cooldown.get(ep.key, 0), time.time() + seconds)

    def snapshot(self) -> dict[str, dict[str, float]]:
        now = time.time()
        with self._lock:
            return {
                k: {"used": self._used(k, now), "cooldown": max(0.0, self._cooldown.get(k, 0) - now)}
                for k in list(self._window) + [c for c in self._cooldown if c not in self._window]
            }


PACER = _Pacer()
HOME_PATIENCE = 8.0     # seconds an agent will wait for its home model before re-routing
RETRY_SOON = 30.0       # a transiently failed endpoint ready within this beats a longer wait elsewhere


def _estimate_tokens(messages: Any) -> int:
    if isinstance(messages, str):
        chars = len(messages)
    else:
        chars = sum(len(str(m.get("content", ""))) for m in messages or [])
    return int(chars / 3.0) + 900  # + system prompt / tool schemas overhead


def _retry_after(err: Exception) -> float:
    if "PerDay" in str(err):                      # daily free-tier quota: park this model for an hour
        return 3600.0
    m = re.search(r"try again in ([\d.]+)\s*(ms|s|m)", str(err), re.I)
    if m:
        v = float(m.group(1))
        return v / 1000 if m.group(2).lower() == "ms" else v * 60 if m.group(2).lower() == "m" else v
    return 20.0


def _kind(err: Exception) -> str:
    s = f"{type(err).__name__} {err}".lower()
    if "413" in s or "request too large" in s:
        return "too_large"
    if "429" in s or "rate limit" in s or "rate_limit" in s or "quota" in s or "resource_exhausted" in s:
        return "rate_limit"
    if "401" in s or "403" in s or "api key" in s or "unauthorized" in s or "permission" in s:
        return "auth"
    if "timeout" in s or "timed out" in s:
        return "timeout"
    if any(c in s for c in ("500", "502", "503", "504", "overloaded", "unavailable")):
        return "server"
    return "error"


class ResilientLLM(BaseLLM):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    llm_type: str = "resilient"
    tier: str = "analysis"
    agent_label: str = "agent"
    max_completion: int = 2600
    telemetry: Any = None

    _inner: dict[str, OpenAICompletion] = PrivateAttr(default_factory=dict)
    _last_endpoint: str = PrivateAttr(default="")

    def __init__(self, tier: str, agent_label: str, telemetry: Any = None, **kw: Any):
        chain = TIERS[tier].chain
        super().__init__(model=chain[0].model, tier=tier, agent_label=agent_label, telemetry=telemetry,
                         temperature=kw.pop("temperature", 0.2), **kw)

    # ------------------------------------------------------------- helpers
    @property
    def chain(self) -> tuple[Endpoint, ...]:
        return tuple(ep for ep in TIERS[self.tier].chain if ep.api_key)

    def _client(self, ep: Endpoint) -> OpenAICompletion:
        if ep.key not in self._inner:
            extra: dict[str, Any] = {}
            if ep.provider == "groq" and "gpt-oss" in ep.model:
                extra["reasoning_effort"] = "low"
            if ep.provider == "groq" and "qwen" in ep.model:
                extra["reasoning_effort"] = "none"   # no hidden thinking tokens: saves TPM budget
            if ep.model.startswith("gemini-"):
                extra["reasoning_effort"] = "low"
            self._inner[ep.key] = OpenAICompletion(
                model=ep.model, api_key=ep.api_key, base_url=ep.base_url, provider="openai",
                temperature=self.temperature, max_completion_tokens=self.max_completion,
                timeout=90, max_retries=0, **extra,
            )
        return self._inner[ep.key]

    def _emit(self, type_: str, **data: Any) -> None:
        if self.telemetry is not None:
            self.telemetry.emit(type_, agent=self.agent_label, **data)

    def _candidates(self, messages: Any, est: int) -> list[Endpoint]:
        """Endpoints allowed for this call, home lane first.

        * Mid tool-use, the model that started the loop goes first (sticky), so a
          conversation is not bounced between models without need.
        * Endpoints whose per-request ceiling is below this request are skipped
          (Groq free tier rejects any single request above its per-minute budget).
        """
        chain = [ep for ep in self.chain if est <= ep.tpm * 0.95] or [max(self.chain, key=lambda e: e.tpm)]
        has_tool_history = isinstance(messages, list) and any(
            m.get("role") == "tool" or m.get("tool_calls") for m in messages if isinstance(m, dict))
        if has_tool_history and self._last_endpoint:
            sticky = [ep for ep in chain if ep.key == self._last_endpoint]
            return sticky + [ep for ep in chain if ep.key != self._last_endpoint]
        return chain

    @staticmethod
    def _gemini_safe(messages: Any) -> Any:
        """Gemini 3 rejects tool calls produced by another model unless they carry a thought
        signature. Google documents a placeholder value for exactly this case."""
        if not isinstance(messages, list):
            return messages
        out = []
        for m in messages:
            if isinstance(m, dict) and m.get("tool_calls"):
                calls = []
                for tc in m["tool_calls"]:
                    if isinstance(tc, dict) and "extra_content" not in tc:
                        tc = {**tc, "extra_content": {"google": {"thought_signature": "skip_thought_signature_validator"}}}
                    calls.append(tc)
                m = {**m, "tool_calls": calls}
            out.append(m)
        return out

    def _pick(self, cands: list[Endpoint], est: int, tried: set[str]) -> tuple[Endpoint, float] | None:
        """Latency-aware choice: minimise (wait for budget + typical response time).

        The preferred (first) endpoint gets a small bonus so agents keep their home model
        when the difference is marginal; this keeps parallel agents on separate lanes.
        """
        home = cands[0]
        if home.key not in tried:
            w = PACER.wait_needed(home, est)
            if w <= HOME_PATIENCE:            # stay on the home lane unless it is clearly congested
                return home, w
        best, best_cost = None, float("inf")
        for ep in cands[1:]:
            if ep.key in tried:
                continue
            wait = PACER.wait_needed(ep, est)
            if wait > 900:            # cooling down for a long time (daily quota / auth)
                continue
            cost = wait + ep.latency
            if cost < best_cost:
                best, best_cost = (ep, wait), cost
        if best is None and home.key not in tried:
            w = PACER.wait_needed(home, est)
            if w <= 900:
                best = (home, w)
        if best is None or best[1] > RETRY_SOON:
            # Everything untried is far off: an endpoint that failed transiently (a 503 cools for seconds) beats
            # minutes of waiting for a rate-limited one. Rate-limited endpoints keep their long cooldown here.
            for ep in cands:
                if ep.key in tried:
                    wait = PACER.wait_needed(ep, est)
                    if wait <= RETRY_SOON and wait + ep.latency < (best_cost if best else float("inf")):
                        best, best_cost = (ep, wait), wait + ep.latency
        return best

    # ---------------------------------------------------------------- core
    def call(self, messages, tools=None, callbacks=None, available_functions=None,
             from_task=None, from_agent=None, response_model=None):
        if not self.chain:
            raise RuntimeError("No API keys configured. Set GROQ_API_KEY and/or GEMINI_API_KEY.")
        est = _estimate_tokens(messages) - 900 + self.max_completion
        last_err: Exception | None = None
        tried: set[str] = set()
        cands = self._candidates(messages, est)
        home = cands[0]
        for attempt in range(len(cands) + 3):
            pick = self._pick(cands, est, tried)
            if pick is None:
                if attempt >= len(cands) + 2:
                    break
                tried = set()                        # every endpoint failed once: wait, then retry all
                self._emit("throttle", model=home.label, wait=10)
                time.sleep(10)
                continue
            ep, wait = pick
            if wait > 0:
                if wait >= 2:
                    self._emit("throttle", model=ep.label, wait=round(wait, 1))
                time.sleep(wait)
            if True:
                if ep is not home and self._last_endpoint != ep.key:
                    self._emit("failover", to_model=ep.label, to_provider=ep.provider,
                               from_model=home.label, reason="budget" if not last_err else _kind(last_err))
                    if self.telemetry is not None:
                        self.telemetry.record_failover(ep.key)
                PACER.reserve(ep, est)
                client = self._client(ep)
                before = client.get_token_usage_summary()
                t = time.time()
                self._emit("llm_start", model=ep.label, provider=ep.provider)
                try:
                    msgs = self._gemini_safe(messages) if ep.model.startswith("gemini-") else messages
                    out = client.call(msgs, tools=tools, callbacks=callbacks,
                                      available_functions=available_functions, from_task=from_task,
                                      from_agent=from_agent, response_model=response_model)
                except Exception as err:  # noqa: BLE001 - classify and fail over
                    kind = _kind(err)
                    last_err = err
                    tried.add(ep.key)
                    if kind != "too_large":
                        PACER.cool(ep, _retry_after(err) if kind == "rate_limit" else 300 if kind == "auth" else 8)
                    self._emit("llm_error", model=ep.label, provider=ep.provider, kind=kind,
                               message=str(err)[:240])
                    continue
                dt = time.time() - t
                after = client.get_token_usage_summary()
                p = max(0, after.prompt_tokens - before.prompt_tokens)
                c = max(0, after.completion_tokens - before.completion_tokens)
                PACER.settle(ep, est, p + c if p + c else est)
                self._last_endpoint = ep.key
                self.model = ep.model
                self.label = ep.label
                if self.telemetry is not None:
                    self.telemetry.record_usage(ep.key, p, c, dt)
                self._emit("llm_end", model=ep.label, provider=ep.provider, prompt_tokens=p,
                           completion_tokens=c, seconds=round(dt, 2))
                return out
        raise RuntimeError(f"All model endpoints failed for {self.agent_label}: {last_err}")

    async def acall(self, messages, tools=None, callbacks=None, available_functions=None,
                    from_task=None, from_agent=None, response_model=None):
        return await asyncio.to_thread(self.call, messages, tools, callbacks, available_functions,
                                       from_task, from_agent, response_model)

    def supports_function_calling(self) -> bool:
        return True

    def supports_stop_words(self) -> bool:
        return False

    def get_context_window_size(self) -> int:
        return 60_000

Writing safeprocure/llm.py


**`safeprocure/data.py`** · Synthetic vendor data with realistic planted patterns (a deteriorating supplier, a cheap-but-defective one, spend concentration, compliance trouble) and data-quality defects for the auditor. Also maps arbitrary uploaded CSV/Excel columns onto the schema.

In [6]:
%%writefile safeprocure/data.py
"""Vendor performance data: synthetic generator + upload normaliser.

All vendor names are fictional. The generator plants realistic business patterns
(a deteriorating supplier, a cheap-but-defective supplier, spend concentration,
compliance trouble, a recovering supplier) plus a few data-quality defects, so the
agents have genuine findings to discover rather than a uniform table.
"""
from __future__ import annotations

import io
import re
from dataclasses import dataclass

import numpy as np
import pandas as pd

CATEGORIES = {
    "Steel & Metals": ["Apex Metalworks", "Sahyadri Steel Forge", "Kaveri Alloys", "Indus Castings", "Vajra Metal Traders"],
    "Packaging": ["NovaPack Solutions", "GreenWrap Packaging", "Shree Corrugators", "PolyShield Films", "Kraftline Boxes"],
    "Electronic Components": ["Vidyut Components", "Quantix Electronics", "Circuitra Systems", "Pinnacle Semicon Traders", "Nexa PCB Works"],
    "Logistics & Freight": ["SwiftHaul Logistics", "Bharat Freight Lines", "Coastal Cargo Movers", "TransEast Carriers", "Ghat Road Express"],
    "Chemicals & Plastics": ["Rasayan Chemicals", "Polymer Hub India", "AquaResin Industries", "Zenith Specialty Chem", "Narmada Polymers"],
}
REGIONS = ["Pune, Maharashtra", "Chennai, Tamil Nadu", "Delhi NCR", "Kolkata, West Bengal", "Ahmedabad, Gujarat"]

COLUMNS = [
    "vendor_id", "vendor_name", "category", "region", "month", "po_count", "po_value_inr",
    "on_time_pct", "fill_rate_pct", "defect_rate_pct", "lead_time_days", "price_variance_pct",
    "response_time_hrs", "compliance_incidents", "capacity_utilization_pct",
]
KPI_COLUMNS = ["on_time_pct", "fill_rate_pct", "defect_rate_pct", "lead_time_days", "price_variance_pct",
               "response_time_hrs", "compliance_incidents", "capacity_utilization_pct"]

ARCHETYPES = {
    "star":          dict(otd=96, defect=0.6, lead=0.85, pv=0.5, resp=6, comp=0.02, util=78),
    "steady":        dict(otd=91, defect=1.3, lead=1.0, pv=1.5, resp=14, comp=0.05, util=70),
    "deteriorating": dict(otd=94, defect=1.0, lead=1.0, pv=1.0, resp=10, comp=0.05, util=93),
    "cheap_poor":    dict(otd=84, defect=4.2, lead=1.1, pv=-4.0, resp=30, comp=0.12, util=65),
    "premium":       dict(otd=97, defect=0.4, lead=0.9, pv=7.5, resp=5, comp=0.0, util=60),
    "compliance":    dict(otd=89, defect=1.8, lead=1.05, pv=0.0, resp=20, comp=0.55, util=72),
    "improving":     dict(otd=78, defect=3.2, lead=1.2, pv=2.0, resp=26, comp=0.1, util=58),
    "volatile":      dict(otd=86, defect=2.2, lead=1.15, pv=3.0, resp=22, comp=0.15, util=68),
}
BASE_LEAD = {"Steel & Metals": 21, "Packaging": 9, "Electronic Components": 35,
             "Logistics & Freight": 4, "Chemicals & Plastics": 14}
BASE_SPEND = {"Steel & Metals": 5.2e6, "Packaging": 1.4e6, "Electronic Components": 4.1e6,
              "Logistics & Freight": 1.9e6, "Chemicals & Plastics": 2.3e6}


@dataclass
class GenSpec:
    n_vendors: int = 20
    months: int = 18
    categories: tuple[str, ...] = tuple(CATEGORIES)
    seed: int = 42
    end_month: str = "2026-09"
    inject_quality_issues: bool = True


def generate(spec: GenSpec | None = None) -> pd.DataFrame:
    spec = spec or GenSpec()
    rng = np.random.default_rng(spec.seed)
    cats = [c for c in spec.categories if c in CATEGORIES] or list(CATEGORIES)
    per_cat = max(2, min(5, round(spec.n_vendors / len(cats))))
    end = pd.Period(spec.end_month, "M")
    months = [str(end - i) for i in range(spec.months - 1, -1, -1)]

    # Make sure the headline archetypes always appear, then fill with a realistic mix.
    must = ["deteriorating", "cheap_poor", "star", "compliance", "improving", "premium"]
    pool = ["steady", "steady", "star", "volatile", "steady", "premium", "volatile"]
    rows, vid = [], 0
    archetype_of: dict[str, str] = {}
    for ci, cat in enumerate(cats):
        names = CATEGORIES[cat][:per_cat]
        # one vendor per category dominates spend (concentration risk in some categories)
        shares = rng.dirichlet(np.ones(len(names)) * 1.4)
        if ci % 2 == 0:
            shares = np.sort(shares)[::-1]
            shares[0] += 0.25
            shares /= shares.sum()
        for vi, name in enumerate(names):
            vid += 1
            arche = must[(vid - 1)] if vid - 1 < len(must) else pool[int(rng.integers(len(pool)))]
            archetype_of[name] = arche
            a = ARCHETYPES[arche]
            region = REGIONS[int(rng.integers(len(REGIONS)))]
            monthly_spend = BASE_SPEND[cat] * shares[vi]
            n = len(months)
            for mi, month in enumerate(months):
                prog = mi / max(1, n - 1)                 # 0 -> 1 across the window
                late = max(0.0, (mi - (n - 7)) / 6)       # 0 until last 6 months, then 0 -> 1
                otd, defect, pv, resp, comp_rate, util = a["otd"], a["defect"], a["pv"], a["resp"], a["comp"], a["util"]
                if arche == "deteriorating":
                    otd -= 24 * late ** 1.3
                    defect += 2.6 * late
                    resp += 18 * late
                    util = min(99, util + 5 * late)
                elif arche == "improving":
                    otd += 15 * prog
                    defect -= 2.2 * prog
                    resp -= 14 * prog
                elif arche == "compliance" and mi >= n - 5:
                    comp_rate += 0.6
                noise = 2.6 if arche == "volatile" else 1.1
                season = 1.6 * np.sin(2 * np.pi * (int(month[-2:]) - 6) / 12)   # monsoon dip Jun-Sep
                otd_v = float(np.clip(otd + rng.normal(0, noise * 1.6) - (season if "Logistics" in cat else season * 0.4), 40, 100))
                defect_v = float(np.clip(defect + rng.normal(0, noise * 0.28), 0.05, 15))
                lead_v = float(max(1, BASE_LEAD[cat] * a["lead"] * (1 + (100 - otd_v) / 160) + rng.normal(0, BASE_LEAD[cat] * 0.06)))
                pv_v = float(pv + rng.normal(0, noise * 0.9))
                resp_v = float(max(1, resp + rng.normal(0, noise * 2.2)))
                comp_v = int(rng.poisson(max(0.0, comp_rate)))
                spend = float(max(5e4, monthly_spend * (1 + 0.12 * np.sin(mi / 2.0)) * rng.lognormal(0, 0.12)))
                po_count = int(max(1, round(spend / rng.uniform(1.6e5, 3.4e5))))
                fill_v = float(np.clip(otd_v + rng.normal(4, 2), 50, 100))
                rows.append([f"V{vid:03d}", f"{name} Pvt Ltd", cat, region, month, po_count, round(spend, -2),
                             round(otd_v, 1), round(fill_v, 1), round(defect_v, 2), round(lead_v, 1), round(pv_v, 2),
                             round(resp_v, 1), comp_v, round(float(np.clip(util + rng.normal(0, 3), 20, 100)), 1)])

    df = pd.DataFrame(rows, columns=COLUMNS)
    if spec.inject_quality_issues and len(df) > 40:
        idx = rng.choice(df.index, size=6, replace=False)
        df.loc[idx[:3], "defect_rate_pct"] = np.nan                       # missing values
        df.loc[idx[3], "lead_time_days"] = df.loc[idx[3], "lead_time_days"] * 18   # data-entry outlier
        df.loc[idx[4], "po_value_inr"] = -abs(df.loc[idx[4], "po_value_inr"])        # sign error
        df = pd.concat([df, df.loc[[idx[5]]]], ignore_index=True)                    # duplicate row
    df.attrs["archetypes"] = {f"{k} Pvt Ltd": v for k, v in archetype_of.items()}
    return df


# ------------------------------------------------------------------ uploads
ALIASES = {
    "vendor_name": ["vendor", "supplier", "supplier_name", "vendor name", "name", "company"],
    "vendor_id": ["supplier_id", "id", "vendor code", "vendor_code", "supplier code"],
    "category": ["commodity", "category_name", "segment", "material_group"],
    "region": ["location", "city", "state", "geography", "zone"],
    "month": ["period", "date", "month_year", "po_month"],
    "po_value_inr": ["spend", "po_value", "value", "amount", "spend_inr", "purchase_value"],
    "po_count": ["orders", "po_count", "number_of_pos", "pos"],
    "on_time_pct": ["otd", "otd_pct", "on_time", "on_time_delivery", "on time delivery %", "otd %", "delivery_pct"],
    "fill_rate_pct": ["fill_rate", "fill rate", "infull_pct"],
    "defect_rate_pct": ["defect", "defects", "defect_rate", "rejection_rate", "rejection %", "quality_defect_pct"],
    "lead_time_days": ["lead_time", "lead time", "leadtime"],
    "price_variance_pct": ["price_variance", "ppv", "price variance %", "cost_variance"],
    "response_time_hrs": ["response_time", "response hrs", "responsiveness_hrs"],
    "compliance_incidents": ["compliance", "incidents", "non_compliance", "audit_findings"],
    "capacity_utilization_pct": ["capacity", "utilization", "capacity_utilisation", "utilisation"],
}
NEUTRAL = {"po_count": 1, "fill_rate_pct": None, "lead_time_days": 14.0, "price_variance_pct": 0.0,
           "response_time_hrs": 24.0, "compliance_incidents": 0, "capacity_utilization_pct": 70.0}


def _norm(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", s.strip().lower()).strip("_")


def normalise_upload(raw: bytes, filename: str = "upload.csv") -> tuple[pd.DataFrame, list[str]]:
    """Map arbitrary column names onto the Safe Procure schema. Returns (df, notes)."""
    if filename.lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(io.BytesIO(raw))
    else:
        df = pd.read_csv(io.BytesIO(raw))
    notes: list[str] = []
    lookup = {_norm(c): c for c in df.columns}
    mapping = {}
    for target, alts in ALIASES.items():
        for cand in [target] + alts:
            if _norm(cand) in lookup:
                mapping[lookup[_norm(cand)]] = target
                break
    df = df.rename(columns=mapping)
    if "vendor_name" not in df.columns:
        raise ValueError("Could not find a vendor/supplier name column.")
    present_kpis = [c for c in ["on_time_pct", "defect_rate_pct", "price_variance_pct", "lead_time_days"] if c in df.columns]
    if len(present_kpis) < 2:
        raise ValueError("Need at least two KPI columns (e.g. on-time %, defect %, price variance %, lead time).")
    if "month" not in df.columns:
        df["month"] = "2026-09"
        notes.append("No date column: treated all rows as a single period (trend & forecast limited).")
    else:
        df["month"] = pd.to_datetime(df["month"], errors="coerce").dt.to_period("M").astype(str)
    if "vendor_id" not in df.columns:
        ids = {n: f"V{i + 1:03d}" for i, n in enumerate(sorted(df["vendor_name"].astype(str).unique()))}
        df["vendor_id"] = df["vendor_name"].astype(str).map(ids)
    for col, default in [("category", "General"), ("region", "Unspecified")]:
        if col not in df.columns:
            df[col] = default
            notes.append(f"No {col} column: defaulted to '{default}'.")
    if "po_value_inr" not in df.columns:
        df["po_value_inr"] = 100000.0
        notes.append("No spend column: equal spend assumed for every row.")
    for col, default in NEUTRAL.items():
        if col not in df.columns:
            df[col] = df["on_time_pct"] if (col == "fill_rate_pct" and "on_time_pct" in df.columns) else default
            notes.append(f"Missing '{col}': filled with neutral default.")
    for col in ["on_time_pct", "defect_rate_pct"]:
        if col not in df.columns:
            df[col] = np.nan
            notes.append(f"Missing '{col}': pillar scored as neutral.")
    mapped = [f"{src} -> {dst}" for src, dst in mapping.items() if src != dst]
    if mapped:
        notes.insert(0, "Mapped columns: " + ", ".join(mapped))
    return df[COLUMNS], notes

Writing safeprocure/data.py


**`safeprocure/analytics.py`** · Deterministic analytics engine: cleaning, spend-weighted KPIs, 5-pillar weighted scorecard, risk, Pareto/HHI spend analysis, damped-trend forecasts with 80% bands, alternative sourcing.

In [7]:
%%writefile safeprocure/analytics.py
"""Deterministic analytics engine.

All numbers the agents reason about come from here, never from the LLM. This makes the
scorecard auditable and lets the UI recompute rankings instantly for what-if scenarios.
"""
from __future__ import annotations

from typing import Any

import numpy as np
import pandas as pd

from .config import DEFAULT_WEIGHTS

PILLARS = ["delivery", "quality", "cost", "responsiveness", "compliance"]


def _lin(x, lo, hi):
    """Map x linearly: lo -> 0, hi -> 100 (works for inverted ranges)."""
    return float(np.clip((x - lo) / (hi - lo) * 100, 0, 100))


# ------------------------------------------------------------------ quality
def clean(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, Any]]:
    d = df.copy()
    issues: list[dict[str, Any]] = []
    n0 = len(d)

    dup = d.duplicated(subset=["vendor_id", "month"], keep="first")
    if dup.any():
        issues.append({"type": "duplicate", "column": "vendor_id+month", "count": int(dup.sum()),
                       "action": "Removed duplicate vendor-month rows",
                       "examples": d.loc[dup, ["vendor_name", "month"]].astype(str).agg(" / ".join, axis=1).head(3).tolist()})
        d = d.loc[~dup]

    neg = d["po_value_inr"] < 0
    if neg.any():
        issues.append({"type": "sign_error", "column": "po_value_inr", "count": int(neg.sum()),
                       "action": "Negative spend treated as sign error (absolute value)",
                       "examples": d.loc[neg, "vendor_name"].head(3).tolist()})
        d.loc[neg, "po_value_inr"] = d.loc[neg, "po_value_inr"].abs()

    for col in ["on_time_pct", "fill_rate_pct", "defect_rate_pct", "lead_time_days", "price_variance_pct",
                "response_time_hrs", "capacity_utilization_pct"]:
        miss = d[col].isna()
        if miss.any() and not miss.all():
            issues.append({"type": "missing", "column": col, "count": int(miss.sum()),
                           "action": "Imputed with the vendor's own median",
                           "examples": d.loc[miss, ["vendor_name", "month"]].astype(str).agg(" / ".join, axis=1).head(3).tolist()})
            d[col] = d.groupby("vendor_id")[col].transform(lambda s: s.fillna(s.median()))
            d[col] = d[col].fillna(d[col].median())

    # robust outliers: > 6 MADs from the category median
    for col in ["lead_time_days", "response_time_hrs"]:
        med = d.groupby("category")[col].transform("median")
        mad = d.groupby("category")[col].transform(lambda s: (s - s.median()).abs().median()) + 1e-9
        out = (d[col] - med).abs() / mad > 6
        if out.any():
            issues.append({"type": "outlier", "column": col, "count": int(out.sum()),
                           "action": "Capped to category median (likely data-entry error)",
                           "examples": [f"{r.vendor_name} {r.month}: {getattr(r, col):.0f}" for r in d.loc[out].head(3).itertuples()]})
            d.loc[out, col] = med[out]

    pct_cols = ["on_time_pct", "fill_rate_pct", "capacity_utilization_pct"]
    for col in pct_cols:
        bad = (d[col] < 0) | (d[col] > 100)
        if bad.any():
            issues.append({"type": "range", "column": col, "count": int(bad.sum()),
                           "action": "Clipped to 0-100", "examples": []})
            d[col] = d[col].clip(0, 100)

    affected = sum(i["count"] for i in issues)
    report = {
        "rows_in": n0, "rows_out": len(d), "vendors": int(d["vendor_id"].nunique()),
        "categories": sorted(d["category"].unique().tolist()), "regions": sorted(d["region"].unique().tolist()),
        "months": [d["month"].min(), d["month"].max()], "n_months": int(d["month"].nunique()),
        "issues": issues, "quality_score": round(max(0.0, 100 - affected / max(1, n0) * 100 * 4), 1),
        "total_spend": float(d["po_value_inr"].sum()),
    }
    return d.sort_values(["vendor_id", "month"]).reset_index(drop=True), report


# ---------------------------------------------------------------- scorecard
def _slope(s: pd.Series) -> float:
    if len(s) < 3:
        return 0.0
    x = np.arange(len(s))
    return float(np.polyfit(x, s.values.astype(float), 1)[0])


def vendor_kpis(d: pd.DataFrame, window: int = 6) -> pd.DataFrame:
    """Spend-weighted KPIs over the recent window, plus trends vs the prior window."""
    out = []
    last = sorted(d["month"].unique())
    recent = set(last[-window:])
    prior = set(last[-2 * window:-window])
    for vid, g in d.groupby("vendor_id"):
        g = g.sort_values("month")
        r = g[g["month"].isin(recent)]
        p = g[g["month"].isin(prior)]
        w = r["po_value_inr"].clip(lower=1)
        wa = lambda c: float(np.average(r[c], weights=w)) if len(r) else float("nan")  # noqa: E731
        tail = g.tail(window)
        rec = {
            "vendor_id": vid, "vendor_name": g["vendor_name"].iloc[0], "category": g["category"].iloc[0],
            "region": g["region"].iloc[0],
            "spend_total": float(g["po_value_inr"].sum()), "spend_recent": float(r["po_value_inr"].sum()),
            "po_count": int(g["po_count"].sum()),
            "on_time_pct": wa("on_time_pct"), "fill_rate_pct": wa("fill_rate_pct"),
            "defect_rate_pct": wa("defect_rate_pct"), "lead_time_days": wa("lead_time_days"),
            "price_variance_pct": wa("price_variance_pct"), "response_time_hrs": wa("response_time_hrs"),
            "compliance_incidents": int(r["compliance_incidents"].sum()),
            "compliance_incidents_total": int(g["compliance_incidents"].sum()),
            "capacity_utilization_pct": float(r["capacity_utilization_pct"].mean()),
            "otd_trend": _slope(tail["on_time_pct"]),                 # pts / month
            "defect_trend": _slope(tail["defect_rate_pct"]),           # pts / month
            "otd_delta": float(r["on_time_pct"].mean() - p["on_time_pct"].mean()) if len(p) else 0.0,
            "defect_delta": float(r["defect_rate_pct"].mean() - p["defect_rate_pct"].mean()) if len(p) else 0.0,
            "otd_volatility": float(g["on_time_pct"].tail(12).std(ddof=0)),
            "otd_series": [round(float(v), 1) for v in g["on_time_pct"].tail(12)],
            "defect_series": [round(float(v), 2) for v in g["defect_rate_pct"].tail(12)],
        }
        out.append(rec)
    k = pd.DataFrame(out)
    k["lead_rel"] = k["lead_time_days"] / k.groupby("category")["lead_time_days"].transform("median")
    return k


def score(k: pd.DataFrame, weights: dict[str, float] | None = None) -> pd.DataFrame:
    w = {**DEFAULT_WEIGHTS, **(weights or {})}
    tot = sum(max(0.0, float(w[p])) for p in PILLARS) or 1.0
    wn = {p: max(0.0, float(w[p])) / tot for p in PILLARS}
    s = k.copy()
    s["delivery"] = [0.6 * _lin(o, 70, 99) + 0.2 * _lin(f, 75, 100) + 0.2 * _lin(l, 1.6, 0.8)
                     for o, f, l in zip(s.on_time_pct, s.fill_rate_pct, s.lead_rel)]
    s["quality"] = [_lin(dr, 5.0, 0.3) for dr in s.defect_rate_pct]
    s["cost"] = [_lin(pv, 10.0, -4.0) for pv in s.price_variance_pct]
    s["responsiveness"] = [_lin(r, 60, 4) for r in s.response_time_hrs]
    s["compliance"] = [max(0.0, 100 - 22 * c) for c in s.compliance_incidents]
    s["composite"] = sum(s[p] * wn[p] for p in PILLARS)
    # internal risk: inverse score + momentum + volatility penalties
    momentum = np.clip(-s["otd_trend"] * 4.0, 0, 18) + np.clip(s["defect_trend"] * 22, 0, 14)
    vol = np.clip((s["otd_volatility"] - 2.5) * 2.0, 0, 10)
    s["internal_risk"] = np.clip(100 - s["composite"] + momentum + vol, 0, 100)
    s["rank"] = s["composite"].rank(ascending=False, method="min").astype(int)
    s["tier"] = pd.cut(s["composite"], [-1, 55, 70, 82, 101], labels=["D", "C", "B", "A"]).astype(str)
    s["weights_used"] = [wn] * len(s)
    return s.sort_values("composite", ascending=False).reset_index(drop=True)


# -------------------------------------------------------------------- spend
def spend_analysis(d: pd.DataFrame, k: pd.DataFrame) -> dict[str, Any]:
    total = float(d["po_value_inr"].sum())
    by_vendor = k[["vendor_name", "category", "spend_total", "price_variance_pct"]].sort_values("spend_total", ascending=False)
    by_vendor = by_vendor.assign(share=by_vendor.spend_total / total, cum_share=by_vendor.spend_total.cumsum() / total)
    pareto_n = int((by_vendor.cum_share <= 0.8).sum() + 1)
    cats = []
    for cat, g in k.groupby("category"):
        sh = g.spend_total / g.spend_total.sum()
        top = g.loc[sh.idxmax()]
        best_pv = float(g.price_variance_pct.min())
        # savings if volume above best-in-category price variance were renegotiated halfway
        gap = ((g.price_variance_pct - best_pv).clip(lower=0) / 100 * g.spend_recent).sum()
        cats.append({
            "category": cat, "spend": float(g.spend_total.sum()), "vendors": int(len(g)),
            "hhi": round(float((sh ** 2).sum() * 10000)), "top_vendor": top.vendor_name,
            "top_share": round(float(sh.max()), 3),
            "concentration_flag": bool(sh.max() > 0.45),
            "overpay_recent": float(((g.price_variance_pct.clip(lower=0) / 100) * g.spend_recent).sum()),
            "savings_opportunity": float(gap * 0.5 * 2),   # annualised (6-month window x2) at 50% capture
        })
    cats.sort(key=lambda c: -c["spend"])
    monthly = d.groupby("month")["po_value_inr"].sum()
    return {
        "total_spend": total, "pareto_vendors_80pct": pareto_n, "vendor_count": int(len(k)),
        "categories": cats,
        "vendors": by_vendor.round(4).to_dict("records"),
        "monthly": {"months": monthly.index.tolist(), "spend": [round(float(v)) for v in monthly.values]},
        "savings_opportunity_total": float(sum(c["savings_opportunity"] for c in cats)),
        "overpay_recent_total": float(sum(c["overpay_recent"] for c in cats)),
    }


# ----------------------------------------------------------------- forecast
def forecast(d: pd.DataFrame, horizon: int = 3, otd_floor: float = 85.0, defect_ceiling: float = 3.0) -> list[dict[str, Any]]:
    res = []
    for vid, g in d.groupby("vendor_id"):
        g = g.sort_values("month").tail(12)
        x = np.arange(len(g))
        months = g["month"].tolist()
        last = pd.Period(months[-1], "M")
        fut = [str(last + i) for i in range(1, horizon + 1)]
        rec = {"vendor_id": vid, "vendor_name": g["vendor_name"].iloc[0], "category": g["category"].iloc[0],
               "history_months": months, "future_months": fut}
        for col, key in [("on_time_pct", "otd"), ("defect_rate_pct", "defect")]:
            y = g[col].values.astype(float)
            # damped trend: fit on last 12, shrink slope 30% to avoid over-extrapolation
            b1, b0 = np.polyfit(x, y, 1) if len(g) >= 3 else (0.0, float(y.mean()))
            b1 *= 0.7
            resid = y - (b0 + b1 / 0.7 * x)
            sd = float(np.std(resid, ddof=1)) if len(y) > 2 else 1.0
            xf = np.arange(len(g), len(g) + horizon)
            fitted_last = b0 + b1 / 0.7 * (len(g) - 1)
            anchor = 0.6 * y[-1] + 0.4 * fitted_last          # start from where the vendor actually is
            pred = anchor + b1 * (xf - (len(g) - 1))
            band = 1.28 * sd * np.sqrt(1 + (xf - x.mean()) ** 2 / max(1, ((x - x.mean()) ** 2).sum()))
            lo, hi = (0, 100) if key == "otd" else (0, 20)
            rec[key] = {"history": [round(float(v), 2) for v in y],
                        "forecast": [round(float(np.clip(v, lo, hi)), 2) for v in pred],
                        "lower": [round(float(np.clip(v, lo, hi)), 2) for v in pred - band],
                        "upper": [round(float(np.clip(v, lo, hi)), 2) for v in pred + band],
                        "slope_per_month": round(float(b1 / 0.7), 3)}
        rec["otd_breach"] = bool(min(rec["otd"]["forecast"]) < otd_floor)
        rec["defect_breach"] = bool(max(rec["defect"]["forecast"]) > defect_ceiling)
        rec["outlook"] = ("deteriorating" if (rec["otd"]["slope_per_month"] < -0.6 or rec["defect"]["slope_per_month"] > 0.08)
                          else "improving" if (rec["otd"]["slope_per_month"] > 0.6 or rec["defect"]["slope_per_month"] < -0.08)
                          else "stable")
        res.append(rec)
    return res


# ------------------------------------------------------------- alternatives
def alternatives(s: pd.DataFrame, vendor_name: str, top: int = 3) -> list[dict[str, Any]]:
    row = s[s.vendor_name.str.lower() == vendor_name.lower()]
    if row.empty:
        row = s[s.vendor_name.str.lower().str.contains(vendor_name.lower().split(" pvt")[0][:12], regex=False)]
    if row.empty:
        return []
    r = row.iloc[0]
    peers = s[(s.category == r.category) & (s.vendor_name != r.vendor_name)].copy()
    peers["headroom_pct"] = (92 - peers.capacity_utilization_pct).clip(lower=0)
    peers["absorbable_spend"] = peers.spend_recent * peers.headroom_pct / peers.capacity_utilization_pct.clip(lower=1)
    peers["fit"] = peers.composite * 0.7 + peers.headroom_pct * 1.2 - peers.internal_risk * 0.2
    peers = peers.sort_values("fit", ascending=False).head(top)
    return [{"vendor_name": p.vendor_name, "region": p.region, "composite": round(p.composite, 1),
             "headroom_pct": round(p.headroom_pct, 1), "absorbable_spend_6m": round(p.absorbable_spend),
             "price_variance_pct": round(p.price_variance_pct, 2), "on_time_pct": round(p.on_time_pct, 1),
             "defect_rate_pct": round(p.defect_rate_pct, 2)} for p in peers.itertuples()]


# -------------------------------------------------------------- combine
def combine_risk(s: pd.DataFrame, market: dict[str, float], ext_weight: float = 0.35) -> pd.DataFrame:
    """Blend internal risk with external market risk for each vendor's category+region."""
    out = s.copy()

    def ext(r):
        for key in (f"{r.category}|{r.region}", r.category, r.region):
            if key in market:
                return float(market[key])
        cat = [v for kk, v in market.items() if kk.startswith(r.category)]
        return float(np.mean(cat)) if cat else 35.0

    out["market_risk"] = out.apply(ext, axis=1)
    out["combined_risk"] = (1 - ext_weight) * out.internal_risk + ext_weight * out.market_risk
    return out


def table_for_llm(s: pd.DataFrame, cols: list[str], n: int = 30) -> str:
    """Compact markdown table (keeps agent prompts small)."""
    t = s[cols].head(n).copy()
    for c in t.columns:
        if t[c].dtype.kind == "f":
            t[c] = t[c].round(1)
    head = "| " + " | ".join(cols) + " |\n|" + "---|" * len(cols) + "\n"
    return head + "\n".join("| " + " | ".join(str(v) for v in row) + " |" for row in t.values)


def inr(v: float) -> str:
    v = float(v)
    if abs(v) >= 1e7:
        return f"₹{v / 1e7:.2f} Cr"
    if abs(v) >= 1e5:
        return f"₹{v / 1e5:.1f} L"
    return f"₹{v:,.0f}"

Writing safeprocure/analytics.py


**`safeprocure/tools.py`** · The **8 custom CrewAI tools**, including live news search that runs several queries in parallel.

In [8]:
%%writefile safeprocure/tools.py
"""Custom CrewAI tools. Each wraps the deterministic analytics engine or live web search.

Tools are created per agent per run, so every call is attributed to an agent in the trace.
"""
from __future__ import annotations

import json
import threading
import time
from dataclasses import dataclass, field
from typing import Any, Optional

import pandas as pd
from crewai.tools import BaseTool
from pydantic import BaseModel, ConfigDict, Field

from . import analytics as A


@dataclass
class RunContext:
    """Everything one analysis run knows. Shared by tools, flow and the web API."""

    df: pd.DataFrame
    quality: dict[str, Any]
    kpis: pd.DataFrame
    scored: pd.DataFrame
    weights: dict[str, float]
    risk_threshold: float = 55.0
    otd_floor: float = 85.0
    defect_ceiling: float = 3.0
    question: str = ""
    focus_categories: list[str] = field(default_factory=list)
    telemetry: Any = None
    sources: list[dict[str, Any]] = field(default_factory=list)
    _lock: threading.Lock = field(default_factory=threading.Lock)
    _search_cache: dict[str, list[dict[str, Any]]] = field(default_factory=dict)

    @property
    def in_scope(self) -> pd.DataFrame:
        s = self.scored
        return s[s.category.isin(self.focus_categories)] if self.focus_categories else s


class _CtxTool(BaseTool):
    model_config = ConfigDict(arbitrary_types_allowed=True)
    ctx: Any = Field(default=None, exclude=True)
    agent_label: str = Field(default="agent", exclude=True)

    def _trace(self, phase: str, **data: Any) -> None:
        if self.ctx is not None and self.ctx.telemetry is not None:
            self.ctx.telemetry.emit(f"tool_{phase}", agent=self.agent_label, tool=self.name, **data)

    def _run(self, **kwargs: Any) -> str:  # pragma: no cover - overridden
        raise NotImplementedError

    def run_traced(self, fn, **kwargs: Any) -> str:
        t = time.time()
        self._trace("start", args={k: v for k, v in kwargs.items() if v not in (None, "", [])})
        try:
            out = fn(**kwargs)
        except Exception as err:  # noqa: BLE001
            out = f"Tool error: {err}"
        self._trace("end", seconds=round(time.time() - t, 2), chars=len(out), preview=out[:160])
        return out


# ----------------------------------------------------------------- 1 audit
class _NoArgs(BaseModel):
    # Groq rejects function schemas with an empty properties object, so expose one optional knob.
    detail: Optional[str] = Field("summary", description="'summary' (default) or 'full'")


class DataQualityTool(_CtxTool):
    name: str = "audit_dataset"
    description: str = ("Profiles the vendor dataset and returns detected data-quality issues "
                        "(duplicates, missing values, outliers, sign errors) and the cleaning applied.")
    args_schema: type[BaseModel] = _NoArgs

    def _run(self, **_: Any) -> str:
        def go() -> str:
            q = self.ctx.quality
            lines = [f"Rows in: {q['rows_in']}, rows after cleaning: {q['rows_out']}, vendors: {q['vendors']}, "
                     f"months: {q['months'][0]}..{q['months'][1]} ({q['n_months']}), quality score: {q['quality_score']}/100",
                     f"Categories: {', '.join(q['categories'])}", f"Regions: {', '.join(q['regions'])}",
                     f"Total spend: {A.inr(q['total_spend'])}", "Issues:"]
            for i in q["issues"] or []:
                lines.append(f"- {i['type']} in {i['column']}: {i['count']} rows -> {i['action']}. e.g. {'; '.join(i['examples'][:2])}")
            if not q["issues"]:
                lines.append("- none detected")
            return "\n".join(lines)
        return self.run_traced(go)


# ------------------------------------------------------------- 2 scorecard
class _ScoreArgs(BaseModel):
    category: Optional[str] = Field(None, description="Optional category filter, e.g. 'Packaging'")
    sort_by: Optional[str] = Field("composite", description="composite | internal_risk | on_time_pct | defect_rate_pct | price_variance_pct")
    top_n: Optional[int] = Field(20, description="Rows to return")


class VendorScorecardTool(_CtxTool):
    name: str = "vendor_scorecard"
    description: str = ("Returns the weighted vendor scorecard: pillar scores (0-100) for delivery, quality, cost, "
                        "responsiveness, compliance, composite score, internal risk, tier, key KPIs and 6-month trends.")
    args_schema: type[BaseModel] = _ScoreArgs

    def _run(self, category: str | None = None, sort_by: str | None = "composite", top_n: int | None = 25, **_: Any) -> str:
        def go(category=None, sort_by="composite", top_n=25) -> str:
            s = self.ctx.in_scope
            if category:
                s = s[s.category.str.lower().str.contains(category.lower()[:8])]
            col = sort_by if sort_by in s.columns else "composite"
            # on-time ascending shows the worst performers first; everything else descending
            s = s.sort_values(col, ascending=col == "on_time_pct")
            w = self.ctx.weights
            hdr = f"Weights: {json.dumps(w)}  (pillars 0-100, higher is better; internal_risk 0-100 higher is worse)\n"
            cols = ["vendor_name", "category", "composite", "tier", "internal_risk", "delivery", "quality", "cost",
                    "responsiveness", "compliance", "on_time_pct", "defect_rate_pct", "price_variance_pct",
                    "otd_trend", "compliance_incidents"]
            return hdr + A.table_for_llm(s, cols, int(top_n or 25))
        return self.run_traced(go, category=category, sort_by=sort_by, top_n=top_n)


# ---------------------------------------------------------- 3 vendor profile
class _VendorArgs(BaseModel):
    vendor_name: str = Field(..., description="Vendor name (partial match allowed)")


class VendorProfileTool(_CtxTool):
    name: str = "vendor_profile"
    description: str = "Returns every KPI, pillar score, trend and the last 12 months of on-time % and defect % for one vendor. Use to verify claims."
    args_schema: type[BaseModel] = _VendorArgs

    def _run(self, vendor_name: str = "", **_: Any) -> str:
        def go(vendor_name="") -> str:
            s = self.ctx.scored
            m = s[s.vendor_name.str.lower().str.contains(vendor_name.lower().replace(" pvt ltd", "")[:14], regex=False)]
            if m.empty:
                return f"No vendor matching '{vendor_name}'. Known: {', '.join(s.vendor_name.head(25))}"
            r = m.iloc[0]
            keys = ["vendor_name", "category", "region", "composite", "tier", "rank", "internal_risk", "delivery", "quality",
                    "cost", "responsiveness", "compliance", "on_time_pct", "fill_rate_pct", "defect_rate_pct", "lead_time_days",
                    "price_variance_pct", "response_time_hrs", "compliance_incidents", "compliance_incidents_total",
                    "capacity_utilization_pct", "otd_trend", "defect_trend", "otd_delta", "defect_delta", "otd_volatility"]
            d = {k: (round(float(r[k]), 2) if isinstance(r[k], (float, int)) and not isinstance(r[k], bool) else r[k]) for k in keys}
            for k in ("market_risk", "combined_risk"):
                if k in r.index:
                    d[k] = round(float(r[k]), 1)
            # Unambiguous names: KPIs are spend-weighted averages of the last 6 months.
            for k in ("on_time_pct", "fill_rate_pct", "defect_rate_pct", "lead_time_days", "price_variance_pct", "response_time_hrs"):
                d[f"{k}_6mo_avg"] = d.pop(k)
            d["spend_total"] = A.inr(r.spend_total)
            d["otd_monthly_last12"] = r.otd_series
            d["defect_monthly_last12"] = r.defect_series
            d["note"] = "combined_risk = 0.65*internal_risk + 0.35*market_risk; *_6mo_avg are spend-weighted 6-month averages"
            return json.dumps(d, default=str)
        return self.run_traced(go, vendor_name=vendor_name)


# ----------------------------------------------------------------- 4 spend
class SpendAnalyticsTool(_CtxTool):
    name: str = "spend_analytics"
    description: str = ("Spend analysis: total spend, Pareto (vendors holding 80% of spend), per-category HHI concentration, "
                        "top-vendor share, price-variance overpayment and annualised savings opportunity.")
    args_schema: type[BaseModel] = _NoArgs

    def _run(self, **_: Any) -> str:
        def go() -> str:
            sp = A.spend_analysis(self.ctx.df, self.ctx.kpis)
            lines = [f"Total spend: {A.inr(sp['total_spend'])} across {sp['vendor_count']} vendors. "
                     f"{sp['pareto_vendors_80pct']} vendors hold 80% of spend.",
                     f"Recent 6-month overpayment vs contract (positive price variance): {A.inr(sp['overpay_recent_total'])}. "
                     f"Annualised savings opportunity: {A.inr(sp['savings_opportunity_total'])}.",
                     "| category | spend | vendors | HHI | top vendor | top share | concentrated | savings/yr |", "|---|---|---|---|---|---|---|---|"]
            for c in sp["categories"]:
                lines.append(f"| {c['category']} | {A.inr(c['spend'])} | {c['vendors']} | {c['hhi']} | {c['top_vendor']} | "
                             f"{c['top_share']:.0%} | {'YES' if c['concentration_flag'] else 'no'} | {A.inr(c['savings_opportunity'])} |")
            lines.append("Top vendors by spend:")
            for v in sp["vendors"][:8]:
                lines.append(f"- {v['vendor_name']} ({v['category']}): {A.inr(v['spend_total'])}, {v['share']:.1%} share, price variance {v['price_variance_pct']:+.1f}%")
            return "\n".join(lines)
        return self.run_traced(go)


# -------------------------------------------------------------- 5 forecast
class _FcArgs(BaseModel):
    vendor_name: Optional[str] = Field(None, description="Optional vendor; omit for all vendors")


class ForecastTool(_CtxTool):
    name: str = "forecast_performance"
    description: str = ("Projects each vendor's on-time % and defect % for the next 3 months using a damped linear trend "
                        "with 80% prediction bands; flags projected breaches of the on-time floor and defect ceiling.")
    args_schema: type[BaseModel] = _FcArgs

    def _run(self, vendor_name: str | None = None, **_: Any) -> str:
        def go(vendor_name=None) -> str:
            fc = A.forecast(self.ctx.df, otd_floor=self.ctx.otd_floor, defect_ceiling=self.ctx.defect_ceiling)
            if vendor_name:
                fc = [f for f in fc if vendor_name.lower()[:10] in f["vendor_name"].lower()] or fc
            lines = [f"Thresholds: on-time floor {self.ctx.otd_floor}%, defect ceiling {self.ctx.defect_ceiling}%. Horizon: {fc[0]['future_months']}",
                     "| vendor | outlook | OTD slope/mo | OTD next 3m | OTD low band | defect next 3m | OTD breach | defect breach |",
                     "|---|---|---|---|---|---|---|---|"]
            fc = sorted(fc, key=lambda f: (f["outlook"] != "deteriorating", not f["otd_breach"], f["otd"]["slope_per_month"]))
            for f in fc:
                lines.append(f"| {f['vendor_name']} | {f['outlook']} | {f['otd']['slope_per_month']:+.2f} | {f['otd']['forecast']} | "
                             f"{min(f['otd']['lower']):.1f} | {f['defect']['forecast']} | {'YES' if f['otd_breach'] else 'no'} | "
                             f"{'YES' if f['defect_breach'] else 'no'} |")
            return "\n".join(lines)
        return self.run_traced(go, vendor_name=vendor_name)


# ---------------------------------------------------------- 6 market search
class _SearchArgs(BaseModel):
    queries: list[str] = Field(..., description="1-8 search queries run in parallel, e.g. ['steel prices supply India', 'Chennai port logistics disruption']")


def _news(query: str) -> list[dict[str, Any]]:
    try:
        from ddgs import DDGS
    except ImportError:
        return []
    for _ in range(2):
        try:
            raw = DDGS().news(query, max_results=4, region="in-en", timelimit="m")
            today = time.strftime("%Y-%m-%d")
            return [{"title": r.get("title", ""), "source": r.get("source", ""),
                     "date": d if (d := (r.get("date") or "")[:10]) <= today else "",
                     "snippet": (r.get("body") or "")[:160], "url": r.get("url", "")} for r in raw]
        except Exception:
            time.sleep(1.0)
    return []


class MarketIntelSearchTool(_CtxTool):
    name: str = "market_intel_search"
    description: str = ("Live web news search (last 30 days) for supply-chain risk signals: price moves, shortages, "
                        "regulation, strikes, weather, port/logistics disruption. Accepts a LIST of queries and runs them "
                        "in parallel. Returns dated headlines with sources, grouped by query.")
    args_schema: type[BaseModel] = _SearchArgs

    def _run(self, queries: list[str] | str | None = None, query: str | None = None, **_: Any) -> str:
        qs = queries if isinstance(queries, list) else [q for q in [queries, query] if q]
        qs = [q.strip() for q in qs if q and q.strip()][:8]

        def go(queries=None) -> str:
            from concurrent.futures import ThreadPoolExecutor
            todo = []
            with self.ctx._lock:
                for q in qs:
                    if q.lower() not in self.ctx._search_cache:
                        todo.append(q)
            with ThreadPoolExecutor(max_workers=min(8, len(todo) or 1)) as ex:
                fetched = dict(zip(todo, ex.map(_news, todo)))
            blocks = []
            with self.ctx._lock:
                for q, res in fetched.items():
                    self.ctx._search_cache[q.lower()] = res
                    for r in res:
                        if r["url"] and all(r["url"] != s["url"] for s in self.ctx.sources):
                            self.ctx.sources.append({**r, "query": q})
                for q in qs:
                    res = self.ctx._search_cache.get(q.lower(), [])
                    if not res:
                        blocks.append(f"## {q}\nNo recent news found. Treat as baseline risk (no specific signal).")
                    else:
                        lines = [f"[{r['date']}] {r['title'][:110]} ({r['source']}) - {r['snippet'][:130]} <{r['url'][:120]}>" for r in res[:3]]
                        blocks.append(f"## {q}\n" + "\n".join(lines))
            return "\n\n".join(blocks)
        return self.run_traced(go, queries=qs)


# ---------------------------------------------------------- 7 alternatives
class AlternativeSourcingTool(_CtxTool):
    name: str = "find_alternatives"
    description: str = ("For a vendor, finds alternative suppliers in the same category ranked by score, spare capacity "
                        "(headroom) and risk, with how much spend each could absorb.")
    args_schema: type[BaseModel] = _VendorArgs

    def _run(self, vendor_name: str = "", **_: Any) -> str:
        def go(vendor_name="") -> str:
            alts = A.alternatives(self.ctx.scored, vendor_name)
            if not alts:
                return f"No alternatives found for '{vendor_name}' (unknown vendor or sole supplier in category)."
            return json.dumps(alts)
        return self.run_traced(go, vendor_name=vendor_name)


# --------------------------------------------------------------- 8 what-if
class _WhatIfArgs(BaseModel):
    weights_json: str = Field(..., description='JSON of pillar weights, e.g. {"delivery":50,"quality":20,"cost":10,"responsiveness":5,"compliance":15}')


class ScenarioTool(_CtxTool):
    name: str = "scenario_rescore"
    description: str = ("Re-scores all vendors under an alternative weighting scenario and reports which vendors change rank "
                        "or tier. Use to test whether a recommendation is robust to different business priorities.")
    args_schema: type[BaseModel] = _WhatIfArgs

    def _run(self, weights_json: str = "{}", **_: Any) -> str:
        def go(weights_json="{}") -> str:
            try:
                w = json.loads(weights_json)
            except json.JSONDecodeError:
                return "weights_json must be valid JSON."
            base = self.ctx.scored.set_index("vendor_name")
            alt = A.score(self.ctx.kpis, w).set_index("vendor_name")
            rows = []
            for v in alt.index:
                rows.append((v, int(base.loc[v, "rank"]), int(alt.loc[v, "rank"]), base.loc[v, "tier"], alt.loc[v, "tier"],
                             round(float(alt.loc[v, "composite"]), 1)))
            moved = [r for r in rows if abs(r[1] - r[2]) >= 2 or r[3] != r[4]]
            out = [f"Scenario weights: {w}. {len(moved)} vendors materially change (>=2 ranks or tier):"]
            out += [f"- {v}: rank {a}->{b}, tier {ta}->{tb}, score {s}" for v, a, b, ta, tb, s in moved] or ["- none (ranking is robust)"]
            return "\n".join(out)
        return self.run_traced(go, weights_json=weights_json)


def make_tools(ctx: RunContext, agent_label: str, names: list[str]) -> list[BaseTool]:
    reg = {
        "audit": DataQualityTool, "scorecard": VendorScorecardTool, "profile": VendorProfileTool,
        "spend": SpendAnalyticsTool, "forecast": ForecastTool, "search": MarketIntelSearchTool,
        "alternatives": AlternativeSourcingTool, "scenario": ScenarioTool,
    }
    return [reg[n](ctx=ctx, agent_label=agent_label) for n in names]

Writing safeprocure/tools.py


**`safeprocure/schemas.py`** · Pydantic output schemas: every agent returns typed, validated JSON.

In [9]:
%%writefile safeprocure/schemas.py
"""Structured (Pydantic) outputs for every agent. Typed outputs make the dashboard reliable."""
from __future__ import annotations

from typing import Literal

from pydantic import BaseModel, Field


class IntakePlan(BaseModel):
    objective: str = Field(..., description="One-sentence restatement of what the user needs")
    question_type: Literal["portfolio_review", "cost_reduction", "risk_scan", "vendor_specific", "consolidation"]
    priority_pillars: list[str] = Field(default_factory=list, description="Pillars that matter most: delivery, quality, cost, responsiveness, compliance")
    vendors_of_interest: list[str] = Field(default_factory=list)
    success_criteria: list[str] = Field(default_factory=list, description="2-4 measurable criteria for a good answer")
    instructions_for_team: str = Field(..., description="Short brief to the analyst agents on what to emphasise")


class AuditFindings(BaseModel):
    verdict: Literal["fit_for_analysis", "usable_with_caveats", "not_reliable"]
    summary: str
    caveats: list[str] = Field(default_factory=list)


class Finding(BaseModel):
    vendor: str
    headline: str = Field(..., description="Short finding, <= 14 words")
    evidence: str = Field(..., description="Specific numbers from the tools")
    severity: Literal["positive", "low", "medium", "high", "critical"] = "medium"


class PerformanceFindings(BaseModel):
    summary: str
    top_performers: list[Finding] = Field(default_factory=list)
    underperformers: list[Finding] = Field(default_factory=list)
    trend_alerts: list[Finding] = Field(default_factory=list)


class SavingsLever(BaseModel):
    lever: str
    categories: list[str] = Field(default_factory=list)
    vendors: list[str] = Field(default_factory=list)
    estimated_annual_value_inr: float = 0
    rationale: str


class CostFindings(BaseModel):
    summary: str
    concentration_risks: list[Finding] = Field(default_factory=list)
    savings_levers: list[SavingsLever] = Field(default_factory=list)


class MarketSignal(BaseModel):
    category: str
    region: str = "All India"
    risk_score: float = Field(..., ge=0, le=100, description="0 = calm, 100 = severe disruption")
    direction: Literal["rising", "stable", "easing"] = "stable"
    headline: str
    source: str = ""
    date: str = ""
    url: str = ""
    implication: str


class MarketFindings(BaseModel):
    summary: str
    signals: list[MarketSignal] = Field(default_factory=list)


class EarlyWarning(BaseModel):
    vendor: str
    metric: Literal["on_time", "defects", "both"]
    projection: str = Field(..., description="e.g. 'OTD 80.1% -> 70.2% by 2026-12'")
    confidence: Literal["low", "medium", "high"]
    recommended_trigger: str


class ForecastFindings(BaseModel):
    summary: str
    early_warnings: list[EarlyWarning] = Field(default_factory=list)
    improving: list[str] = Field(default_factory=list)


class VendorDecision(BaseModel):
    vendor: str
    decision: Literal["Keep", "Watch", "Develop", "Replace"]
    confidence: float = Field(..., ge=0, le=1)
    rationale: str = Field(..., description="One sentence (<= 30 words) citing the decisive numbers")
    actions: list[str] = Field(default_factory=list, description="1-2 concrete next actions, <= 12 words each")


class StrategyDecision(BaseModel):
    portfolio_summary: str
    robustness_note: str = Field("", description="Result of scenario re-scoring: is the ranking robust?")
    decisions: list[VendorDecision]


class ShiftPlan(BaseModel):
    to_vendor: str
    share_of_volume_pct: float


class MitigationPlan(BaseModel):
    vendor: str
    risk: str
    alternatives: list[ShiftPlan] = Field(default_factory=list)
    days_30: list[str] = Field(default_factory=list)
    days_60: list[str] = Field(default_factory=list)
    days_90: list[str] = Field(default_factory=list)
    expected_impact: str


class MitigationFindings(BaseModel):
    summary: str
    plans: list[MitigationPlan] = Field(default_factory=list)


class QACheck(BaseModel):
    claim: str
    status: Literal["verified", "corrected", "unsupported"]
    note: str


class QAReview(BaseModel):
    verdict: Literal["approve", "revise"]
    confidence: float = Field(..., ge=0, le=1)
    checks: list[QACheck] = Field(default_factory=list)
    required_changes: list[str] = Field(default_factory=list)


class KeyNumber(BaseModel):
    label: str
    value: str


class Action(BaseModel):
    action: str
    owner: str
    timeline: str
    impact: str


class ExecutiveBrief(BaseModel):
    headline: str = Field(..., description="<= 16 words")
    answer: str = Field(..., description="Direct answer to the user's question, 2-4 sentences")
    summary: list[str] = Field(..., description="3 short paragraphs")
    key_numbers: list[KeyNumber] = Field(default_factory=list)
    top_actions: list[Action] = Field(default_factory=list)

Writing safeprocure/schemas.py


**`safeprocure/agents.py`** · The agent roster: role, goal, backstory, model tier and tools for each specialist.

In [10]:
%%writefile safeprocure/agents.py
"""The Safe Procure agent roster: 10 specialists + 1 copilot.

Each agent has a role, goal and backstory (CrewAI's persona model), a model tier
(see config.TIERS) and the tools it is allowed to use.
"""
from __future__ import annotations

from dataclasses import dataclass

from crewai import Agent

from .llm import ResilientLLM
from .tools import RunContext, make_tools


@dataclass(frozen=True)
class AgentSpec:
    key: str
    name: str
    role: str
    goal: str
    backstory: str
    tier: str
    tools: tuple[str, ...]
    max_iter: int = 6
    max_completion: int = 1800


ROSTER: dict[str, AgentSpec] = {s.key: s for s in [
    AgentSpec("intake", "Intake Router", "Procurement Intake Router",
              "Translate the user's request into a precise analysis plan the team can execute.",
              "A chief-of-staff to the CPO who turns vague asks into crisp, measurable briefs.", "intake", ()),
    AgentSpec("audit", "Data Quality Auditor", "Vendor Data Quality Auditor",
              "Decide whether the vendor dataset is reliable enough to base sourcing decisions on.",
              "An ex-internal-auditor who never lets a decision rest on dirty data.", "lite", ("audit",), 3),
    AgentSpec("performance", "Performance Analyst", "Supplier Performance Analyst",
              "Identify the best and worst performing vendors and the trends behind them, with numbers.",
              "A supplier-quality engineer who reads OTD, defect and lead-time data like an ECG.", "analysis",
              ("scorecard", "profile"), 5),
    AgentSpec("cost", "Cost & Spend Analyst", "Category Spend Analyst",
              "Find spend concentration risk and quantified savings levers.",
              "A category manager obsessed with Pareto charts, HHI and price variance.", "analysis_b", ("spend", "scorecard"), 4),
    AgentSpec("market", "Market Risk Researcher", "Supply Market Risk Researcher",
              "Score current external supply risk for each category and key region from live news.",
              "A supply-market intelligence analyst who scans news daily for disruptions before they hit the plant.",
              "research", ("search",), 4, 2200),
    AgentSpec("forecast", "Forecast Agent", "Supplier Performance Forecaster",
              "Project next-quarter vendor performance and raise early warnings before thresholds are breached.",
              "A demand-and-supply planner who trusts trends but respects uncertainty bands.", "analysis_b", ("forecast",), 4),
    AgentSpec("strategy", "Procurement Strategist", "Chief Procurement Strategist",
              "Assign every vendor a Keep / Watch / Develop / Replace decision that balances performance, cost and risk.",
              "A CPO with 20 years of sourcing experience who defends every call with evidence.", "reasoning",
              ("profile", "scenario"), 6, 3200),
    AgentSpec("mitigation", "Mitigation Planner", "Supply Risk Mitigation Planner",
              "For each high-risk vendor, design a dual-sourcing shift and a 30/60/90-day action plan.",
              "A supply-continuity lead who has rescued production lines from supplier failures.", "analysis",
              ("alternatives", "profile"), 6, 2400),
    AgentSpec("qa", "QA Reviewer", "Independent Decision QA Reviewer",
              "Verify every material claim against the data and reject recommendations the numbers do not support.",
              "A sceptical internal-audit partner. Runs on a different model family from the strategist to avoid shared blind spots.",
              "review", ("profile", "scorecard"), 6, 2200),
    AgentSpec("brief", "Executive Writer", "Executive Briefing Writer",
              "Write a board-ready brief that answers the user's question directly with numbers and actions.",
              "A former strategy consultant who writes for CFOs: short, quantified, decisive.", "reasoning", (), 3, 2400),
    AgentSpec("copilot", "Procurement Copilot", "Procurement Copilot",
              "Answer follow-up questions about this analysis accurately, using tools to check numbers.",
              "The team's always-on analyst who knows every number in the report.", "analysis",
              ("profile", "scorecard", "scenario", "alternatives"), 5),
]}

FLOW_ORDER = ["intake", "audit", "performance", "cost", "market", "forecast", "strategy", "mitigation", "qa", "brief"]


def build_agent(key: str, ctx: RunContext) -> Agent:
    s = ROSTER[key]
    llm = ResilientLLM(s.tier, s.name, telemetry=ctx.telemetry, max_completion=s.max_completion)
    return Agent(
        role=s.role, goal=s.goal, backstory=s.backstory, llm=llm,
        tools=make_tools(ctx, s.name, list(s.tools)), max_iter=s.max_iter,
        allow_delegation=False, verbose=False, respect_context_window=True,
    )

Writing safeprocure/agents.py


**`safeprocure/grounding.py`** · Grounding checks in code: percentages, rupee savings and vendor tier/KPI claims in agent prose must match computed numbers.

In [11]:
%%writefile safeprocure/grounding.py
"""Grounding checks for agent prose: numbers in text must trace to numbers computed in code.

* ground_percents: strips percentages that match no computed figure (forward-looking claims need computed projections).
* ground_savings: rupee savings claims must equal a computed savings figure.
* check_vendor_claims: tier and KPI claims about a named vendor are compared with that vendor's data (used by the copilot).
"""
from __future__ import annotations

import re
from typing import Any

from . import analytics as A

# Percent claims in agent prose: "8-10%", "by ~1.5%", "to >95%", "↓15%", "from 68.5%".
_N = r"\d+(?:\.\d+)?"
_PCT = re.compile(rf"(?P<pre>\b(?:by|of|to|from|at)\s+)?(?P<cmp>(?:[~≈><≥≤↑↓]=?|approx(?:imately|\.)?|about|around|up to|over|under|"
                  rf"above|below|nearly|almost)\s*)?(?P<a>{_N})(?:\s*%?\s*[-–—]\s*(?P<b>{_N})|\s+to\s+(?P<c>{_N}))?\s*(?:%|\s?percent\b)", re.I)
_PROJ_BEFORE = re.compile(r"(improv|increas|rais|rise|rising|lift|boost|reduc|cut|lower|drop|fall|decreas|declin|achiev|reach|target|"
                          r"stabili[sz]|bring|gain|grow|shrink|mitigat|expect|project|will|down|up\b|[↑↓])(?:(?!\.\s|[;:,]).)*$", re.I)
_PROJ_AFTER = re.compile(r"^\s*(reduction|improvement|increase|decrease|drop|gain|rise|lower|higher|better|faster|fewer|less|more|cut|uplift)", re.I)
_VOLUME_BEFORE = re.compile(r"(shift|realloc|mov|transfer|transition|allocat|split)\w*\s+(?:\w+\s+){0,2}$", re.I)
_VOLUME_AFTER = re.compile(r"^\s*(?:of\s+(?:\w+\s+){0,2})?(volume|share|spend)\b", re.I)


def _known(v: str, pool: set[float]) -> bool:
    """Match at the claim's own precision: '1.5' covers 1.45-1.55; '96' covers 95.5-96.99 (rounded or truncated)."""
    x, half = float(v), 0.5 * 10 ** -len(v.partition(".")[2]) + 1e-9
    return any(x - half <= k < x + (half if "." in v else 1) for k in pool)


def _tidy(s: str) -> str:
    s = re.sub(r"\(\s*\)", "", s)
    s = re.sub(rf"\bfrom\s+{_N}\s*%\s*(?=[,.;:)]|\band\b|$)", "", s)   # orphaned "from X%" once its "to Y%" went
    s = re.sub(r"\b(from|to|by|of|at)\s*(?=[,.;:)]|$)", "", s)
    s = re.sub(r"\s+([,.;:)])", r"\1", s)
    return re.sub(r"\s{2,}", " ", s).strip(" ,;")


def ground_percents(text: str, general: set[float], projected: set[float]) -> tuple[str, list[str]]:
    """Remove percentages that do not trace to computed numbers. Forward-looking claims (improve, reduce, ↓, '...reduction')
    may only use computed projections; plain mentions may use any computed KPI. Returns (text, removed claims)."""
    if not isinstance(text, str) or "%" not in text and "percent" not in text.lower():
        return text, []
    removed: list[str] = []

    def fix(m: re.Match) -> str:
        before = text[max(0, m.start() - 60):m.start()] + (m.group("pre") or "") + (m.group("cmp") or "")
        after = text[m.end():m.end() + 25]
        # Forward-looking: a change verb in the clause, "...% reduction", "to/by X%", or a bound ("> 95%", "~1.5%", "up to").
        proj = bool(_PROJ_BEFORE.search(before) or _PROJ_AFTER.search(after) or (m.group("cmp") or "").strip()
                    or (m.group("pre") or "").strip().lower() in ("to", "by"))
        if _VOLUME_BEFORE.search(before) or _VOLUME_AFTER.search(after):
            proj = False   # a volume share being moved, not an outcome
        pool = projected if proj else general
        vals = [x for x in (m.group("a"), m.group("b"), m.group("c")) if x]
        if all(_known(v, pool) for v in vals):
            return m.group(0)
        removed.append(m.group(0).strip())
        return " "
    out = _PCT.sub(fix, text)
    return (_tidy(out), removed) if removed else (text, [])


# ------------------------------------------------------------------ rupees
_INR = re.compile(r"(?:₹|INR|Rs\.?)\s?(?P<v>\d[\d,]*(?:\.\d+)?)\s*(?P<u>Cr\b|crore|L\b|lakh|lac\b)?|(?P<v2>\d[\d,]*(?:\.\d+)?)\s*(?P<u2>Cr\b|crore|L\b|lakh)\s*(?:INR|rupees)?", re.I)
_SAVING = re.compile(r"sav|opportunit|captur|unlock|recover", re.I)


def parse_inr(raw: str, unit: str | None) -> float:
    v = float(raw.replace(",", ""))
    u = (unit or "").lower()
    return v * 1e7 if u in ("cr", "crore") else v * 1e5 if u in ("l", "lakh", "lac") else v


def ground_savings(text: str, allowed: list[float], replacement: float) -> tuple[str, list[str]]:
    """A rupee amount in a savings sentence must equal a computed savings figure (to its shown precision);
    otherwise it is replaced by the computed figure. Returns (text, corrected claims)."""
    if not isinstance(text, str) or not _SAVING.search(text):
        return text, []
    fixed: list[str] = []

    def fix(m: re.Match) -> str:
        raw, unit = (m.group("v"), m.group("u")) if m.group("v") else (m.group("v2"), m.group("u2"))
        left = re.split(r"[,;]|\.\s|\b(?:while|and|but|whereas)\b", text[max(0, m.start() - 70):m.start()])[-1]
        right = re.split(r"[,;]|\.\s|\.$|\b(?:while|and|but|whereas)\b", text[m.end():m.end() + 40])[0]
        if not _SAVING.search(left + right):   # only amounts in a savings clause are checked
            return m.group(0)
        val = parse_inr(raw, unit)
        step = parse_inr("0." + "0" * (len(raw.partition(".")[2]) - 1) + "5", unit) if "." in raw else parse_inr("0.5", unit) * 2
        if any(abs(val - a) <= step + 1 for a in allowed):
            return m.group(0)
        fixed.append(m.group(0).strip())
        return A.inr(replacement)
    out = _INR.sub(fix, text)
    return (out, fixed) if fixed else (text, [])


def savings_figures(result_or_cost: dict, total: float) -> list[float]:
    levers = [float(lv.get("estimated_annual_value_inr") or 0) for lv in result_or_cost.get("savings_levers", [])]
    return [total, *levers, sum(levers)]


# ------------------------------------------------------- vendor-specific claims
_METRICS = [("on_time_pct", "on-time delivery", r"on[\s-]?time|\bOTD\b|deliver"), ("defect_rate_pct", "defect rate", r"defect"),
            ("price_variance_pct", "price variance", r"price\s+variance|overpa"), ("fill_rate_pct", "fill rate", r"fill"),
            ("capacity_utilization_pct", "capacity utilisation", r"capacity|utili[sz]")]
_TIER = re.compile(r"\btier[\s-]?(?P<t>[A-D])\b|\b(?P<t2>[A-D])[\s-]tier\b")
_SCENARIO = re.compile(r"\b(would|could|if|scenario|weight\w*|re-?scor\w*|drops?|dropping|falls?|moves?|becomes?|rises?|climbs?|"
                       r"improves?|to tier|into tier)\b", re.I)
_REFERENCE = re.compile(r"\b(threshold|mark|target|floor|ceiling|limit|benchmark|standard|cut-?off|bar|criteri\w*|requirement|"
                        r"minimum|maximum|tolerance|goal)\b", re.I)
_NEGATED = re.compile(r"\b(not|nor|no|never|fails?|failing|misses|missing)\b|n't\b", re.I)
_ALL = re.compile(r"\b(both|all|each|either)\b[^.;]{0,60}$", re.I)
_UPPER = re.compile(r"[>≥]|above|over|more than|at least|exceed", re.I)
_LOWER = re.compile(r"[<≤]|below|under|less than|at most", re.I)


def _vendor_aliases(vendors: list[dict]) -> list[tuple[str, dict]]:
    """(alias, vendor) pairs: full name, name without 'Pvt Ltd', and the first word when it is unique."""
    firsts: dict[str, int] = {}
    for v in vendors:
        firsts[v["vendor"].split()[0].lower()] = firsts.get(v["vendor"].split()[0].lower(), 0) + 1
    out = []
    for v in vendors:
        short = v["vendor"].replace(" Pvt Ltd", "")
        out += [(v["vendor"], v), (short, v)]
        first = v["vendor"].split()[0]
        if firsts[first.lower()] == 1 and len(first) > 3:
            out.append((first, v))
    return sorted(out, key=lambda a: -len(a[0]))


def _mentions(sentence: str, aliases: list[tuple[str, dict]]) -> list[tuple[int, dict]]:
    found, taken = [], []
    for alias, v in aliases:
        for m in re.finditer(rf"\b{re.escape(alias)}\b", sentence, re.I):
            if not any(s <= m.start() < e for s, e in taken):
                taken.append((m.start(), m.end()))
                found.append((m.start(), v))
    return sorted(found, key=lambda x: x[0])


def _subjects(sentence: str, pos: int, mentions: list[tuple[int, dict]]) -> list[dict]:
    before = [v for p, v in mentions if p < pos]
    if not before:
        return []
    m = _ALL.search(sentence[:pos])
    if m:
        uniq = list({v["vendor"]: v for v in before}.values())
        return uniq[-2:] if m.group(1).lower() in ("both", "either") else uniq
    return [before[-1]]


def check_vendor_claims(text: str, vendors: list[dict]) -> list[str]:
    """Compare tier and KPI claims about named vendors with their data. Each vendor dict needs 'vendor', 'tier', 'kpis'.
    Returns human-readable corrections (empty when every checked claim holds)."""
    aliases = _vendor_aliases(vendors)
    issues: list[str] = []
    for sent in re.split(r"(?<=[.!?;])\s+|\n+", text or ""):
        ments = _mentions(sent, aliases)
        if not ments:
            continue
        # What-if answers move vendors between tiers ("would fall to Tier D"); KPIs never change under re-weighting.
        tier_claims = [] if _SCENARIO.search(sent) else _TIER.finditer(sent)
        for m in tier_claims:
            claimed = m.group("t") or m.group("t2")
            for v in _subjects(sent, m.start(), ments):
                if v.get("tier") and v["tier"] != claimed:
                    issues.append(f"{v['vendor']} is tier {v['tier']}, not tier {claimed}")
        for m in _PCT.finditer(sent):
            if m.group("b") or m.group("c"):
                continue
            window = sent[max(0, m.start() - 45):m.start()]
            after = re.split(r",|;|\band\b|\bwhile\b", sent[m.end():m.end() + 25])[0]   # "96% and defect..." is not about defects
            near = []   # (distance, key, label): the metric word closest to the number wins
            for key, label, pat in _METRICS:
                near += [(len(window) - h.end(), key, label) for h in re.finditer(pat, window, re.I)]
                near += [(h.start() + 1, key, label) for h in re.finditer(pat, after, re.I)]
            if not near:
                continue
            metric = min(near)[1:]
            clause = re.split(r"[,;]|\bwhich\b|\bwhile\b", window)[-1]
            if _REFERENCE.search(clause + " " + after) or _NEGATED.search(clause):
                continue   # "the 95% threshold", "does not meet 95%": a yardstick, not the vendor's own value
            x, cmp_ = float(m.group("a")), (m.group("cmp") or "") + window[-12:]
            for v in _subjects(sent, m.start(), ments):
                actual = (v.get("kpis") or {}).get(metric[0])
                if actual is None:
                    continue
                ok = (actual >= x - 0.5 if _UPPER.search(cmp_) else actual <= x + 0.05 if _LOWER.search(cmp_)
                      else _known(m.group("a"), {float(actual)}))
                if not ok:
                    issues.append(f"{v['vendor']} {metric[1]} is {actual:g}%, not {m.group(0).strip()}")
    return list(dict.fromkeys(issues))

Writing safeprocure/grounding.py


**`safeprocure/flow.py`** · **SafeProcureFlow**: the CrewAI Flow with parallel fan-out (`and_`), the risk-gate `@router`, the QA revision loop, guardrails, the grounding guard and deterministic fallbacks for every stage.

In [12]:
%%writefile safeprocure/flow.py
"""SafeProcureFlow: the multi-agent workflow (CrewAI Flow).

    intake -> audit -> [performance | cost | market]  (parallel)
           -> forecast -> strategy -> (risk gate) -> mitigation? -> QA -> (revise loop?) -> brief

* Parallel fan-out/fan-in with and_().
* A router sends high-risk portfolios through the Mitigation Planner, low-risk ones straight to QA.
* The QA Reviewer can send the strategy back for ONE revision with required changes.
* Every stage has a deterministic fallback, so a run always completes (degraded stages are labelled).
"""
from __future__ import annotations

import asyncio
import json
import re
import time
import uuid
from difflib import get_close_matches
from typing import Any

import pandas as pd
from crewai import Crew, Process, Task
from crewai.flow import Flow, and_, listen, or_, router, start
from pydantic import BaseModel

from . import analytics as A
from .agents import ROSTER, build_agent
from .grounding import ground_percents, ground_savings, savings_figures
from .schemas import (AuditFindings, CostFindings, ExecutiveBrief, ForecastFindings, IntakePlan, MarketFindings,
                      MitigationFindings, PerformanceFindings, QAReview, StrategyDecision)
from .telemetry import RunTelemetry
from .tools import RunContext, make_tools


def _quarter() -> str:
    t = time.localtime()
    q = (t.tm_mon - 1) // 3 + 1
    nq, ny = (q % 4) + 1, t.tm_year + (1 if q == 4 else 0)
    names = {1: "Jan-Mar", 2: "Apr-Jun", 3: "Jul-Sep", 4: "Oct-Dec"}
    return f"current quarter Q{q} {t.tm_year} = {names[q]} {t.tm_year}; next quarter Q{nq} {ny}"


class SPState(BaseModel):
    plan: dict = {}
    audit: dict = {}
    performance: dict = {}
    cost: dict = {}
    market: dict = {}
    forecast: dict = {}
    strategy: dict = {}
    mitigation: dict = {}
    qa: dict = {}
    brief: dict = {}
    revisions: int = 0
    degraded: list[str] = []
    stage_seconds: dict = {}
    path: list[str] = []
    grounding: list[str] = []


def _j(x: Any, limit: int = 3500) -> str:
    s = json.dumps(x, default=str, ensure_ascii=False)
    return s if len(s) <= limit else s[:limit] + "...(truncated)"


def _coerce(raw: str, model: type[BaseModel]) -> dict | None:
    m = re.search(r"\{.*\}", raw or "", re.S)
    if not m:
        return None
    try:
        return model.model_validate_json(m.group(0)).model_dump()
    except Exception:
        return None


class SafeProcureFlow(Flow[SPState]):
    def __init__(self, ctx: RunContext, **kw: Any):
        super().__init__(**kw)
        self._ctx = ctx

    # ------------------------------------------------------------ plumbing
    @property
    def ctx(self) -> RunContext:
        return self._ctx

    def _vendor_names(self) -> list[str]:
        return self.ctx.in_scope.vendor_name.tolist()

    def _match_vendor(self, name: str) -> str | None:
        names = self.ctx.scored.vendor_name.tolist()
        if name in names:
            return name
        low = {n.lower(): n for n in names}
        key = name.lower().strip()
        for n_low, n in low.items():
            if key and (key in n_low or n_low.replace(" pvt ltd", "") in key):
                return n
        hit = get_close_matches(key, list(low), n=1, cutoff=0.6)
        return low[hit[0]] if hit else None

    def _ground(self, stage: str, items: list[dict], key: str = "vendor") -> list[dict]:
        """Drop findings about vendors that do not exist; canonicalise the rest."""
        kept, dropped = [], []
        for it in items or []:
            m = self._match_vendor(str(it.get(key, "")))
            if m:
                kept.append({**it, key: m})
            else:
                dropped.append(str(it.get(key, "")))
        if dropped:
            self.ctx.telemetry.emit("grounding", stage=stage, dropped=dropped)
            self.state.grounding.extend(f"{stage}: removed finding about unknown vendor '{d}'" for d in dropped)
        return kept

    def _project(self, plan: dict) -> dict | None:
        """Blended OTD/defects on the at-risk vendor's volume once the chosen shares move to the alternatives."""
        s = self.ctx.scored.set_index("vendor_name")
        if plan.get("vendor") not in s.index:
            return None
        r = s.loc[plan["vendor"]]
        shares = [(a["to_vendor"], max(0.0, float(a.get("share_of_volume_pct") or 0))) for a in plan.get("alternatives", [])
                  if a.get("to_vendor") in s.index]
        total = sum(sh for _, sh in shares)
        if total <= 0:
            return None
        if total > 100:
            shares, total = [(v, sh * 100 / total) for v, sh in shares], 100.0
        keep = 100 - total
        otd = (keep * r.on_time_pct + sum(sh * s.loc[v].on_time_pct for v, sh in shares)) / 100
        dfx = (keep * r.defect_rate_pct + sum(sh * s.loc[v].defect_rate_pct for v, sh in shares)) / 100
        return {"shift_pct": round(total, 1), "otd_now": round(float(r.on_time_pct), 1), "otd_after": round(float(otd), 1),
                "defect_now": round(float(r.defect_rate_pct), 2), "defect_after": round(float(dfx), 2)}

    @staticmethod
    def _projection_text(p: dict) -> str:
        return (f"OTD {p['otd_now']:.1f}% → {p['otd_after']:.1f}%, defects {p['defect_now']:.2f}% → {p['defect_after']:.2f}% "
                f"on this volume after shifting {p['shift_pct']:g}% (computed from current KPIs)")

    def _pct_pools(self, plans: list[dict]) -> tuple[set[float], set[float]]:
        """(general, projected): percentages an agent may cite in plain statements vs. in forward-looking claims."""
        c = self.ctx
        rules = {float(c.otd_floor), float(c.defect_ceiling), float(c.risk_threshold)}
        setup = {65.0, 35.0} | {float(w) for w in (c.weights or {}).values()}   # risk blend + pillar weights
        targets = {p.get("vendor") for p in plans}
        fc: set[float] = set()        # forecast end points of projected breaches + recent history of planned vendors
        hist: set[float] = set()      # any vendor's recent history (plain mentions only)
        for f in A.forecast(c.df, otd_floor=c.otd_floor, defect_ceiling=c.defect_ceiling):
            for m in ("otd", "defect"):
                recent = {float(x) for x in f[m].get("history", [])[-3:]}
                hist |= recent
                if f["otd_breach"] or f["defect_breach"]:
                    fc.add(float(f[m]["forecast"][-1]))
                if f["vendor_name"] in targets:
                    fc |= recent
        plan_vals: set[float] = set()
        s = c.scored.set_index("vendor_name")
        moved: set[float] = set()   # volume shares are plan inputs, not outcomes: plain mentions only
        for p in plans:
            moved |= {float(a.get("share_of_volume_pct") or 0) for a in p.get("alternatives", [])}
            pr = p.get("projection") or {}
            moved.add(float(pr.get("shift_pct", 0)))
            plan_vals |= {float(v) for k, v in pr.items() if k != "shift_pct"}
            if p.get("vendor") in s.index:
                plan_vals |= {float(s.loc[p["vendor"]][k]) for k in ("on_time_pct", "defect_rate_pct")}
        pct_cols = ("on_time_pct", "defect_rate_pct", "price_variance_pct", "fill_rate_pct", "capacity_utilization_pct")
        kpis = {float(x) for col in pct_cols if col in c.scored.columns for x in c.scored[col].dropna()}
        shares = {x["top_share"] * 100 for x in A.spend_analysis(c.df, c.kpis)["categories"]}
        projected = rules | fc | plan_vals
        return projected | kpis | shares | moved | hist | setup, projected

    def _ground_text(self, stage: str, x: Any, general: set[float], projected: set[float], drop_sentences: bool = False) -> Any:
        if isinstance(x, str):
            if drop_sentences:   # secondary prose: drop a sentence with an unsupported claim rather than patch it
                kept, removed = [], []
                for sent in re.split(r"(?<=[.!?])\s+(?=[A-Z])", x):
                    bad = ground_percents(sent, general, projected)[1]
                    if bad:
                        removed.extend(bad)
                    else:
                        kept.append(sent)
                out = " ".join(kept)
            else:
                out, removed = ground_percents(x, general, projected)
            if removed:
                self.state.grounding.extend(f"{stage}: removed unsupported projection '{r}'" for r in removed)
                self.ctx.telemetry.emit("grounding_pct", stage=stage, removed=removed)
            return out
        if isinstance(x, list):
            return [self._ground_text(stage, i, general, projected) for i in x]
        if isinstance(x, dict):
            return {k: self._ground_text(stage, v, general, projected) for k, v in x.items()}
        return x

    def _stage(self, key: str, description: str, expected: str, out_model: type[BaseModel], fallback) -> dict:
        tel: RunTelemetry = self.ctx.telemetry
        spec = ROSTER[key]
        tel.emit("stage_start", stage=key, agent=spec.name, tier=spec.tier, tools=list(spec.tools))
        t = time.time()
        status, data = "ok", None
        try:
            agent = build_agent(key, self.ctx)
            task = Task(description=description, agent=agent, output_pydantic=out_model,
                        expected_output=expected + " Give the final answer as plain JSON text, never as a tool call.")
            res = Crew(agents=[agent], tasks=[task], process=Process.sequential, verbose=False).kickoff()
            if getattr(res, "pydantic", None) is not None:
                data = res.pydantic.model_dump()
            else:
                data = _coerce(res.raw, out_model)
            if data is None:
                raise ValueError("Agent did not return valid structured output")
        except Exception as err:  # noqa: BLE001 - never let a stage kill the run
            tel.emit("stage_error", stage=key, agent=spec.name, message=str(err)[:300])
            status, data = "degraded", fallback()
            self.state.degraded.append(key)
        secs = round(time.time() - t, 1)
        self.state.stage_seconds[key] = secs
        self.state.path.append(key)
        tel.emit("stage_end", stage=key, agent=spec.name, status=status, seconds=secs, output=data)
        return data

    async def _astage(self, *a, **kw) -> dict:
        return await asyncio.to_thread(self._stage, *a, **kw)

    def _scope_note(self) -> str:
        c = self.ctx
        cats = ", ".join(c.focus_categories) if c.focus_categories else "all categories"
        today = time.strftime("%Y-%m-%d")
        return (f"Today is {today} (calendar: {_quarter()}). Scope: {cats}. Vendors in scope: {len(c.in_scope)}. "
                f"Pillar weights: {c.weights}. Risk threshold (combined risk 0-100): {c.risk_threshold}. "
                f"On-time floor {c.otd_floor}%, defect ceiling {c.defect_ceiling}%.\n"
                f"The ONLY vendors that exist (use these exact names, never invent others): {', '.join(self._vendor_names())}.\n"
                "Money is INR; write amounts with L (lakh) or Cr (crore) exactly as the tools show them, never M or bn.")

    # ------------------------------------------------------------ 1 intake
    @start()
    def intake(self):
        c = self.ctx
        q = c.question or "Run a full vendor portfolio risk review and recommend what to do with each vendor."
        desc = (f"User request: \"{q}\"\n{self._scope_note()}\n"
                f"Dataset: {c.quality['vendors']} vendors, categories {c.quality['categories']}, "
                f"months {c.quality['months']}, total spend {A.inr(c.quality['total_spend'])}.\n"
                f"Vendor names: {', '.join(self._vendor_names())}.\n"
                "Produce the analysis plan. Classify the question, pick the pillars that matter most for THIS request, "
                "list any vendors explicitly mentioned, define 2-4 success criteria, and write a short brief for the team.")

        def fb():
            ql = q.lower()
            qt = ("cost_reduction" if any(w in ql for w in ("cost", "sav", "cheap", "price")) else
                  "risk_scan" if "risk" in ql else "consolidation" if "consolidat" in ql else "portfolio_review")
            return IntakePlan(objective=q, question_type=qt, priority_pillars=["delivery", "quality", "cost"],
                              success_criteria=["Every vendor has a decision", "Risks quantified"],
                              instructions_for_team="Full portfolio review.").model_dump()
        self.state.plan = self._stage("intake", desc, "An IntakePlan JSON object.", IntakePlan, fb)
        return self.state.plan

    # ------------------------------------------------------------- 2 audit
    @listen(intake)
    def audit(self):
        desc = ("Call audit_dataset once. Judge whether the data is fit for sourcing decisions. "
                "Summarise the issues found and the cleaning applied, and list caveats the decision-makers must know.")

        def fb():
            q = self.ctx.quality
            v = "fit_for_analysis" if q["quality_score"] >= 90 else "usable_with_caveats" if q["quality_score"] >= 70 else "not_reliable"
            return AuditFindings(verdict=v, summary=f"Quality score {q['quality_score']}/100; {len(q['issues'])} issue types auto-cleaned.",
                                 caveats=[i["action"] for i in q["issues"]]).model_dump()
        self.state.audit = self._stage("audit", desc, "An AuditFindings JSON object.", AuditFindings, fb)
        return self.state.audit

    # ---------------------------------------------- 3a/3b/3c parallel analysts
    @listen(audit)
    async def performance(self):
        desc = (f"{self._scope_note()}\nTeam brief: {self.state.plan.get('instructions_for_team', '')}\n"
                "Call vendor_scorecard ONCE (sort_by internal_risk) and vendor_profile for at most 2 of the most "
                "concerning vendors. Report top performers, underperformers and trend alerts (deteriorating or improving "
                "momentum). Every finding must cite exact numbers from the tools.")

        def fb():
            s = self.ctx.in_scope
            top = [{"vendor": r.vendor_name, "headline": f"Tier {r.tier}, score {r.composite:.0f}", "evidence": f"OTD {r.on_time_pct:.1f}%, defects {r.defect_rate_pct:.2f}%", "severity": "positive"} for r in s.head(3).itertuples()]
            bot = [{"vendor": r.vendor_name, "headline": f"Tier {r.tier}, risk {r.internal_risk:.0f}", "evidence": f"OTD {r.on_time_pct:.1f}%, defects {r.defect_rate_pct:.2f}%", "severity": "high"} for r in s.sort_values("internal_risk", ascending=False).head(3).itertuples()]
            return PerformanceFindings(summary="Rule-based summary (agent unavailable).", top_performers=top, underperformers=bot).model_dump()
        out = await self._astage("performance", desc, "A PerformanceFindings JSON object.", PerformanceFindings, fb)
        for k in ("top_performers", "underperformers", "trend_alerts"):
            out[k] = self._ground("performance", out.get(k, []))
        self.state.performance = out
        return self.state.performance

    @listen(audit)
    async def cost(self):
        desc = (f"{self._scope_note()}\nTeam brief: {self.state.plan.get('instructions_for_team', '')}\n"
                "Call spend_analytics. Identify categories with dangerous single-vendor concentration (share > 45% or HHI > 2500) "
                "and quantify 2-4 savings levers (renegotiation of positive price variance, volume consolidation to best-value "
                "vendors, etc.) in INR per year. Every lever value must be derived from the tool's per-category savings opportunity; "
                "the levers together cannot exceed the tool's annualised savings opportunity. Use only vendor names from the tools.")

        def fb():
            sp = A.spend_analysis(self.ctx.df, self.ctx.kpis)
            conc = [{"vendor": c["top_vendor"], "headline": f"{c['top_share']:.0%} of {c['category']} spend", "evidence": f"HHI {c['hhi']}", "severity": "high"} for c in sp["categories"] if c["concentration_flag"]]
            lev = [{"lever": "Renegotiate positive price variance", "categories": [c["category"] for c in sp["categories"]], "vendors": [], "estimated_annual_value_inr": sp["savings_opportunity_total"], "rationale": "Bring prices to best-in-category variance."}]
            return CostFindings(summary="Rule-based summary (agent unavailable).", concentration_risks=conc, savings_levers=lev).model_dump()
        out = await self._astage("cost", desc, "A CostFindings JSON object.", CostFindings, fb)
        sp = A.spend_analysis(self.ctx.df, self.ctx.kpis)
        # Concentration facts are computed, not generated: exact and never hallucinated.
        out["concentration_risks"] = [
            {"vendor": c["top_vendor"], "headline": f"{c['top_share']:.0%} of {c['category']} spend with one vendor",
             "evidence": f"HHI {c['hhi']:,} across {c['vendors']} vendors; category spend {A.inr(c['spend'])}",
             "severity": "high" if c["top_share"] > 0.6 else "medium"}
            for c in sp["categories"] if c["concentration_flag"]]
        # One savings basis everywhere: the annualised renegotiation opportunity (also the brief's key number).
        # Full overpayment is a different, larger figure; adding it on top double-counted the same variance.
        cap = sp["savings_opportunity_total"]
        known_cats = {c["category"] for c in sp["categories"]}
        levers = []
        for lv in out.get("savings_levers", []):
            vs = [m for m in (self._match_vendor(v) for v in lv.get("vendors", [])) if m]
            bad = [v for v in lv.get("vendors", []) if not self._match_vendor(v)]
            text = lv.get("rationale", "")
            if bad:
                self.ctx.telemetry.emit("grounding", stage="cost", dropped=bad)
                self.state.grounding.extend(f"cost: removed unknown vendor '{b}' from a savings lever" for b in bad)
                text = "Consolidate volume toward the best-value existing vendors in this category."
            val = float(lv.get("estimated_annual_value_inr") or 0)
            room = max(0.0, cap - sum(x["estimated_annual_value_inr"] for x in levers))   # cap applies to the levers together
            if val > room:
                self.state.grounding.append(f"cost: capped '{lv.get('lever')}' from {A.inr(val)} to {A.inr(room)} "
                                            f"(levers together cannot exceed the data-supported {A.inr(cap)})")
                val = room
            if val < 1:   # rounding residue counts as nothing left
                self.state.grounding.append(f"cost: dropped '{lv.get('lever')}' (no data-supported savings left after earlier levers)")
                continue
            levers.append({**lv, "vendors": vs, "rationale": text, "estimated_annual_value_inr": round(val),
                           "categories": [c for c in lv.get("categories", []) if c in known_cats] or lv.get("categories", [])})
        out["savings_levers"] = levers
        self.state.cost = out
        return self.state.cost

    @listen(audit)
    async def market(self):
        s = self.ctx.in_scope
        cats = sorted(s.category.unique().tolist())
        top_regions = s.groupby("region").spend_total.sum().sort_values(ascending=False).head(2).index.tolist()
        desc = (f"Today is {time.strftime('%Y-%m-%d')}. Categories in scope: {cats}. Highest-spend regions: {top_regions}.\n"
                "Call market_intel_search ONCE with a list of queries: one per category (e.g. '<category> India supply prices "
                f"shortage') and one per listed region about logistics or disruption (at most {len(cats) + 2} queries). "
                "Then output one signal per category (region 'All India') plus one per region searched (category 'All'). "
                "Score risk 0-100 from the evidence (35 = normal baseline, >60 = material disruption). Quote the real headline, "
                "source, date and url from the results; ignore results unrelated to Indian supply chains. "
                "Never invent news; if nothing relevant was found, say so and score 35.")

        def fb():
            return MarketFindings(summary="Live market search unavailable; baseline risk assumed.",
                                  signals=[{"category": c, "region": "All India", "risk_score": 35, "headline": "No live signal",
                                            "implication": "Baseline risk"} for c in cats]).model_dump()
        tel = self.ctx.telemetry
        mark = len(tel.events)
        out = await self._astage("market", desc, "A MarketFindings JSON object.", MarketFindings, fb)
        searched = any(e["type"] == "tool_start" and e.get("tool") == "market_intel_search" for e in tel.events[mark:])
        if not searched:
            # The agent answered without searching: run the search ourselves, then let it reason over real results.
            self.state.grounding.append("market: agent skipped the news search; search was run by the guard and the agent re-ran on real results")
            tool = make_tools(self.ctx, ROSTER["market"].name, ["search"])[0]
            queries = [f"{c} India supply prices shortage" for c in cats] + [f"{r.split(',')[0]} logistics disruption India" for r in top_regions]
            results = await asyncio.to_thread(tool._run, queries=queries)
            out = await self._astage("market", desc + "\n\nSearch results already retrieved by market_intel_search (cite only these):\n"
                                     + results[:6000], "A MarketFindings JSON object.", MarketFindings, fb)
        # Citation check: a signal may only cite an article that was actually retrieved in this run.
        by_url = {s["url"]: s for s in self.ctx.sources if s.get("url")}
        for sig in out.get("signals", []):
            url = (sig.get("url") or "").strip()
            if url in by_url:
                src = by_url[url]
                sig["source"], sig["date"] = src.get("source", sig.get("source")), src.get("date", sig.get("date"))
                continue
            area = f"{sig.get('category')}/{sig.get('region')}"
            key = (sig.get("category") if sig.get("category") not in ("All", "") else sig.get("region", "")).split(",")[0].split("&")[0].strip().lower()
            match = next((s for s in self.ctx.sources if key and key in s.get("query", "").lower()
                          and (not sig.get("headline") or s["title"][:40].lower() in str(sig.get("headline")).lower()
                               or str(sig.get("headline"))[:40].lower() in s["title"].lower())), None)                 or next((s for s in self.ctx.sources if key and key in s.get("query", "").lower()), None)
            if match:
                if url or sig.get("source"):
                    self.state.grounding.append(f"market: re-linked the {area} citation to a retrieved article (agent's link did not match)")
                sig.update(headline=match["title"], source=match["source"], date=match["date"], url=match["url"])
            else:
                if url or sig.get("source"):
                    self.state.grounding.append(f"market: removed unverifiable citation '{sig.get('source') or url}' for {area}")
                    tel.emit("grounding", stage="market", dropped=[sig.get("source") or url])
                sig.update(source="", url="", date="", risk_score=min(float(sig.get("risk_score", 35)), 35.0),
                           headline="No verified news signal in the last 30 days", direction="stable")
        out["sources"] = list(self.ctx.sources)
        self.state.market = out
        return out

    # ---------------------------------------------------------- 4 forecast
    @listen(and_(performance, cost, market))
    def forecast(self):
        # Blend external risk into the scorecard now that market signals exist.
        mk: dict[str, float] = {}
        for sig in self.state.market.get("signals", []):
            cat, reg = sig.get("category", ""), sig.get("region", "")
            if cat and cat.lower() not in ("all", "all categories"):
                mk[cat] = float(sig["risk_score"])
            elif reg:
                mk[reg] = float(sig["risk_score"])
        for col in ("market_risk", "combined_risk"):
            if col in self.ctx.scored.columns:
                self.ctx.scored = self.ctx.scored.drop(columns=[col])
        scored = self.ctx.scored.copy()
        cat_map = {c: v for c, v in mk.items() if c in set(scored.category)}
        reg_map = {r: v for r, v in mk.items() if r in set(scored.region)}

        def ext(row):
            vals = [cat_map.get(row.category, 35.0)]
            if row.region in reg_map:
                vals.append(reg_map[row.region])
            return 0.7 * vals[0] + 0.3 * vals[-1] if len(vals) > 1 else vals[0]
        scored["market_risk"] = scored.apply(ext, axis=1)
        scored["combined_risk"] = 0.65 * scored.internal_risk + 0.35 * scored.market_risk
        self.ctx.scored = scored
        self.ctx.telemetry.emit("risk_blend", flagged=int((scored.combined_risk >= self.ctx.risk_threshold).sum()))

        desc = (f"{self._scope_note()}\nCall forecast_performance once (all vendors). Raise early warnings for vendors projected "
                "to breach the on-time floor or defect ceiling within 3 months, with a concrete monitoring trigger for each. "
                "Note the 80% band: low confidence when the band is wide. List improving vendors separately.")

        def fb():
            fc = A.forecast(self.ctx.df, otd_floor=self.ctx.otd_floor, defect_ceiling=self.ctx.defect_ceiling)
            ew = [{"vendor": f["vendor_name"], "metric": "both" if f["otd_breach"] and f["defect_breach"] else "on_time" if f["otd_breach"] else "defects",
                   "projection": f"OTD {f['otd']['forecast'][-1]}%, defects {f['defect']['forecast'][-1]}%", "confidence": "medium",
                   "recommended_trigger": "Weekly review"} for f in fc if f["otd_breach"] or f["defect_breach"]]
            return ForecastFindings(summary="Rule-based projection (agent unavailable).", early_warnings=ew,
                                    improving=[f["vendor_name"] for f in fc if f["outlook"] == "improving"]).model_dump()
        out = self._stage("forecast", desc, "A ForecastFindings JSON object.", ForecastFindings, fb)
        out["early_warnings"] = self._ground("forecast", out.get("early_warnings", []))
        # Every projected breach from the model is guaranteed to appear, whatever the agent wrote.
        have = {w["vendor"] for w in out["early_warnings"]}
        for f in A.forecast(self.ctx.df, otd_floor=self.ctx.otd_floor, defect_ceiling=self.ctx.defect_ceiling):
            if (f["otd_breach"] or f["defect_breach"]) and f["vendor_name"] not in have and f["vendor_name"] in set(self.ctx.in_scope.vendor_name):
                band = max(u - l for u, l in zip(f["otd"]["upper"], f["otd"]["lower"]))
                metric = "both" if f["otd_breach"] and f["defect_breach"] else "on_time" if f["otd_breach"] else "defects"
                proj = (f"OTD {f['otd']['history'][-1]:.1f}% -> {f['otd']['forecast'][-1]:.1f}% by {f['future_months'][-1]}" if f["otd_breach"]
                        else f"Defects {f['defect']['history'][-1]:.2f}% -> {f['defect']['forecast'][-1]:.2f}% by {f['future_months'][-1]}")
                out["early_warnings"].append({"vendor": f["vendor_name"], "metric": metric, "projection": proj,
                                              "confidence": "high" if band < 8 else "medium" if band < 16 else "low",
                                              "recommended_trigger": "Weekly performance review; escalate if the next month misses the threshold",
                                              "source": "model"})
        out["improving"] = [m for m in (self._match_vendor(v) for v in out.get("improving", [])) if m]
        self.state.forecast = out
        return self.state.forecast

    # ---------------------------------------------------------- 5 strategy
    def _strategy_table(self) -> str:
        fc = {f["vendor_name"]: f["outlook"] for f in A.forecast(self.ctx.df, otd_floor=self.ctx.otd_floor, defect_ceiling=self.ctx.defect_ceiling)}
        s = self.ctx.in_scope.copy()
        s["outlook"] = s.vendor_name.map(fc)
        return A.table_for_llm(s, ["vendor_name", "category", "composite", "tier", "internal_risk", "market_risk", "combined_risk",
                                   "on_time_pct", "defect_rate_pct", "price_variance_pct", "otd_trend", "compliance_incidents",
                                   "capacity_utilization_pct", "outlook"], 40)

    def _rule_decisions(self) -> list[dict]:
        th = self.ctx.risk_threshold
        out = []
        for r in self.ctx.in_scope.itertuples():
            cr = r.combined_risk
            if cr >= th + 12 or (r.tier == "D"):
                d, conf = "Replace", 0.7
            elif cr >= th:
                d, conf = "Watch", 0.65
            elif r.composite < 72 and r.otd_trend > 0.4:
                d, conf = "Develop", 0.6
            elif r.composite < 70:
                d, conf = "Watch", 0.55
            else:
                d, conf = "Keep", 0.75
            out.append({"vendor": r.vendor_name, "decision": d, "confidence": conf,
                        "rationale": f"Rule-based: composite {r.composite:.0f}, combined risk {cr:.0f} vs threshold {th:.0f}.",
                        "actions": [], "source": "rule"})
        return out

    @listen(or_(forecast, "revise"))
    def strategy(self):
        st = self.state
        revision = ""
        if st.revisions and st.qa:
            revision = ("\nREVISION REQUIRED by the QA Reviewer. Fix these issues:\n- " + "\n- ".join(st.qa.get("required_changes", []))
                        + f"\nYour previous decisions: {_j([{k: d[k] for k in ('vendor', 'decision')} for d in st.strategy.get('decisions', [])], 1800)}")
        desc = (f"User objective: {st.plan.get('objective')}\nPriority pillars: {st.plan.get('priority_pillars')}\n{self._scope_note()}\n"
                f"Data audit: {st.audit.get('verdict')} - {_j(st.audit.get('caveats', []), 400)}\n"
                f"Performance findings: {_j(st.performance, 1500)}\nCost findings: {_j(st.cost, 1300)}\n"
                f"Market signals: {_j([{k: s.get(k) for k in ('category', 'region', 'risk_score', 'direction', 'headline')} for s in st.market.get('signals', [])], 1300)}\n"
                f"Forecast warnings: {_j(st.forecast.get('early_warnings', []), 1100)}\n\n"
                f"Decision table (combined_risk = 65% internal + 35% market):\n{self._strategy_table()}\n{revision}\n\n"
                "Decide for EVERY vendor in the table with this rubric. Keep: tier A or B, combined_risk at least 8 below the "
                "threshold and no deteriorating outlook. Watch: combined_risk within 8 of the threshold, a deteriorating outlook, "
                "or rising market risk. Develop: tier C with an improving or stable outlook and meaningful spend (worth fixing). "
                "Replace: combined_risk above the threshold with a deteriorating or volatile record, or repeated compliance breaches. "
                "Vendors with combined_risk above the threshold must not be 'Keep' unless you justify it. "
                "Before finalising, call scenario_rescore once with a delivery-heavy and once with a cost-heavy weighting and "
                "report in robustness_note whether your Replace/Watch calls survive. Cite numbers in every rationale. "
                "Use the exact vendor names.")
        data = self._stage("strategy", desc, "A StrategyDecision JSON object covering every vendor.", StrategyDecision,
                           lambda: StrategyDecision(portfolio_summary="Rule-based decisions (agent unavailable).",
                                                    decisions=self._rule_decisions()).model_dump())
        # Reconcile names and fill any vendor the agent skipped with the rule engine.
        rules = {d["vendor"]: d for d in self._rule_decisions()}
        fixed, seen = [], set()
        for d in data.get("decisions", []):
            m = self._match_vendor(d.get("vendor", ""))
            if m and m not in seen and m in rules:
                fixed.append({**d, "vendor": m, "source": d.get("source", "agent")})
                seen.add(m)
        for v, d in rules.items():
            if v not in seen:
                fixed.append(d)
        data["decisions"] = fixed
        st.strategy = data
        return data

    @router(strategy)
    def risk_gate(self):
        th = self.ctx.risk_threshold
        decisions = self.state.strategy.get("decisions", [])
        replace = sorted(d["vendor"] for d in decisions if d["decision"] == "Replace")
        high = self.ctx.in_scope[self.ctx.in_scope.combined_risk >= th + 8].vendor_name.tolist()
        targets = sorted(set(replace) | set(high))
        prev = sorted(p.get("vendor") for p in self.state.mitigation.get("plans", []))
        route = "mitigate" if targets and targets != prev else "review"
        reason = "high_risk" if route == "mitigate" else ("already_planned" if targets else "no_high_risk")
        self.ctx.telemetry.emit("route", stage="risk_gate", route=route, targets=targets, reason=reason)
        self._mitigation_targets = targets
        return route

    # -------------------------------------------------------- 6 mitigation
    @listen("mitigate")
    def mitigation(self):
        targets = getattr(self, "_mitigation_targets", [])[:5]
        decisions = {d["vendor"]: d for d in self.state.strategy.get("decisions", [])}
        alt_facts = {t: [{k: a[k] for k in ("vendor_name", "composite", "headroom_pct", "on_time_pct", "defect_rate_pct")}
                         for a in A.alternatives(self.ctx.scored, t, 3)] for t in targets}
        desc = (f"{self._scope_note()}\nHigh-risk vendors needing a continuity plan: {targets}.\nStrategist view: "
                f"{_j([decisions.get(t, {}) for t in targets], 1500)}\n"
                f"Data-supported alternatives in the same category (ONLY these may receive volume): {_j(alt_facts, 1600)}\n"
                "For EACH vendor: call find_alternatives, choose 1-2 alternatives and the % of volume to shift (respect their "
                "headroom), and write concrete 30/60/90-day actions (e.g. qualification audit, trial PO, contract clauses, "
                "safety stock). State the expected impact qualitatively: the system computes the blended OTD/defect "
                "projection from the shares you choose, so do not invent percentage improvements.")

        def fb():
            plans = []
            for t in targets:
                alts = A.alternatives(self.ctx.scored, t, 2)
                plans.append({"vendor": t, "risk": "Combined risk above tolerance",
                              "alternatives": [{"to_vendor": a["vendor_name"], "share_of_volume_pct": min(40.0, a["headroom_pct"] * 2)} for a in alts],
                              "days_30": ["Qualification audit of alternatives", "Raise safety stock"], "days_60": ["Trial POs"],
                              "days_90": ["Shift agreed volume"], "expected_impact": "Reduced single-source exposure"})
            return MitigationFindings(summary="Rule-based plan (agent unavailable).", plans=plans).model_dump()
        out = self._stage("mitigation", desc, "A MitigationFindings JSON object.", MitigationFindings, fb)
        out["plans"] = self._ground("mitigation", out.get("plans", []))
        for plan in out["plans"]:
            allowed = {a["vendor_name"]: a for a in A.alternatives(self.ctx.scored, plan["vendor"], 4)}
            good, bad = [], []
            for a in plan.get("alternatives", []):
                m = self._match_vendor(str(a.get("to_vendor", "")))
                (good if m in allowed else bad).append({**a, "to_vendor": m or a.get("to_vendor")})
            if bad:
                names = [b["to_vendor"] for b in bad]
                self.ctx.telemetry.emit("grounding", stage="mitigation", dropped=names)
                self.state.grounding.extend(f"mitigation: replaced unsupported alternative '{n}' for {plan['vendor']}" for n in names)
                text = " ".join(sum([plan.get(k, []) for k in ("days_30", "days_60", "days_90")], []) + [plan.get("expected_impact", "")])
                if any(n and n in text for n in names):   # plan wording names an invented supplier: rewrite it
                    plan["days_30"] = ["Run qualification audits at the alternative vendors", "Raise safety stock for critical SKUs"]
                    plan["days_60"] = ["Issue trial purchase orders and track OTD/defects weekly"]
                    plan["days_90"] = ["Shift the agreed volume and amend contracts"]
            if not good:
                good = [{"to_vendor": a["vendor_name"], "share_of_volume_pct": round(min(40.0, a["headroom_pct"] * 2), 0)}
                        for a in list(allowed.values())[:2] if a["headroom_pct"] > 0]
            plan["alternatives"] = good
            plan["projection"] = self._project(plan)
        # Expected impact: the computed blend leads; agent prose keeps only percentages that trace to computed numbers.
        general, projected = self._pct_pools(out["plans"])
        for plan in out["plans"]:
            prose = self._ground_text("mitigation", plan.get("expected_impact", ""), general, projected, drop_sentences=bool(plan["projection"]))
            if plan["projection"]:
                plan["expected_impact"] = self._projection_text(plan["projection"]) +(f". {prose}" if prose else ".")
            else:
                plan["expected_impact"] = prose
        self.state.mitigation = out
        return self.state.mitigation

    # ----------------------------------------------------------------- 7 QA
    def _guardrails(self) -> list[str]:
        """Deterministic checks run before the QA agent (cheap, exact)."""
        th = self.ctx.risk_threshold
        s = self.ctx.scored.set_index("vendor_name")
        flags = []
        for d in self.state.strategy.get("decisions", []):
            if d["vendor"] not in s.index:
                continue
            r = s.loc[d["vendor"]]
            if d["decision"] == "Keep" and r.combined_risk >= th + 5:
                flags.append(f"{d['vendor']}: 'Keep' but combined risk {r.combined_risk:.0f} > threshold {th:.0f}")
            if d["decision"] == "Replace" and r.tier == "A":
                flags.append(f"{d['vendor']}: 'Replace' but tier A (score {r.composite:.0f})")
            for num in re.findall(r"(\d+\.\d+)\s*%", d.get("rationale", ""))[:2]:
                v = float(num)
                cands = [r.on_time_pct, r.defect_rate_pct, r.price_variance_pct, r.fill_rate_pct, r.capacity_utilization_pct]
                if all(abs(v - c) > 1.0 for c in cands) and v > 5:
                    flags.append(f"{d['vendor']}: rationale cites {v}% which matches no KPI for this vendor")
        return flags

    @router(or_("review", mitigation))
    def qa(self):
        st = self.state
        flags = self._guardrails()
        self.ctx.telemetry.emit("guardrails", stage="qa", flags=flags)
        decs = st.strategy.get("decisions", [])
        material = [d for d in decs if d["decision"] != "Keep" or any(d["vendor"] in f for f in flags)]
        keeps = [d["vendor"].replace(" Pvt Ltd", "") for d in decs if d not in material]
        lines = "\n".join(f"- {d['vendor']} | {d['decision']} | conf {d.get('confidence', 0):.2f} | {str(d.get('rationale', ''))[:150]}"
                          for d in material)
        desc = (f"Review the strategist's decisions as an independent auditor.\nNon-Keep decisions:\n{lines}\n"
                f"Keep decisions ({len(keeps)}): {', '.join(keeps)}\n"
                f"Mitigation summary: {_j(st.mitigation.get('summary', 'none'), 300)}\n"
                f"Automated guardrail flags: {flags or 'none'}\n{self._scope_note()}\n"
                "Use vendor_profile to verify the numbers behind at most 4 of the most consequential decisions (Replace calls "
                "first, then flagged vendors). Mark each check verified / corrected / unsupported. Choose 'revise' only for "
                "material errors (wrong numbers, decision contradicts data); otherwise 'approve'. List required changes if revising.")
        data = self._stage("qa", desc, "A QAReview JSON object.", QAReview,
                           lambda: QAReview(verdict="approve", confidence=0.6,
                                            checks=[{"claim": f, "status": "unsupported", "note": "Automated guardrail"} for f in flags],
                                            required_changes=[]).model_dump())
        data["guardrail_flags"] = flags
        data["round"] = st.revisions + 1
        st.qa = data
        if data.get("verdict") == "revise" and st.revisions < 1:
            st.revisions += 1
            self.ctx.telemetry.emit("route", stage="qa", route="revise")
            return "revise"
        self.ctx.telemetry.emit("route", stage="qa", route="publish")
        return "publish"

    # -------------------------------------------------------------- 8 brief
    @listen("publish")
    def brief(self):
        st, c = self.state, self.ctx
        sp = A.spend_analysis(c.df, c.kpis)
        dec = pd.Series([d["decision"] for d in st.strategy.get("decisions", [])]).value_counts().to_dict()
        at_risk = c.in_scope[c.in_scope.combined_risk >= c.risk_threshold]
        facts = {
            "total_spend": A.inr(sp["total_spend"]), "vendors": len(c.in_scope),
            "spend_at_risk": A.inr(at_risk.spend_total.sum()), "vendors_above_risk_threshold": len(at_risk),
            "annual_savings_opportunity": A.inr(sp["savings_opportunity_total"]),
            "concentrated_categories": [x["category"] for x in sp["categories"] if x["concentration_flag"]],
        }
        desc = (f"{self._scope_note()}\nUser question: \"{c.question or st.plan.get('objective')}\"\nKey facts (exact, use these): {_j(facts)}\n"
                f"Decision counts (shown separately in the UI; do not make it a key number): {dec}\n"
                f"FINAL decisions after QA (use exactly; do not call a Watch vendor 'Replace'): "
                + "; ".join(f"{k} = {', '.join(d['vendor'].replace(' Pvt Ltd', '') for d in st.strategy.get('decisions', []) if d['decision'] == k) or 'none'}"
                            for k in ("Replace", "Watch", "Develop")) + "\n"
                f"Strategy: {_j(st.strategy.get('portfolio_summary'), 600)} | Robustness: {_j(st.strategy.get('robustness_note'), 300)}\n"
                f"Replace/Watch: {_j([d for d in st.strategy.get('decisions', []) if d['decision'] in ('Replace', 'Watch')], 1800)}\n"
                f"Mitigation: {_j([{k: p.get(k) for k in ('vendor', 'alternatives', 'projection')} for p in st.mitigation.get('plans', [])], 1200)}\n"
                "Projected impact: cite ONLY the computed 'projection' numbers above (blended OTD/defects on the shifted vendor's "
                "volume) or forecast figures; never invent percentage improvements or risk reductions.\n"
                f"Savings levers: {_j(st.cost.get('savings_levers', []), 900)}\n"
                f"Savings: quote only {facts['annual_savings_opportunity']} (annual opportunity) or the lever values above, "
                "which are already capped to it; never another savings figure.\nMarket:{_j(st.market.get('summary'), 400)}\n"
                f"QA: {st.qa.get('verdict')} (confidence {st.qa.get('confidence')})\n"
                "Write the executive brief: answer the user's question directly first; 3 short paragraphs; 4-6 key numbers; "
                "3-5 actions with owner (role), timeline and quantified impact. Key number values must be short (max 14 characters, "
                "e.g. '₹5.63 Cr' or '2 of 20'). Timelines use real dates from today onward. No numbers or vendors that are not "
                "in the inputs.")

        def fb():
            return ExecutiveBrief(headline=f"{len(at_risk)} vendors above risk tolerance; {facts['spend_at_risk']} spend exposed",
                                  answer="See decisions and mitigation plans.", summary=[st.strategy.get("portfolio_summary", "")],
                                  key_numbers=[{"label": k.replace('_', ' '), "value": str(v)} for k, v in facts.items() if k != "decisions"]).model_dump()
        st.brief = self._stage("brief", desc, "An ExecutiveBrief JSON object.", ExecutiveBrief, fb)
        # Final scrub: any name the grounding guard removed upstream must not resurface in the brief.
        removed = {n for e in self.ctx.telemetry.events if e["type"] == "grounding" and e.get("stage") != "market"
                   for n in e.get("dropped", []) if len(n) > 3}
        if removed:
            def scrub(x):
                if isinstance(x, str):
                    for n in removed:
                        x = x.replace(n, "an alternative qualified vendor")
                    return x
                if isinstance(x, list):
                    return [scrub(i) for i in x]
                if isinstance(x, dict):
                    return {k: scrub(v) for k, v in x.items()}
                return x
            st.brief = scrub(st.brief)
        # Projected percentages must trace to computed numbers; a stripped action impact gets the computed blend instead.
        plans = st.mitigation.get("plans", [])
        general, projected = self._pct_pools(plans)
        for k in ("headline", "answer", "summary"):
            st.brief[k] = self._ground_text("brief", st.brief.get(k), general, projected)
        for a in st.brief.get("top_actions", []):
            before = a.get("impact", "")
            a["action"] = self._ground_text("brief", a.get("action", ""), general, projected)
            a["impact"] = self._ground_text("brief", before, general, projected)
            if a["impact"] != before or not a["impact"]:
                text = f"{a['action']} {before}".lower()
                hit = next((p for p in plans if p.get("projection") and any(
                    n.lower().replace(" pvt ltd", "") in text for n in [p["vendor"]] + [x["to_vendor"] for x in p.get("alternatives", [])])), None)
                if hit:   # the computed blend replaces the stripped claim outright
                    a["impact"] = self._projection_text(hit["projection"]).replace(" (computed from current KPIs)", "")
        # Rupee savings must be the computed opportunity or a (capped) lever: one savings story across key numbers and prose.
        allowed = savings_figures(st.cost, sp["savings_opportunity_total"])
        levers_total = allowed[-1] if len(allowed) > 2 else sp["savings_opportunity_total"]
        def fix_savings(x):
            if isinstance(x, list):
                return [fix_savings(i) for i in x]
            out, bad = ground_savings(x, allowed, levers_total)
            st.grounding.extend(f"brief: corrected savings figure '{b}' to {A.inr(levers_total)} (computed)" for b in bad)
            return out
        for k in ("headline", "answer", "summary"):
            st.brief[k] = fix_savings(st.brief.get(k))
        for a in st.brief.get("top_actions", []):
            a["impact"] = fix_savings(a.get("impact", ""))
        st.brief["facts"] = {**facts, "decisions": dec}
        # Key numbers must be traceable to computed facts (or the user's own thresholds); anything else is dropped.
        def norm(v):
            return re.sub(r"[\s₹,]", "", str(v)).lower()
        allowed = {norm(v) for v in facts.values()} | {norm(len(v)) for v in facts.values() if isinstance(v, list)} | {norm(f"{len(at_risk)} of {len(c.in_scope)}"), norm(f"{c.otd_floor:g}%"),
                                                        norm(f"{c.defect_ceiling:g}%"), norm(f"{c.risk_threshold:g}")}
        kept = [k for k in st.brief.get("key_numbers", []) if norm(k.get("value", "")) in allowed]
        dropped_kn = [k for k in st.brief.get("key_numbers", []) if k not in kept]
        if dropped_kn:
            st.grounding.extend(f"brief: removed key number '{k.get('label')}: {k.get('value')}' (not in computed facts)" for k in dropped_kn)
        if len(kept) < 4:
            labels = {"total_spend": "Total spend", "spend_at_risk": "Spend at risk", "vendors_above_risk_threshold": "Vendors above tolerance",
                      "annual_savings_opportunity": "Annual savings opportunity", "vendors": "Vendors analysed"}
            have = {norm(k["value"]) for k in kept}
            for key, label in labels.items():
                v = facts[key] if key != "vendors_above_risk_threshold" else f"{len(at_risk)} of {len(c.in_scope)}"
                if norm(v) not in have and len(kept) < 5:
                    kept.append({"label": label, "value": str(v)})
        st.brief["key_numbers"] = kept[:6]
        return st.brief


# =================================================================== runner
def prepare_context(df: pd.DataFrame, *, weights: dict | None = None, risk_threshold: float = 55, question: str = "",
                    focus_categories: list[str] | None = None, otd_floor: float = 85, defect_ceiling: float = 3.0,
                    telemetry: RunTelemetry | None = None) -> RunContext:
    d, q = A.clean(df)
    k = A.vendor_kpis(d)
    s = A.score(k, weights)
    s["market_risk"] = 35.0
    s["combined_risk"] = 0.65 * s.internal_risk + 0.35 * s.market_risk
    from .config import DEFAULT_WEIGHTS
    return RunContext(df=d, quality=q, kpis=k, scored=s, weights={**DEFAULT_WEIGHTS, **(weights or {})},
                      risk_threshold=risk_threshold, otd_floor=otd_floor, defect_ceiling=defect_ceiling, question=question,
                      focus_categories=[c for c in (focus_categories or []) if c in set(s.category)],
                      telemetry=telemetry or RunTelemetry(uuid.uuid4().hex[:8]))


def run_analysis(ctx: RunContext) -> dict[str, Any]:
    tel = ctx.telemetry
    tel.emit("run_start", question=ctx.question, vendors=len(ctx.in_scope), weights=ctx.weights)
    flow = SafeProcureFlow(ctx)
    try:
        flow.kickoff()
    except Exception as err:  # noqa: BLE001
        tel.emit("run_error", message=str(err)[:400])
    result = build_result(ctx, flow.state)
    tel.emit("run_end", usage=result["usage"], degraded=flow.state.degraded)
    tel.close()
    return result


def build_result(ctx: RunContext, st: SPState) -> dict[str, Any]:
    s = ctx.scored
    dec = {d["vendor"]: d for d in st.strategy.get("decisions", [])}
    fc = {f["vendor_name"]: f for f in A.forecast(ctx.df, otd_floor=ctx.otd_floor, defect_ceiling=ctx.defect_ceiling)}
    vendors = []
    for r in s.itertuples():
        vendors.append({
            "vendor": r.vendor_name, "id": r.vendor_id, "category": r.category, "region": r.region,
            "in_scope": bool(r.vendor_name in set(ctx.in_scope.vendor_name)),
            "composite": round(r.composite, 1), "tier": r.tier, "rank": int(r.rank),
            "pillars": {p: round(getattr(r, p), 1) for p in A.PILLARS},
            "internal_risk": round(r.internal_risk, 1), "market_risk": round(r.market_risk, 1), "combined_risk": round(r.combined_risk, 1),
            "kpis": {k: round(float(getattr(r, k)), 2) for k in ("on_time_pct", "fill_rate_pct", "defect_rate_pct", "lead_time_days",
                                                                   "price_variance_pct", "response_time_hrs", "capacity_utilization_pct",
                                                                   "otd_trend", "defect_trend")},
            "compliance_incidents": int(r.compliance_incidents), "spend_total": float(r.spend_total), "spend_recent": float(r.spend_recent),
            "otd_series": r.otd_series, "defect_series": r.defect_series,
            "forecast": fc.get(r.vendor_name), "decision": dec.get(r.vendor_name),
        })
    return {
        "run_id": ctx.telemetry.run_id, "created": time.strftime("%Y-%m-%d %H:%M"), "question": ctx.question,
        "config": {"weights": ctx.weights, "risk_threshold": ctx.risk_threshold, "otd_floor": ctx.otd_floor,
                   "defect_ceiling": ctx.defect_ceiling, "focus_categories": ctx.focus_categories},
        "quality": ctx.quality, "vendors": vendors, "spend": A.spend_analysis(ctx.df, ctx.kpis),
        "plan": st.plan, "audit": st.audit, "performance": st.performance, "cost": st.cost, "market": st.market,
        "forecast": st.forecast, "strategy": st.strategy, "mitigation": st.mitigation, "qa": st.qa, "brief": st.brief,
        "path": st.path, "grounding": st.grounding, "revisions": st.revisions, "degraded": st.degraded, "stage_seconds": st.stage_seconds,
        "usage": ctx.telemetry.usage_summary(), "events": ctx.telemetry.events,
    }

Writing safeprocure/flow.py


**`safeprocure/copilot.py`** · Procurement Copilot: a tool-using agent for follow-up questions on a finished run.

In [13]:
%%writefile safeprocure/copilot.py
"""Procurement Copilot: conversational follow-up on a finished run.

With a live run context, the copilot is a tool-using CrewAI agent (it can re-check any vendor,
re-score scenarios, find alternatives). For a saved run it answers from the result snapshot.
"""
from __future__ import annotations

import asyncio
import contextlib
import inspect
import io
import json
import time
from concurrent.futures import ThreadPoolExecutor
from typing import Any

from .agents import build_agent
from .grounding import _INR, check_vendor_claims, ground_savings, parse_inr, savings_figures
from .llm import ResilientLLM
from .telemetry import RunTelemetry


def _snapshot(result: dict[str, Any]) -> str:
    vendors = [{"vendor": v["vendor"], "category": v["category"], "tier": v["tier"], "score": v["composite"],
                "risk": v["combined_risk"], "otd": v["kpis"]["on_time_pct"], "defect": v["kpis"]["defect_rate_pct"],
                "price_var": v["kpis"]["price_variance_pct"], "decision": (v.get("decision") or {}).get("decision")}
               for v in result["vendors"]]
    brief = result.get("brief", {})
    return json.dumps({"question": result.get("question"), "headline": brief.get("headline"), "answer": brief.get("answer"),
                       "facts": brief.get("facts"), "vendors": vendors,
                       "mitigation": [{"vendor": p.get("vendor"), "alternatives": p.get("alternatives")} for p in result.get("mitigation", {}).get("plans", [])],
                       "market": [{k: s.get(k) for k in ("category", "region", "risk_score", "headline")} for s in result.get("market", {}).get("signals", [])]},
                      default=str)[:9000]


def answer(result: dict[str, Any], message: str, history: list[dict[str, str]], ctx: Any = None) -> dict[str, Any]:
    t = time.time()
    convo = "\n".join(f"{m.get('role', 'user')}: {m.get('content', '')[:500]}" for m in history)
    prompt = (f"You are the Safe Procure copilot. Analysis snapshot (authoritative): {_snapshot(result)}\n"
              f"Conversation so far:\n{convo}\nUser: {message}\n"
              "Answer in <= 120 words of plain prose or short bullet points (no tables, no headings), cite exact numbers, use the vendor names as written. If the user asks a what-if about "
              "weights, use the scenario tool when available. Never invent data.")
    tools_used: list[str] = []
    if ctx is not None:
        tel = RunTelemetry("chat")
        old = ctx.telemetry
        ctx.telemetry = tel
        try:
            agent = build_agent("copilot", ctx)
            with contextlib.redirect_stdout(io.StringIO()):   # hide CrewAI's console panels
                out = agent.kickoff(prompt)
                if inspect.iscoroutine(out):   # inside Jupyter/Flows CrewAI returns a coroutine: run it on a fresh loop
                    with ThreadPoolExecutor(1) as ex:
                        out = ex.submit(asyncio.run, out).result()
            text = getattr(out, "raw", None) or str(out)
        finally:
            ctx.telemetry = old
        tools_used = [e["tool"] for e in tel.events if e["type"] == "tool_start"]
        models = sorted({e["model"] for e in tel.events if e["type"] == "llm_end"})
    else:
        llm = ResilientLLM("analysis", "Procurement Copilot")
        text = llm.call(prompt)
        models = [getattr(llm, "label", llm.model)]
    text, corrections = _verified(str(text).strip(), prompt, result, models)
    return {"answer": text, "tools": tools_used, "models": models, "seconds": round(time.time() - t, 1),
            "corrections": corrections}


def _fact_check(text: str, result: dict[str, Any]) -> list[str]:
    return check_vendor_claims(text, result.get("vendors", []))


def _verified(text: str, prompt: str, result: dict[str, Any], models: list[str]) -> tuple[str, list[str]]:
    """Check vendor tier/KPI claims against the run data; on errors ask for one corrected rewrite, and if errors remain,
    state the correct figures under the answer. Savings amounts are pinned to the computed figures."""
    issues = _fact_check(text, result)
    if issues:
        try:
            llm = ResilientLLM("analysis", "Procurement Copilot")
            draft = str(llm.call(f"{prompt}\n\nYour draft answer:\n{text}\n\nA check against the data found errors: "
                                 f"{'; '.join(issues)}. Rewrite the answer with these facts corrected (same length, plain prose, "
                                 "do not mention the check).")).strip()
            label = getattr(llm, "label", llm.model)
            if label not in models:
                models.append(label)
            if draft and len(_fact_check(draft, result)) < len(issues):
                text = draft
        except Exception:  # noqa: BLE001 - the deterministic note below still corrects the answer
            pass
        left = _fact_check(text, result)
        if left:
            text += "\n\nData check: " + "; ".join(left) + "."
    m = _INR.search(str((result.get("brief", {}).get("facts") or {}).get("annual_savings_opportunity", "")))
    if m and m.group("v"):
        allowed = savings_figures(result.get("cost", {}), parse_inr(m.group("v"), m.group("u")))
        text, fixed = ground_savings(text, allowed, allowed[-1] if len(allowed) > 2 else allowed[0])
        issues += [f"savings figure {f} replaced with the computed value" for f in fixed]
    return text, issues

Writing safeprocure/copilot.py


**`safeprocure/report.py`** · Self-contained HTML executive report.

In [14]:
%%writefile safeprocure/report.py
"""Self-contained executive report (single HTML file, print-ready, no external assets)."""
from __future__ import annotations

import html
from collections import Counter
from typing import Any

from .analytics import inr

DEC_COLOR = {"Keep": "#0B7A4E", "Watch": "#A85A06", "Develop": "#0A6B9E", "Replace": "#BE2A21"}


def _e(x: Any) -> str:
    return html.escape(str(x if x is not None else ""))


def _short(n: str) -> str:
    return str(n).replace(" Pvt Ltd", "")


def _risk_color(r: float) -> str:
    return "#0B7A4E" if r < 25 else "#6E8F1A" if r < 40 else "#C08A05" if r < 55 else "#D7600F" if r < 70 else "#BE2A21"


def render_report(r: dict[str, Any]) -> str:
    b, qa, u = r.get("brief", {}), r.get("qa", {}), r.get("usage", {})
    vendors = [v for v in r["vendors"] if v.get("in_scope", True)]
    dec = Counter((v.get("decision") or {}).get("decision", "—") for v in vendors)
    total = sum(dec[k] for k in DEC_COLOR) or 1
    bar = "".join(f'<span style="flex:{dec[k]};background:{c}">{dec[k]}</span>' for k, c in DEC_COLOR.items() if dec[k])
    legend = " ".join(f'<i style="background:{c}"></i>{k} {dec[k]}' for k, c in DEC_COLOR.items())
    facts = "".join(f"<div><dt>{_e(k['label'])}</dt><dd>{_e(k['value'])}</dd></div>" for k in b.get("key_numbers", []))
    paras = "".join(f"<p>{_e(p)}</p>" for p in b.get("summary", []))
    actions = "".join(
        f"<tr><td class=n>{i + 1:02d}</td><td><b>{_e(a['action'])}</b></td><td>{_e(a['owner'])}</td><td>{_e(a['timeline'])}</td><td>{_e(a['impact'])}</td></tr>"
        for i, a in enumerate(b.get("top_actions", [])))
    rows = "".join(
        f"<tr><td class=n>{v['rank']}</td><td><b>{_e(_short(v['vendor']))}</b><br><small>{_e(v['category'])} · {_e(v['region'])}</small></td>"
        f"<td class=n>{v['composite']:.1f} ({v['tier']})</td>"
        f"<td class=n><span class=dot style='background:{_risk_color(v['combined_risk'])}'></span>{v['combined_risk']:.0f}</td>"
        f"<td class=n>{v['kpis']['on_time_pct']:.1f}%</td><td class=n>{v['kpis']['defect_rate_pct']:.2f}%</td>"
        f"<td class=n>{v['kpis']['price_variance_pct']:+.1f}%</td><td class=n>{_e(inr(v['spend_total']))}</td>"
        f"<td><span class=chip style='color:{DEC_COLOR.get((v.get('decision') or {}).get('decision'), '#555')}'>{_e((v.get('decision') or {}).get('decision', '—'))}</span>"
        f"<br><small>{_e((v.get('decision') or {}).get('rationale', ''))}</small></td></tr>"
        for v in sorted(vendors, key=lambda x: -x["combined_risk"]))
    plans = "".join(
        f"<div class=plan><h3>{_e(_short(p['vendor']))}</h3><p class=muted>{_e(p.get('risk'))}</p>"
        f"<p><b>Shift volume to:</b> {', '.join(f'{_e(_short(a['to_vendor']))} ({a['share_of_volume_pct']:.0f}%)' for a in p.get('alternatives', [])) or 'no qualified alternative'}</p>"
        f"<div class=phases>{''.join(f'<div><h4>{lbl}</h4><ul>' + ''.join(f'<li>{_e(x)}</li>' for x in p.get(key, [])) + '</ul></div>' for lbl, key in (('30 days', 'days_30'), ('60 days', 'days_60'), ('90 days', 'days_90')))}</div>"
        f"<p><b>Expected impact:</b> {_e(p.get('expected_impact'))}</p></div>"
        for p in r.get("mitigation", {}).get("plans", []))
    signals = "".join(
        f"<tr><td><b>{_e(s['category'] if s['category'] != 'All' else s['region'])}</b></td><td class=n>{s['risk_score']:.0f}</td><td>{_e(s.get('direction'))}</td>"
        f"<td>{_e(s.get('headline'))}<br><small>{_e(s.get('source'))} {_e(s.get('date'))}"
        + (f" · <a href='{_e(s['url'])}'>source</a>" if s.get("url") else "") + "</small></td></tr>"
        for s in sorted(r.get("market", {}).get("signals", []), key=lambda s: -s["risk_score"]))
    checks = "".join(f"<li><b>{_e(c['status'])}</b>: {_e(c['claim'])} <small>{_e(c['note'])}</small></li>" for c in qa.get("checks", []))
    grounding = "".join(f"<li>{_e(g)}</li>" for g in r.get("grounding", [])) or "<li>No ungrounded claims were found.</li>"
    models = ", ".join(f"{k.split(':', 1)[1]} ({v['calls']} calls)" for k, v in u.get("per_model", {}).items())

    return f"""<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Safe Procure report {_e(r['run_id'])}</title>
<style>
:root{{--ink:#15171C;--ink2:#474C58;--ink3:#676D7B;--line:#E4E1D9;--bg:#F4F3EF;--brand:#3E3AE0}}
*{{box-sizing:border-box}} body{{margin:0;background:var(--bg);color:var(--ink);font:14px/1.55 system-ui,-apple-system,"Segoe UI",sans-serif}}
main{{max-width:1040px;margin:0 auto;padding:40px 28px 60px}} .sheet{{background:#fff;border:1px solid var(--line);border-radius:16px;padding:40px 44px;margin-bottom:18px}}
header.top{{display:flex;justify-content:space-between;align-items:center;margin-bottom:22px;color:var(--ink3);font-size:13px}}
header.top b{{color:var(--ink);font-size:15px}} h1{{font:400 40px/1.05 Georgia,"Times New Roman",serif;letter-spacing:-.01em;margin:6px 0 16px;max-width:26ch}}
h2{{font-size:17px;margin:0 0 14px}} h3{{font-size:15px;margin:0}} h4{{font-size:12px;color:var(--ink3);margin:0 0 6px}}
.q{{color:var(--ink3);font-style:italic}} .answer{{font-size:16.5px;max-width:68ch}} p{{max-width:72ch;color:var(--ink2)}}
dl.facts{{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));border-top:1px solid var(--line);margin:22px 0 0}}
dl.facts div{{padding:12px 12px 0 0}} dt{{font-size:12px;color:var(--ink3)}} dd{{margin:2px 0 0;font:500 21px ui-monospace,Consolas,monospace}}
.dist{{display:flex;height:30px;border-radius:8px;overflow:hidden;gap:3px;margin:8px 0}} .dist span{{display:grid;place-items:center;color:#fff;font:600 12px ui-monospace,monospace}}
.legend i{{display:inline-block;width:9px;height:9px;border-radius:3px;margin:0 5px 0 12px}} table{{width:100%;border-collapse:collapse;font-size:13px}}
th{{text-align:left;color:var(--ink3);font-weight:600;font-size:12px;border-bottom:1px solid var(--line);padding:7px 8px}} td{{border-bottom:1px solid var(--line);padding:9px 8px;vertical-align:top}}
td.n{{font-family:ui-monospace,Consolas,monospace;white-space:nowrap}} small{{color:var(--ink3)}} .dot{{display:inline-block;width:9px;height:9px;border-radius:50%;margin-right:6px}}
.chip{{font-weight:700}} .plan{{border:1px solid var(--line);border-radius:12px;padding:18px;margin-top:12px}} .muted{{color:var(--ink3)}}
.phases{{display:grid;grid-template-columns:repeat(3,1fr);gap:12px;margin:10px 0}} .phases ul{{margin:0;padding-left:18px;color:var(--ink2)}}
ul{{color:var(--ink2)}} footer{{color:var(--ink3);font-size:12px;text-align:center}}
@media print{{body{{background:#fff}} .sheet{{border:0;padding:0;break-inside:avoid-page}} main{{padding:0}}}}
@media (max-width:700px){{.sheet{{padding:24px 20px}} .phases{{grid-template-columns:1fr}} h1{{font-size:30px}}}}
</style></head><body><main>
<section class="sheet">
<header class="top"><span><b>Safe Procure</b> · Executive vendor risk brief</span><span>Run {_e(r['run_id'])} · {_e(r.get('created'))}</span></header>
<div class="q">“{_e(r.get('question') or r.get('plan', {}).get('objective'))}”</div>
<h1>{_e(b.get('headline'))}</h1>
<p class="answer">{_e(b.get('answer'))}</p>{paras}
<dl class="facts">{facts}</dl>
</section>
<section class="sheet"><h2>Decisions</h2><div class="dist">{bar}</div><div class="legend">{legend}</div>
<h2 style="margin-top:26px">Priority actions</h2><table><thead><tr><th>#</th><th>Action</th><th>Owner</th><th>Timeline</th><th>Impact</th></tr></thead><tbody>{actions}</tbody></table></section>
<section class="sheet"><h2>Vendor scorecard (sorted by combined risk)</h2><table><thead><tr><th>Rank</th><th>Vendor</th><th>Score</th><th>Risk</th><th>On-time</th><th>Defects</th><th>Price var.</th><th>Spend</th><th>Decision</th></tr></thead><tbody>{rows}</tbody></table></section>
{'<section class="sheet"><h2>Mitigation plans</h2>' + plans + '</section>' if plans else ''}
<section class="sheet"><h2>Market intelligence (live news)</h2><table><thead><tr><th>Area</th><th>Risk</th><th>Trend</th><th>Signal</th></tr></thead><tbody>{signals}</tbody></table></section>
<section class="sheet"><h2>Assurance</h2><p>Independent QA verdict: <b>{_e(qa.get('verdict'))}</b> · {r.get('revisions', 0)} revision round(s).</p><ul>{checks}</ul>
<h4 style="margin-top:16px">Grounding guard</h4><ul>{grounding}</ul>
<p class="muted">Data quality score {r['quality']['quality_score']}/100 · {u.get('llm_calls')} model calls · {u.get('total_tokens')} tokens · {u.get('failovers')} automatic failovers · {u.get('elapsed')} s. Models: {_e(models)}.</p></section>
<footer>Generated by Safe Procure, a 10-agent CrewAI workflow. Vendor names in the sample dataset are fictional.</footer>
</main></body></html>"""

Writing safeprocure/report.py


**`safeprocure/server.py`** · FastAPI backend for the web app: datasets, what-if re-scoring, background runs, Server-Sent Events, copilot, downloads.

In [15]:
%%writefile safeprocure/server.py
"""FastAPI backend for the Safe Procure web app.

Endpoints
  GET  /                          the single-page app
  GET  /api/health                key + model status
  POST /api/dataset/sample        generate a synthetic dataset (user-controlled size/seed/categories)
  POST /api/dataset/upload        upload CSV/XLSX; columns auto-mapped
  POST /api/dataset/{id}/score    instant deterministic scorecard for any weights (what-if, no LLM)
  POST /api/runs                  launch the multi-agent flow in the background
  GET  /api/runs/{id}/events      Server-Sent Events stream of the live agent trace
  GET  /api/runs/{id}             final result JSON
  POST /api/runs/{id}/chat        ask the Procurement Copilot about this run
  GET  /api/runs/{id}/report      downloadable executive report (HTML)
  GET  /api/runs/{id}/csv         downloadable scorecard (CSV)
  GET  /api/runs/saved/latest     last completed run (offline backup)
"""
from __future__ import annotations

import asyncio
import io
import json
import threading
import uuid
from pathlib import Path
from typing import Any

import pandas as pd
from fastapi import FastAPI, File, HTTPException, UploadFile
from fastapi.responses import FileResponse, HTMLResponse, JSONResponse, Response, StreamingResponse
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel

from . import analytics as A
from . import data as D
from .config import DEFAULT_WEIGHTS, ENDPOINTS, TIERS, keys_status
from .flow import prepare_context, run_analysis
from .llm import PACER
from .telemetry import RunTelemetry

ROOT = Path(__file__).parent
WEB = ROOT / "web"
RUNS_DIR = Path.cwd() / "runs"
RUNS_DIR.mkdir(exist_ok=True)

app = FastAPI(title="Safe Procure", docs_url="/api/docs")
DATASETS: dict[str, dict[str, Any]] = {}
RUNS: dict[str, dict[str, Any]] = {}


def _clean_json(o: Any) -> Any:
    return json.loads(json.dumps(o, default=str))


def _dataset_payload(ds_id: str, df: pd.DataFrame, notes: list[str]) -> dict[str, Any]:
    d, q = A.clean(df)
    k = A.vendor_kpis(d)
    s = A.score(k)
    DATASETS[ds_id] = {"df": df, "notes": notes}
    preview = df.head(40).fillna("").to_dict("records")
    return _clean_json({
        "dataset_id": ds_id, "notes": notes, "quality": q, "columns": list(df.columns), "preview": preview,
        "row_count": len(df), "scorecard": _score_rows(s),
        "monthly_spend": A.spend_analysis(d, k)["monthly"],
    })


def _score_rows(s: pd.DataFrame) -> list[dict[str, Any]]:
    return [{"vendor": r.vendor_name, "category": r.category, "region": r.region, "composite": round(r.composite, 1),
             "tier": r.tier, "rank": int(r.rank), "internal_risk": round(r.internal_risk, 1), "spend": float(r.spend_total),
             "pillars": {p: round(getattr(r, p), 1) for p in A.PILLARS},
             "on_time_pct": round(r.on_time_pct, 1), "defect_rate_pct": round(r.defect_rate_pct, 2),
             "price_variance_pct": round(r.price_variance_pct, 2), "otd_trend": round(r.otd_trend, 2),
             "otd_series": r.otd_series} for r in s.itertuples()]


# ------------------------------------------------------------------ health
@app.get("/api/health")
def health():
    ks = keys_status()
    return {"keys": ks, "tiers": {k: [e.label for e in t.chain] for k, t in TIERS.items()},
            "endpoints": [{"label": e.label, "provider": e.provider, "model": e.model, "tpm": e.tpm, "rpm": e.rpm,
                           "available": ks.get(e.provider, False)} for e in ENDPOINTS],
            "pacer": PACER.snapshot(), "default_weights": DEFAULT_WEIGHTS, "categories": list(D.CATEGORIES),
            "saved_run": (RUNS_DIR / "latest.json").exists()}


# ---------------------------------------------------------------- datasets
class SampleReq(BaseModel):
    n_vendors: int = 20
    months: int = 18
    seed: int = 42
    categories: list[str] | None = None
    inject_quality_issues: bool = True


@app.post("/api/dataset/sample")
def sample(req: SampleReq):
    spec = D.GenSpec(n_vendors=max(5, min(25, req.n_vendors)), months=max(6, min(36, req.months)), seed=req.seed,
                     categories=tuple(req.categories or D.CATEGORIES), inject_quality_issues=req.inject_quality_issues)
    df = D.generate(spec)
    return _dataset_payload(uuid.uuid4().hex[:10], df, ["Synthetic dataset: all vendor names are fictional."])


@app.post("/api/dataset/upload")
async def upload(file: UploadFile = File(...)):
    raw = await file.read()
    if len(raw) > 15_000_000:
        raise HTTPException(413, "File too large (max 15 MB).")
    try:
        df, notes = D.normalise_upload(raw, file.filename or "upload.csv")
    except Exception as err:  # noqa: BLE001
        raise HTTPException(422, f"Could not read this file: {err}") from err
    return _dataset_payload(uuid.uuid4().hex[:10], df, notes)


@app.get("/api/dataset/{ds_id}/csv")
def dataset_csv(ds_id: str):
    ds = DATASETS.get(ds_id) or HTTPException(404)
    if isinstance(ds, HTTPException):
        raise ds
    buf = io.StringIO()
    ds["df"].to_csv(buf, index=False)
    return Response(buf.getvalue(), media_type="text/csv",
                    headers={"Content-Disposition": 'attachment; filename="safe_procure_vendors.csv"'})


class ScoreReq(BaseModel):
    weights: dict[str, float] = DEFAULT_WEIGHTS


@app.post("/api/dataset/{ds_id}/score")
def rescore(ds_id: str, req: ScoreReq):
    ds = DATASETS.get(ds_id)
    if not ds:
        raise HTTPException(404, "Dataset expired; reload sample or upload again.")
    d, _ = A.clean(ds["df"])
    return _clean_json({"scorecard": _score_rows(A.score(A.vendor_kpis(d), req.weights))})


# -------------------------------------------------------------------- runs
class RunReq(BaseModel):
    dataset_id: str
    question: str = ""
    weights: dict[str, float] = DEFAULT_WEIGHTS
    risk_threshold: float = 55
    otd_floor: float = 85
    defect_ceiling: float = 3.0
    focus_categories: list[str] = []


def _worker(run_id: str, ctx) -> None:
    try:
        result = run_analysis(ctx)
        result = _clean_json(result)
        RUNS[run_id]["result"] = result
        (RUNS_DIR / f"{run_id}.json").write_text(json.dumps(result), encoding="utf-8")
        if not result.get("degraded") or len(result["degraded"]) < 4:
            (RUNS_DIR / "latest.json").write_text(json.dumps(result), encoding="utf-8")
    except Exception as err:  # noqa: BLE001
        ctx.telemetry.emit("run_error", message=str(err)[:400])
        ctx.telemetry.close()
    finally:
        RUNS[run_id]["done"] = True


@app.post("/api/runs")
def start_run(req: RunReq):
    ds = DATASETS.get(req.dataset_id)
    if not ds:
        raise HTTPException(404, "Dataset expired; reload sample or upload again.")
    if not any(keys_status().values()):
        raise HTTPException(400, "No API keys configured on the server.")
    run_id = uuid.uuid4().hex[:8]
    tel = RunTelemetry(run_id)
    ctx = prepare_context(ds["df"], weights=req.weights, risk_threshold=req.risk_threshold, question=req.question.strip()[:600],
                          focus_categories=req.focus_categories, otd_floor=req.otd_floor, defect_ceiling=req.defect_ceiling,
                          telemetry=tel)
    RUNS[run_id] = {"telemetry": tel, "ctx": ctx, "result": None, "done": False}
    threading.Thread(target=_worker, args=(run_id, ctx), daemon=True).start()
    return {"run_id": run_id, "vendors_in_scope": len(ctx.in_scope)}


@app.get("/api/runs/{run_id}/events")
async def events(run_id: str):
    run = RUNS.get(run_id)
    if not run:
        raise HTTPException(404, "Unknown run")
    tel: RunTelemetry = run["telemetry"]

    async def gen():
        seq = 0
        while True:
            batch = await asyncio.to_thread(tel.since, seq, 10.0)
            for ev in batch:
                yield f"data: {json.dumps(ev, default=str)}\n\n"
            seq += len(batch)
            if tel.closed and seq >= len(tel.events):
                yield "event: done\ndata: {}\n\n"
                return
            if not batch:
                yield ": keep-alive\n\n"
    return StreamingResponse(gen(), media_type="text/event-stream",
                             headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})


@app.get("/api/runs/{run_id}/poll")
def poll(run_id: str, since: int = 0):
    """Polling fallback for proxies that buffer Server-Sent Events."""
    run = RUNS.get(run_id)
    if not run:
        raise HTTPException(404, "Unknown run")
    tel: RunTelemetry = run["telemetry"]
    evs = tel.events[since:]
    return {"events": _clean_json(evs), "done": bool(tel.closed)}


def _load_result(run_id: str) -> dict[str, Any]:
    if run_id == "latest":
        p = RUNS_DIR / "latest.json"
    else:
        run = RUNS.get(run_id)
        if run and run.get("result"):
            return run["result"]
        p = RUNS_DIR / f"{run_id}.json"
    if p.exists():
        return json.loads(p.read_text(encoding="utf-8"))
    raise HTTPException(404, "Result not ready")


@app.get("/api/runs/saved/latest")
def latest():
    return _load_result("latest")


@app.get("/api/runs/{run_id}")
def get_run(run_id: str):
    run = RUNS.get(run_id)
    if run and not run["done"]:
        return JSONResponse({"status": "running", "events": len(run["telemetry"].events)}, status_code=202)
    if run and run["done"] and not run.get("result"):
        raise HTTPException(410, "Run failed before producing a result.")
    return _load_result(run_id)


class ChatReq(BaseModel):
    message: str
    history: list[dict[str, str]] = []


@app.post("/api/runs/{run_id}/chat")
def chat(run_id: str, req: ChatReq):
    from .copilot import answer
    result = _load_result(run_id)
    run = RUNS.get(run_id)
    return answer(result, req.message[:800], req.history[-8:], run["ctx"] if run else None)


@app.get("/api/runs/{run_id}/report")
def report(run_id: str):
    from .report import render_report
    html = render_report(_load_result(run_id))
    return HTMLResponse(html, headers={"Content-Disposition": f'attachment; filename="safe_procure_report_{run_id}.html"'})


@app.get("/api/runs/{run_id}/csv")
def run_csv(run_id: str):
    r = _load_result(run_id)
    rows = [{"vendor": v["vendor"], "category": v["category"], "region": v["region"], "rank": v["rank"], "tier": v["tier"],
             "composite": v["composite"], **{f"pillar_{k}": x for k, x in v["pillars"].items()},
             "internal_risk": v["internal_risk"], "market_risk": v["market_risk"], "combined_risk": v["combined_risk"],
             "decision": (v.get("decision") or {}).get("decision", ""), "confidence": (v.get("decision") or {}).get("confidence", ""),
             "rationale": (v.get("decision") or {}).get("rationale", ""), "spend_inr": v["spend_total"]} for v in r["vendors"]]
    buf = io.StringIO()
    pd.DataFrame(rows).to_csv(buf, index=False)
    return Response(buf.getvalue(), media_type="text/csv",
                    headers={"Content-Disposition": f'attachment; filename="safe_procure_scorecard_{run_id}.csv"'})


# --------------------------------------------------------------- frontend
@app.get("/")
def index():
    return FileResponse(WEB / "index.html", headers={"Cache-Control": "no-cache"})


app.mount("/static", StaticFiles(directory=WEB), name="static")

Writing safeprocure/server.py


In [16]:
%%writefile safeprocure/__init__.py
"""Safe Procure: multi-agent vendor risk intelligence."""

Writing safeprocure/__init__.py


**Web app**: hand-built single-page app (no UI framework): agent mission control, 9 insight tabs, copilot. Expand to view.

In [17]:
%%writefile safeprocure/web/index.html
<!doctype html>
<html lang="en" data-theme="auto">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">
<title>Safe Procure</title>
<meta name="description" content="Multi-agent vendor risk intelligence built on CrewAI.">
<meta name="theme-color" content="#F4F3EF" media="(prefers-color-scheme: light)">
<meta name="theme-color" content="#0C0E12" media="(prefers-color-scheme: dark)">
<link rel="icon" href="data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 32 32'%3E%3Cpath d='M16 2.5 27 6.6v8.9c0 6.9-4.6 11.8-11 14-6.4-2.2-11-7.1-11-14V6.6L16 2.5Z' fill='%233E3AE0'/%3E%3Cpath d='m10.5 16.2 3.7 3.6 7.3-7.6' stroke='white' stroke-width='2.6' fill='none' stroke-linecap='round' stroke-linejoin='round'/%3E%3C/svg%3E">
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Geist:wght@400;500;600;700&family=Geist+Mono:wght@400;500;600&family=Instrument+Serif:ital@0;1&display=swap" rel="stylesheet">
<link rel="stylesheet" href="/static/styles.css">
<script>
  (function () {
    try { var t = localStorage.getItem('sp-theme'); if (t) document.documentElement.dataset.theme = t; } catch (e) {}
  })();
</script>
</head>
<body>
<a class="skip" href="#main">Skip to content</a>

<header class="topbar">
  <a class="brand" href="#/configure" aria-label="Safe Procure home">
    <svg class="brand-mark" viewBox="0 0 32 32" aria-hidden="true">
      <path d="M16 2.5 27 6.6v8.9c0 6.9-4.6 11.8-11 14-6.4-2.2-11-7.1-11-14V6.6L16 2.5Z" class="bm-shield"/>
      <circle cx="11" cy="17.5" r="2.1" class="bm-node"/><circle cx="16" cy="11.5" r="2.1" class="bm-node"/><circle cx="21" cy="17.5" r="2.1" class="bm-node"/>
      <path d="M11 17.5 16 11.5 21 17.5M11 17.5h10" class="bm-link"/>
    </svg>
    <span class="brand-word">Safe<span>Procure</span></span>
  </a>

  <nav class="steps" aria-label="Workflow">
    <span class="steps-ind" aria-hidden="true"></span>
    <a href="#/configure" data-nav="configure"><svg class="i"><use href="#i-sliders"/></svg><span>Configure</span></a>
    <a href="#/run" data-nav="run"><svg class="i"><use href="#i-network"/></svg><span>Agents</span></a>
    <a href="#/insights" data-nav="insights"><svg class="i"><use href="#i-chart"/></svg><span>Insights</span></a>
  </nav>

  <div class="top-actions">
    <button class="pal-btn" id="palette-btn" type="button" aria-haspopup="dialog" aria-expanded="false" aria-controls="palette" aria-keyshortcuts="Control+K">
      <svg class="i"><use href="#i-search"/></svg><span>Jump to</span><kbd aria-hidden="true">Ctrl K</kbd>
    </button>
    <div class="status-pill" id="status-pill">
      <span class="dot" aria-hidden="true"></span><span class="label" role="status" aria-live="polite">Checking models</span>
    </div>
    <button class="icon-btn" id="theme-toggle" type="button" aria-label="Switch colour theme" title="Switch colour theme">
      <svg class="i i-sun"><use href="#i-sun"/></svg><svg class="i i-moon"><use href="#i-moon"/></svg>
    </button>
  </div>
</header>

<main id="main" tabindex="-1">
  <!-- ======================================================= CONFIGURE -->
  <section class="view" id="view-configure" data-view="configure" aria-labelledby="cfg-title">
    <div class="cfg-hero">
      <div class="hero-copy">
        <p class="kicker"><span class="kdot" aria-hidden="true"></span><span id="kicker">Procurement observatory</span></p>
        <h1 id="cfg-title" class="display">Ask the crew. <em>Watch them work.</em></h1>
        <p class="lede">Load vendor data, tell the agents what matters, and ask your question. Ten specialist agents will audit, analyse, research live markets, forecast, decide and self-review.</p>
        <div class="ask">
          <label class="sr-only" for="in-question">Question for the agents</label>
          <textarea id="in-question" rows="2" maxlength="600" placeholder="Which vendors put our Q4 production at risk, and what should we do about each one?"></textarea>
          <span class="ask-glow" aria-hidden="true"></span>
        </div>
        <div class="prompt-chips" id="prompt-chips"><button type="button">Which vendors put our Q4 production at risk, and what should we do about each one?</button><button type="button">Where can we cut 10% of cost without hurting on-time delivery?</button><button type="button">How exposed are we to steel and logistics disruption right now?</button><button type="button">Which vendors deserve more volume next quarter?</button></div>
      </div>
      <svg class="crew-ring" id="crew-ring" viewBox="-120 -120 240 240" aria-hidden="true"></svg>
    </div>

    <div class="cfg-grid">
      <!-- Data -->
      <div class="panel" id="panel-data">
        <div class="panel-head">
          <h2>Vendor data</h2>
          <div class="seg" role="tablist" aria-label="Data source">
            <button role="tab" aria-selected="true" data-src="sample" id="tab-sample" aria-controls="src-sample">Generate sample</button>
            <button role="tab" aria-selected="false" data-src="upload" id="tab-upload" aria-controls="src-upload">Upload file</button>
          </div>
        </div>

        <div id="src-sample" role="tabpanel" aria-labelledby="tab-sample" class="src">
          <div class="field-row">
            <label class="field">
              <span class="field-label">Vendors <output id="o-nv">20</output></span>
              <input type="range" id="in-nv" min="5" max="25" value="20">
            </label>
            <label class="field">
              <span class="field-label">Months of history <output id="o-mo">18</output></span>
              <input type="range" id="in-mo" min="6" max="36" value="18">
            </label>
          </div>
          <div class="field-row">
            <div class="field">
              <span class="field-label" id="lbl-cats">Categories</span>
              <div class="chips" id="gen-cats" role="group" aria-labelledby="lbl-cats"></div>
            </div>
          </div>
          <div class="field-row compact">
            <label class="field seed">
              <span class="field-label">Seed</span>
              <span class="seed-wrap"><input type="number" id="in-seed" value="42" min="1" max="99999" inputmode="numeric">
              <button type="button" class="icon-btn sm" id="btn-dice" aria-label="Random seed" title="Random seed"><svg class="i"><use href="#i-dice"/></svg></button></span>
            </label>
            <label class="switch">
              <input type="checkbox" id="in-issues" checked>
              <span class="track" aria-hidden="true"></span>
              <span>Plant data-quality defects for the auditor</span>
            </label>
            <button type="button" class="btn" id="btn-generate"><svg class="i"><use href="#i-refresh"/></svg>Generate dataset</button>
          </div>
        </div>

        <div id="src-upload" role="tabpanel" aria-labelledby="tab-upload" class="src" hidden>
          <label class="drop" id="drop">
            <input type="file" id="in-file" accept=".csv,.xlsx,.xls">
            <svg class="i xl"><use href="#i-upload"/></svg>
            <strong>Drop a CSV or Excel file</strong>
            <span>Columns are mapped automatically (vendor, month, on-time %, defect %, price variance, spend…). Missing columns get neutral defaults and are disclosed.</span>
          </label>
          <p class="hint">Need the format? <a href="#" id="dl-template">Download the current dataset as a template</a>.</p>
        </div>

        <div class="dataset" id="dataset" aria-live="polite">
          <div class="skeleton-block" aria-hidden="true"></div>
        </div>
      </div>

      <!-- Mission -->
      <div class="panel" id="panel-mission">
        <div class="panel-head"><h2>What matters most</h2>
          <div class="presets" id="presets" role="group" aria-label="Weight presets"></div>
        </div>
        <div class="weights" id="weights"></div>
        <div class="weight-bar" id="weight-bar" aria-hidden="true"></div>

        <div class="panel-head sub"><h2>Scope and tolerance</h2></div>
        <div class="field">
          <span class="field-label" id="lbl-scope">Categories to analyse</span>
          <div class="chips" id="scope-cats" role="group" aria-labelledby="lbl-scope"></div>
        </div>
        <div class="field-row three">
          <label class="field"><span class="field-label">Risk tolerance <output id="o-th">55</output></span>
            <input type="range" id="in-th" min="35" max="75" value="55"><span class="field-help">Combined risk above this is flagged</span></label>
          <label class="field"><span class="field-label">On-time floor <output id="o-otd">85%</output></span>
            <input type="range" id="in-otd" min="75" max="95" value="85"></label>
          <label class="field"><span class="field-label">Defect ceiling <output id="o-def">3.0%</output></span>
            <input type="range" id="in-def" min="1" max="6" step="0.5" value="3"></label>
        </div>
      </div>

      <!-- Live preview -->
      <div class="panel" id="panel-preview">
        <div class="panel-head">
          <h2>Instant scorecard</h2>
          <span class="meta">Recomputed on every change. No AI involved yet.</span>
        </div>
        <ol class="rank-list" id="rank-list" tabindex="0" aria-label="Vendor ranking under the current weights"></ol>
        <p class="sr-only" id="rank-sr" role="status" aria-live="polite"></p>
      </div>
    </div>

    <div class="launch-bar">
      <div class="launch-info" id="launch-info">Ready when you are.</div>
      <div class="launch-actions">
        <button type="button" class="btn ghost" id="btn-saved" hidden><svg class="i"><use href="#i-history"/></svg><span>Open last saved run</span></button>
        <button type="button" class="btn primary lg" id="btn-run" disabled><svg class="i"><use href="#i-play"/></svg>Run the 10-agent analysis</button>
      </div>
    </div>
  </section>

  <!-- ============================================================ RUN -->
  <section class="view" id="view-run" data-view="run" aria-labelledby="run-title" hidden>
    <div class="view-head run-head">
      <div>
        <h1 id="run-title" class="display">Agents at work</h1>
        <p class="lede" id="run-question">No analysis running. Configure one first.</p>
      </div>
      <div class="run-side">
      <button type="button" class="btn ghost sm" id="btn-replay" hidden><svg class="i"><use href="#i-history"/></svg>Replay this run</button>
      <dl class="run-stats" id="run-stats">
        <div><dt>Elapsed</dt><dd id="st-time">0:00</dd></div>
        <div><dt>Stages</dt><dd id="st-stages">0/10</dd></div>
        <div><dt>LLM calls</dt><dd id="st-calls">0</dd></div>
        <div><dt>Tokens</dt><dd id="st-tokens">0</dd></div>
        <div><dt>Failovers</dt><dd id="st-fail">0</dd></div>
      </dl>
      </div>
    </div>

    <div class="run-grid">
      <div class="panel graph-panel">
        <div class="panel-head"><h2>Workflow</h2><span class="meta" id="graph-meta">CrewAI Flow · parallel fan-out · risk gate · QA revision loop</span></div>
        <div class="graph-wrap"><svg id="flow-graph" role="group" aria-label="Agent workflow graph. Select an agent to inspect it."></svg></div>
        <ol class="flow-list" id="flow-list" aria-label="Agent workflow stages"></ol>
        <div class="inspector" id="inspector"></div>
        <p class="sr-only" id="run-sr" role="status" aria-live="polite"></p>
      </div>

      <div class="panel trace-panel">
        <div class="panel-head"><h2>Live trace</h2>
          <label class="switch sm"><input type="checkbox" id="trace-verbose"><span class="track" aria-hidden="true"></span><span>Show model calls</span></label>
        </div>
        <ol class="trace" id="trace" aria-live="off"></ol>
      </div>

      <div class="panel lanes-panel">
        <div class="panel-head"><h2>Model router</h2><span class="meta">Tokens used in the last 60 s against each model's free-tier budget</span></div>
        <div class="lanes" id="lanes"></div>
        <p class="lane-legend"><span><i class="l-ok"></i>within budget</span><span><i class="l-hot"></i>above 75% of budget</span><span><i class="l-cool"></i>refused a call, cooling down</span></p>
      </div>
    </div>

    <div class="launch-bar" id="run-done-bar" hidden>
      <div class="launch-info" id="run-done-info"></div>
      <div class="launch-actions" id="run-done-actions"><a class="btn primary lg" href="#/insights"><svg class="i"><use href="#i-chart"/></svg>Open insights</a></div>
    </div>
  </section>

  <!-- ======================================================= INSIGHTS -->
  <section class="view" id="view-insights" data-view="insights" aria-labelledby="ins-title" hidden>
    <div class="ins-head">
      <h1 id="ins-title" class="sr-only">Insights</h1>
      <div class="tabs" role="tablist" aria-label="Insight sections" id="ins-tabs"></div>
      <div class="ins-actions">
        <a class="btn ghost sm" id="dl-report" href="#" download aria-label="Download executive report"><svg class="i"><use href="#i-download"/></svg><span>Report</span></a>
        <a class="btn ghost sm" id="dl-csv" href="#" download aria-label="Download scorecard CSV"><svg class="i"><use href="#i-table"/></svg><span>CSV</span></a>
      </div>
    </div>
    <div id="ins-body"></div>
  </section>
</main>

<!-- Copilot -->
<button class="copilot-fab" id="copilot-fab" type="button" aria-controls="copilot" aria-expanded="false" hidden>
  <svg class="i"><use href="#i-spark"/></svg><span>Ask the copilot</span>
</button>
<aside class="copilot" id="copilot" aria-label="Procurement copilot" hidden>
  <div class="cp-head">
    <div><strong>Procurement Copilot</strong><span>Tool-using agent with this run's data</span></div>
    <button class="icon-btn" id="copilot-close" type="button" aria-label="Close copilot"><svg class="i"><use href="#i-x"/></svg></button>
  </div>
  <ol class="cp-log" id="cp-log" aria-live="polite"></ol>
  <div class="cp-suggest" id="cp-suggest"></div>
  <form class="cp-form" id="cp-form">
    <label class="sr-only" for="cp-input">Message</label>
    <input id="cp-input" autocomplete="off" placeholder="Ask about any vendor, risk or scenario…" maxlength="800">
    <button class="icon-btn primary" type="submit" aria-label="Send"><svg class="i"><use href="#i-send"/></svg></button>
  </form>
</aside>

<div class="drawer-scrim" id="scrim" hidden></div>
<div class="pal-scrim" id="pal-scrim" hidden></div>
<div class="palette" id="palette" role="dialog" aria-modal="true" aria-label="Command palette" hidden>
  <div class="pal-field">
    <svg class="i" aria-hidden="true"><use href="#i-search"/></svg>
    <input id="pal-input" role="combobox" aria-expanded="true" aria-controls="pal-list" aria-autocomplete="list" aria-label="Jump to a view, insight, vendor or action" placeholder="Jump to a view, insight, vendor or action…" autocomplete="off" spellcheck="false">
    <kbd aria-hidden="true">Esc</kbd>
  </div>
  <ul class="pal-list" id="pal-list" role="listbox" aria-label="Commands"></ul>
  <div class="pal-foot"><span id="pal-count" role="status" aria-live="polite"></span><span aria-hidden="true"><kbd>↑</kbd><kbd>↓</kbd> move · <kbd>Enter</kbd> open · <kbd>/</kbd> or <kbd>Ctrl K</kbd> anywhere</span></div>
</div>
<div class="drawer" id="drawer" role="dialog" aria-modal="true" aria-labelledby="drawer-title" hidden>
  <div class="drawer-inner" id="drawer-inner"></div>
</div>

<div class="toast-zone" id="toasts" role="status" aria-live="polite" aria-atomic="false" aria-relevant="additions"></div>

<!-- Icon sprite (stroke icons, 24px grid, 1.75 stroke) -->
<svg width="0" height="0" style="position:absolute" aria-hidden="true">
  <defs>
    <symbol id="i-sliders" viewBox="0 0 24 24"><path d="M4 6h10M18 6h2M4 12h4M12 12h8M4 18h12M20 18h0"/><circle cx="16" cy="6" r="2"/><circle cx="10" cy="12" r="2"/><circle cx="18" cy="18" r="2"/></symbol>
    <symbol id="i-network" viewBox="0 0 24 24"><rect x="9" y="3" width="6" height="5" rx="1.5"/><rect x="3" y="16" width="6" height="5" rx="1.5"/><rect x="15" y="16" width="6" height="5" rx="1.5"/><path d="M12 8v4M6 16v-2a2 2 0 0 1 2-2h8a2 2 0 0 1 2 2v2"/></symbol>
    <symbol id="i-chart" viewBox="0 0 24 24"><path d="M4 20V10M10 20V4M16 20v-7M22 20H2"/></symbol>
    <symbol id="i-sun" viewBox="0 0 24 24"><circle cx="12" cy="12" r="4"/><path d="M12 2v2M12 20v2M4.9 4.9l1.4 1.4M17.7 17.7l1.4 1.4M2 12h2M20 12h2M4.9 19.1l1.4-1.4M17.7 6.3l1.4-1.4"/></symbol>
    <symbol id="i-moon" viewBox="0 0 24 24"><path d="M20 14.5A8 8 0 1 1 9.5 4a6.5 6.5 0 0 0 10.5 10.5Z"/></symbol>
    <symbol id="i-dice" viewBox="0 0 24 24"><rect x="3.5" y="3.5" width="17" height="17" rx="3.5"/><circle cx="8.5" cy="8.5" r="1" fill="currentColor"/><circle cx="15.5" cy="15.5" r="1" fill="currentColor"/><circle cx="12" cy="12" r="1" fill="currentColor"/></symbol>
    <symbol id="i-refresh" viewBox="0 0 24 24"><path d="M20 11a8 8 0 0 0-14.8-4M4 4v4h4M4 13a8 8 0 0 0 14.8 4M20 20v-4h-4"/></symbol>
    <symbol id="i-upload" viewBox="0 0 24 24"><path d="M12 16V4M7 9l5-5 5 5M4 16v2a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2v-2"/></symbol>
    <symbol id="i-history" viewBox="0 0 24 24"><path d="M3 12a9 9 0 1 0 3-6.7L3 8M3 3v5h5M12 7v5l3 2"/></symbol>
    <symbol id="i-play" viewBox="0 0 24 24"><path d="M7 4.5v15l12-7.5-12-7.5Z"/></symbol>
    <symbol id="i-download" viewBox="0 0 24 24"><path d="M12 4v12M7 11l5 5 5-5M4 20h16"/></symbol>
    <symbol id="i-table" viewBox="0 0 24 24"><rect x="3" y="4" width="18" height="16" rx="2"/><path d="M3 10h18M3 15h18M9 10v10"/></symbol>
    <symbol id="i-spark" viewBox="0 0 24 24"><path d="M12 3l1.8 5.2L19 10l-5.2 1.8L12 17l-1.8-5.2L5 10l5.2-1.8L12 3ZM19 16l.8 2.2L22 19l-2.2.8L19 22l-.8-2.2L16 19l2.2-.8L19 16Z"/></symbol>
    <symbol id="i-x" viewBox="0 0 24 24"><path d="M6 6l12 12M18 6 6 18"/></symbol>
    <symbol id="i-send" viewBox="0 0 24 24"><path d="M4 12 20 4l-6 16-3-7-7-1Z"/></symbol>
    <symbol id="i-check" viewBox="0 0 24 24"><path d="m5 12.5 4.5 4.5L19 7.5"/></symbol>
    <symbol id="i-alert" viewBox="0 0 24 24"><path d="M12 3 2.5 20h19L12 3ZM12 10v4.5M12 17.5v.01"/></symbol>
    <symbol id="i-arrow-up" viewBox="0 0 24 24"><path d="M12 19V5M6 11l6-6 6 6"/></symbol>
    <symbol id="i-arrow-down" viewBox="0 0 24 24"><path d="M12 5v14M6 13l6 6 6-6"/></symbol>
    <symbol id="i-arrow-right" viewBox="0 0 24 24"><path d="M5 12h14M13 6l6 6-6 6"/></symbol>
    <symbol id="i-search" viewBox="0 0 24 24"><circle cx="11" cy="11" r="7"/><path d="m20 20-4-4"/></symbol>
    <symbol id="i-tool" viewBox="0 0 24 24"><path d="M14.5 6.5a4 4 0 0 0-5.6 4.9L3.5 16.8a1.8 1.8 0 0 0 2.6 2.6l5.4-5.4a4 4 0 0 0 4.9-5.6l-2.6 2.6-2.1-.5-.5-2.1 2.6-2.6-.3-.3Z"/></symbol>
    <symbol id="i-shuffle" viewBox="0 0 24 24"><path d="M4 7h3.5c4.5 0 4.5 10 9 10H20M4 17h3.5c1.6 0 2.6-1.3 3.4-3M14 10c.8-1.7 1.8-3 3-3H20M17 4l3 3-3 3M17 14l3 3-3 3"/></symbol>
    <symbol id="i-globe" viewBox="0 0 24 24"><circle cx="12" cy="12" r="9"/><path d="M3 12h18M12 3c2.5 2.6 3.8 5.6 3.8 9s-1.3 6.4-3.8 9c-2.5-2.6-3.8-5.6-3.8-9S9.5 5.6 12 3Z"/></symbol>
    <symbol id="i-shield" viewBox="0 0 24 24"><path d="M12 3 19.5 6v5.6c0 4.6-3.1 7.9-7.5 9.4-4.4-1.5-7.5-4.8-7.5-9.4V6L12 3Z"/><path d="m9 12 2.2 2.2L15.5 10"/></symbol>
    <symbol id="i-external" viewBox="0 0 24 24"><path d="M14 4h6v6M20 4l-9 9M18 14v4a2 2 0 0 1-2 2H6a2 2 0 0 1-2-2V8a2 2 0 0 1 2-2h4"/></symbol>
    <symbol id="i-flag" viewBox="0 0 24 24"><path d="M5 21V4M5 4h11l-2 4 2 4H5"/></symbol>
    <symbol id="i-gauge" viewBox="0 0 24 24"><path d="M4 18a8 8 0 1 1 16 0"/><path d="m12 18 4-6"/></symbol>
  </defs>
</svg>

<script id="echarts-js" src="https://cdn.jsdelivr.net/npm/echarts@5.5.1/dist/echarts.min.js" async></script>
<script type="module" src="/static/app.js"></script>
</body>
</html>

Writing safeprocure/web/index.html


In [18]:
%%writefile safeprocure/web/styles.css
/* Width-matched local fallbacks (ratios measured in Chrome) so the web-font swap does not re-wrap text and shift layout. */
@font-face { font-family: "Geist Fallback"; src: local("Arial"); size-adjust: 102.3%; }
@font-face { font-family: "Geist Mono Fallback"; src: local("Consolas"), local("Menlo"); size-adjust: 109.1%; ascent-override: 92.6%; descent-override: 27.5%; line-gap-override: 0%; }
@font-face { font-family: "Instrument Serif Fallback"; src: local("Times New Roman"), local("Times"); size-adjust: 83.6%; ascent-override: 118.4%; descent-override: 37.1%; line-gap-override: 0%; }
@font-face { font-family: "Instrument Serif Fallback"; font-style: italic; src: local("Times New Roman Italic"), local("TimesNewRomanPS-ItalicMT"), local("Times Italic"); size-adjust: 89.2%; ascent-override: 111%; descent-override: 34.8%; line-gap-override: 0%; }
/* Safe Procure — design tokens + components
   World: an instrument panel for sourcing decisions. Warm paper surfaces, ink text,
   one indigo action colour, four decision colours that never mean anything else. */

:root {
  --font-ui: "Geist", "Geist Fallback", ui-sans-serif, system-ui, -apple-system, "Segoe UI", sans-serif;
  --font-num: "Geist Mono", "Geist Mono Fallback", ui-monospace, "SFMono-Regular", Consolas, monospace;
  --font-display: "Instrument Serif", "Instrument Serif Fallback", "Iowan Old Style", Georgia, serif;

  --bg: #F4F3EF;
  --surface: #FFFFFF;
  --surface-2: #FAF9F6;
  --surface-3: #F1EFE9;
  --ink: #15171C;
  --ink-2: #474C58;
  --ink-3: #676D7B;
  --line: #E4E1D9;
  --line-2: #D3CFC4;
  --brand: #3E3AE0;
  --brand-2: #2F2BC4;
  --brand-ink: #FFFFFF;
  --brand-soft: #ECEBFD;
  --brand-line: #C9C7F7;
  --focus: #3E3AE0;

  --keep: #0B7A4E;     --keep-soft: #E2F3EA;     --keep-line: #A9D9C0;
  --watch: #A85A06;    --watch-soft: #FBF0DF;    --watch-line: #EBCB98;
  --develop: #0A6B9E;  --develop-soft: #E1EFF7;  --develop-line: #A6CDE3;
  --replace: #BE2A21;  --replace-soft: #FBE6E3;  --replace-line: #EDB2AB;

  --risk-0: #0B7A4E; --risk-1: #6E8F1A; --risk-2: #C08A05; --risk-3: #D7600F; --risk-4: #BE2A21;

  --shadow-1: 0 1px 2px rgba(20, 22, 30, .05), 0 2px 8px rgba(20, 22, 30, .04);
  --shadow-2: 0 8px 28px rgba(20, 22, 30, .10), 0 2px 6px rgba(20, 22, 30, .05);
  --shadow-3: 0 24px 64px rgba(20, 22, 30, .18), 0 4px 12px rgba(20, 22, 30, .08);
  --radius: 14px;
  --radius-sm: 9px;
  --ease: cubic-bezier(.16, 1, .3, 1);
  --topbar-h: 60px;
  color-scheme: light;
}

@media (prefers-color-scheme: dark) {
  :root:not([data-theme="light"]) {
    --bg: #0C0E12; --surface: #13161C; --surface-2: #171A21; --surface-3: #1C2029;
    --ink: #ECEDF0; --ink-2: #B0B5C1; --ink-3: #8D93A2; --line: #242933; --line-2: #313744;
    --brand: #8E8BFF; --brand-2: #A5A3FF; --brand-ink: #0C0E12; --brand-soft: #1E1D3D; --brand-line: #3D3A82; --focus: #A5A3FF;
    --keep: #4CC793; --keep-soft: #10261D; --keep-line: #1E5A40;
    --watch: #F0A84A; --watch-soft: #2A1E0E; --watch-line: #5E4218;
    --develop: #5EB6E8; --develop-soft: #0F2230; --develop-line: #1E4D6B;
    --replace: #F2766C; --replace-soft: #2E1311; --replace-line: #6A2722;
    --risk-0: #4CC793; --risk-1: #A6C74C; --risk-2: #E8C04A; --risk-3: #F08A45; --risk-4: #F2766C;
    --shadow-1: 0 1px 2px rgba(0, 0, 0, .4); --shadow-2: 0 10px 30px rgba(0, 0, 0, .45); --shadow-3: 0 24px 64px rgba(0, 0, 0, .6);
    color-scheme: dark;
  }
}
:root[data-theme="dark"] {
  --bg: #0C0E12; --surface: #13161C; --surface-2: #171A21; --surface-3: #1C2029;
  --ink: #ECEDF0; --ink-2: #B0B5C1; --ink-3: #8D93A2; --line: #242933; --line-2: #313744;
  --brand: #8E8BFF; --brand-2: #A5A3FF; --brand-ink: #0C0E12; --brand-soft: #1E1D3D; --brand-line: #3D3A82; --focus: #A5A3FF;
  --keep: #4CC793; --keep-soft: #10261D; --keep-line: #1E5A40;
  --watch: #F0A84A; --watch-soft: #2A1E0E; --watch-line: #5E4218;
  --develop: #5EB6E8; --develop-soft: #0F2230; --develop-line: #1E4D6B;
  --replace: #F2766C; --replace-soft: #2E1311; --replace-line: #6A2722;
  --risk-0: #4CC793; --risk-1: #A6C74C; --risk-2: #E8C04A; --risk-3: #F08A45; --risk-4: #F2766C;
  --shadow-1: 0 1px 2px rgba(0, 0, 0, .4); --shadow-2: 0 10px 30px rgba(0, 0, 0, .45); --shadow-3: 0 24px 64px rgba(0, 0, 0, .6);
  color-scheme: dark;
}

/* ------------------------------------------------------------------ base */
*, *::before, *::after { box-sizing: border-box; }
html { -webkit-text-size-adjust: 100%; scroll-padding-top: calc(var(--topbar-h) + 12px); }
body {
  margin: 0; background: var(--bg); color: var(--ink); font: 400 14px/1.5 var(--font-ui);
  font-feature-settings: "ss01", "cv11"; -webkit-font-smoothing: antialiased; text-rendering: optimizeLegibility;
  min-height: 100dvh; caret-color: var(--brand);
}
::selection { background: var(--brand); color: var(--brand-ink); }
:focus-visible { outline: 2px solid var(--focus); outline-offset: 2px; border-radius: 6px; }
a { color: var(--brand); text-underline-offset: 3px; text-decoration-thickness: 1px; }
h1, h2, h3, h4 { margin: 0; font-weight: 600; letter-spacing: -0.01em; text-wrap: balance; }
p { margin: 0; text-wrap: pretty; }
button, input, textarea, select { font: inherit; color: inherit; }
button { cursor: pointer; }
[hidden] { display: none !important; }
.sr-only { position: absolute; width: 1px; height: 1px; overflow: hidden; clip: rect(0 0 0 0); white-space: nowrap; }
.skip { position: absolute; left: 12px; top: -40px; background: var(--brand); color: var(--brand-ink); padding: 8px 12px; border-radius: 8px; z-index: 100; }
.skip:focus { top: 10px; }
.num, .mono { font-family: var(--font-num); font-variant-numeric: tabular-nums; letter-spacing: -0.01em; }
* { scrollbar-width: thin; scrollbar-color: var(--line-2) transparent; }
::-webkit-scrollbar { width: 10px; height: 10px; }
::-webkit-scrollbar-thumb { background: var(--line-2); border-radius: 10px; border: 3px solid transparent; background-clip: padding-box; }

.i { width: 18px; height: 18px; fill: none; stroke: currentColor; stroke-width: 1.75; stroke-linecap: round; stroke-linejoin: round; flex: none; }
.i.sm { width: 15px; height: 15px; }
.i.xl { width: 30px; height: 30px; stroke-width: 1.5; }

/* --------------------------------------------------------------- topbar */
.topbar {
  position: sticky; top: 0; z-index: 30; height: var(--topbar-h);
  display: grid; grid-template-columns: 1fr auto 1fr; align-items: center; gap: 16px;
  padding: 0 max(20px, env(safe-area-inset-left)); background: color-mix(in oklab, var(--bg) 86%, transparent);
  backdrop-filter: saturate(1.4) blur(14px); -webkit-backdrop-filter: saturate(1.4) blur(14px);
  border-bottom: 1px solid var(--line);
}
.brand { display: inline-flex; align-items: center; gap: 10px; color: var(--ink); text-decoration: none; justify-self: start; }
.brand-mark { width: 28px; height: 28px; }
.bm-shield { fill: var(--brand); }
.bm-node { fill: var(--brand-ink); }
.bm-link { stroke: var(--brand-ink); stroke-width: 1.6; fill: none; stroke-linecap: round; opacity: .85; }
.brand-word { font-weight: 650; font-size: 16px; letter-spacing: -0.02em; }
.brand-word span { color: var(--ink-3); font-weight: 500; margin-left: 2px; }

.steps { display: flex; gap: 4px; padding: 4px; background: var(--surface-3); border-radius: 12px; border: 1px solid var(--line); }
.steps a {
  display: inline-flex; align-items: center; gap: 8px; padding: 7px 14px; border-radius: 9px; color: var(--ink-2);
  text-decoration: none; font-weight: 500; transition: background .2s var(--ease), color .2s var(--ease);
}
.steps a:hover { color: var(--ink); }
.steps a[aria-current="page"] { background: var(--surface); color: var(--ink); box-shadow: var(--shadow-1); }
.steps a.locked { opacity: .45; pointer-events: none; }
.steps a .count { font-family: var(--font-num); font-size: 11px; padding: 1px 6px; border-radius: 99px; background: var(--brand); color: var(--brand-ink); }

.top-actions { display: flex; align-items: center; gap: 10px; justify-self: end; }
.status-pill {
  display: inline-flex; align-items: center; gap: 8px; height: 32px; padding: 0 12px; border-radius: 99px;
  border: 1px solid var(--line); background: var(--surface); color: var(--ink-2); font-size: 12.5px; font-weight: 500; white-space: nowrap;
}
.status-pill .dot { width: 8px; height: 8px; border-radius: 50%; background: var(--ink-3); }
.status-pill.ok .dot { background: var(--keep); box-shadow: 0 0 0 3px var(--keep-soft); }
.status-pill.warn .dot { background: var(--watch); box-shadow: 0 0 0 3px var(--watch-soft); }
.status-pill.bad .dot { background: var(--replace); box-shadow: 0 0 0 3px var(--replace-soft); }
.status-pill.live .dot { background: var(--brand); animation: pulse 1.4s var(--ease) infinite; }

.icon-btn {
  display: inline-grid; place-items: center; width: 36px; height: 36px; border-radius: 10px; border: 1px solid var(--line);
  background: var(--surface); color: var(--ink-2); transition: background .15s, color .15s, border-color .15s;
}
.icon-btn:hover { color: var(--ink); border-color: var(--line-2); }
.icon-btn.sm { width: 32px; height: 32px; border-radius: 8px; }
.icon-btn.primary { background: var(--brand); border-color: var(--brand); color: var(--brand-ink); }
.icon-btn.primary:hover { background: var(--brand-2); }
#theme-toggle .i-moon { display: none; }
:root[data-theme="dark"] #theme-toggle .i-sun { display: none; }
:root[data-theme="dark"] #theme-toggle .i-moon { display: block; }
:root[data-theme="dark"] .dist span { color: #0C0E12; text-shadow: none; }
@media (prefers-color-scheme: dark) {
  :root:not([data-theme="light"]) .dist span { color: #0C0E12; text-shadow: none; }
  :root:not([data-theme="light"]) #theme-toggle .i-sun { display: none; }
  :root:not([data-theme="light"]) #theme-toggle .i-moon { display: block; }
}

/* ---------------------------------------------------------------- layout */
main { padding: 28px max(20px, env(safe-area-inset-left)) 120px; max-width: 1480px; margin: 0 auto; outline: none; }
.view { animation: viewIn .5s var(--ease) both; }
@keyframes viewIn { from { opacity: 0; transform: translateY(6px); } to { opacity: 1; transform: none; } }
.view-head { display: flex; justify-content: space-between; align-items: flex-end; gap: 24px; margin-bottom: 22px; }
.display { font-family: var(--font-display); font-weight: 400; font-size: clamp(34px, 4.2vw, 52px); line-height: 1.02; letter-spacing: -0.015em; }
.lede { color: var(--ink-2); font-size: 15px; max-width: 68ch; margin-top: 10px; }

.panel { background: var(--surface); border: 1px solid var(--line); border-radius: var(--radius); padding: 20px; min-width: 0; }
.panel-head { display: flex; align-items: center; justify-content: space-between; gap: 12px; margin-bottom: 14px; flex-wrap: wrap; }
.panel-head h2 { font-size: 15px; font-weight: 600; }
.panel-head.sub { margin-top: 24px; padding-top: 18px; border-top: 1px solid var(--line); }
.meta { color: var(--ink-3); font-size: 12.5px; }

/* --------------------------------------------------------------- controls */
.btn {
  display: inline-flex; align-items: center; justify-content: center; gap: 8px; height: 38px; padding: 0 16px; border-radius: 10px;
  border: 1px solid var(--line-2); background: var(--surface); color: var(--ink); font-weight: 550; font-size: 13.5px;
  text-decoration: none; white-space: nowrap; transition: transform .15s var(--ease), background .15s, border-color .15s, box-shadow .2s;
}
.btn:hover { border-color: var(--ink-3); }
.btn:active { transform: translateY(1px); }
.btn.primary { background: var(--brand); border-color: var(--brand); color: var(--brand-ink); box-shadow: 0 6px 18px -6px color-mix(in oklab, var(--brand) 60%, transparent); }
.btn.primary:hover { background: var(--brand-2); border-color: var(--brand-2); }
.btn.ghost { background: transparent; }
.btn.lg { height: 48px; padding: 0 22px; font-size: 15px; border-radius: 12px; }
.btn.sm { height: 32px; padding: 0 12px; font-size: 12.5px; border-radius: 8px; }
.btn[disabled], .btn[aria-disabled="true"] { opacity: .5; cursor: not-allowed; box-shadow: none; }
.btn.loading { position: relative; color: transparent !important; }
.btn.loading::after { content: ""; position: absolute; width: 18px; height: 18px; border-radius: 50%; border: 2px solid var(--brand-ink); border-right-color: transparent; animation: spin .7s linear infinite; }
.btn:not(.primary).loading::after { border-color: var(--ink); border-right-color: transparent; }

.seg { display: inline-flex; padding: 3px; gap: 2px; background: var(--surface-3); border: 1px solid var(--line); border-radius: 10px; }
.seg button { border: 0; background: transparent; padding: 6px 12px; border-radius: 7px; color: var(--ink-2); font-weight: 500; font-size: 13px; }
.seg button[aria-selected="true"] { background: var(--surface); color: var(--ink); box-shadow: var(--shadow-1); }

.field { display: flex; flex-direction: column; gap: 8px; min-width: 0; }
.field-row { display: grid; grid-template-columns: 1fr 1fr; gap: 18px 22px; margin-bottom: 18px; }
.field-row.three { grid-template-columns: repeat(3, 1fr); }
.field:has(#scope-cats) { margin-bottom: 18px; }
.field-row.compact { display: flex; align-items: flex-end; flex-wrap: wrap; gap: 14px 18px; margin-bottom: 4px; }
.field-label { display: flex; justify-content: space-between; align-items: baseline; font-size: 12.5px; font-weight: 550; color: var(--ink-2); }
.field-label output { font-family: var(--font-num); color: var(--ink); font-weight: 500; }
.field-help { font-size: 12px; color: var(--ink-3); }
.hint { margin-top: 10px; color: var(--ink-3); font-size: 12.5px; }

input[type="range"] { -webkit-appearance: none; appearance: none; width: 100%; height: 24px; background: transparent; margin: 0; --p: 50%; }
input[type="range"]::-webkit-slider-runnable-track { height: 6px; border-radius: 99px; background: linear-gradient(to right, var(--brand) var(--p), var(--line) var(--p)); }
input[type="range"]::-moz-range-track { height: 6px; border-radius: 99px; background: var(--line); }
input[type="range"]::-moz-range-progress { height: 6px; border-radius: 99px; background: var(--brand); }
input[type="range"]::-webkit-slider-thumb { -webkit-appearance: none; margin-top: -7px; width: 20px; height: 20px; border-radius: 50%; background: var(--surface); border: 2px solid var(--brand); box-shadow: var(--shadow-1); transition: transform .15s var(--ease); }
input[type="range"]::-moz-range-thumb { width: 16px; height: 16px; border-radius: 50%; background: var(--surface); border: 2px solid var(--brand); }
input[type="range"]:active::-webkit-slider-thumb { transform: scale(1.12); }
input[type="range"]:focus-visible { outline: none; }
input[type="range"]:focus-visible::-webkit-slider-thumb { box-shadow: 0 0 0 4px var(--brand-soft), 0 0 0 6px var(--focus); }

input[type="number"], input[type="search"], input:not([type]), textarea, select {
  width: 100%; border: 1px solid var(--line-2); background: var(--surface-2); border-radius: 10px; padding: 9px 12px; color: var(--ink);
  transition: border-color .15s, box-shadow .15s, background .15s;
}
input:hover, textarea:hover { border-color: var(--ink-3); }
input:focus, textarea:focus, select:focus { outline: none; border-color: var(--brand); box-shadow: 0 0 0 3px var(--brand-soft); background: var(--surface); }
textarea { resize: vertical; min-height: 84px; line-height: 1.5; font-size: 15px; }
::placeholder { color: var(--ink-3); opacity: 1; }
.seed { width: 140px; }
.seed-wrap { display: flex; gap: 6px; }
.seed input { font-family: var(--font-num); }

.switch { display: inline-flex; align-items: center; gap: 10px; font-size: 13px; color: var(--ink-2); cursor: pointer; user-select: none; min-height: 38px; }
.switch input { position: absolute; opacity: 0; width: 1px; height: 1px; }
.switch .track { width: 36px; height: 21px; border-radius: 99px; background: var(--line-2); position: relative; transition: background .2s var(--ease); flex: none; }
.switch .track::after { content: ""; position: absolute; top: 2.5px; left: 2.5px; width: 16px; height: 16px; border-radius: 50%; background: var(--surface); box-shadow: var(--shadow-1); transition: transform .25s var(--ease); }
.switch input:checked + .track { background: var(--brand); }
.switch input:checked + .track::after { transform: translateX(15px); }
.switch input:focus-visible + .track { outline: 2px solid var(--focus); outline-offset: 2px; }
.switch.sm { font-size: 12.5px; min-height: 28px; }

.chips { display: flex; flex-wrap: wrap; gap: 6px; }
.chip {
  display: inline-flex; align-items: center; gap: 6px; min-height: 32px; padding: 0 12px; border-radius: 99px; border: 1px solid var(--line-2);
  background: var(--surface); color: var(--ink-2); font-size: 12.5px; font-weight: 500; transition: all .15s var(--ease);
}
.chip:hover { color: var(--ink); border-color: var(--ink-3); }
.chip[aria-pressed="true"] { background: var(--brand-soft); border-color: var(--brand-line); color: var(--brand-2); }
:root[data-theme="dark"] .chip[aria-pressed="true"] { color: var(--brand-2); }
.chip .tick { width: 14px; height: 14px; display: none; }
.chip[aria-pressed="true"] .tick { display: block; }

.prompt-chips { display: flex; flex-wrap: wrap; gap: 6px; margin-top: 10px; }
.prompt-chips button {
  border: 1px dashed var(--line-2); background: transparent; border-radius: 99px; padding: 6px 12px; color: var(--ink-2); font-size: 12.5px; text-align: left;
}
.prompt-chips button:hover { border-style: solid; color: var(--ink); border-color: var(--brand-line); background: var(--brand-soft); }

/* ----------------------------------------------------------- configure */
.cfg-grid { display: grid; grid-template-columns: minmax(0, 1.08fr) minmax(0, 1fr) minmax(280px, .72fr); gap: 16px; align-items: start; }
#panel-preview { position: sticky; top: calc(var(--topbar-h) + 16px); max-height: calc(100dvh - var(--topbar-h) - 120px); display: flex; flex-direction: column; }
.src { padding-bottom: 6px; }

.drop {
  display: flex; flex-direction: column; align-items: center; justify-content: center; text-align: center; gap: 8px; padding: 30px 20px;
  border: 1.5px dashed var(--line-2); border-radius: var(--radius); background: var(--surface-2); color: var(--ink-2); cursor: pointer; transition: all .2s var(--ease);
}
.drop strong { color: var(--ink); font-size: 15px; }
.drop span { max-width: 52ch; font-size: 12.5px; }
.drop input { position: absolute; opacity: 0; width: 1px; height: 1px; }
.drop.over, .drop:hover { border-color: var(--brand); background: var(--brand-soft); color: var(--ink); }
.drop:focus-within { outline: 2px solid var(--focus); outline-offset: 2px; }

.dataset { margin-top: 18px; padding-top: 18px; border-top: 1px solid var(--line); }
.ds-facts { display: flex; flex-wrap: wrap; gap: 6px 22px; align-items: baseline; margin-bottom: 14px; }
.ds-facts div { display: flex; flex-direction: column; }
.ds-facts dt { font-size: 12px; color: var(--ink-3); }
.ds-facts dd { margin: 0; font-family: var(--font-num); font-size: 19px; font-weight: 500; letter-spacing: -0.02em; }
.issues { list-style: none; padding: 0; margin: 0 0 14px; display: grid; gap: 6px; }
.issues li { display: grid; grid-template-columns: auto 1fr; gap: 10px; align-items: start; font-size: 13px; color: var(--ink-2); padding: 8px 10px; background: var(--surface-2); border-radius: 9px; border: 1px solid var(--line); }
.issues li .i { color: var(--watch); margin-top: 1px; }
.issues li b { color: var(--ink); font-weight: 600; }
.notes { font-size: 12.5px; color: var(--ink-3); margin: 0 0 12px; padding-left: 18px; }
.mini-chart { height: 120px; margin: 4px -6px 0; }
details.preview { margin-top: 12px; }
details.preview summary { cursor: pointer; color: var(--ink-2); font-size: 13px; font-weight: 500; list-style: none; display: inline-flex; align-items: center; gap: 6px; }
details.preview summary::-webkit-details-marker { display: none; }
details.preview summary::before { content: ""; width: 7px; height: 7px; border-right: 1.75px solid currentColor; border-bottom: 1.75px solid currentColor; transform: rotate(-45deg); transition: transform .2s var(--ease); }
details.preview[open] summary::before { transform: rotate(45deg); }
.table-scroll { overflow: auto; max-height: 300px; margin-top: 10px; border: 1px solid var(--line); border-radius: 10px; }
table.data { border-collapse: collapse; width: 100%; font-size: 12.5px; }
table.data th { position: sticky; top: 0; background: var(--surface-3); text-align: left; font-weight: 600; color: var(--ink-2); padding: 7px 10px; white-space: nowrap; z-index: 1; }
table.data td { padding: 6px 10px; border-top: 1px solid var(--line); white-space: nowrap; font-family: var(--font-num); font-variant-numeric: tabular-nums; }
table.data td.txt { font-family: var(--font-ui); }

.weights { display: grid; gap: 10px; }
.weight { display: grid; grid-template-columns: 128px 1fr 46px; align-items: center; gap: 12px; }
.weight label { font-size: 13px; font-weight: 550; display: flex; align-items: center; gap: 8px; }
.weight label .sw { width: 9px; height: 9px; border-radius: 3px; }
.weight output { font-family: var(--font-num); text-align: right; font-size: 13px; color: var(--ink-2); }
.weight-bar { display: flex; height: 8px; border-radius: 99px; overflow: hidden; margin-top: 14px; gap: 2px; }
.weight-bar span { transition: flex-grow .35s var(--ease); min-width: 2px; }
.presets { display: flex; gap: 6px; flex-wrap: wrap; }
.presets button { border: 1px solid var(--line); background: var(--surface-2); border-radius: 8px; padding: 4px 10px; font-size: 12px; color: var(--ink-2); }
.presets button:hover { color: var(--ink); border-color: var(--line-2); }
.presets button[aria-pressed="true"] { border-color: var(--brand-line); background: var(--brand-soft); color: var(--brand-2); }

.rank-list { list-style: none; margin: 0 -8px; padding: 0; overflow: auto; flex: 1; min-height: 0; }
.rank-list li {
  display: grid; grid-template-columns: 26px minmax(0, 1fr) auto; gap: 10px; align-items: center; padding: 8px; border-radius: 9px;
  transition: background .2s;
}
.rank-list li:hover { background: var(--surface-2); }
.rank-list .rk { font-family: var(--font-num); color: var(--ink-3); font-size: 12px; text-align: right; }
.rank-list .nm { overflow: hidden; }
.rank-list .nm b { display: block; font-weight: 550; font-size: 13px; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }
.rank-list .nm span { font-size: 11.5px; color: var(--ink-3); }
.rank-list .sc { display: flex; align-items: center; gap: 8px; font-family: var(--font-num); font-size: 13px; }
.rank-list .mv { font-size: 11px; width: 30px; display: inline-flex; align-items: center; justify-content: flex-end; gap: 1px; }
.rank-list .mv.up { color: var(--keep); } .rank-list .mv.down { color: var(--replace); }
.rank-list .mv .i { width: 12px; height: 12px; stroke-width: 2.2; }
.rank-list li.flash { animation: flash 1s var(--ease); }
@keyframes flash { from { background: var(--brand-soft); } to { background: transparent; } }

.tier { display: inline-grid; place-items: center; width: 22px; height: 22px; border-radius: 6px; font-weight: 650; font-size: 11.5px; font-family: var(--font-num); }
.tier.A { background: var(--keep-soft); color: var(--keep); }
.tier.B { background: var(--develop-soft); color: var(--develop); }
.tier.C { background: var(--watch-soft); color: var(--watch); }
.tier.D { background: var(--replace-soft); color: var(--replace); }

.launch-bar {
  position: sticky; bottom: 16px; z-index: 20; margin-top: 18px; display: flex; align-items: center; justify-content: space-between; gap: 16px;
  padding: 12px 12px 12px 20px; background: color-mix(in oklab, var(--surface) 92%, transparent); backdrop-filter: blur(12px); -webkit-backdrop-filter: blur(12px);
  border: 1px solid var(--line); border-radius: 16px; box-shadow: var(--shadow-2);
}
.launch-info { color: var(--ink-2); font-size: 13.5px; }
.launch-info b { color: var(--ink); font-weight: 600; }
.launch-actions { display: flex; gap: 10px; flex: none; }

.skeleton-block { height: 160px; border-radius: 10px; background: linear-gradient(90deg, var(--surface-3), var(--surface-2), var(--surface-3)); background-size: 200% 100%; animation: shimmer 1.4s linear infinite; }
@keyframes shimmer { to { background-position: -200% 0; } }

/* ------------------------------------------------------------------ run */
.run-head { align-items: flex-end; }
.run-side { display: flex; flex-direction: column; align-items: flex-end; gap: 10px; }
.replay-tag { display: inline-flex; align-items: center; gap: 6px; font-size: 12px; font-weight: 600; color: var(--develop); background: var(--develop-soft); padding: 3px 10px; border-radius: 99px; margin-left: 10px; vertical-align: middle; font-family: var(--font-ui); }
.run-stats { display: flex; gap: 6px; margin: 0; flex-wrap: wrap; }
.run-stats div { background: var(--surface); border: 1px solid var(--line); border-radius: 12px; padding: 8px 14px; min-width: 92px; }
.run-stats dt { font-size: 11.5px; color: var(--ink-3); }
.run-stats dd { margin: 0; font-family: var(--font-num); font-size: 18px; font-weight: 500; letter-spacing: -0.02em; }
.run-grid { display: grid; grid-template-columns: minmax(0, 1.55fr) minmax(320px, 1fr); grid-template-rows: auto auto; gap: 16px; }
.graph-panel { grid-row: span 2; display: flex; flex-direction: column; }
.graph-wrap { position: relative; width: 100%; overflow-x: auto; }
#flow-graph { display: block; width: 100%; min-width: 640px; height: auto; }

.g-edge { fill: none; stroke: var(--line-2); stroke-width: 1.6; }
.g-edge.live { stroke: var(--brand); stroke-dasharray: 5 7; animation: flow 0.9s linear infinite; }
.g-edge.done { stroke: color-mix(in oklab, var(--brand) 55%, var(--line-2)); }
.g-edge.loop { stroke-dasharray: 3 5; }
.g-edge.skipped { opacity: .35; }
@keyframes flow { to { stroke-dashoffset: -24; } }
.g-node { cursor: pointer; }
.g-node rect.card { fill: var(--surface); stroke: var(--line-2); stroke-width: 1.2; transition: stroke .3s, fill .3s; }
.g-node:hover rect.card { stroke: var(--ink-3); }
.g-node:focus-visible { outline: none; }
.g-node:focus-visible rect.card { stroke: var(--focus); stroke-width: 2.2; }
.g-node text.t { fill: var(--ink); font: 600 12.5px var(--font-ui); }
.g-node text.s { fill: var(--ink-3); font: 500 10.5px var(--font-num); }
.g-node .halo { fill: none; stroke: var(--brand); stroke-width: 2; opacity: 0; }
.g-node .badge { fill: var(--surface-3); }
.g-node .badge-ic { stroke: var(--ink-3); fill: none; stroke-width: 1.8; stroke-linecap: round; stroke-linejoin: round; }
.g-node.active rect.card { stroke: var(--brand); fill: var(--brand-soft); }
.g-node.active .halo { animation: halo 1.6s var(--ease) infinite; }
.g-node.active .badge { fill: var(--brand); }
.g-node.active .badge-ic { stroke: var(--brand-ink); }
.g-node.done rect.card { stroke: color-mix(in oklab, var(--brand) 45%, var(--line-2)); }
.g-node.done .badge { fill: var(--keep); }
.g-node.done .badge-ic { stroke: #fff; }
.g-node.degraded .badge { fill: var(--watch); }
.g-node.degraded rect.card { stroke: var(--watch-line); }
.g-node.failed rect.card { stroke: var(--replace); fill: var(--replace-soft); stroke-width: 1.6; }
.g-node.failed .badge { fill: var(--replace); }
.g-node.failed .badge-ic { stroke: #fff; }
.g-node.skipped { opacity: .4; }
.g-node.selected rect.card { stroke: var(--ink); stroke-width: 2; }
.g-gate polygon { fill: var(--surface); stroke: var(--line-2); stroke-width: 1.2; }
.g-gate.active polygon { stroke: var(--brand); fill: var(--brand-soft); }
.g-gate.done polygon { stroke: var(--ink-3); }
.g-gate text { fill: var(--ink-2); font: 600 10px var(--font-ui); }
.g-route { fill: var(--ink-3); font: 500 10px var(--font-num); }
.g-route.on { fill: var(--brand); }
@keyframes halo { 0% { opacity: .7; transform: scale(1); } 100% { opacity: 0; transform: scale(1.08); } }
.g-node .halo { transform-box: fill-box; transform-origin: center; }

.flow-list { display: none; list-style: none; margin: 0; padding: 0; }
.flow-list > li { position: relative; padding-left: 14px; border-left: 2px solid var(--line); margin-left: 13px; padding-bottom: 8px; }
.flow-list > li.fl-group { display: grid; gap: 6px; padding-top: 2px; }
.fl-par { font-size: 11px; color: var(--ink-3); font-weight: 600; }
.fl-item { display: flex; align-items: center; gap: 10px; width: 100%; text-align: left; border: 1px solid var(--line-2); background: var(--surface); border-radius: 11px; padding: 9px 12px; color: var(--ink); }
.fl-item b { display: block; font-size: 13.5px; font-weight: 600; }
.fl-item small { display: block; font: 500 11px var(--font-num); color: var(--ink-3); }
.fl-dot { flex: none; width: 24px; height: 24px; border-radius: 50%; display: grid; place-items: center; background: var(--surface-3); color: var(--ink-3); font: 600 11px var(--font-num); }
.fl-item.active { border-color: var(--brand); background: var(--brand-soft); }
.fl-item.active .fl-dot { background: var(--brand); color: var(--brand-ink); animation: pulse 1.4s var(--ease) infinite; }
.fl-item.done .fl-dot { background: var(--keep); color: #fff; }
.fl-item.degraded .fl-dot { background: var(--watch); color: #fff; }
.fl-item.failed { border-color: var(--replace); background: var(--replace-soft); }
.fl-item.failed .fl-dot { background: var(--replace); color: #fff; }
.fl-item.failed small { color: var(--ink-2); }
.g-node.failed text.s { fill: var(--ink-2); }
.fl-item.skipped { opacity: .45; }
.inspector { margin-top: 16px; padding-top: 16px; border-top: 1px solid var(--line); min-height: 120px; }
.insp-head { display: flex; justify-content: space-between; align-items: flex-start; gap: 12px; margin-bottom: 10px; }
.insp-head h3 { font-size: 16px; }
.insp-head p { color: var(--ink-3); font-size: 12.5px; margin-top: 2px; }
.kv { display: flex; flex-wrap: wrap; gap: 6px; margin: 6px 0 10px; }
.tag { display: inline-flex; align-items: center; gap: 6px; height: 24px; padding: 0 9px; border-radius: 7px; background: var(--surface-3); color: var(--ink-2); font-size: 12px; font-weight: 500; white-space: nowrap; }
.tag.brand { background: var(--brand-soft); color: var(--brand-2); }
.tag .i { width: 13px; height: 13px; }
.insp-out { font-size: 13px; color: var(--ink-2); display: grid; gap: 8px; }
.insp-out .k { font-weight: 600; color: var(--ink); }
pre.json { margin: 8px 0 0; max-height: 260px; overflow: auto; font: 12px/1.5 var(--font-num); background: var(--surface-2); border: 1px solid var(--line); border-radius: 10px; padding: 12px; color: var(--ink-2); white-space: pre-wrap; word-break: break-word; }

.trace-panel { display: flex; flex-direction: column; max-height: 620px; }
.trace { list-style: none; margin: 0 -6px; padding: 0; overflow: auto; flex: 1; min-height: 240px; scroll-behavior: smooth; }
.trace li {
  display: grid; grid-template-columns: 44px 22px minmax(0, 1fr); gap: 8px; align-items: start; padding: 7px 6px; border-radius: 8px; font-size: 12.5px;
  animation: rowIn .45s var(--ease) both;
}
@keyframes rowIn { from { opacity: 0; transform: translateY(4px); } to { opacity: 1; transform: none; } }
.trace li:hover { background: var(--surface-2); }
.trace .tt { font-family: var(--font-num); color: var(--ink-3); font-size: 11.5px; padding-top: 1px; }
.trace .ic { width: 22px; height: 22px; border-radius: 6px; display: grid; place-items: center; background: var(--surface-3); color: var(--ink-2); }
.trace .ic .i { width: 13px; height: 13px; }
.trace .tx { color: var(--ink-2); overflow-wrap: anywhere; }
.trace .tx b { color: var(--ink); font-weight: 600; }
.trace .tx code { font: 11.5px var(--font-num); background: var(--surface-3); padding: 1px 5px; border-radius: 5px; color: var(--ink); }
.trace .tx .q { display: block; color: var(--ink-3); font-size: 12px; margin-top: 2px; }
.trace li.k-stage .ic { background: var(--brand-soft); color: var(--brand-2); }
.trace li.k-done .ic { background: var(--keep-soft); color: var(--keep); }
.trace li.k-warn .ic { background: var(--watch-soft); color: var(--watch); }
.trace li.k-bad .ic { background: var(--replace-soft); color: var(--replace); }
.trace li.k-route .ic { background: var(--develop-soft); color: var(--develop); }
.trace li.v { display: none; }
.trace.verbose li.v { display: grid; }
.trace-empty { color: var(--ink-3); font-size: 13px; padding: 12px 6px; }

.lanes { display: grid; gap: 10px; }
.lane { display: grid; grid-template-columns: 150px minmax(0, 1fr) 84px; gap: 12px; align-items: center; font-size: 12.5px; }
.lane .nm { display: flex; flex-direction: column; }
.lane .nm b { font-weight: 600; font-size: 12.5px; }
.lane .nm span { color: var(--ink-3); font-size: 11px; }
.lane .bar { height: 10px; border-radius: 99px; background: var(--surface-3); overflow: hidden; position: relative; }
.lane .bar i { position: absolute; inset: 0 auto 0 0; border-radius: 99px; background: var(--brand); transition: width .6s var(--ease), background .3s; }
.lane .bar i.hot { background: var(--watch); }
.lane .bar i.cool { background: var(--replace); }
.lane .vl { font-family: var(--font-num); text-align: right; color: var(--ink-2); font-size: 12px; }
.lane.idle .bar { opacity: .5; }
.lane.idle .vl { color: var(--ink-3); }
.lane-legend { display: flex; flex-wrap: wrap; gap: 6px 16px; margin: 14px 0 0; font-size: 12px; color: var(--ink-3); }
.lane-legend i { display: inline-block; width: 18px; height: 6px; border-radius: 9px; margin-right: 6px; vertical-align: middle; }
.l-ok { background: var(--brand); } .l-hot { background: var(--watch); } .l-cool { background: var(--replace); }

/* -------------------------------------------------------------- insights */
.ins-head { position: sticky; top: var(--topbar-h); z-index: 15; display: flex; justify-content: space-between; align-items: center; gap: 12px; margin: -28px calc(-1 * max(20px, env(safe-area-inset-left))) 20px; padding: 12px max(20px, env(safe-area-inset-left)); background: color-mix(in oklab, var(--bg) 90%, transparent); backdrop-filter: blur(12px); -webkit-backdrop-filter: blur(12px); border-bottom: 1px solid var(--line); }
.tabs { display: flex; gap: 2px; overflow-x: auto; scrollbar-width: none; }
.tabs::-webkit-scrollbar { display: none; }
h1[tabindex="-1"]:focus { outline: none; }
.tabs.more-r { -webkit-mask-image: linear-gradient(90deg, #000 calc(100% - 44px), transparent); mask-image: linear-gradient(90deg, #000 calc(100% - 44px), transparent); }
.tabs.more-l { -webkit-mask-image: linear-gradient(90deg, transparent, #000 44px); mask-image: linear-gradient(90deg, transparent, #000 44px); }
.tabs.more-l.more-r { -webkit-mask-image: linear-gradient(90deg, transparent, #000 44px, #000 calc(100% - 44px), transparent); mask-image: linear-gradient(90deg, transparent, #000 44px, #000 calc(100% - 44px), transparent); }
.tabs button { border: 0; background: transparent; padding: 8px 12px; border-radius: 9px; color: var(--ink-2); font-weight: 500; white-space: nowrap; display: inline-flex; gap: 6px; align-items: center; }
.tabs button:hover { color: var(--ink); background: var(--surface-3); }
.tabs button[aria-selected="true"] { color: var(--ink); background: var(--surface); box-shadow: var(--shadow-1), inset 0 0 0 1px var(--line); }
.tabs .n { font-family: var(--font-num); font-size: 11px; color: var(--ink-3); }
.ins-actions { display: flex; gap: 8px; flex: none; }
.tab-body { animation: viewIn .45s var(--ease) both; }

.grid-2 { display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 16px; }
.grid-3 { display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 16px; }
.span-2 { grid-column: span 2; }
.stack { display: grid; gap: 16px; }

/* Brief */
.brief { display: grid; grid-template-columns: minmax(0, 1.6fr) minmax(300px, 1fr); gap: 16px; align-items: start; }
.brief .stack { align-content: start; }
#ins-body .whatif > .panel:first-child { position: sticky; top: calc(var(--topbar-h) + 76px); }
.brief-main { padding: 32px 34px; }
.brief-q { font-size: 13px; color: var(--ink-3); display: flex; gap: 8px; align-items: baseline; margin-bottom: 14px; }
.brief-q span { color: var(--ink-2); font-style: italic; }
.brief h2.headline { font-family: var(--font-display); font-weight: 400; font-size: clamp(30px, 3.4vw, 44px); line-height: 1.06; letter-spacing: -0.015em; max-width: 24ch; }
.brief .answer { font-size: 16.5px; line-height: 1.6; color: var(--ink); margin-top: 18px; max-width: 66ch; }
.brief .para { color: var(--ink-2); font-size: 14.5px; line-height: 1.65; margin-top: 12px; max-width: 70ch; }
.facts { display: grid; grid-template-columns: repeat(auto-fit, minmax(150px, 1fr)); border-top: 1px solid var(--line); margin-top: 26px; }
.facts div { padding: 14px 16px 2px 0; }
.facts dt { font-size: 12px; color: var(--ink-3); }
.facts dd { margin: 2px 0 0; font-family: var(--font-num); font-size: 22px; letter-spacing: -0.02em; font-weight: 500; }
.facts dd.long { font-size: 15px; font-family: var(--font-ui); letter-spacing: 0; line-height: 1.35; padding-top: 4px; }
.qa-seal { display: flex; gap: 12px; align-items: center; padding: 14px 16px; border-radius: 12px; background: var(--keep-soft); border: 1px solid var(--keep-line); color: var(--ink); }
.qa-seal.revise { background: var(--watch-soft); border-color: var(--watch-line); }
.qa-seal .i { color: var(--keep); width: 26px; height: 26px; }
.qa-seal.revise .i { color: var(--watch); }
.qa-seal b { display: block; font-size: 13.5px; }
.qa-seal span { font-size: 12.5px; color: var(--ink-2); }

.dist { display: flex; height: 34px; border-radius: 10px; overflow: hidden; gap: 3px; margin: 10px 0 12px; }
.dist span { display: grid; place-items: center; color: #fff; text-shadow: 0 1px 1px rgba(0,0,0,.25); font: 600 12px var(--font-num); min-width: 26px; transition: flex-grow .6s var(--ease); }
.legend { display: flex; flex-wrap: wrap; gap: 6px 16px; font-size: 12.5px; color: var(--ink-2); }
.legend i { display: inline-block; width: 9px; height: 9px; border-radius: 3px; margin-right: 6px; }
.bg-keep { background: var(--keep); } .bg-watch { background: var(--watch); } .bg-develop { background: var(--develop); } .bg-replace { background: var(--replace); }

.actions-list { list-style: none; margin: 0; padding: 0; display: grid; gap: 0; }
.actions-list li { display: grid; grid-template-columns: 28px minmax(0, 1fr); gap: 12px; padding: 14px 0; border-top: 1px solid var(--line); }
.actions-list li:first-child { border-top: 0; padding-top: 4px; }
.actions-list .n { font-family: var(--font-num); color: var(--brand); font-weight: 600; font-size: 13px; padding-top: 1px; }
.actions-list b { font-weight: 600; display: block; }
.actions-list .sub { color: var(--ink-3); font-size: 12.5px; margin-top: 3px; display: flex; flex-wrap: wrap; gap: 4px 14px; }

.decision { display: inline-flex; align-items: center; gap: 6px; height: 24px; padding: 0 10px; border-radius: 99px; font-size: 12px; font-weight: 600; border: 1px solid transparent; white-space: nowrap; }
.decision::before { content: ""; width: 7px; height: 7px; border-radius: 50%; background: currentColor; }
.decision.Keep { background: var(--keep-soft); color: var(--keep); border-color: var(--keep-line); }
.decision.Watch { background: var(--watch-soft); color: var(--watch); border-color: var(--watch-line); }
.decision.Develop { background: var(--develop-soft); color: var(--develop); border-color: var(--develop-line); }
.decision.Replace { background: var(--replace-soft); color: var(--replace); border-color: var(--replace-line); }
.src-tag { font-size: 10.5px; color: var(--ink-2); font-weight: 500; margin-left: 4px; }

/* Vendors table */
.toolbar { display: flex; gap: 10px; flex-wrap: wrap; align-items: center; margin-bottom: 12px; }
.search { position: relative; flex: 1 1 220px; max-width: 340px; }
.search .i { position: absolute; left: 11px; top: 50%; transform: translateY(-50%); color: var(--ink-3); }
.search input { padding-left: 36px; }
input[type="search"], select { height: 38px; }
input[type="search"]::-webkit-search-cancel-button { filter: grayscale(1); }
select { width: auto; padding-right: 30px; appearance: none; background-image: linear-gradient(45deg, transparent 50%, var(--ink-3) 50%), linear-gradient(135deg, var(--ink-3) 50%, transparent 50%); background-position: calc(100% - 15px) 50%, calc(100% - 10px) 50%; background-size: 5px 5px; background-repeat: no-repeat; }
.vt-wrap { overflow-x: auto; margin: 0 -20px; padding: 0 20px; }
table.vt { width: 100%; border-collapse: separate; border-spacing: 0; font-size: 13px; min-width: 900px; }
table.vt th { text-align: left; font-size: 12px; font-weight: 550; color: var(--ink-3); padding: 8px 10px; border-bottom: 1px solid var(--line); white-space: nowrap; }
table.vt th button { all: unset; cursor: pointer; display: inline-flex; gap: 4px; align-items: center; }
table.vt th button:hover, table.vt th[aria-sort] button { color: var(--ink); }
table.vt th button:focus-visible { outline: 2px solid var(--focus); outline-offset: 2px; border-radius: 4px; }
table.vt td { padding: 10px; border-bottom: 1px solid var(--line); vertical-align: middle; }
table.vt tbody tr { cursor: pointer; transition: background .15s; }
table.vt tbody tr:hover { background: var(--surface-2); }
table.vt tbody tr:focus-visible { outline: 2px solid var(--focus); outline-offset: -2px; }
table.vt .vn b { display: block; font-weight: 600; }
table.vt .vn span { font-size: 12px; color: var(--ink-3); }
table.vt td.n { font-family: var(--font-num); font-variant-numeric: tabular-nums; white-space: nowrap; }
table.vt .otd { display: flex; align-items: center; gap: 8px; }
.scorebar { display: flex; align-items: center; gap: 8px; }
.scorebar .b { width: 64px; height: 6px; border-radius: 99px; background: var(--surface-3); overflow: hidden; }
.scorebar .b i { display: block; height: 100%; border-radius: 99px; background: var(--ink-2); }
.riskdot { display: inline-flex; align-items: center; gap: 7px; }
.riskdot::before { content: ""; width: 9px; height: 9px; border-radius: 50%; background: var(--rc, var(--ink-3)); }
.spark { width: 84px; height: 24px; display: block; }
.spark path { fill: none; stroke: var(--ink-2); stroke-width: 1.5; }
.spark .end { fill: var(--ink); }
.delta-up { color: var(--keep); } .delta-down { color: var(--replace); }
.empty { padding: 40px 20px; text-align: center; color: var(--ink-3); }
.empty b { display: block; color: var(--ink); font-size: 15px; margin-bottom: 4px; }

.chart { width: 100%; height: 380px; }
.chart.tall { height: 460px; }
.chart.short { height: 260px; }

/* Market */
.signals { display: grid; gap: 10px; }
.signal { display: grid; grid-template-columns: 54px minmax(0, 1fr); gap: 14px; padding: 14px; border-radius: 12px; border: 1px solid var(--line); background: var(--surface-2); }
.signal .score { font-family: var(--font-num); font-size: 22px; font-weight: 500; text-align: center; line-height: 1; padding-top: 4px; }
.signal .score small { display: block; font-size: 10.5px; color: var(--ink-3); margin-top: 6px; font-family: var(--font-ui); }
.signal .sdot { display: block; width: 10px; height: 10px; border-radius: 50%; margin: 0 auto 6px; }
.checks li.as-btn { padding: 0; display: block; }
.row-btn { all: unset; box-sizing: border-box; display: grid; grid-template-columns: 22px minmax(0, 1fr); gap: 10px; width: 100%; padding: 10px 12px; border-radius: 10px; cursor: pointer; }
.row-btn:hover { background: var(--surface-3); }
.row-btn:focus-visible { outline: 2px solid var(--focus); outline-offset: -2px; }
.checks .unsupported .row-btn .i { color: var(--replace); } .checks .corrected .row-btn .i { color: var(--watch); }
.signal h3 { font-size: 14px; display: flex; flex-wrap: wrap; gap: 8px; align-items: center; }
.signal .hl { margin-top: 6px; color: var(--ink); font-size: 13.5px; }
.signal .im { color: var(--ink-2); font-size: 13px; margin-top: 4px; }
.signal .src { margin-top: 8px; font-size: 12px; color: var(--ink-3); display: flex; gap: 10px; flex-wrap: wrap; align-items: center; }
.signal .src a { display: inline-flex; gap: 4px; align-items: center; }
.signal .src .i { width: 12px; height: 12px; }
.dir { font-size: 11.5px; font-weight: 600; padding: 2px 8px; border-radius: 99px; }
.dir.rising { background: var(--replace-soft); color: var(--replace); }
.dir.stable { background: var(--surface-3); color: var(--ink-2); }
.dir.easing { background: var(--keep-soft); color: var(--keep); }
.sources { list-style: none; padding: 0; margin: 0; display: grid; gap: 2px; font-size: 12.5px; }
.sources li { padding: 8px 0; border-top: 1px solid var(--line); display: grid; gap: 2px; }
.sources li:first-child { border-top: 0; }
.sources a { color: var(--ink); text-decoration: none; font-weight: 500; }
.sources a:hover { color: var(--brand); text-decoration: underline; }
.sources span { color: var(--ink-3); font-size: 12px; }
.more { margin-top: 10px; }

/* Plan */
.plan { padding: 22px; }
.plan + .plan { margin-top: 16px; }
.plan-head { display: flex; justify-content: space-between; gap: 16px; flex-wrap: wrap; margin-bottom: 16px; }
.plan-head h2 { font-size: 18px; }
.plan-head p { color: var(--ink-2); font-size: 13px; margin-top: 4px; max-width: 70ch; }
.shift { display: flex; flex-wrap: wrap; gap: 10px; align-items: center; margin-bottom: 18px; }
.shift .from, .shift .to { padding: 8px 12px; border-radius: 10px; border: 1px solid var(--line); font-size: 13px; background: var(--surface-2); }
.shift .from { border-color: var(--replace-line); background: var(--replace-soft); }
.shift .to { border-color: var(--keep-line); background: var(--keep-soft); }
.shift .to b, .shift .from b { font-weight: 600; }
.shift .pct { font-family: var(--font-num); font-weight: 600; color: var(--keep); margin-left: 6px; }
.phases { display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 0; border: 1px solid var(--line); border-radius: 12px; overflow: hidden; }
.phase { padding: 14px 16px; border-left: 1px solid var(--line); }
.phase:first-child { border-left: 0; }
.phase h3, .phase h4 { font-size: 12.5px; color: var(--ink-3); font-weight: 600; margin-bottom: 8px; font-family: var(--font-num); }
.phase ul { margin: 0; padding-left: 16px; display: grid; gap: 6px; font-size: 13px; color: var(--ink-2); }
.impact { margin-top: 14px; font-size: 13px; color: var(--ink-2); }
.impact b { color: var(--ink); }

/* What-if */
.whatif { display: grid; grid-template-columns: minmax(280px, .8fr) minmax(0, 1.6fr); gap: 16px; align-items: start; }
.whatif .weights { gap: 14px; }
.wf-table td.mv { font-family: var(--font-num); }
.flip { font-size: 11.5px; font-weight: 600; color: var(--watch); }

/* Audit */
.checks { list-style: none; margin: 0; padding: 0; display: grid; gap: 8px; }
.checks li { display: grid; grid-template-columns: 22px minmax(0, 1fr); gap: 10px; padding: 10px 12px; border-radius: 10px; background: var(--surface-2); border: 1px solid var(--line); font-size: 13px; }
.checks li .i { width: 18px; height: 18px; margin-top: 1px; }
.checks .verified .i { color: var(--keep); } .checks .corrected .i { color: var(--watch); } .checks .unsupported .i { color: var(--replace); }
.checks b { font-weight: 600; }
.checks .note { color: var(--ink-2); display: block; margin-top: 2px; }
table.usage { width: 100%; border-collapse: collapse; font-size: 13px; }
table.usage th { text-align: left; color: var(--ink-3); font-weight: 550; font-size: 12px; padding: 6px 8px; border-bottom: 1px solid var(--line); }
table.usage td { padding: 8px; border-bottom: 1px solid var(--line); font-family: var(--font-num); }
table.usage td:first-child { font-family: var(--font-ui); font-weight: 550; }

/* Drawer */
.drawer-scrim { position: fixed; inset: 0; background: rgba(10, 12, 18, .32); z-index: 40; animation: fade .3s var(--ease); }
.drawer { position: fixed; top: 0; right: 0; bottom: 0; width: min(680px, 100vw); background: var(--surface); z-index: 41; box-shadow: var(--shadow-3); border-left: 1px solid var(--line); animation: slideIn .45s var(--ease); display: flex; }
.drawer-inner { overflow: auto; padding: 26px 28px 40px; flex: 1; }
@keyframes slideIn { from { transform: translateX(40px); opacity: 0; } to { transform: none; opacity: 1; } }
@keyframes fade { from { opacity: 0; } to { opacity: 1; } }
.dw-head { display: flex; justify-content: space-between; gap: 16px; align-items: flex-start; margin-bottom: 18px; }
.dw-head h2 { font-family: var(--font-display); font-weight: 400; font-size: 34px; line-height: 1.05; }
.dw-head p { color: var(--ink-3); margin-top: 6px; font-size: 13px; }
.dw-sec { margin-top: 22px; }
.dw-sec h3 { font-size: 13px; color: var(--ink-3); font-weight: 600; margin-bottom: 10px; }
.pillars { display: grid; gap: 8px; }
.pillar { display: grid; grid-template-columns: 120px minmax(0, 1fr) 40px; gap: 10px; align-items: center; font-size: 13px; }
.pillar .b { height: 8px; background: var(--surface-3); border-radius: 99px; overflow: hidden; }
.pillar .b i { display: block; height: 100%; border-radius: 99px; }
.pillar .v { font-family: var(--font-num); text-align: right; }
.kpis { display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 0; border: 1px solid var(--line); border-radius: 12px; overflow: hidden; }
.kpis div { padding: 10px 12px; border-top: 1px solid var(--line); border-left: 1px solid var(--line); }
.kpis div:nth-child(3n+1) { border-left: 0; }
.kpis div:nth-child(-n+3) { border-top: 0; }
.kpis dt { font-size: 11.5px; color: var(--ink-3); }
.kpis dd { margin: 2px 0 0; font-family: var(--font-num); font-size: 16px; }
.rationale { font-size: 14px; line-height: 1.6; color: var(--ink); }
.rationale + ul { margin: 10px 0 0; padding-left: 18px; color: var(--ink-2); font-size: 13.5px; display: grid; gap: 4px; }

/* Copilot */
.copilot-fab {
  position: fixed; right: max(20px, env(safe-area-inset-right)); bottom: 22px; z-index: 35; height: 48px; padding: 0 18px 0 14px; border-radius: 99px;
  display: inline-flex; align-items: center; gap: 8px; border: 0; background: var(--ink); color: var(--bg); font-weight: 600; box-shadow: var(--shadow-3);
  transition: transform .2s var(--ease);
}
.copilot-fab:hover { transform: translateY(-2px); }
.copilot {
  position: fixed; right: max(16px, env(safe-area-inset-right)); bottom: 16px; z-index: 36; width: min(420px, calc(100vw - 32px)); height: min(640px, calc(100dvh - 100px));
  background: var(--surface); border: 1px solid var(--line); border-radius: 18px; box-shadow: var(--shadow-3); display: flex; flex-direction: column; overflow: hidden;
  animation: popIn .35s var(--ease);
}
@keyframes popIn { from { opacity: 0; transform: translateY(12px) scale(.98); } to { opacity: 1; transform: none; } }
.cp-head { display: flex; justify-content: space-between; align-items: center; padding: 14px 14px 12px 18px; border-bottom: 1px solid var(--line); }
.cp-head strong { display: block; font-size: 14px; }
.cp-head span { font-size: 12px; color: var(--ink-3); }
.cp-log { list-style: none; margin: 0; padding: 16px; overflow: auto; flex: 1; display: flex; flex-direction: column; gap: 12px; }
.cp-log li { max-width: 88%; font-size: 13.5px; line-height: 1.55; animation: rowIn .35s var(--ease) both; }
.cp-log li.u { align-self: flex-end; background: var(--brand); color: var(--brand-ink); padding: 9px 13px; border-radius: 14px 14px 4px 14px; }
.cp-log li.a { align-self: flex-start; background: var(--surface-3); padding: 10px 13px; border-radius: 14px 14px 14px 4px; white-space: pre-wrap; }
.cp-log li.a.md { white-space: normal; }
.cp-log li.a ul { margin: 4px 0; padding-left: 18px; }
.cp-log li.a code.tbl { font: 11.5px var(--font-num); white-space: pre; display: inline-block; max-width: 100%; overflow-x: auto; }
.cp-log li.a .via { display: block; font-size: 11px; color: var(--ink-3); margin-top: 6px; white-space: normal; }
.cp-log li.typing { display: inline-flex; gap: 4px; }
.cp-log li.typing i { width: 6px; height: 6px; border-radius: 50%; background: var(--ink-3); animation: bounce 1s infinite; }
.cp-log li.typing i:nth-child(2) { animation-delay: .15s; } .cp-log li.typing i:nth-child(3) { animation-delay: .3s; }
@keyframes bounce { 0%, 60%, 100% { transform: none; opacity: .5; } 30% { transform: translateY(-4px); opacity: 1; } }
.cp-suggest { display: flex; gap: 6px; flex-wrap: wrap; padding: 0 16px 10px; }
.cp-suggest button { border: 1px solid var(--line-2); background: var(--surface); border-radius: 99px; padding: 5px 10px; font-size: 12px; color: var(--ink-2); }
.cp-suggest button:hover { color: var(--ink); border-color: var(--brand-line); }
.cp-form { display: flex; gap: 8px; padding: 12px; border-top: 1px solid var(--line); }
.cp-form input { flex: 1; }

/* Toasts */
.toast-zone { position: fixed; left: 50%; bottom: 24px; transform: translateX(-50%); z-index: 60; display: grid; gap: 8px; width: min(440px, calc(100vw - 32px)); pointer-events: none; transition: bottom .2s var(--ease); }
.toast { pointer-events: auto; background: var(--ink); color: var(--bg); padding: 12px 16px; border-radius: 12px; box-shadow: var(--shadow-3); font-size: 13.5px; display: flex; gap: 10px; align-items: flex-start; animation: popIn .3s var(--ease); }
.toast.err { background: #B3261E; color: #fff; }

@keyframes pulse { 0% { box-shadow: 0 0 0 0 color-mix(in oklab, var(--brand) 50%, transparent); } 100% { box-shadow: 0 0 0 8px transparent; } }
@keyframes spin { to { transform: rotate(360deg); } }

/* ---------------------------------------------------------- responsive */
@media (max-width: 1240px) {
  .cfg-grid { grid-template-columns: minmax(0, 1fr) minmax(0, 1fr); }
  #panel-preview { grid-column: span 2; position: static; max-height: none; }
  .rank-list { columns: 2; column-gap: 16px; }
  .rank-list li { break-inside: avoid; }
}
@media (max-width: 1060px) {
  table.vlist { min-width: 0; }
  table.vlist .c-defect, table.vlist .c-pv { display: none; }
  #ins-body .whatif > .panel:first-child { position: static; }
  .run-grid { grid-template-columns: 1fr; }
  .graph-panel { grid-row: auto; }
  .brief, .whatif { grid-template-columns: 1fr; }
}
@media (max-width: 920px) {
  .status-pill .lx { display: none; }
  table.vlist .c-rank, table.vlist .c-spend_total { display: none; }
}
@media (max-width: 860px) {
  .topbar { grid-template-columns: auto 1fr auto; }
  .steps { justify-self: center; }
  .cfg-grid, .grid-2, .grid-3 { grid-template-columns: 1fr; }
  #panel-preview, .span-2 { grid-column: auto; }
  .rank-list { columns: 1; }
  .field-row, .field-row.three { grid-template-columns: 1fr; }
  .view-head { flex-direction: column; align-items: flex-start; }
  .phases { grid-template-columns: 1fr; }
  .phase { border-left: 0; border-top: 1px solid var(--line); }
  .phase:first-child { border-top: 0; }
  .brief-main { padding: 24px 20px; }
  .kpis { grid-template-columns: repeat(2, minmax(0, 1fr)); }
  .kpis div:nth-child(3n+1) { border-left: 1px solid var(--line); }
  .kpis div:nth-child(2n+1) { border-left: 0; }
  .kpis div:nth-child(3) { border-top: 1px solid var(--line); }
}
@media (max-width: 860px) {
  .graph-wrap { display: none; }
  .flow-list { display: block; }
}
@media (max-width: 720px) {
  /* vendor table becomes a stacked list: name + decision, then score / risk / on-time */
  .vlist-wrap { overflow: visible; margin: 0; padding: 0; }
  table.vlist, table.vlist thead, table.vlist tbody { display: block; }
  table.vlist thead tr { display: flex; flex-wrap: wrap; gap: 6px; padding: 2px 0 12px; border-bottom: 1px solid var(--line); }
  table.vlist thead tr::before { content: 'Sort'; font-size: 12px; color: var(--ink-3); align-self: center; margin-right: 2px; }
  table.vlist th { border: 1px solid var(--line); border-radius: 99px; padding: 5px 11px; background: var(--surface-2); }
  table.vlist th[aria-sort] { border-color: var(--ink-3); }
  table.vlist tbody tr { display: grid; grid-template-columns: minmax(0, 1fr) auto auto; grid-template-areas: "vn vn dec" "score risk otd"; gap: 8px 14px; padding: 12px 4px; border-bottom: 1px solid var(--line); align-items: center; }
  table.vlist td { padding: 0; border: 0; }
  table.vlist td.c-vendor { grid-area: vn; }
  table.vlist td.c-decision { grid-area: dec; justify-self: end; }
  table.vlist td.c-composite { grid-area: score; }
  table.vlist td.c-combined_risk { grid-area: risk; }
  table.vlist td.c-otd { grid-area: otd; }
  table.vlist td.c-combined_risk::before, table.vlist td.c-otd::before { content: attr(data-label); font: 500 11.5px var(--font-ui); color: var(--ink-3); margin-right: 6px; }
  table.vlist td.c-otd { display: flex; align-items: center; }
  table.vlist td.c-otd svg { display: none; }
  table.vlist tbody tr td[colspan] { grid-column: 1 / -1; }
  .steps a span { position: absolute; width: 1px; height: 1px; overflow: hidden; clip: rect(0 0 0 0); white-space: nowrap; }
  .steps a { padding: 7px 11px; }
  .status-pill .label { display: none; }
  .status-pill { padding: 0 10px; }
}
@media (max-width: 560px) {
  main { padding-top: 20px; }
  .run-side { align-items: stretch; width: 100%; }
  .brand-word { display: none; }
  .panel { padding: 16px; border-radius: 12px; }
  .weight { grid-template-columns: 112px 1fr 40px; gap: 8px; }
  .weight label { font-size: 12.5px; }
  .ins-actions .btn { width: 32px; padding: 0; }
  .ins-head { gap: 8px; }
  .launch-bar { padding: 8px; bottom: 10px; gap: 8px; }
  .launch-info { display: none; }
  .launch-actions { flex: 1; flex-wrap: nowrap; }
  .launch-actions .btn.lg { flex: 1; font-size: 14px; padding: 0 14px; }
  #run-done-bar { flex-direction: column; align-items: stretch; padding: 12px; }
  #run-done-bar .launch-info { display: block; font-size: 13px; padding: 0 4px; }
  #btn-saved span { position: absolute; width: 1px; height: 1px; overflow: hidden; clip: rect(0 0 0 0); white-space: nowrap; }
  #btn-saved { width: 48px; height: 48px; padding: 0; }
  .lane { grid-template-columns: 124px minmax(0, 1fr) 70px; gap: 8px; }
  .lane .nm span { display: none; }
  .run-stats div { min-width: 0; flex: 1 1 30%; padding: 6px 10px; }
  .copilot-fab span { position: absolute; width: 1px; height: 1px; overflow: hidden; clip: rect(0 0 0 0); white-space: nowrap; }
  .copilot-fab { width: 52px; padding: 0; justify-content: center; }
  .ins-actions .btn span { position: absolute; width: 1px; height: 1px; overflow: hidden; clip: rect(0 0 0 0); white-space: nowrap; }
  .drawer-inner { padding: 20px 18px 40px; }
  .dw-head h2 { font-size: 28px; }
  .pillar { grid-template-columns: 96px minmax(0, 1fr) 36px; }
}

@media (prefers-reduced-motion: reduce) {
  *, *::before, *::after { animation-duration: .001ms !important; animation-iteration-count: 1 !important; transition-duration: .001ms !important; scroll-behavior: auto !important; }
  .g-edge.live { stroke-dasharray: none; }
}
@media print {
  .topbar, .launch-bar, .copilot-fab, .copilot, .ins-head { display: none !important; }
  body { background: #fff; }
}

/* =====================================================================
   OBSERVATORY LAYER
   A living sky behind the app (fx.js), floating instrument chrome, cinematic
   view changes, a decision orbit, a command palette and tactile controls.
   Decision colours keep their single meaning; motion stops under
   prefers-reduced-motion.
   ===================================================================== */
:root { --spring: linear(0, .009, .035 2.1%, .141, .281 6.7%, .723 12.9%, .938 16.7%, 1.017, 1.077, 1.121, 1.149 24.3%, 1.159, 1.163, 1.161, 1.154 29.9%, 1.129 32.8%, 1.051 39.6%, 1.017 43.1%, .991, .977 51%, .974 53.8%, .975 57.1%, .997 69.8%, 1.003 76.9%, 1); --glass: color-mix(in oklab, var(--surface) 90%, transparent); }
:root[data-theme="dark"] { --bg: #07080D; --glass: color-mix(in oklab, var(--surface) 86%, transparent); }
@media (prefers-color-scheme: dark) { :root:not([data-theme="light"]) { --bg: #07080D; --glass: color-mix(in oklab, var(--surface) 86%, transparent); } }

body {
  background:
    radial-gradient(1200px 700px at 85% -10%, color-mix(in oklab, var(--brand) 13%, transparent), transparent 70%),
    radial-gradient(900px 600px at -10% 30%, color-mix(in oklab, var(--develop) 9%, transparent), transparent 70%),
    radial-gradient(800px 600px at 60% 110%, color-mix(in oklab, var(--keep) 7%, transparent), transparent 70%),
    var(--bg);
  background-attachment: fixed;
}
.sky { position: fixed; inset: 0; width: 100vw; height: 100vh; z-index: 0; pointer-events: none; }
main { position: relative; z-index: 1; }

/* floating instrument bar */
.topbar {
  top: 10px; margin: 10px auto 0; width: min(1480px, calc(100% - 24px)); border-radius: 18px; border: 1px solid var(--line);
  background: color-mix(in oklab, var(--surface) 72%, transparent); box-shadow: var(--shadow-2); padding: 0 12px 0 16px;
  view-transition-name: topbar;
}
.steps { position: relative; background: color-mix(in oklab, var(--surface-3) 80%, transparent); }
.steps a { position: relative; z-index: 1; }
.steps a[aria-current="page"] { background: transparent; box-shadow: none; }
.steps-ind {
  position: absolute; z-index: 0; top: 4px; bottom: 4px; left: 0; width: 0; border-radius: 9px; background: var(--surface);
  box-shadow: var(--shadow-1), 0 0 0 1px var(--line), 0 6px 20px -8px color-mix(in oklab, var(--brand) 70%, transparent);
  transition: transform .6s var(--spring), width .6s var(--spring);
}
.steps-ind::after { content: ""; position: absolute; left: 22%; right: 22%; bottom: -1px; height: 2px; border-radius: 2px; background: var(--brand); box-shadow: 0 0 10px var(--brand); }
.pal-btn {
  display: inline-flex; align-items: center; gap: 8px; height: 36px; padding: 0 8px 0 11px; border-radius: 10px; border: 1px solid var(--line);
  background: var(--surface); color: var(--ink-2); font-size: 13px; transition: border-color .15s, color .15s;
}
.pal-btn:hover { color: var(--ink); border-color: var(--brand-line); }
kbd { font: 500 11px var(--font-num); padding: 2px 6px; border-radius: 6px; border: 1px solid var(--line-2); background: var(--surface-2); color: var(--ink-2); box-shadow: 0 1px 0 var(--line-2); }
.ins-head { top: calc(var(--topbar-h) + 18px); margin: -14px 0 20px; padding: 8px 10px; border: 1px solid var(--line); border-radius: 16px; background: color-mix(in oklab, var(--surface) 78%, transparent); box-shadow: var(--shadow-1); }
.tabs button[aria-selected="true"] { box-shadow: var(--shadow-1), inset 0 0 0 1px var(--line), inset 0 -2px 0 var(--brand); }

/* glass panels with a cursor spotlight */
.panel, .launch-bar { background: var(--glass); backdrop-filter: blur(14px) saturate(1.2); -webkit-backdrop-filter: blur(14px) saturate(1.2); box-shadow: var(--shadow-1); }
.panel { position: relative; }
.panel::after, .launch-bar::after {
  content: ""; position: absolute; inset: -1px; border-radius: inherit; pointer-events: none; opacity: 0; transition: opacity .35s;
  background: radial-gradient(420px circle at var(--mx, 50%) var(--my, 50%), color-mix(in oklab, var(--brand) 9%, transparent), transparent 60%);
}
.panel::before {
  content: ""; position: absolute; inset: -1px; border-radius: inherit; pointer-events: none; padding: 1px; opacity: 0; transition: opacity .35s;
  background: radial-gradient(260px circle at var(--mx, 50%) var(--my, 50%), color-mix(in oklab, var(--brand) 70%, transparent), transparent 70%);
  -webkit-mask: linear-gradient(#000 0 0) content-box, linear-gradient(#000 0 0); -webkit-mask-composite: xor; mask: linear-gradient(#000 0 0) content-box exclude, linear-gradient(#000 0 0);
}
.panel.lit::after, .panel.lit::before, .launch-bar.lit::after { opacity: 1; }

/* configure hero: the question is the stage */
.cfg-hero { display: grid; grid-template-columns: minmax(0, 1fr) 300px; gap: 28px; align-items: center; margin: 18px 0 26px; }
.cfg-hero .display { font-size: clamp(42px, 5.6vw, 78px); line-height: .98; letter-spacing: -0.025em; }
.cfg-hero .display em { font-style: italic; color: var(--brand); }
.kicker { display: inline-flex; align-items: center; gap: 8px; font: 500 12px var(--font-num); letter-spacing: .04em; text-transform: uppercase; color: var(--ink-2); margin-bottom: 14px; padding: 5px 12px 5px 10px; border: 1px solid var(--line); border-radius: 99px; background: var(--glass); }
.kdot { width: 7px; height: 7px; border-radius: 50%; background: var(--keep); box-shadow: 0 0 0 3px var(--keep-soft); animation: pulse 2s var(--ease) infinite; }
@property --ang { syntax: "<angle>"; inherits: false; initial-value: 0deg; }
@keyframes spinAng { to { --ang: 360deg; } }
.ask { position: relative; margin-top: 22px; border-radius: 18px; padding: 1.5px; background: conic-gradient(from var(--ang), var(--brand), var(--develop), var(--keep), var(--watch), var(--brand)); animation: spinAng 8s linear infinite; box-shadow: 0 18px 50px -24px color-mix(in oklab, var(--brand) 70%, transparent); }
.ask textarea {
  display: block; border: 0; border-radius: 16.5px; background: var(--surface); min-height: 0; padding: 18px 22px; resize: none;
  font: 400 clamp(19px, 1.9vw, 25px)/1.35 var(--font-display); letter-spacing: -0.005em; color: var(--ink);
}
.ask textarea:hover { border: 0; }
.ask textarea:focus { box-shadow: none; background: var(--surface); }
.ask:focus-within { outline: 2px solid var(--focus); outline-offset: 3px; }
.ask-glow { position: absolute; inset: 10% 6% -18%; z-index: -1; filter: blur(30px); background: conic-gradient(var(--brand), var(--develop), var(--keep), var(--brand)); opacity: .22; border-radius: 30px; pointer-events: none; }
.cfg-hero .prompt-chips { margin-top: 14px; }
.cfg-hero .prompt-chips button { background: var(--glass); }

.crew-ring { width: 300px; height: 300px; overflow: visible; }
.cr-orbit { fill: none; stroke: var(--line-2); stroke-dasharray: 2 6; transform-origin: center; transform-box: fill-box; animation: crSpin 60s linear infinite; }
.cr-orbit.in { animation-direction: reverse; animation-duration: 40s; }
@keyframes crSpin { to { transform: rotate(360deg); } }
.cr-link { stroke: var(--brand); stroke-width: 1; opacity: .12; animation: crCall 5s var(--ease) infinite; animation-delay: calc(var(--i) * .5s); }
.cr-node circle { fill: var(--surface); stroke: var(--line-2); stroke-width: 1.2; transition: fill .3s, stroke .3s; }
.cr-node text { font: 600 10px var(--font-num); fill: var(--ink-2); }
.cr-node { animation: crNode 5s var(--ease) infinite; animation-delay: calc(var(--i) * .5s); }
@keyframes crCall { 0%, 100% { opacity: .12; } 8% { opacity: .8; } 22% { opacity: .12; } }
@keyframes crNode { 0%, 100% { filter: none; } 8% { filter: drop-shadow(0 0 6px var(--brand)); } 22% { filter: none; } }
.cr-shield { fill: var(--brand); filter: drop-shadow(0 6px 18px color-mix(in oklab, var(--brand) 60%, transparent)); }
.cr-tick { fill: none; stroke: var(--brand-ink); stroke-width: 3; stroke-linecap: round; stroke-linejoin: round; }
body:has(#btn-run:hover) .cr-node circle, body:has(#btn-run:focus-visible) .cr-node circle { fill: var(--brand); stroke: var(--brand); }
body:has(#btn-run:hover) .cr-node text, body:has(#btn-run:focus-visible) .cr-node text { fill: var(--brand-ink); }
body:has(#btn-run:hover) .cr-link { opacity: .7; animation: none; }

/* living scorecard: each row carries a tide of its score in its tier colour */
.rank-list li { position: relative; isolation: isolate; }
.rank-list li::before { content: ""; position: absolute; inset: 3px 4px; z-index: -1; border-radius: 8px; background: linear-gradient(90deg, color-mix(in oklab, var(--tc, var(--brand)) 13%, transparent), transparent var(--sc, 50%)); }

/* tactile controls */
.pressed { transform: scale(.94) !important; transition: transform .08s !important; }
.sprung { animation: sprung .6s var(--spring); }
@keyframes sprung { from { transform: scale(.94); } to { transform: none; } }
.btn.primary.lg { transform: translate(var(--mgx, 0), var(--mgy, 0)); transition: transform .35s var(--spring), background .15s, box-shadow .2s; position: relative; overflow: hidden; isolation: isolate; }
.btn.primary.lg::before { content: ""; position: absolute; inset: 0; z-index: -1; background: linear-gradient(105deg, transparent 35%, rgba(255, 255, 255, .35) 50%, transparent 65%); transform: translateX(-120%); animation: sheen 4.5s var(--ease) infinite 1.5s; }
@keyframes sheen { 0%, 70% { transform: translateX(-120%); } 100% { transform: translateX(120%); } }
#btn-run:not([disabled]) { box-shadow: 0 10px 34px -10px color-mix(in oklab, var(--brand) 85%, transparent), 0 0 0 4px color-mix(in oklab, var(--brand) 14%, transparent); }

/* agents: mission theatre */
.graph-panel { background:
  radial-gradient(600px 400px at 50% 30%, color-mix(in oklab, var(--brand) 8%, transparent), transparent 70%),
  radial-gradient(circle, color-mix(in oklab, var(--ink) 10%, transparent) 1px, transparent 1.2px) 0 0 / 22px 22px,
  var(--glass); }
.g-edge.live { stroke-width: 2.2; filter: drop-shadow(0 0 4px color-mix(in oklab, var(--brand) 70%, transparent)); }
.g-packet { fill: var(--brand); filter: drop-shadow(0 0 5px var(--brand)); pointer-events: none; }
.g-node.active rect.card { filter: drop-shadow(0 0 14px color-mix(in oklab, var(--brand) 45%, transparent)); }
.g-node.done .badge { filter: drop-shadow(0 0 5px color-mix(in oklab, var(--keep) 70%, transparent)); }
.run-stats div { position: relative; background: var(--glass); backdrop-filter: blur(10px); -webkit-backdrop-filter: blur(10px); }
.run-stats dd { font-size: 22px; transition: color .4s; }
:root[data-sky="run"] .run-stats dd { color: var(--brand); text-shadow: 0 0 18px color-mix(in oklab, var(--brand) 50%, transparent); }

/* decision orbit */
.orbit { position: relative; width: min(100%, 380px); margin: -4px auto 10px; touch-action: none; user-select: none; cursor: grab; }
.orbit.grab { cursor: grabbing; }
.orbit svg { display: block; width: 100%; height: auto; overflow: visible; }
.o-ring { fill: none; stroke-width: 1.2; stroke-dasharray: 3 5; opacity: .6; }
.o-ring.keep { stroke: var(--keep); } .o-ring.develop { stroke: var(--develop); } .o-ring.watch { stroke: var(--watch); } .o-ring.replace { stroke: var(--replace); }
.o-lbl { font: 600 9.5px var(--font-num); letter-spacing: .06em; text-transform: uppercase; fill: var(--ink-2); paint-order: stroke; stroke: var(--surface); stroke-width: 4px; }
.o-core { fill: var(--surface); stroke: var(--brand); stroke-width: 1.5; filter: drop-shadow(0 0 16px color-mix(in oklab, var(--brand) 55%, transparent)); }
.o-n { font: 400 24px var(--font-display); fill: var(--ink); }
.o-nl { font: 500 8.5px var(--font-num); fill: var(--ink-2); text-transform: uppercase; letter-spacing: .08em; }
.o-star { cursor: pointer; }
.o-hit { fill: transparent; }
.o-dot { stroke: var(--surface); stroke-width: 1.5; transition: transform .3s var(--spring); transform-box: fill-box; transform-origin: center; }
.o-star.keep .o-dot { fill: var(--keep); } .o-star.develop .o-dot { fill: var(--develop); } .o-star.watch .o-dot { fill: var(--watch); }
.o-star.replace .o-dot { fill: var(--replace); filter: drop-shadow(0 0 6px var(--replace)); }
.o-star.hot .o-dot { transform: scale(1.5); }
.orbit-tip { position: absolute; left: 0; top: 0; z-index: 3; pointer-events: none; display: grid; gap: 1px; padding: 7px 10px; border-radius: 10px; background: var(--ink); color: var(--bg); font-size: 12px; white-space: nowrap; box-shadow: var(--shadow-2); }
.orbit-tip span { opacity: .85; font-size: 11px; }
.orbit.bang .o-core { animation: coreBang 1.2s var(--spring); transform-box: fill-box; transform-origin: center; }
@keyframes coreBang { from { transform: scale(.2); } to { transform: none; } }

/* command palette */
.pal-scrim { position: fixed; inset: 0; z-index: 70; background: color-mix(in oklab, var(--bg) 40%, transparent); backdrop-filter: blur(6px); -webkit-backdrop-filter: blur(6px); animation: fade .2s both; }
.palette {
  position: fixed; z-index: 71; left: 50%; top: 12vh; width: min(640px, calc(100vw - 24px)); transform: translateX(-50%);
  background: color-mix(in oklab, var(--surface) 95%, transparent); border: 1px solid var(--line-2); border-radius: 18px;
  box-shadow: var(--shadow-3), 0 0 0 1px color-mix(in oklab, var(--brand) 18%, transparent), 0 30px 120px -30px color-mix(in oklab, var(--brand) 60%, transparent);
  overflow: hidden; animation: palIn .45s var(--spring) both; backdrop-filter: blur(20px); -webkit-backdrop-filter: blur(20px);
}
@keyframes palIn { from { opacity: 0; transform: translateX(-50%) translateY(-14px) scale(.96); filter: blur(6px); } to { opacity: 1; transform: translateX(-50%); filter: none; } }
.pal-field { display: flex; align-items: center; gap: 10px; padding: 14px 16px; border-bottom: 1px solid var(--line); }
.pal-field input { border: 0; background: transparent; padding: 4px 0; font-size: 16px; }
.pal-field input:focus { box-shadow: none; background: transparent; }
.pal-list { list-style: none; margin: 0; padding: 6px; max-height: min(52vh, 440px); overflow: auto; }
.pal-group { font: 600 11px var(--font-num); text-transform: uppercase; letter-spacing: .08em; color: var(--ink-2); padding: 10px 10px 4px; }
.pal-opt { display: grid; grid-template-columns: 30px minmax(0, 1fr) auto; align-items: center; gap: 10px; padding: 8px 10px; border-radius: 10px; cursor: pointer; }
.pal-opt[aria-selected="true"] { background: var(--brand-soft); box-shadow: inset 2px 0 0 var(--brand); }
.pal-ic { display: grid; place-items: center; width: 30px; height: 30px; border-radius: 8px; background: var(--surface-3); color: var(--ink-2); }
.pal-opt[aria-selected="true"] .pal-ic { background: var(--brand); color: var(--brand-ink); }
.pal-lb { display: flex; flex-direction: column; min-width: 0; font-weight: 550; }
.pal-lb small { font-weight: 400; color: var(--ink-2); font-size: 12px; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }
.pal-empty { padding: 22px; text-align: center; color: var(--ink-2); }
.pal-foot { display: flex; justify-content: space-between; gap: 12px; padding: 9px 14px; border-top: 1px solid var(--line); font-size: 12px; color: var(--ink-2); flex-wrap: wrap; }
.pal-foot kbd { font-size: 10.5px; padding: 1px 5px; }

/* copilot: aurora ring on the launcher, spring-in panel */
.copilot-fab { isolation: isolate; overflow: visible; transform: translate(var(--mgx, 0), var(--mgy, 0)); transition: transform .35s var(--spring); }
.copilot-fab::before { content: ""; position: absolute; inset: -3px; z-index: -2; border-radius: inherit; background: conic-gradient(from var(--ang), var(--brand), var(--develop), var(--keep), var(--watch), var(--brand)); animation: spinAng 5s linear infinite; }
.copilot-fab::after { content: ""; position: absolute; inset: 0; z-index: -1; border-radius: inherit; background: var(--ink); }
.copilot { background: color-mix(in oklab, var(--surface) 94%, transparent); backdrop-filter: blur(18px); -webkit-backdrop-filter: blur(18px); animation: cpIn .5s var(--spring) both; transform-origin: 100% 100%; }
@keyframes cpIn { from { opacity: 0; transform: scale(.6) translateY(30px); filter: blur(8px); } to { opacity: 1; transform: none; filter: none; } }

/* cinematic view changes */
:root[data-vt="iris"]::view-transition-old(root) { animation: vtSink .6s cubic-bezier(.7, 0, .2, 1) both; }
:root[data-vt="iris"]::view-transition-new(root) { animation: vtIris .6s cubic-bezier(.7, 0, .2, 1) both; }
:root[data-vt="theme"]::view-transition-old(root) { animation: none; }
:root[data-vt="theme"]::view-transition-new(root) { animation: vtIris .7s cubic-bezier(.7, 0, .2, 1) both; }
@keyframes vtIris { from { clip-path: circle(0 at var(--vx, 50%) var(--vy, 50%)); } to { clip-path: circle(150vmax at var(--vx, 50%) var(--vy, 50%)); } }
@keyframes vtSink { to { transform: scale(.93); filter: blur(8px) brightness(.7); opacity: .3; } }
:root[data-vt="tab"] #ins-body { view-transition-name: tabbody; }
:root[data-vt="tab"]::view-transition-old(root), :root[data-vt="tab"]::view-transition-new(root) { animation: none; }
::view-transition-old(tabbody) { animation: tabOut .3s cubic-bezier(.4, 0, 1, 1) both; }
::view-transition-new(tabbody) { animation: tabIn .5s var(--spring) both; }
@keyframes tabOut { to { opacity: 0; transform: translateX(calc(var(--tdir, 1) * -48px)); filter: blur(4px); } }
@keyframes tabIn { from { opacity: 0; transform: translateX(calc(var(--tdir, 1) * 64px)); filter: blur(4px); } }

@media (max-width: 1060px) { .cfg-hero { grid-template-columns: minmax(0, 1fr) 220px; } .crew-ring { width: 220px; height: 220px; } }
@media (max-width: 860px) {
  .cfg-hero { grid-template-columns: 1fr; margin-top: 6px; }
  .crew-ring { display: none; }
  .ins-head { top: calc(var(--topbar-h) + 14px); }
}
@media (max-width: 1060px) {
  .pal-btn span, .pal-btn kbd { position: absolute; width: 1px; height: 1px; overflow: hidden; clip: rect(0 0 0 0); white-space: nowrap; }
  .pal-btn { width: 36px; padding: 0; justify-content: center; }
}
@media (max-width: 820px) { .brand-word { display: none; } }
@media (max-width: 720px) {
  .topbar { width: calc(100% - 16px); top: 8px; margin-top: 8px; padding: 0 8px 0 12px; }
  .pal-foot span[aria-hidden] { display: none; }
  .palette { top: 8vh; }
}
@media (max-width: 560px) {
  .kicker .kx { display: none; }
  .cfg-hero .display { font-size: 40px; }
  .topbar { gap: 8px; }
  .top-actions { gap: 6px; }
}
@media (prefers-reduced-motion: reduce) {
  .ask, .copilot-fab::before { animation: none; }
  .btn.primary.lg::before { display: none; }
  .btn.primary.lg, .copilot-fab { transform: none; }
  .steps-ind { transition: none; }
}
@media print { .sky, .palette, .pal-scrim { display: none !important; } }

/* JS-built controls (chips, weight sliders) appear in one step: the grid stays invisible until boot has built them
   (revealing is not a layout shift). Failsafe: shown after 3 s even if the script never arrives. */
.cfg-grid:not(.ready) { visibility: hidden; animation: cfgReveal 0s 3s forwards; }
@keyframes cfgReveal { to { visibility: visible; } }

Writing safeprocure/web/styles.css


In [19]:
%%writefile safeprocure/web/ui.js
// Small DOM + formatting helpers shared by every view.

export const $ = (sel, root = document) => root.querySelector(sel);
export const $$ = (sel, root = document) => [...root.querySelectorAll(sel)];

export function h(tag, attrs = {}, ...children) {
  const el = document.createElement(tag);
  for (const [k, v] of Object.entries(attrs || {})) {
    if (v == null) continue;
    if (k.startsWith('aria-')) { el.setAttribute(k, String(v)); continue; }
    if (v === false) continue;
    if (k === 'class') el.className = v;
    else if (k === 'html') el.innerHTML = v;
    else if (k.startsWith('on')) el.addEventListener(k.slice(2), v);
    else if (k === 'style' && typeof v === 'object') {
      for (const [sk, sv] of Object.entries(v)) { if (sk.startsWith('--')) el.style.setProperty(sk, sv); else el.style[sk] = sv; }
    }
    else el.setAttribute(k, v === true ? '' : v);
  }
  for (const c of children.flat()) {
    if (c == null || c === false) continue;
    el.append(c instanceof Node ? c : document.createTextNode(String(c)));
  }
  return el;
}

export const icon = (name, cls = '') => {
  const svg = document.createElementNS('http://www.w3.org/2000/svg', 'svg');
  svg.setAttribute('class', `i ${cls}`.trim());
  svg.setAttribute('aria-hidden', 'true');
  const use = document.createElementNS('http://www.w3.org/2000/svg', 'use');
  use.setAttribute('href', `#i-${name}`);
  svg.append(use);
  return svg;
};

export const esc = (s) => String(s ?? '').replace(/[&<>"']/g, (c) => ({ '&': '&amp;', '<': '&lt;', '>': '&gt;', '"': '&quot;', "'": '&#39;' }[c]));

export function inr(v) {
  v = Number(v) || 0;
  const a = Math.abs(v);
  if (a >= 1e7) return `₹${(v / 1e7).toFixed(2)} Cr`;
  if (a >= 1e5) return `₹${(v / 1e5).toFixed(1)} L`;
  return `₹${Math.round(v).toLocaleString('en-IN')}`;
}
export const pct = (v, d = 1) => `${Number(v).toFixed(d)}%`;
export const num = (v, d = 0) => Number(v).toLocaleString('en-IN', { maximumFractionDigits: d, minimumFractionDigits: d });
export const short = (name) => String(name || '').replace(/ Pvt Ltd$/i, '');
export const clock = (s) => `${Math.floor(s / 60)}:${String(Math.floor(s % 60)).padStart(2, '0')}`;

export function css(name) {
  return getComputedStyle(document.documentElement).getPropertyValue(name).trim();
}

export function riskColor(r) {
  const stops = ['--risk-0', '--risk-1', '--risk-2', '--risk-3', '--risk-4'];
  const i = r < 25 ? 0 : r < 40 ? 1 : r < 55 ? 2 : r < 70 ? 3 : 4;
  return `var(${stops[i]})`;
}

export function sparkline(values, w = 84, hgt = 24) {
  if (!values || values.length < 2) return '';
  const min = Math.min(...values), max = Math.max(...values), span = max - min || 1;
  const pts = values.map((v, i) => [i * (w - 4) / (values.length - 1) + 2, hgt - 3 - (v - min) / span * (hgt - 6)]);
  const d = pts.map((p, i) => `${i ? 'L' : 'M'}${p[0].toFixed(1)} ${p[1].toFixed(1)}`).join(' ');
  const [lx, ly] = pts[pts.length - 1];
  return `<svg class="spark" viewBox="0 0 ${w} ${hgt}" aria-hidden="true"><path d="${d}"/><circle class="end" cx="${lx}" cy="${ly}" r="2.2"/></svg>`;
}

// ECharts loads async from the CDN so the UI can render first; run cb now or once the library arrives.
export function onEcharts(cb) {
  if (window.echarts) { cb(); return; }
  document.getElementById('echarts-js')?.addEventListener('load', () => cb(), { once: true });
}

// Score tier letter; screen readers hear "tier A" instead of a bare letter glued to the score.
export const tierChip = (t) => h('span', { class: `tier ${t}` }, h('span', { class: 'sr-only' }, ` tier ${t} `), h('span', { 'aria-hidden': 'true' }, t));

// Bottom space taken by visible docked controls (launch bar, run-done bar, copilot button) that sit under the toast column.
function dockOffset(zone) {
  const z = zone.getBoundingClientRect();
  let off = 0;
  for (const el of document.querySelectorAll('.launch-bar, .copilot-fab')) {
    const r = el.getBoundingClientRect();
    if (!r.width || !r.height || r.top >= innerHeight || r.right < z.left || r.left > z.right) continue;
    off = Math.max(off, innerHeight - r.top);
  }
  return off;
}

export function toast(msg, kind = '') {
  const t = h('div', { class: `toast ${kind}` }, msg);
  const zone = document.getElementById('toasts');
  zone.append(t);
  requestAnimationFrame(() => { zone.style.bottom = `${Math.max(24, dockOffset(zone) + 12)}px`; });
  setTimeout(() => { t.style.transition = 'opacity .3s'; t.style.opacity = '0'; setTimeout(() => t.remove(), 300); }, kind === 'err' ? 6500 : 3800);
}

export async function api(path, opts = {}) {
  const init = { ...opts, headers: { ...(opts.body && !(opts.body instanceof FormData) ? { 'Content-Type': 'application/json' } : {}), ...(opts.headers || {}) } };
  if (init.body && typeof init.body === 'object' && !(init.body instanceof FormData)) init.body = JSON.stringify(init.body);
  const r = await fetch(path, init);
  if (!r.ok) {
    let detail = `${r.status}`;
    try { detail = (await r.json()).detail || detail; } catch (e) { /* ignore */ }
    throw Object.assign(new Error(detail), { status: r.status });
  }
  return r.status === 202 ? { _pending: true, ...(await r.json()) } : r.json();
}

export function debounce(fn, ms) {
  let t;
  return (...a) => { clearTimeout(t); t = setTimeout(() => fn(...a), ms); };
}

export function setRangeFill(input) {
  const p = ((input.value - input.min) / (input.max - input.min)) * 100;
  input.style.setProperty('--p', `${p}%`);
}

export const DECISIONS = ['Keep', 'Watch', 'Develop', 'Replace'];
export const decisionVar = (d) => `var(--${String(d || 'keep').toLowerCase()})`;

export const store = {
  get(k, d = null) { try { const v = localStorage.getItem(k); return v == null ? d : JSON.parse(v); } catch (e) { return d; } },
  set(k, v) { try { localStorage.setItem(k, JSON.stringify(v)); } catch (e) { /* private mode */ } },
};

Writing safeprocure/web/ui.js


In [20]:
%%writefile safeprocure/web/graph.js
// Live workflow graph: draws the CrewAI Flow and animates it from telemetry events.

const NS = 'http://www.w3.org/2000/svg';
const W = 214, H = 54;

export const NODES = {
  intake:      { x: 430, y: 44,  name: 'Intake Router',        role: 'Turns the question into a plan', n: 1 },
  audit:       { x: 430, y: 136, name: 'Data Quality Auditor', role: 'Checks the data is decision-grade', n: 2 },
  performance: { x: 150, y: 244, name: 'Performance Analyst',  role: 'Delivery, quality, trends', n: 3 },
  cost:        { x: 430, y: 244, name: 'Cost & Spend Analyst', role: 'Concentration and savings', n: 4 },
  market:      { x: 710, y: 244, name: 'Market Risk Researcher', role: 'Live news by category and region', n: 5 },
  forecast:    { x: 430, y: 352, name: 'Forecast Agent',       role: 'Next-quarter projections', n: 6 },
  strategy:    { x: 430, y: 448, name: 'Procurement Strategist', role: 'Keep, Watch, Develop, Replace', n: 7 },
  mitigation:  { x: 700, y: 532, name: 'Mitigation Planner',   role: 'Dual sourcing and 30/60/90 plan', n: 8 },
  qa:          { x: 430, y: 616, name: 'QA Reviewer',          role: 'Verifies claims, can send back', n: 9 },
  brief:       { x: 430, y: 712, name: 'Executive Writer',     role: 'Board-ready answer', n: 10 },
};
const GATE = { x: 430, y: 532 };

const EDGES = [
  ['intake', 'audit'], ['audit', 'performance'], ['audit', 'cost'], ['audit', 'market'],
  ['performance', 'forecast'], ['cost', 'forecast'], ['market', 'forecast'], ['forecast', 'strategy'],
  ['strategy', 'gate'], ['gate', 'mitigation', 'high risk'], ['gate', 'qa', 'low risk'], ['mitigation', 'qa'],
  ['qa', 'brief', 'approve'], ['qa', 'strategy', 'revise', true],
];

function el(tag, attrs = {}, parent) {
  const e = document.createElementNS(NS, tag);
  for (const [k, v] of Object.entries(attrs)) e.setAttribute(k, v);
  if (parent) parent.append(e);
  return e;
}

function anchor(id, side) {
  if (id === 'gate') {
    return { top: [GATE.x, GATE.y - 22], bottom: [GATE.x, GATE.y + 22], right: [GATE.x + 22, GATE.y], left: [GATE.x - 22, GATE.y] }[side];
  }
  const n = NODES[id];
  return { top: [n.x, n.y - H / 2], bottom: [n.x, n.y + H / 2], left: [n.x - W / 2, n.y], right: [n.x + W / 2, n.y] }[side];
}

function edgePath(a, b, loop) {
  if (loop) {
    const [x1, y1] = anchor(a, 'left');
    const [x2, y2] = anchor(b, 'left');
    return `M${x1} ${y1} C ${x1 - 150} ${y1}, ${x2 - 150} ${y2}, ${x2} ${y2}`;
  }
  if (a === 'gate' && b === 'mitigation') {
    const [x1, y1] = anchor('gate', 'right');
    const [x2, y2] = anchor('mitigation', 'left');
    return `M${x1} ${y1} L ${x2} ${y2}`;
  }
  if (a === 'mitigation' && b === 'qa') {
    const [x1, y1] = anchor('mitigation', 'bottom');
    const [x2, y2] = anchor('qa', 'right');
    return `M${x1} ${y1} C ${x1} ${y2}, ${x1} ${y2}, ${x2} ${y2}`;
  }
  const [x1, y1] = anchor(a, 'bottom');
  const [x2, y2] = anchor(b, 'top');
  const my = (y1 + y2) / 2;
  return `M${x1} ${y1} C ${x1} ${my}, ${x2} ${my}, ${x2} ${y2}`;
}

const BADGE_IDLE = 'M-3 0h6'; // small dash
const fit = (s, n = 27) => (s.length > n ? `${s.slice(0, n - 1)}…` : s);
const MODEL_SHORT = (s) => s.replace('Gemini 3.1 Flash-Lite', 'Flash-Lite').replace('GPT-OSS ', 'GPT-OSS·').replace('Qwen 3.8 27B', 'Qwen 3.8');
const BADGE_DONE = 'M-4.5 0.5 -1.3 3.6 4.6 -3.2';
const BADGE_WARN = 'M0 -4v4.2M0 4v.1';

export class FlowGraph {
  constructor(svg, onSelect) {
    this.svg = svg;
    this.onSelect = onSelect;
    this.state = {};
    this.selected = null;
    this.build();
  }

  build() {
    const s = this.svg;
    s.innerHTML = '';
    s.setAttribute('viewBox', '0 0 860 750');
    s.setAttribute('role', 'group');
    const edges = el('g', {}, s);
    this.packetLayer = null;
    this.edges = {};
    for (const [a, b, label, loop] of EDGES) {
      const p = el('path', { class: `g-edge${loop ? ' loop' : ''}`, d: edgePath(a, b, loop), 'marker-end': 'url(#arrow)' }, edges);
      this.edges[`${a}>${b}`] = p;
      if (label) {
        let tx, ty;
        if (loop) { tx = 214; ty = 534; }
        else if (b === 'mitigation') { tx = 486; ty = 522; }
        else if (b === 'qa' && a === 'gate') { tx = 442; ty = 576; }
        else { tx = 442; ty = 668; }
        const t = el('text', { class: 'g-route', x: tx, y: ty }, edges);
        t.textContent = label;
        this.edges[`${a}>${b}`].label = t;
      }
    }
    const defs = el('defs', {}, s);
    const m = el('marker', { id: 'arrow', viewBox: '0 0 10 10', refX: '8.5', refY: '5', markerWidth: '6', markerHeight: '6', orient: 'auto-start-reverse' }, defs);
    el('path', { d: 'M0 1.5 8.5 5 0 8.5', fill: 'none', stroke: 'var(--ink-3)', 'stroke-width': '1.4', 'stroke-linecap': 'round', 'stroke-linejoin': 'round' }, m);

    this.packetLayer = el('g', { class: 'g-packets', 'aria-hidden': 'true' }, s);
    const gate = el('g', { class: 'g-gate', transform: `translate(${GATE.x} ${GATE.y})` }, s);
    el('polygon', { points: '0,-22 22,0 0,22 -22,0' }, gate);
    const gt = el('text', { x: -30, y: 4, 'text-anchor': 'end' }, gate);
    gt.textContent = 'Risk gate';
    this.gate = gate;

    this.nodes = {};
    for (const [id, n] of Object.entries(NODES)) {
      const g = el('g', { class: 'g-node', transform: `translate(${n.x - W / 2} ${n.y - H / 2})`, tabindex: '0', role: 'button', 'aria-label': `${n.name}: idle` }, s);
      el('rect', { class: 'halo', x: -3, y: -3, width: W + 6, height: H + 6, rx: 15 }, g);
      el('rect', { class: 'card', width: W, height: H, rx: 12 }, g);
      el('circle', { class: 'badge', cx: 24, cy: H / 2, r: 12 }, g);
      const bi = el('path', { class: 'badge-ic', d: BADGE_IDLE, transform: `translate(24 ${H / 2})` }, g);
      const num = el('text', { class: 's', x: 24, y: H / 2 + 3.6, 'text-anchor': 'middle' }, g);
      num.textContent = n.n;
      const t = el('text', { class: 't', x: 44, y: 23 }, g);
      t.textContent = n.name;
      const sub = el('text', { class: 's', x: 44, y: 40 }, g);
      sub.textContent = fit(n.role, 27);
      g.addEventListener('click', () => this.select(id));
      g.addEventListener('keydown', (e) => { if (e.key === 'Enter' || e.key === ' ') { e.preventDefault(); this.select(id); } });
      this.nodes[id] = { g, bi, num, sub, t };
    }
  }

  buildList(ol) {
    this.list = {};
    if (!ol) return;
    ol.innerHTML = '';
    const order = [['intake'], ['audit'], ['performance', 'cost', 'market'], ['forecast'], ['strategy'], ['mitigation'], ['qa'], ['brief']];
    for (const group of order) {
      const li = document.createElement('li');
      li.className = group.length > 1 ? 'fl-group' : '';
      if (group.length > 1) { const t = document.createElement('span'); t.className = 'fl-par'; t.textContent = 'In parallel'; li.append(t); }
      for (const id of group) {
        const n = NODES[id];
        const b = document.createElement('button');
        b.type = 'button'; b.className = 'fl-item idle';
        b.innerHTML = `<span class="fl-dot" aria-hidden="true">${n.n}</span><span class="fl-txt"><b>${n.name}</b><small>${n.role}</small></span>`;
        b.addEventListener('click', () => this.select(id));
        li.append(b);
        this.list[id] = b;
      }
      ol.append(li);
    }
  }

  reset() {
    this.state = {};
    for (const id of Object.keys(NODES)) this.set(id, 'idle');
    Object.values(this.edges).forEach((p) => { p.classList.remove('live', 'done', 'skipped'); p.label?.classList.remove('on'); });
    this.gate.setAttribute('class', 'g-gate');
  }

  set(id, status, model) {
    const n = this.nodes[id];
    if (!n) return;
    this.state[id] = status;
    n.g.setAttribute('class', `g-node ${status}${this.selected === id ? ' selected' : ''}`);
    n.g.setAttribute('aria-label', `${NODES[id].name}: ${status}`);
    const showNum = status === 'idle' || status === 'skipped';
    n.num.style.display = showNum ? '' : 'none';
    n.bi.style.display = showNum ? 'none' : '';
    n.bi.setAttribute('d', status === 'done' ? BADGE_DONE : status === 'degraded' || status === 'failed' ? BADGE_WARN : 'M0 0');
    if (status === 'active') {
      n.bi.setAttribute('d', 'M-4 0a4 4 0 1 0 8 0');
      // SMIL ignores the CSS reduced-motion rule, so only spin when motion is allowed.
      n.bi.innerHTML = matchMedia('(prefers-reduced-motion: reduce)').matches ? '' : '<animateTransform attributeName="transform" type="rotate" from="0" to="360" dur="0.9s" repeatCount="indefinite" additive="sum"/>';
    } else {
      n.bi.innerHTML = '';
    }
    if (model) n.sub.textContent = fit(MODEL_SHORT(model));
    const li = this.list?.[id];
    if (li) {
      li.className = `fl-item ${status}`;
      li.querySelector('.fl-dot').textContent = status === 'done' ? '✓' : status === 'degraded' || status === 'failed' ? '!' : NODES[id].n;
      if (model) li.querySelector('small').textContent = model;
    }
    // a finished agent hands its output on: data packets fly along every outgoing edge
    if (status === 'done' || status === 'degraded') {
      for (const key of Object.keys(this.edges)) if (key.startsWith(id + '>') && key !== 'qa>strategy') this.packet(key);
    }
    // edges into an active node animate; edges into done node settle
    for (const [key, p] of Object.entries(this.edges)) {
      const [, b] = key.split('>');
      if (b === id || (id === 'strategy' && key === 'forecast>strategy')) {
        if (status === 'active') { p.classList.add('live'); p.classList.remove('done'); }
        if (status === 'done' || status === 'degraded') { p.classList.remove('live'); p.classList.add('done'); }
        if (status === 'failed') p.classList.remove('live');
      }
    }
  }

  model(id, label) {
    const n = this.nodes[id];
    if (n) n.sub.textContent = fit(MODEL_SHORT(label));
    const li = this.list?.[id];
    if (li) li.querySelector('small').textContent = label;
  }

  // Comet-like packet travelling along an edge (skipped under reduced motion).
  packet(key, color) {
    const path = this.edges[key];
    if (!path || !this.packetLayer || matchMedia('(prefers-reduced-motion: reduce)').matches) return;
    const len = path.getTotalLength();
    const dots = [0, 1, 2, 3].map((i) => el('circle', { class: 'g-packet', r: 4.2 - i * 0.9, style: `opacity:${1 - i * 0.22}${color ? `;fill:${color}` : ''}` }, this.packetLayer));
    const t0 = performance.now(), dur = 520 + len * 1.1;
    const step = (now) => {
      const p = (now - t0) / dur;
      dots.forEach((d, i) => {
        const q = Math.min(1, Math.max(0, p - i * 0.045)), e = q < 0.5 ? 2 * q * q : 1 - Math.pow(-2 * q + 2, 2) / 2;
        const pt = path.getPointAtLength(e * len);
        d.setAttribute('cx', pt.x.toFixed(1)); d.setAttribute('cy', pt.y.toFixed(1));
      });
      if (p < 1 + dots.length * 0.045 && this.svg.isConnected) requestAnimationFrame(step); else dots.forEach((d) => d.remove());
    };
    requestAnimationFrame(step);
  }

  route(stage, route) {
    if (stage === 'risk_gate') {
      this.packet(route === 'mitigate' ? 'gate>mitigation' : 'gate>qa', route === 'mitigate' ? 'var(--replace)' : null);
      this.gate.setAttribute('class', 'g-gate done');
      const hi = route === 'mitigate';
      this.edges['strategy>gate'].classList.add('done');
      this.edges['gate>mitigation'].classList.toggle('done', hi);
      this.edges['gate>mitigation'].label?.classList.toggle('on', hi);
      this.edges['gate>qa'].classList.toggle('done', !hi);
      this.edges['gate>qa'].label?.classList.toggle('on', !hi);
      if (!hi && this.state.mitigation !== 'done') this.set('mitigation', 'skipped');
    }
    if (stage === 'qa') {
      const rev = route === 'revise';
      this.packet(rev ? 'qa>strategy' : 'qa>brief', rev ? 'var(--watch)' : 'var(--keep)');
      this.edges['qa>strategy'].classList.toggle('live', rev);
      this.edges['qa>strategy'].label?.classList.toggle('on', rev);
      if (!rev) { this.edges['qa>brief'].classList.add('done'); this.edges['qa>brief'].label?.classList.add('on'); this.edges['qa>strategy'].classList.remove('live'); }
    }
  }

  select(id) {
    if (this.selected && this.nodes[this.selected]) this.nodes[this.selected].g.classList.remove('selected');
    this.selected = id;
    this.nodes[id]?.g.classList.add('selected');
    this.onSelect?.(id);
  }
}

Writing safeprocure/web/graph.js


In [21]:
%%writefile safeprocure/web/fx.js
// Observatory layer: a generative star field behind the app that reacts to the cursor, the run and the decisions.
// Purely visual. Every effect is a no-op under prefers-reduced-motion, and the loop sleeps when the tab is hidden.

const RM = matchMedia('(prefers-reduced-motion: reduce)');
const css = (n) => getComputedStyle(document.documentElement).getPropertyValue(n).trim();

const F = {
  cv: null, cx: null, w: 0, h: 0, dpr: 1, stars: [], fx: [], mx: -1e4, my: -1e4, tx: 0, ty: 0, px: 0, py: 0,
  mode: 'calm', speed: 0, warp: 0, mix: null, raf: 0, last: 0, ink: '20,22,30', dark: false, egg: 0,
};

const DEC = ['keep', 'watch', 'develop', 'replace'];
function rgb(v) {
  const c = document.createElement('canvas').getContext('2d');
  c.fillStyle = v || '#888'; const s = c.fillStyle;
  if (s.startsWith('#')) { const n = parseInt(s.slice(1), 16); return `${n >> 16 & 255},${n >> 8 & 255},${n & 255}`; }
  return s.replace(/rgba?\(|\)|\s/g, '').split(',').slice(0, 3).join(',');
}
function palette() {
  const dark = document.documentElement.dataset.theme === 'dark' || (document.documentElement.dataset.theme !== 'light' && matchMedia('(prefers-color-scheme: dark)').matches);
  F.dark = dark;
  F.ink = dark ? '226,230,255' : '40,42,70';
  F.brand = rgb(css('--brand'));
  F.dec = Object.fromEntries(DEC.map((d) => [d, rgb(css(`--${d}`))]));
}

function seed() {
  const n = Math.min(240, Math.round((F.w * F.h) / 7000));
  F.stars = Array.from({ length: n }, () => ({
    x: Math.random() * F.w, y: Math.random() * F.h, z: Math.random() * 0.85 + 0.15,
    tw: Math.random() * Math.PI * 2, c: null, vx: (Math.random() - 0.5) * 0.06, vy: (Math.random() - 0.5) * 0.06,
  }));
  tint();
}

function resize() {
  F.dpr = Math.min(2, devicePixelRatio || 1);
  F.w = innerWidth; F.h = innerHeight;
  F.cv.width = F.w * F.dpr; F.cv.height = F.h * F.dpr;
  F.cx.setTransform(F.dpr, 0, 0, F.dpr, 0, 0);
  seed();
  if (RM.matches) frame(performance.now(), true);
}

// Star colours follow the decision mix on Insights; elsewhere they are ink/brand.
function tint() {
  const m = F.mode === 'insights' && F.mix;
  const total = m ? DEC.reduce((a, d) => a + (m[d] || 0), 0) : 0;
  F.stars.forEach((s, i) => {
    if (total && i % 3 === 0) {
      let r = Math.random() * total, k = 'keep';
      for (const d of DEC) { r -= m[d] || 0; if (r <= 0) { k = d; break; } }
      s.c = F.dec[k];
    } else s.c = i % 7 === 0 ? F.brand : null;
  });
}

function frame(t, once = false) {
  const dt = Math.min(50, t - (F.last || t)); F.last = t;
  const { cx, w, h } = F;
  cx.clearRect(0, 0, w, h);
  // eased parallax toward the cursor
  F.px += (F.tx - F.px) * 0.05; F.py += (F.ty - F.py) * 0.05;
  const target = F.mode === 'run' ? 1 : 0;
  F.speed += (target - F.speed) * 0.02;
  F.warp = Math.max(0, F.warp - dt / 900);
  const cxm = w / 2, cym = h / 2;
  const near = [];
  for (const s of F.stars) {
    // drift; during a run the field streams outward from the centre like a slow jump
    const dx = s.x - cxm, dy = s.y - cym, d = Math.hypot(dx, dy) || 1;
    const out = (F.speed * 0.35 + F.warp * 14) * s.z;
    s.x += s.vx * dt * 0.6 + (dx / d) * out * dt * 0.06;
    s.y += s.vy * dt * 0.6 + (dy / d) * out * dt * 0.06;
    if (s.x < -20 || s.x > w + 20 || s.y < -20 || s.y > h + 20) {
      if (F.speed > 0.2 || F.warp > 0) { const a = Math.random() * 6.283, r = Math.random() * 60; s.x = cxm + Math.cos(a) * r; s.y = cym + Math.sin(a) * r; }
      else { s.x = (s.x + w) % w; s.y = (s.y + h) % h; }
    }
    // cursor gravity: stars lean toward the pointer
    const ox = s.x + F.px * s.z * 18, oy = s.y + F.py * s.z * 18;
    const mdx = F.mx - ox, mdy = F.my - oy, md = Math.hypot(mdx, mdy);
    let gx = 0, gy = 0;
    if (md < 170) { const f = (1 - md / 170) * 10 * s.z; gx = (mdx / md) * f; gy = (mdy / md) * f; near.push([ox + gx, oy + gy, s]); }
    s.tw += dt * 0.002;
    const a = (0.25 + 0.55 * s.z) * (0.75 + 0.25 * Math.sin(s.tw)) * (F.dark ? 1 : 0.55);
    const col = s.c || F.ink;
    const X = ox + gx, Y = oy + gy;
    if (F.warp > 0.02 || F.speed > 0.3) {
      const len = (F.warp * 60 + F.speed * 6) * s.z;
      cx.strokeStyle = `rgba(${col},${a})`; cx.lineWidth = s.z * 1.4;
      cx.beginPath(); cx.moveTo(X, Y); cx.lineTo(X - (dx / d) * len, Y - (dy / d) * len); cx.stroke();
    } else {
      cx.fillStyle = `rgba(${col},${a})`;
      cx.beginPath(); cx.arc(X, Y, s.z * (s.c ? 1.7 : 1.2), 0, 6.283); cx.fill();
    }
  }
  // constellation lines around the cursor
  if (near.length > 1) {
    cx.lineWidth = 0.7;
    for (let i = 0; i < near.length; i++) for (let j = i + 1; j < near.length; j++) {
      const [x1, y1] = near[i], [x2, y2] = near[j], dd = Math.hypot(x1 - x2, y1 - y2);
      if (dd < 90) { cx.strokeStyle = `rgba(${F.brand},${(1 - dd / 90) * (F.dark ? 0.45 : 0.3)})`; cx.beginPath(); cx.moveTo(x1, y1); cx.lineTo(x2, y2); cx.stroke(); }
    }
  }
  // transient effects: pulses and bursts
  F.fx = F.fx.filter((e) => {
    e.t += dt;
    const p = e.t / e.life;
    if (p >= 1) return false;
    if (e.kind === 'ring') {
      cx.strokeStyle = `rgba(${e.c},${(1 - p) * 0.7})`; cx.lineWidth = 2 * (1 - p) + 0.4;
      cx.beginPath(); cx.arc(e.x, e.y, e.r0 + p * e.r, 0, 6.283); cx.stroke();
    } else {
      e.x += e.vx * dt; e.y += e.vy * dt; e.vy += 0.00045 * dt; e.vx *= 0.985; e.vy *= 0.985;
      cx.fillStyle = `rgba(${e.c},${1 - p})`;
      cx.beginPath(); cx.arc(e.x, e.y, e.s * (1 - p * 0.5), 0, 6.283); cx.fill();
    }
    return true;
  });
  if (!once) F.raf = requestAnimationFrame(frame);
}

function start() { if (!F.raf && !RM.matches && !document.hidden) { F.last = 0; F.raf = requestAnimationFrame(frame); } }
function stop() { cancelAnimationFrame(F.raf); F.raf = 0; }

export function initFx() {
  const cv = document.createElement('canvas');
  cv.className = 'sky'; cv.setAttribute('aria-hidden', 'true');
  document.body.prepend(cv);
  F.cv = cv; F.cx = cv.getContext('2d');
  palette(); resize();
  addEventListener('resize', () => { resize(); });
  addEventListener('pointermove', (e) => {
    F.mx = e.clientX; F.my = e.clientY;
    F.tx = (e.clientX / F.w - 0.5) * 2; F.ty = (e.clientY / F.h - 0.5) * 2;
    spotlight(e);
  }, { passive: true });
  document.addEventListener('pointerleave', () => { F.mx = F.my = -1e4; });
  document.addEventListener('visibilitychange', () => (document.hidden ? stop() : start()));
  RM.addEventListener('change', () => { if (RM.matches) { stop(); frame(performance.now(), true); } else start(); });
  initPress(); initMagnet(); initEgg();
  start();
}

export function setMode(mode, mix) {
  F.mode = mode; if (mix) F.mix = mix;
  tint();
  document.documentElement.dataset.sky = mode;
}
export function retheme() { palette(); tint(); if (RM.matches) frame(performance.now(), true); }

export function warp() { if (RM.matches) return; F.warp = 1; start(); }

export function pulse(x, y, color = 'brand') {
  if (RM.matches) return;
  const c = color === 'brand' ? F.brand : F.dec[color] || color;
  F.fx.push({ kind: 'ring', x, y, r0: 6, r: 90, t: 0, life: 900, c });
}
export function pulseAt(el, color) {
  if (!el) return;
  const r = el.getBoundingClientRect();
  if (r.width) pulse(r.left + r.width / 2, r.top + r.height / 2, color);
}

// Particle burst in the decision colours (e.g. run complete).
export function burst(x = F.w / 2, y = F.h / 2, mix = { keep: 1, watch: 1, develop: 1, replace: 1 }, n = 110) {
  if (RM.matches) return;
  const keys = DEC.flatMap((d) => Array(Math.max(0, Math.round((mix[d] || 0) * 4))).fill(d));
  for (let i = 0; i < n; i++) {
    const a = Math.random() * 6.283, v = 0.15 + Math.random() * 0.55;
    F.fx.push({ kind: 'p', x, y, vx: Math.cos(a) * v, vy: Math.sin(a) * v - 0.18, s: 1.5 + Math.random() * 2.2, t: 0, life: 1100 + Math.random() * 900, c: F.dec[keys[i % keys.length] || 'keep'] });
  }
  F.fx.push({ kind: 'ring', x, y, r0: 10, r: 260, t: 0, life: 1200, c: F.brand });
  start();
}

/* ------------------------------------------------------------ delight */
// Cursor spotlight on the panel under the pointer.
let litPanel = null;
function spotlight(e) {
  const p = e.target.closest?.('.panel, .launch-bar, .run-stats div');
  if (litPanel && litPanel !== p) litPanel.classList.remove('lit');
  litPanel = p;
  if (!p) return;
  const r = p.getBoundingClientRect();
  p.style.setProperty('--mx', `${e.clientX - r.left}px`); p.style.setProperty('--my', `${e.clientY - r.top}px`);
  p.classList.add('lit');
}

// Spring press: buttons squash on press and spring back on release.
function initPress() {
  document.addEventListener('pointerdown', (e) => {
    const b = e.target.closest?.('.btn, .chip, .icon-btn, .steps a, [role="tab"], .prompt-chips button, .presets button, .copilot-fab');
    if (!b || RM.matches) return;
    b.classList.remove('sprung'); b.classList.add('pressed');
    const up = () => { b.classList.remove('pressed'); void b.offsetWidth; b.classList.add('sprung'); setTimeout(() => b.classList.remove('sprung'), 600); removeEventListener('pointerup', up); removeEventListener('pointercancel', up); };
    addEventListener('pointerup', up); addEventListener('pointercancel', up);
  });
}

// Magnetic primary buttons lean toward the cursor.
function initMagnet() {
  document.addEventListener('pointermove', (e) => {
    if (RM.matches || e.pointerType !== 'mouse') return;
    for (const b of document.querySelectorAll('.btn.primary.lg, .copilot-fab')) {
      const r = b.getBoundingClientRect();
      if (!r.width) continue;
      const dx = e.clientX - (r.left + r.width / 2), dy = e.clientY - (r.top + r.height / 2);
      const near = Math.abs(dx) < r.width / 2 + 40 && Math.abs(dy) < r.height / 2 + 40;
      b.style.setProperty('--mgx', near ? `${dx * 0.14}px` : '0px');
      b.style.setProperty('--mgy', near ? `${dy * 0.22}px` : '0px');
    }
  }, { passive: true });
}

// Easter egg: the Konami code (or five clicks on the logo) recolours the sky with every decision colour.
function initEgg() {
  const code = ['ArrowUp', 'ArrowUp', 'ArrowDown', 'ArrowDown', 'ArrowLeft', 'ArrowRight', 'ArrowLeft', 'ArrowRight', 'b', 'a'];
  let i = 0;
  document.addEventListener('keydown', (e) => { i = e.key === code[i] ? i + 1 : e.key === code[0] ? 1 : 0; if (i === code.length) { i = 0; egg(); } });
  let clicks = 0, t = 0;
  document.querySelector('.brand')?.addEventListener('click', () => { const n = Date.now(); clicks = n - t < 600 ? clicks + 1 : 1; t = n; if (clicks >= 5) { clicks = 0; egg(); } });
}
function egg() {
  F.stars.forEach((s, i) => { s.c = F.dec[DEC[i % 4]]; });
  burst(F.w / 2, F.h / 2, { keep: 1, watch: 1, develop: 1, replace: 1 }, 200);
  warp();
  document.dispatchEvent(new CustomEvent('sp-egg'));
}

Writing safeprocure/web/fx.js


In [22]:
%%writefile safeprocure/web/orbit.js
// Decision Orbit: every in-scope vendor is a star on the ring of its decision (Keep innermost, Replace outermost).
// Drag to spin it, select a star for the dossier. Opening it plays a "big bang" where the stars burst out of the core.
// Pointer convenience only: the same vendors stay reachable by keyboard in "Needs attention" and the Vendors tab.
import { short, inr } from './ui.js';
import { burst } from './fx.js';

const NS = 'http://www.w3.org/2000/svg';
const RINGS = [['Keep', 74], ['Develop', 112], ['Watch', 150], ['Replace', 188]];
const RM = matchMedia('(prefers-reduced-motion: reduce)');
let played = new WeakSet();

const el = (tag, attrs = {}, parent) => {
  const e = document.createElementNS(NS, tag);
  for (const [k, v] of Object.entries(attrs)) e.setAttribute(k, v);
  if (parent) parent.append(e);
  return e;
};

export function decisionOrbit(result, vendors, onPick) {
  const wrap = document.createElement('div');
  wrap.className = 'orbit';
  const c = Object.fromEntries(RINGS.map(([d]) => [d, vendors.filter((v) => v.decision?.decision === d)]));
  const svg = el('svg', { viewBox: '-215 -215 430 430', role: 'img', 'aria-label': `Decision orbit: ${RINGS.map(([d]) => `${c[d].length} ${d}`).join(', ')}. Drag to rotate; vendors are also listed under Needs attention and in the Vendors tab.` });
  wrap.append(svg);
  const tip = document.createElement('div');
  tip.className = 'orbit-tip'; tip.setAttribute('aria-hidden', 'true'); tip.hidden = true;
  wrap.append(tip);

  const defs = el('defs', {}, svg);
  const g = el('radialGradient', { id: 'orbit-core' }, defs);
  el('stop', { offset: '0%', 'stop-color': 'var(--brand)', 'stop-opacity': '.55' }, g);
  el('stop', { offset: '100%', 'stop-color': 'var(--brand)', 'stop-opacity': '0' }, g);
  el('circle', { r: 64, fill: 'url(#orbit-core)', class: 'o-glow' }, svg);

  const maxSpend = Math.max(...vendors.map((v) => v.spend_total), 1);
  const stars = [];
  const rings = RINGS.map(([d, r], ri) => {
    el('circle', { r, class: `o-ring ${d.toLowerCase()}` }, svg);
    const lbl = el('text', { class: `o-lbl ${d.toLowerCase()}`, x: 0, y: -r - 5, 'text-anchor': 'middle' }, svg);
    lbl.textContent = `${d} · ${c[d].length}`;
    const grp = el('g', { class: 'o-grp' }, svg);
    const list = c[d];
    list.forEach((v, i) => {
      const a0 = (i / Math.max(1, list.length)) * Math.PI * 2 + ri * 0.7;
      const s = el('g', { class: `o-star ${d.toLowerCase()}` }, grp);
      const size = 4.5 + Math.sqrt(v.spend_total / maxSpend) * 8;
      el('circle', { r: size + 6, class: 'o-hit' }, s);
      el('circle', { r: size, class: 'o-dot' }, s);
      s.addEventListener('pointerenter', () => { tip.hidden = false; tip.innerHTML = `<b>${short(v.vendor)}</b><span>${d} · risk ${Math.round(v.combined_risk)} · ${inr(v.spend_total)}</span>`; s.classList.add('hot'); });
      s.addEventListener('pointerleave', () => { tip.hidden = true; s.classList.remove('hot'); });
      s.addEventListener('click', () => { if (!dragMoved) onPick(v.vendor); });
      stars.push({ s, r, a0, ri, i, size, v });
    });
    return { grp, speed: [0.00011, -0.00008, 0.00006, -0.00004][ri] };
  });
  // core
  el('circle', { r: 30, class: 'o-core' }, svg);
  const n = el('text', { class: 'o-n', y: 4, 'text-anchor': 'middle' }, svg);
  n.textContent = vendors.length;
  const nl = el('text', { class: 'o-nl', y: 18, 'text-anchor': 'middle' }, svg);
  nl.textContent = 'vendors';

  // drag-to-spin with inertia
  let spin = 0, vel = 0, dragging = false, dragMoved = false, lastA = 0, lastT = 0;
  const angleOf = (e) => { const b = svg.getBoundingClientRect(); return Math.atan2(e.clientY - (b.top + b.height / 2), e.clientX - (b.left + b.width / 2)); };
  svg.addEventListener('pointerdown', (e) => { dragging = true; dragMoved = false; lastA = angleOf(e); lastT = performance.now(); svg.setPointerCapture(e.pointerId); wrap.classList.add('grab'); });
  svg.addEventListener('pointermove', (e) => {
    const rb = wrap.getBoundingClientRect();
    tip.style.transform = `translate(${e.clientX - rb.left + 14}px, ${e.clientY - rb.top + 10}px)`;
    if (!dragging) return;
    const a = angleOf(e), now = performance.now();
    let d = a - lastA; if (d > Math.PI) d -= 2 * Math.PI; if (d < -Math.PI) d += 2 * Math.PI;
    if (Math.abs(d) > 0.01) dragMoved = true;
    spin += d; vel = d / Math.max(8, now - lastT); lastA = a; lastT = now;
    if (RM.matches) draw(now);
  });
  const end = () => { dragging = false; wrap.classList.remove('grab'); setTimeout(() => { dragMoved = false; }, 0); };
  svg.addEventListener('pointerup', end); svg.addEventListener('pointercancel', end);

  // big bang intro (plays the first time the orbit is on screen) + continuous orbit
  let t0 = performance.now();
  const intro = !RM.matches && !played.has(result);
  played.add(result);
  const ease = (p) => (p >= 1 ? 1 : 1 - Math.pow(2, -10 * p) * Math.cos(p * 6.2)); // springy overshoot
  let raf = 0, visible = true;
  function draw(now) {
    const t = now - t0;
    if (!dragging) { spin += vel * 16; vel *= 0.95; }
    for (const st of stars) {
      const delay = 120 + st.ri * 140 + st.i * 35;
      const p = intro ? Math.max(0, Math.min(1, (t - delay) / 1100)) : 1;
      const e = ease(p);
      const a = st.a0 + spin + (RM.matches ? 0 : t * rings[st.ri].speed);
      const rr = st.r * e;
      st.s.setAttribute('transform', `translate(${(Math.cos(a) * rr).toFixed(2)} ${(Math.sin(a) * rr).toFixed(2)}) scale(${(0.2 + 0.8 * Math.min(1, p * 1.6)).toFixed(3)})`);
    }
  }
  function loop(now) { draw(now); raf = wrap.isConnected && visible && !RM.matches ? requestAnimationFrame(loop) : 0; }
  let banged = !intro;
  const bang = () => {
    banged = true; t0 = performance.now();
    const b = svg.getBoundingClientRect();
    const mix = Object.fromEntries(RINGS.map(([d]) => [d.toLowerCase(), c[d].length]));
    setTimeout(() => burst(b.left + b.width / 2, b.top + b.height / 2, mix, 90), 80);
    wrap.classList.add('bang');
  };
  const io = new IntersectionObserver(([en]) => {
    visible = en.isIntersecting;
    if (visible && !banged) bang();
    if (visible && !raf && !RM.matches) raf = requestAnimationFrame(loop);
  }, { threshold: 0.35 });
  requestAnimationFrame(() => {
    if (!wrap.isConnected) return;
    io.observe(wrap);
    draw(performance.now());
  });
  return wrap;
}

Writing safeprocure/web/orbit.js

In [23]:
%%writefile safeprocure/web/palette.js
// Command palette (Ctrl/⌘ K or "/"): jump to any view, insight tab or vendor, or run an action, from the keyboard.
// ARIA combobox + listbox pattern; focus returns to where it was on close.
import { h, icon } from './ui.js';

let getCommands = () => [];
let items = [], active = 0, lastFocus = null;

const $ = (s) => document.querySelector(s);

function score(label, q) {
  if (!q) return 1;
  const L = label.toLowerCase();
  const idx = L.indexOf(q);
  if (idx >= 0) return 100 - idx;
  let i = 0, s = 0;
  for (const ch of q) { const j = L.indexOf(ch, i); if (j < 0) return 0; s += j === i ? 3 : 1; i = j + 1; }
  return s;
}

function render() {
  const q = $('#pal-input').value.trim().toLowerCase();
  const all = getCommands().filter((c) => !c.hidden);
  items = all.map((c) => ({ ...c, s: score(`${c.group} ${c.label} ${c.keys || ''}`, q) })).filter((c) => c.s > 0);
  if (q) items.sort((a, b) => b.s - a.s);
  items = items.slice(0, 40);
  active = Math.min(active, Math.max(0, items.length - 1));
  const ul = $('#pal-list');
  ul.innerHTML = '';
  let group = '';
  items.forEach((c, i) => {
    if (c.group !== group && !q) { group = c.group; ul.append(h('li', { role: 'presentation', class: 'pal-group' }, group)); }
    const li = h('li', { role: 'option', id: `pal-opt-${i}`, 'aria-selected': i === active, class: 'pal-opt' },
      h('span', { class: 'pal-ic', 'aria-hidden': 'true' }, icon(c.icon || 'arrow-right')),
      h('span', { class: 'pal-lb' }, c.label, c.sub ? h('small', {}, c.sub) : null),
      c.keys ? h('kbd', {}, c.keys) : null);
    li.addEventListener('pointermove', () => { if (active !== i) { active = i; mark(); } });
    li.addEventListener('click', () => exec(i));
    ul.append(li);
  });
  if (!items.length) ul.append(h('li', { role: 'presentation', class: 'pal-empty' }, 'No match. Try a view, a tab or a vendor name.'));
  $('#pal-count').textContent = `${items.length} result${items.length === 1 ? '' : 's'}`;
  mark();
}
function mark() {
  document.querySelectorAll('#pal-list [role=option]').forEach((li, i) => li.setAttribute('aria-selected', i === active));
  const inp = $('#pal-input');
  if (items.length) inp.setAttribute('aria-activedescendant', `pal-opt-${active}`); else inp.removeAttribute('aria-activedescendant');
  document.getElementById(`pal-opt-${active}`)?.scrollIntoView({ block: 'nearest' });
}
function exec(i) {
  const c = items[i];
  if (!c) return;
  close(false);
  setTimeout(() => c.run(), 0);
}

export function openPalette() {
  const d = $('#palette');
  if (!d.hidden) return;
  lastFocus = document.activeElement;
  d.hidden = false; $('#pal-scrim').hidden = false;
  $('#pal-input').value = ''; active = 0;
  render();
  $('#pal-input').focus();
  $('#palette-btn')?.setAttribute('aria-expanded', 'true');
}
function close(restore = true) {
  const d = $('#palette');
  if (d.hidden) return;
  d.hidden = true; $('#pal-scrim').hidden = true;
  $('#palette-btn')?.setAttribute('aria-expanded', 'false');
  if (restore) lastFocus?.focus?.();
}

export function initPalette(commands) {
  getCommands = commands;
  const inp = $('#pal-input');
  inp.addEventListener('input', () => { active = 0; render(); });
  inp.addEventListener('keydown', (e) => {
    if (e.key === 'ArrowDown') { e.preventDefault(); active = (active + 1) % Math.max(1, items.length); mark(); }
    else if (e.key === 'ArrowUp') { e.preventDefault(); active = (active - 1 + items.length) % Math.max(1, items.length); mark(); }
    else if (e.key === 'Home' && e.ctrlKey) { active = 0; mark(); }
    else if (e.key === 'Enter') { e.preventDefault(); exec(active); }
    else if (e.key === 'Escape') { e.preventDefault(); e.stopPropagation(); close(); }
    else if (e.key === 'Tab') { e.preventDefault(); } // single-field dialog: keep focus inside
  });
  $('#pal-scrim').addEventListener('click', () => close());
  $('#palette-btn')?.addEventListener('click', openPalette);
  document.addEventListener('keydown', (e) => {
    const typing = /^(INPUT|TEXTAREA|SELECT)$/.test(document.activeElement?.tagName) || document.activeElement?.isContentEditable;
    if ((e.ctrlKey || e.metaKey) && e.key.toLowerCase() === 'k') { e.preventDefault(); $('#palette').hidden ? openPalette() : close(); }
    else if (!typing && !e.ctrlKey && !e.metaKey && !e.altKey && (e.key === '/' || e.key === '?') && $('#palette').hidden && $('#drawer').hidden) { e.preventDefault(); openPalette(); }
  });
}

Writing safeprocure/web/palette.js


In [24]:
%%writefile safeprocure/web/app.js
// Safe Procure — app shell, Configure view, live Run view, Copilot.
import { $, $$, h, icon, esc, inr, num, short, clock, api, debounce, setRangeFill, toast, store, sparkline, tierChip, onEcharts } from './ui.js';
import { FlowGraph, NODES } from './graph.js';
import { renderInsights, rerenderCharts, selectTab, TAB_LIST, openVendor } from './insights.js';
import { initFx, setMode, retheme, warp, burst, pulseAt } from './fx.js';
import { initPalette } from './palette.js';

const RM = matchMedia('(prefers-reduced-motion: reduce)');
// Cinematic view changes: an iris opens from where the user clicked (View Transitions API; plain swap where unsupported).
let lastPoint = null, vtSeq = 0;
addEventListener('pointerdown', (e) => { lastPoint = [e.clientX, e.clientY]; }, true);
export function cinematic(fn, kind = 'iris') {
  if (!document.startViewTransition || RM.matches || !S.booted) { fn(); return; }
  const [x, y] = lastPoint || [innerWidth / 2, innerHeight / 2];
  const root = document.documentElement;
  root.style.setProperty('--vx', `${x}px`); root.style.setProperty('--vy', `${y}px`);
  root.dataset.vt = kind;
  lastPoint = null;
  const t = document.startViewTransition(fn);
  t.ready.catch(() => {}); // a newer transition skipped this one; the DOM update still ran
  const mine = ++vtSeq;
  t.finished.catch(() => {}).finally(() => { if (mine === vtSeq) delete root.dataset.vt; });
}
const decisionMix = (r) => {
  const m = { keep: 0, watch: 0, develop: 0, replace: 0 };
  (r?.vendors || []).filter((v) => v.in_scope).forEach((v) => { const d = v.decision?.decision?.toLowerCase(); if (d in m) m[d] += 1; });
  return m;
};

const PILLARS = [
  { key: 'delivery', label: 'Delivery', color: '#3E3AE0' },
  { key: 'quality', label: 'Quality', color: '#0A6B9E' },
  { key: 'cost', label: 'Cost', color: '#0B7A4E' },
  { key: 'responsiveness', label: 'Responsiveness', color: '#A85A06' },
  { key: 'compliance', label: 'Compliance', color: '#BE2A21' },
];
const PRESETS = {
  Balanced: { delivery: 30, quality: 25, cost: 20, responsiveness: 10, compliance: 15 },
  'Delivery first': { delivery: 50, quality: 20, cost: 10, responsiveness: 10, compliance: 10 },
  'Cost first': { delivery: 15, quality: 15, cost: 50, responsiveness: 5, compliance: 15 },
  'Risk averse': { delivery: 25, quality: 25, cost: 5, responsiveness: 10, compliance: 35 },
};
const PROMPTS = [
  'Which vendors put our Q4 production at risk, and what should we do about each one?',
  'Where can we cut 10% of cost without hurting on-time delivery?',
  'How exposed are we to steel and logistics disruption right now?',
  'Which vendors deserve more volume next quarter?',
];
const CAT_SHORT = { 'Steel & Metals': 'Steel', 'Packaging': 'Packaging', 'Electronic Components': 'Electronics', 'Logistics & Freight': 'Logistics', 'Chemicals & Plastics': 'Chemicals' };

export const S = {
  health: null, dataset: null, weights: { ...PRESETS.Balanced }, genCats: null, scopeCats: [],
  baselineRanks: {}, prevRanks: {}, run: null, result: null, route: 'configure',
};

/* ================================================================ routing */
function route() {
  const r = (location.hash.replace('#/', '') || 'configure').split('/')[0];
  const target = ['configure', 'run', 'insights'].includes(r) ? r : 'configure';
  if (target === 'insights' && !S.result) {
    if (S.run && !S.run.done) { location.hash = '#/run'; return; }
    if (S.health?.saved_run) { loadSaved(); return; }
    location.hash = '#/configure'; return;
  }
  if (target === 'run' && !S.run) {
    if (S.result) { S.route = 'run'; showReplayIdle(); }
    else if (S.health?.saved_run) { loadSaved(true); return; }
    else { location.hash = '#/configure'; return; }
  }
  const changed = S.route !== target || !S.shown;
  S.shown = true;
  if (changed) cinematic(() => showView(target)); else showView(target);
}
function showView(target) {
  S.route = target;
  setMode(target === 'run' ? (S.run && !S.run.done ? 'run' : 'calm') : target, target === 'insights' ? decisionMix(S.result) : null);
  $$('.view').forEach((v) => { v.hidden = v.dataset.view !== target; });
  $$('[data-nav]').forEach((a) => { if (a.dataset.nav !== target) a.removeAttribute('aria-current'); else a.setAttribute('aria-current', 'page'); });
  $('#copilot-fab').hidden = !(target === 'insights' && S.result);
  if (target !== 'insights') closeCopilot();
  if (target === 'insights') renderInsights(S.result);
  window.scrollTo({ top: 0, behavior: 'instant' in window ? 'instant' : 'auto' });
  // Move focus to the new view's heading so screen readers announce the page change without reading the whole view.
  const h1 = $(`.view[data-view="${target}"] h1`);
  if (h1) { h1.tabIndex = -1; h1.focus({ preventScroll: true }); } else $('#main').focus({ preventScroll: true });
  requestAnimationFrame(moveNavIndicator);
}
// Liquid indicator that slides under the active workflow step.
function moveNavIndicator() {
  const a = $('.steps a[aria-current="page"]'), ind = $('.steps-ind');
  if (!a || !ind) return;
  const n = $('.steps').getBoundingClientRect(), r = a.getBoundingClientRect();
  ind.style.transform = `translateX(${r.left - n.left}px)`; ind.style.width = `${r.width}px`;
  $('.steps').dataset.step = a.dataset.nav;
}
function lockNav() {
  $('[data-nav="run"]').classList.toggle('locked', !S.run && !S.result && !S.health?.saved_run);
  $('#btn-replay').hidden = !S.result || (S.run && !S.run.done);
  $('[data-nav="insights"]').classList.toggle('locked', !S.result && !S.health?.saved_run);
}

/* ================================================================== theme */
function syncThemeButton() {
  const cur = document.documentElement.dataset.theme;
  const dark = cur === 'dark' || (cur !== 'light' && matchMedia('(prefers-color-scheme: dark)').matches);
  const b = $('#theme-toggle');
  b.setAttribute('aria-label', 'Dark theme'); b.setAttribute('aria-pressed', String(dark));
}
function initTheme() {
  syncThemeButton();
  $('#theme-toggle').addEventListener('click', () => {
    const cur = document.documentElement.dataset.theme;
    const sysDark = matchMedia('(prefers-color-scheme: dark)').matches;
    const isDark = cur === 'dark' || (cur === 'auto' && sysDark);
    const next = isDark ? 'light' : 'dark';
    cinematic(() => {
      document.documentElement.dataset.theme = next;
      store.set('sp-theme', next); try { localStorage.setItem('sp-theme', next); } catch (e) { /* */ }
      retheme();
      rerenderCharts();
      renderMiniChart();
      syncThemeButton();
    }, 'theme');
  });
  matchMedia('(prefers-color-scheme: dark)').addEventListener('change', () => { retheme(); rerenderCharts(); renderMiniChart(); });
}

/* ================================================================= health */
async function loadHealth() {
  const pill = $('#status-pill');
  try {
    S.health = await api('/api/health');
    const ready = S.health.endpoints.filter((e) => e.available);
    const provs = new Set(ready.map((e) => e.provider));
    pill.className = `status-pill ${ready.length ? 'ok' : 'bad'}`;
    pill.querySelector('.label').innerHTML = ready.length ? `${ready.length} models<span class="lx"> · ${provs.size} providers</span> ready` : 'No API keys configured';
    pill.title = ready.map((e) => e.label).join(', ');
    $('#btn-saved').hidden = !S.health.saved_run;
  } catch (e) {
    pill.className = 'status-pill bad';
    pill.querySelector('.label').textContent = 'Server unreachable';
    toast('Cannot reach the Safe Procure server. Is it still running?', 'err');
  }
  lockNav();
}

/* ============================================================== configure */
function initConfigure() {
  // data source tabs
  $$('#panel-data [role="tab"]').forEach((b) => b.addEventListener('click', () => {
    $$('#panel-data [role="tab"]').forEach((x) => x.setAttribute('aria-selected', x === b));
    $('#src-sample').hidden = b.dataset.src !== 'sample';
    $('#src-upload').hidden = b.dataset.src !== 'upload';
  }));
  $$('#panel-data [role="tab"]').forEach((b, i, all) => b.addEventListener('keydown', (e) => {
    if (e.key === 'ArrowRight' || e.key === 'ArrowLeft') { const n = all[(i + (e.key === 'ArrowRight' ? 1 : all.length - 1)) % all.length]; n.focus(); n.click(); }
  }));

  // ranges with outputs
  const bindRange = (id, out, fmt = (v) => v, cb) => {
    const r = $(id);
    $(out).setAttribute('aria-hidden', 'true'); // the slider itself announces the value via aria-valuetext
    const upd = () => { setRangeFill(r); $(out).textContent = fmt(r.value); r.setAttribute('aria-valuetext', String(fmt(r.value))); cb?.(); };
    r.addEventListener('input', upd); upd();
  };
  bindRange('#in-nv', '#o-nv');
  bindRange('#in-mo', '#o-mo');
  bindRange('#in-th', '#o-th', (v) => v, () => updateLaunchInfo());
  bindRange('#in-otd', '#o-otd', (v) => `${v}%`);
  bindRange('#in-def', '#o-def', (v) => `${Number(v).toFixed(1)}%`);

  $('#btn-dice').addEventListener('click', () => { $('#in-seed').value = Math.floor(Math.random() * 99999) + 1; });
  $('#btn-generate').addEventListener('click', generate);

  // upload
  const drop = $('#drop');
  const file = $('#in-file');
  file.addEventListener('change', () => file.files[0] && upload(file.files[0]));
  ['dragenter', 'dragover'].forEach((ev) => drop.addEventListener(ev, (e) => { e.preventDefault(); drop.classList.add('over'); }));
  ['dragleave', 'drop'].forEach((ev) => drop.addEventListener(ev, (e) => { e.preventDefault(); drop.classList.remove('over'); }));
  drop.addEventListener('drop', (e) => e.dataTransfer.files[0] && upload(e.dataTransfer.files[0]));
  $('#dl-template').addEventListener('click', (e) => { e.preventDefault(); if (S.dataset) location.href = `/api/dataset/${S.dataset.dataset_id}/csv`; });

  // question
  const pc = $('#prompt-chips');
  // Chips ship in the HTML (no layout shift when the script arrives); build them only if missing.
  if (!pc.children.length) PROMPTS.forEach((p) => pc.append(h('button', { type: 'button' }, p)));
  $$('button', pc).forEach((b) => b.addEventListener('click', () => { $('#in-question').value = b.textContent; $('#in-question').focus(); updateLaunchInfo(); }));
  $('#in-question').addEventListener('input', debounce(updateLaunchInfo, 200));

  // weights
  const presets = $('#presets');
  Object.keys(PRESETS).forEach((name) => presets.append(h('button', { type: 'button', 'aria-pressed': name === 'Balanced', onclick: () => applyPreset(name) }, name)));
  const wbox = $('#weights');
  PILLARS.forEach((p) => {
    const id = `w-${p.key}`;
    const input = h('input', { type: 'range', id, min: 0, max: 60, value: S.weights[p.key] });
    const out = h('output', { id: `${id}-o`, for: id, 'aria-hidden': 'true' }, '');
    input.addEventListener('input', () => { S.weights[p.key] = Number(input.value); markPreset(); renderWeights(); rescoreSoon(); });
    wbox.append(h('div', { class: 'weight' }, h('label', { for: id }, h('span', { class: 'sw', style: { background: p.color } }), p.label), input, out));
  });
  renderWeights();

  $('#btn-run').addEventListener('click', startRun);
  $('#btn-saved').addEventListener('click', () => loadSaved());
  $('#btn-replay').addEventListener('click', replay);
}

function applyPreset(name) {
  S.weights = { ...PRESETS[name] };
  PILLARS.forEach((p) => { $(`#w-${p.key}`).value = S.weights[p.key]; });
  markPreset(); renderWeights(); rescoreSoon();
}
function markPreset() {
  const match = Object.entries(PRESETS).find(([, w]) => PILLARS.every((p) => w[p.key] === S.weights[p.key]))?.[0];
  $$('#presets button').forEach((b) => b.setAttribute('aria-pressed', b.textContent === match));
}
function renderWeights() {
  const total = PILLARS.reduce((a, p) => a + S.weights[p.key], 0) || 1;
  PILLARS.forEach((p) => {
    const r = $(`#w-${p.key}`); setRangeFill(r);
    const share = `${Math.round((S.weights[p.key] / total) * 100)}%`;
    $(`#w-${p.key}-o`).textContent = share;
    r.setAttribute('aria-valuetext', `${share} of total weight`);
  });
  const bar = $('#weight-bar');
  bar.innerHTML = '';
  PILLARS.forEach((p) => bar.append(h('span', { style: { flexGrow: S.weights[p.key] || 0.001, background: p.color } })));
}

function renderGenCats() {
  const cats = S.health?.categories || Object.keys(CAT_SHORT);
  if (!S.genCats) S.genCats = [...cats];
  const box = $('#gen-cats'); box.innerHTML = '';
  cats.forEach((c) => {
    const b = h('button', { type: 'button', class: 'chip', 'aria-pressed': S.genCats.includes(c) }, icon('check', 'tick'), CAT_SHORT[c] || c);
    b.addEventListener('click', () => {
      const on = S.genCats.includes(c);
      if (on && S.genCats.length <= 2) { toast('Keep at least two categories.'); return; }
      S.genCats = on ? S.genCats.filter((x) => x !== c) : [...S.genCats, c];
      b.setAttribute('aria-pressed', !on);
    });
    box.append(b);
  });
}

function renderScopeCats() {
  const cats = S.dataset?.quality.categories || [];
  const box = $('#scope-cats'); box.innerHTML = '';
  S.scopeCats = S.scopeCats.filter((c) => cats.includes(c));
  const all = h('button', { type: 'button', class: 'chip', 'aria-pressed': S.scopeCats.length === 0 }, icon('check', 'tick'), 'All categories');
  all.addEventListener('click', () => { S.scopeCats = []; renderScopeCats(); updateLaunchInfo(); renderRanks(S.dataset.scorecard); });
  box.append(all);
  cats.forEach((c) => {
    const b = h('button', { type: 'button', class: 'chip', 'aria-pressed': S.scopeCats.includes(c) }, icon('check', 'tick'), CAT_SHORT[c] || c);
    b.addEventListener('click', () => {
      S.scopeCats = S.scopeCats.includes(c) ? S.scopeCats.filter((x) => x !== c) : [...S.scopeCats, c];
      if (S.scopeCats.length === cats.length) S.scopeCats = [];
      renderScopeCats(); updateLaunchInfo(); renderRanks(S.dataset.scorecard);
    });
    box.append(b);
  });
}

async function generate() {
  const btn = $('#btn-generate');
  btn.classList.add('loading'); btn.disabled = true;
  try {
    const ds = await api('/api/dataset/sample', { method: 'POST', body: {
      n_vendors: Number($('#in-nv').value), months: Number($('#in-mo').value), seed: Number($('#in-seed').value) || 42,
      categories: S.genCats, inject_quality_issues: $('#in-issues').checked } });
    setDataset(ds);
  } catch (e) { toast(`Could not generate data: ${e.message}`, 'err'); }
  finally { btn.classList.remove('loading'); btn.disabled = false; }
}

async function upload(f) {
  const fd = new FormData(); fd.append('file', f);
  const box = $('#dataset');
  box.innerHTML = '<div class="skeleton-block" aria-hidden="true"></div>';
  try {
    const ds = await api('/api/dataset/upload', { method: 'POST', body: fd });
    setDataset(ds);
    toast(`Loaded ${f.name}: ${ds.quality.vendors} vendors.`);
  } catch (e) {
    box.innerHTML = '';
    box.append(h('div', { class: 'empty' }, h('b', {}, 'That file could not be used'), e.message));
    toast(e.message, 'err');
  }
}

function setDataset(ds) {
  S.dataset = ds;
  S.baselineRanks = Object.fromEntries(ds.scorecard.map((r) => [r.vendor, r.rank]));
  S.prevRanks = {};
  renderDataset();
  renderScopeCats();
  rescore();
  $('#btn-run').disabled = false;
  updateLaunchInfo();
}

function renderDataset() {
  const ds = S.dataset, q = ds.quality;
  const box = $('#dataset'); box.innerHTML = '';
  box.append(h('dl', { class: 'ds-facts' },
    h('div', {}, h('dt', {}, 'Vendors'), h('dd', {}, q.vendors)),
    h('div', {}, h('dt', {}, 'Period'), h('dd', {}, `${q.n_months} mo`)),
    h('div', {}, h('dt', {}, 'Spend'), h('dd', {}, inr(q.total_spend))),
    h('div', {}, h('dt', {}, 'Rows'), h('dd', {}, num(q.rows_in))),
    h('div', {}, h('dt', {}, 'Quality'), h('dd', {}, `${q.quality_score}`)),
  ));
  if (q.issues.length) {
    const ul = h('ul', { class: 'issues', 'aria-label': 'Data issues found and auto-cleaned' });
    q.issues.forEach((i) => ul.append(h('li', {}, icon('alert'), h('span', { html: `<b>${i.count} ${esc(i.type.replace('_', ' '))}</b> in <code class="mono">${esc(i.column)}</code>. ${esc(i.action)}.` }))));
    box.append(ul);
  }
  if (ds.notes?.length) box.append(h('ul', { class: 'notes' }, ds.notes.map((n) => h('li', {}, n))));
  box.append(h('div', { class: 'mini-chart', id: 'mini-chart', role: 'img', 'aria-label': 'Monthly spend across all vendors' }));
  renderMiniChart();
  const cols = ds.columns;
  const table = h('table', { class: 'data' },
    h('thead', {}, h('tr', {}, cols.map((c) => h('th', { scope: 'col' }, c)))),
    h('tbody', {}, ds.preview.map((r) => h('tr', {}, cols.map((c) => h('td', { class: typeof r[c] === 'string' ? 'txt' : '' }, r[c]))))));
  box.append(h('details', { class: 'preview' }, h('summary', {}, `Preview first ${ds.preview.length} of ${num(ds.row_count)} rows`), h('div', { class: 'table-scroll', tabindex: '0' }, table)));
}

function renderMiniChart() {
  const node = $('#mini-chart');
  if (!node || !window.echarts || !S.dataset) return;
  const css = (n) => getComputedStyle(document.documentElement).getPropertyValue(n).trim();
  const ch = echarts.getInstanceByDom(node) || echarts.init(node, null, { renderer: 'svg' });
  const m = S.dataset.monthly_spend;
  ch.setOption({
    animationDuration: 600, grid: { left: 6, right: 6, top: 10, bottom: 20 },
    xAxis: { type: 'category', data: m.months, axisLine: { show: false }, axisTick: { show: false }, axisLabel: { color: css('--ink-3'), fontSize: 10, interval: Math.ceil(m.months.length / 6) - 1, fontFamily: 'Geist Mono' } },
    yAxis: { type: 'value', show: false },
    tooltip: { trigger: 'axis', valueFormatter: (v) => inr(v), backgroundColor: css('--surface'), borderColor: css('--line'), textStyle: { color: css('--ink'), fontFamily: 'Geist' } },
    series: [{ type: 'bar', data: m.spend, barWidth: '62%', itemStyle: { color: css('--brand'), borderRadius: [3, 3, 0, 0], opacity: 0.85 } }],
  }, true);
}

const rescoreSoon = debounce(() => rescore(), 140);
async function rescore() {
  if (!S.dataset) return;
  try {
    const r = await api(`/api/dataset/${S.dataset.dataset_id}/score`, { method: 'POST', body: { weights: S.weights } });
    S.dataset.scorecard = r.scorecard;
    renderRanks(r.scorecard);
  } catch (e) { toast(e.message, 'err'); }
}

function renderRanks(rows) {
  const list = $('#rank-list');
  const scoped = S.scopeCats.length ? rows.filter((r) => S.scopeCats.includes(r.category)) : rows;
  const prev = S.prevRanks;
  // FLIP: remember where each vendor row was, re-render, then glide rows from old to new positions.
  const before = Object.fromEntries($$('li[data-v]', list).map((li) => [li.dataset.v, li.getBoundingClientRect()]));
  list.innerHTML = '';
  scoped.forEach((r, i) => {
    const base = S.baselineRanks[r.vendor];
    const mv = base ? base - r.rank : 0;
    const li = h('li', { class: prev[r.vendor] && prev[r.vendor] !== r.rank ? 'flash' : '', 'data-v': r.vendor, style: { '--tc': `var(--${{ A: 'keep', B: 'develop', C: 'watch', D: 'replace' }[r.tier] || 'brand'})`, '--sc': `${Math.max(4, r.composite)}%` } },
      h('span', { class: 'rk' }, i + 1),
      h('span', { class: 'nm' }, h('b', { title: r.vendor }, short(r.vendor)), h('span', {}, `${CAT_SHORT[r.category] || r.category} · risk ${Math.round(r.internal_risk)}`)),
      h('span', { class: 'sc' },
        h('span', { class: `mv ${mv > 0 ? 'up' : mv < 0 ? 'down' : ''}`, title: mv ? 'Rank change vs balanced weights' : null, 'aria-hidden': 'true' },
          mv ? icon(mv > 0 ? 'arrow-up' : 'arrow-down') : '', mv ? h('span', {}, Math.abs(mv)) : ''),
        mv ? h('span', { class: 'sr-only' }, `${mv > 0 ? 'up' : 'down'} ${Math.abs(mv)} places versus balanced weights, `) : '',
        h('span', {}, h('span', { class: 'sr-only' }, 'score '), r.composite.toFixed(1)), tierChip(r.tier)));
    list.append(li);
  });
  if (!RM.matches && Object.keys(before).length) {
    $$('li[data-v]', list).forEach((li) => {
      const o = before[li.dataset.v];
      if (!o) return;
      const n = li.getBoundingClientRect(), dx = o.left - n.left, dy = o.top - n.top;
      if (Math.abs(dx) < 1 && Math.abs(dy) < 1) return;
      li.animate([{ transform: `translate(${dx}px, ${dy}px)`, zIndex: 2 }, { transform: 'none', zIndex: 2 }], { duration: 650, easing: 'cubic-bezier(.2,1.35,.4,1)' });
    });
  }
  // One short announcement per rescore instead of re-reading the whole list.
  const moved = Object.keys(prev).length ? rows.filter((r) => prev[r.vendor] && prev[r.vendor] !== r.rank).length : 0;
  if (Object.keys(prev).length && scoped.length) $('#rank-sr').textContent = `Ranking updated. ${moved ? `${moved} vendors changed rank. ` : 'No rank changes. '}Top vendor: ${short(scoped[0].vendor)}.`;
  S.prevRanks = Object.fromEntries(rows.map((r) => [r.vendor, r.rank]));
}

function updateLaunchInfo() {
  const info = $('#launch-info');
  const noKeys = S.health && !S.health.endpoints.some((e) => e.available);
  $('#btn-run').disabled = !S.dataset || noKeys;
  if (noKeys) { info.innerHTML = '<b>No model keys on the server.</b> Set GROQ_API_KEY and/or GEMINI_API_KEY, restart, and the agents will be ready. The instant scorecard still works.'; return; }
  if (!S.dataset) { info.textContent = 'Load or generate a dataset to begin.'; return; }
  const n = S.scopeCats.length ? S.dataset.scorecard.filter((r) => S.scopeCats.includes(r.category)).length : S.dataset.scorecard.length;
  const q = $('#in-question').value.trim();
  info.innerHTML = `<b>${n} vendors</b> in scope · risk tolerance <b>${$('#in-th').value}</b> · ${q ? 'custom question' : 'full portfolio review'}. Typical run: about 1.5 minutes on free-tier models.`;
}

/* ==================================================================== run */
const RUN = { graph: null, es: null, timer: null, t0: 0, stats: {}, lanes: {}, events: [], stageOut: {}, stageModel: {}, pollSeq: 0 };

async function startRun() {
  if (!S.dataset) return;
  const btn = $('#btn-run');
  btn.classList.add('loading'); btn.disabled = true;
  try {
    const body = {
      dataset_id: S.dataset.dataset_id, question: $('#in-question').value.trim(), weights: S.weights,
      risk_threshold: Number($('#in-th').value), otd_floor: Number($('#in-otd').value), defect_ceiling: Number($('#in-def').value),
      focus_categories: S.scopeCats,
    };
    const r = await api('/api/runs', { method: 'POST', body });
    RUN.replayTimers?.forEach(clearTimeout); RUN.replaying = false;
    S.run = { id: r.run_id, done: false, question: body.question || 'Full vendor portfolio risk review' };
    S.result = null;
    store.set('sp-last-run', r.run_id);
    warp();
    beginRunView();
    location.hash = '#/run';
  } catch (e) {
    toast(`Could not start: ${e.message}`, 'err');
  } finally { btn.classList.remove('loading'); btn.disabled = false; lockNav(); }
}

function beginRunView(live = true) {
  Object.assign(RUN, { t0: Date.now(), stats: { stages: 0, calls: 0, tokens: 0, fail: 0 }, events: [], stageOut: {}, stageModel: {}, pollSeq: 0, lanes: {} });
  $('#run-question').textContent = `“${S.run.question}”`;
  $('#trace').innerHTML = '';
  $('#trace').append(h('li', { class: 'trace-empty' }, 'Connecting to the agent crew…'));
  $('#run-done-bar').hidden = true;
  $('#inspector').innerHTML = '';
  RUN.graph.reset();
  renderLanes();
  clearInterval(RUN.timer);
  RUN.timer = setInterval(tickRun, 500);
  setMode('run');
  const pill = $('#status-pill');
  if (live) {
    RUN.timeScale = 1; RUN.replaying = false;
    pill.className = 'status-pill live'; pill.querySelector('.label').textContent = 'Agents running';
    connect();
  }
  $('#btn-replay').hidden = true;
}

function connect() {
  RUN.es?.close();
  let gotAny = false;
  const es = new EventSource(`/api/runs/${S.run.id}/events`);
  RUN.es = es;
  es.onmessage = (m) => { gotAny = true; const ev = JSON.parse(m.data); if (ev.seq >= RUN.pollSeq) { RUN.pollSeq = ev.seq + 1; onEvent(ev); } };
  es.addEventListener('done', () => { es.close(); finishRun(); });
  es.onerror = () => { es.close(); if (!S.run.done) poll(); };
  setTimeout(() => { if (!gotAny && !S.run.done) { es.close(); poll(); } }, 6000);
}

async function poll() {
  if (S.run.done) return;
  try {
    const r = await api(`/api/runs/${S.run.id}/poll?since=${RUN.pollSeq}`);
    r.events.forEach((ev) => { if (ev.seq >= RUN.pollSeq) { RUN.pollSeq = ev.seq + 1; onEvent(ev); } });
    if (r.done) { finishRun(); return; }
  } catch (e) { /* transient */ }
  setTimeout(poll, 1200);
}

function tickRun() {
  const s = ((Date.now() - RUN.t0) / 1000) * (RUN.timeScale || 1);
  $('#st-time').textContent = clock(s);
  $('#st-stages').textContent = `${RUN.stats.stages}/10`;
  $('#st-calls').textContent = num(RUN.stats.calls);
  $('#st-tokens').textContent = num(RUN.stats.tokens);
  $('#st-fail').textContent = num(RUN.stats.fail);
  renderLanes();
}

const STAGE_OF = Object.fromEntries(Object.entries(NODES).map(([k, n]) => [n.name, k]));

function onEvent(ev) {
  RUN.events.push(ev);
  const stage = ev.stage || STAGE_OF[ev.agent];
  switch (ev.type) {
    case 'stage_start': RUN.graph.set(ev.stage, 'active'); break;
    case 'stage_end':
      RUN.stats.stages = new Set(RUN.events.filter((e) => e.type === 'stage_end').map((e) => e.stage)).size;
      RUN.stageOut[ev.stage] = ev.output;
      RUN.graph.set(ev.stage, ev.status === 'ok' ? 'done' : 'degraded', RUN.stageModel[ev.stage] ? `${RUN.stageModel[ev.stage]} · ${ev.seconds}s` : `${ev.seconds}s`);
      if (RUN.graph.selected === ev.stage || !RUN.graph.selected) inspect(ev.stage);
      pulseAt(RUN.graph.nodes[ev.stage]?.g.querySelector('.card'), ev.status === 'ok' ? 'keep' : 'watch');
      $('#run-sr').textContent = `${NODES[ev.stage]?.name || ev.agent} ${ev.status === 'ok' ? 'finished' : 'fell back to rules'}. ${RUN.stats.stages} of 10 stages done.`;
      break;
    case 'llm_end':
      RUN.stats.calls += 1; RUN.stats.tokens += (ev.prompt_tokens || 0) + (ev.completion_tokens || 0);
      (RUN.lanes[ev.model] ||= []).push({ t: ev.t, tok: (ev.prompt_tokens || 0) + (ev.completion_tokens || 0) });
      if (stage) { RUN.stageModel[stage] = ev.model; if (RUN.graph.state[stage] === 'active') RUN.graph.model(stage, `on ${ev.model}`); }
      break;
    case 'llm_start':
      if (stage && RUN.graph.state[stage] === 'active') RUN.graph.model(stage, `thinking · ${ev.model}`);
      break;
    case 'tool_start':
      if (stage && RUN.graph.state[stage] === 'active') RUN.graph.model(stage, `using ${ev.tool}`);
      break;
    case 'failover': RUN.stats.fail += 1; break;
    case 'route': RUN.graph.route(ev.stage, ev.route); pulseAt(ev.stage === 'risk_gate' ? RUN.graph.gate : RUN.graph.nodes.qa?.g, ev.route === 'mitigate' || ev.route === 'revise' ? 'replace' : 'brand'); break;
    default: break;
  }
  appendTrace(ev);
}

function argsText(a) {
  if (!a || !Object.keys(a).length) return '';
  return Object.entries(a).map(([k, v]) => `${k}=${Array.isArray(v) ? `[${v.length}]` : JSON.stringify(v).slice(0, 60)}`).join(', ');
}

function stageSummary(o) {
  if (!o) return '';
  return o.headline || o.summary || o.portfolio_summary || o.objective || (o.verdict ? `Verdict: ${o.verdict}` : '') || '';
}

function appendTrace(ev) {
  const list = $('#trace');
  list.querySelector('.trace-empty')?.remove();
  let cls = '', ic = 'network', html = '', verbose = false;
  const A = `<b>${esc(ev.agent || '')}</b>`;
  switch (ev.type) {
    case 'run_start': cls = 'k-stage'; ic = 'play'; html = `Run started with <b>${ev.vendors}</b> vendors in scope`; break;
    case 'stage_start': cls = 'k-stage'; ic = 'network'; html = `${A} started`; break;
    case 'stage_end': cls = ev.status === 'ok' ? 'k-done' : 'k-warn'; ic = ev.status === 'ok' ? 'check' : 'alert';
      html = `${A} finished in ${ev.seconds}s<span class="q">${esc(String(stageSummary(ev.output)).slice(0, 180))}</span>`; break;
    case 'stage_error': cls = 'k-bad'; ic = 'alert'; html = `${A} hit an error, so deterministic rules took over for this stage<span class="q">${esc(ev.message)}</span>`; break;
    case 'tool_start': ic = 'tool'; html = `${A} called <code>${esc(ev.tool)}(${esc(argsText(ev.args))})</code>`; break;
    case 'tool_end': ic = 'tool'; verbose = true; html = `<code>${esc(ev.tool)}</code> returned ${num(ev.chars)} chars in ${ev.seconds}s`; break;
    case 'llm_end': ic = 'spark'; verbose = true; html = `${A} ← ${esc(ev.model)} · ${num(ev.prompt_tokens)} in / ${num(ev.completion_tokens)} out · ${ev.seconds}s`; break;
    case 'llm_error': cls = 'k-warn'; ic = 'alert'; html = `${esc(ev.model)} refused a call from ${A} (${esc(ev.kind.replace('_', ' '))})`; break;
    case 'failover': cls = 'k-warn'; ic = 'shuffle'; html = `Router moved ${A} to <b>${esc(ev.to_model)}</b> (${esc(ev.reason)})`; break;
    case 'throttle': ic = 'gauge'; verbose = true; html = `Pacing ${A}: waiting ${ev.wait}s for ${esc(ev.model)} budget`; break;
    case 'risk_blend': cls = 'k-route'; ic = 'globe'; html = `Market risk blended into the scorecard: <b>${ev.flagged}</b> vendors above tolerance`; break;
    case 'route': cls = 'k-route'; ic = 'flag';
      html = ev.stage === 'risk_gate' ? `Risk gate → <b>${ev.route === 'mitigate' ? 'Mitigation Planner' : ev.reason === 'already_planned' ? 'QA (mitigation plan already covers these vendors)' : 'QA (no high-risk vendors)'}</b>${ev.targets?.length ? `<span class="q">${esc(ev.targets.map(short).join(', '))}</span>` : ''}`
        : `QA verdict → <b>${ev.route === 'revise' ? 'send back to Strategist' : 'publish'}</b>`; break;
    case 'guardrails': cls = ev.flags?.length ? 'k-warn' : 'k-done'; ic = 'shield'; html = `Guardrails ran: <b>${ev.flags?.length || 0}</b> automated flags`; break;
    case 'run_end': cls = 'k-done'; ic = 'check'; html = `Run complete · ${num(ev.usage?.llm_calls)} LLM calls · ${num(ev.usage?.total_tokens)} tokens`; break;
    case 'run_error': cls = 'k-bad'; ic = 'alert'; html = `Run error: ${esc(ev.message)}`; break;
    default: return;
  }
  const li = h('li', { class: `${cls}${verbose ? ' v' : ''}` }, h('span', { class: 'tt' }, clock(ev.t)), h('span', { class: 'ic' }, icon(ic)), h('span', { class: 'tx', html }));
  const nearBottom = list.scrollHeight - list.scrollTop - list.clientHeight < 80;
  list.append(li);
  if (nearBottom) list.scrollTop = list.scrollHeight;
}

function renderLanes() {
  const box = $('#lanes');
  if (!S.health) return;
  const now = RUN.events.length ? RUN.events[RUN.events.length - 1].t + (S.run && !S.run.done ? 0 : 0) : 0;
  const tNow = S.run && !S.run.done ? ((Date.now() - RUN.t0) / 1000) * (RUN.timeScale || 1) : now;
  const rows = S.health.endpoints.filter((e) => e.available);
  if (box.children.length !== rows.length) {
    box.innerHTML = '';
    rows.forEach((e) => box.append(h('div', { class: 'lane idle', 'data-lane': e.label },
      h('div', { class: 'nm' }, h('b', {}, e.label), h('span', {}, `${e.provider === 'groq' ? 'Groq' : 'Google'} · ${e.tpm >= 1000 ? `${Math.round(e.tpm / 1000)}k` : e.tpm} tokens/min`)),
      h('div', { class: 'bar', role: 'meter', 'aria-label': `${e.label} budget used`, 'aria-valuemin': 0, 'aria-valuemax': 100, 'aria-valuenow': 0 }, h('i', { style: { width: '0%' } })),
      h('div', { class: 'vl' }, '0'))));
  }
  const cooling = new Set(RUN.events.filter((e) => e.type === 'llm_error' && tNow - e.t < 30).map((e) => e.model));
  rows.forEach((e) => {
    const lane = box.querySelector(`[data-lane="${CSS.escape(e.label)}"]`);
    const used = (RUN.lanes[e.label] || []).filter((x) => tNow - x.t <= 60).reduce((a, x) => a + x.tok, 0);
    const calls = (RUN.lanes[e.label] || []).length;
    const p = Math.min(100, (used / e.tpm) * 100);
    const bar = lane.querySelector('.bar i');
    bar.style.width = `${p}%`;
    bar.className = cooling.has(e.label) ? 'cool' : p > 75 ? 'hot' : '';
    lane.querySelector('.bar').setAttribute('aria-valuenow', Math.round(p));
    lane.querySelector('.vl').textContent = calls ? `${used >= 1000 ? `${(used / 1000).toFixed(1)}k` : used} · ${calls}×` : '—';
    lane.classList.toggle('idle', !calls);
  });
}

function inspect(id) {
  const n = NODES[id];
  const box = $('#inspector');
  const evs = RUN.events.filter((e) => e.stage === id || STAGE_OF[e.agent] === id);
  const tools = [...new Set(evs.filter((e) => e.type === 'tool_start').map((e) => e.tool))];
  const models = [...new Set(evs.filter((e) => e.type === 'llm_end').map((e) => e.model))];
  const end = evs.find((e) => e.type === 'stage_end');
  const out = RUN.stageOut[id];
  box.innerHTML = '';
  box.append(h('div', { class: 'insp-head' },
    h('div', {}, h('h3', {}, n.name), h('p', {}, n.role)),
    end ? h('span', { class: `decision ${end.status === 'ok' ? 'Keep' : 'Watch'}` }, end.status === 'ok' ? `Done in ${end.seconds}s` : 'Rule fallback') : RUN.graph.state[id] === 'failed' ? h('span', { class: 'decision Replace' }, 'Stopped here')
      : h('span', { class: 'tag' }, RUN.graph.state[id] === 'active' ? 'Working…' : RUN.graph.state[id] === 'skipped' ? 'Skipped by router' : 'Waiting')));
  box.append(h('div', { class: 'kv' },
    models.map((m) => h('span', { class: 'tag brand' }, icon('spark'), m)),
    tools.map((t) => h('span', { class: 'tag' }, icon('tool'), t)),
    h('span', { class: 'tag' }, `${evs.filter((e) => e.type === 'llm_end').length} LLM calls`)));
  if (out) {
    const sum = stageSummary(out);
    if (sum) box.append(h('p', { class: 'insp-out' }, sum));
    box.append(h('details', {}, h('summary', { class: 'meta' }, 'Structured output (JSON)'), h('pre', { class: 'json' }, JSON.stringify(out, null, 2))));
  } else {
    const err = RUN.graph.state[id] === 'failed' && [...RUN.events].reverse().find((e) => e.type === 'run_error' || e.type === 'stage_error');
    box.append(h('p', { class: 'meta' }, err ? `The run ended before this agent produced output: ${err.message}` : 'Output appears here when the agent finishes. Select any node to inspect it.'));
  }
}

async function finishRun() {
  if (S.run.done) return;
  S.run.done = true;
  clearInterval(RUN.timer); tickRun();
  // A fatal run_error or HTTP 410 means no result is coming; otherwise retry briefly while the result is written.
  const fatal = RUN.events.some((e) => e.type === 'run_error');
  for (let i = 0; i < (fatal ? 1 : 20); i++) {
    try {
      const r = await api(`/api/runs/${S.run.id}`);
      if (!r._pending) { S.result = r; break; }
    } catch (e) { if (e.status === 410) break; }
    await new Promise((res) => setTimeout(res, 800));
  }
  const pill = $('#status-pill');
  if (S.result) {
    pill.className = 'status-pill ok'; pill.querySelector('.label').textContent = 'Analysis ready';
    const u = S.result.usage;
    $('#run-done-info').innerHTML = `<b>Done in ${clock(u.elapsed)}</b> · ${num(u.llm_calls)} model calls across ${Object.keys(u.per_model).length} models · ${num(u.failovers)} automatic failovers${S.result.degraded.length ? ` · ${S.result.degraded.length} stage(s) on rule fallback` : ''}`;
    $('#run-done-actions').innerHTML = '<a class="btn primary lg" href="#/insights"><svg class="i"><use href="#i-chart"/></svg>Open insights</a>';
    $('#run-done-bar').hidden = false;
    toast('Analysis complete. Insights are ready.');
    setMode('calm'); burst(innerWidth / 2, innerHeight * 0.45, decisionMix(S.result), 160);
    if (S.health) S.health.saved_run = true;
  } else {
    pill.className = 'status-pill bad'; pill.querySelector('.label').textContent = 'Run failed';
    setMode('calm');
    showRunFailure();
  }
  lockNav();
}

// Failed run: say where and why it stopped, mark that node on the graph, and offer a one-click retry.
function showRunFailure() {
  const order = Object.keys(NODES);
  const err = [...RUN.events].reverse().find((e) => e.type === 'run_error' || e.type === 'stage_error');
  const active = order.filter((id) => RUN.graph.state[id] === 'active');
  const lastDone = [...RUN.events].reverse().find((e) => e.type === 'stage_end')?.stage;
  active.forEach((id) => RUN.graph.set(id, 'failed'));
  const focusNode = active[0] || lastDone;
  if (focusNode) RUN.graph.select(focusNode);
  const last = RUN.events[RUN.events.length - 1];
  if (last?.t) $('#st-time').textContent = clock(last.t);
  const names = (ids) => new Intl.ListFormat('en', { type: 'conjunction' }).format(ids.map((id) => NODES[id].name));
  const where = active.length ? `at ${names(active)}` : lastDone ? `after ${NODES[lastDone].name}` : 'before any agent finished';
  const why = err?.message ? ` ${esc(String(err.message).split('\n')[0].slice(0, 140).replace(/\.$/, ''))}.` : '';
  $('#run-done-info').innerHTML = `<b>Run stopped ${esc(where)}.</b>${why} ${RUN.stats.stages} of 10 stages finished. Your data and settings are kept.`;
  const actions = $('#run-done-actions');
  actions.innerHTML = '';
  actions.append(
    h('a', { class: 'btn lg', href: '#/configure' }, icon('sliders'), 'Adjust settings'),
    h('button', { class: 'btn primary lg', type: 'button', id: 'btn-retry', 'aria-describedby': 'run-done-info', onclick: () => startRun() }, icon('refresh'), 'Retry run'));
  $('#run-done-bar').hidden = false;
  toast(`Run stopped ${active.length > 1 ? `with ${active.length} agents mid-task` : where}. Retry, or adjust settings first.`, 'err');
  $('#btn-retry').focus({ preventScroll: true });
}

async function loadSaved(toRun = false) {
  try {
    S.result = await api('/api/runs/saved/latest');
    lockNav();
    const want = toRun ? '#/run' : '#/insights';
    if (location.hash !== want) location.hash = want; else route();
  } catch (e) { toast('No saved run yet.', 'err'); location.hash = '#/configure'; }
}

/* ================================================================ replay */
function showReplayIdle() {
  if (RUN.replaying) return;
  $('#run-question').innerHTML = `“${esc(S.result.question || S.result.plan?.objective || 'Full vendor portfolio review')}”<span class="replay-tag">Saved run · ${esc(S.result.created || '')}</span>`;
  $('#trace').innerHTML = '';
  $('#trace').append(h('li', { class: 'trace-empty' }, 'This run finished earlier. Press “Replay this run” to watch the agents work through it again at 6× speed.'));
  RUN.graph.reset();
  $('#btn-replay').hidden = false;
}

function replay() {
  const r = S.result;
  if (!r?.events?.length) { toast('This run has no recorded trace.', 'err'); return; }
  RUN.replayTimers?.forEach(clearTimeout);
  S.run = { id: r.run_id, done: false, question: r.question || r.plan?.objective || 'Full vendor portfolio review', replay: true };
  RUN.replaying = true;
  beginRunView(false);
  $('#run-question').innerHTML = `“${esc(S.run.question)}”<span class="replay-tag">Replay · 6×</span>`;
  const speed = 6;
  RUN.t0 = Date.now();
  RUN.timeScale = speed;
  RUN.replayTimers = r.events.map((ev) => setTimeout(() => onEvent(ev), (ev.t * 1000) / speed));
  const end = r.events[r.events.length - 1].t;
  RUN.replayTimers.push(setTimeout(() => { RUN.replaying = false; S.run.done = true; clearInterval(RUN.timer); RUN.timeScale = 1; finishReplay(); }, (end * 1000) / speed + 300));
}

function finishReplay() {
  const u = S.result.usage;
  setMode('calm'); burst(innerWidth / 2, innerHeight * 0.45, decisionMix(S.result), 120);
  $('#st-time').textContent = clock(u.elapsed);
  $('#run-done-info').innerHTML = `<b>Recorded run: ${clock(u.elapsed)}</b> · ${num(u.llm_calls)} model calls across ${Object.keys(u.per_model).length} models · ${num(u.failovers)} automatic failovers`;
  $('#run-done-bar').hidden = false;
  lockNav();
}

/* =============================================================== copilot */
// Minimal, safe markdown: escape first, then bold / italics / bullet lists / line breaks.
function miniMd(text) {
  const lines = esc(String(text || '')).split('\n');
  let out = '', inList = false;
  for (let line of lines) {
    line = line.replace(/\*\*(.+?)\*\*/g, '<b>$1</b>').replace(/(^|\s)\*(\S.*?)\*(?=\s|$)/g, '$1<i>$2</i>');
    const m = line.match(/^\s*(?:[-*•]|\d+\.)\s+(.*)$/);
    if (m) { if (!inList) { out += '<ul>'; inList = true; } out += `<li>${m[1]}</li>`; continue; }
    if (inList) { out += '</ul>'; inList = false; }
    if (/^\s*\|/.test(line)) { out += `<code class="tbl">${line}</code><br>`; continue; }
    out += line.trim() ? `${line}<br>` : '<br>';
  }
  if (inList) out += '</ul>';
  return out.replace(/(<br>){3,}/g, '<br><br>');
}
const CP = { history: [] };
function initCopilot() {
  const fab = $('#copilot-fab'), panel = $('#copilot');
  fab.addEventListener('click', () => (panel.hidden ? openCopilot() : closeCopilot()));
  $('#copilot-close').addEventListener('click', closeCopilot);
  document.addEventListener('keydown', (e) => { if (e.key === 'Escape' && !panel.hidden) closeCopilot(); });
  $('#cp-form').addEventListener('submit', (e) => { e.preventDefault(); const v = $('#cp-input').value.trim(); if (v) ask(v); });
}
function openCopilot() {
  const panel = $('#copilot');
  panel.hidden = false; $('#copilot-fab').setAttribute('aria-expanded', 'true');
  if (!$('#cp-log').children.length) {
    const brief = S.result?.brief || {};
    $('#cp-log').append(h('li', { class: 'a' }, `I have this run's full scorecard, decisions and plans. ${brief.headline ? `Headline: ${brief.headline}` : ''}`));
    const reps = (S.result?.vendors || []).filter((v) => v.decision?.decision === 'Replace').map((v) => short(v.vendor));
    const sugg = [reps[0] ? `Why replace ${reps[0]}?` : 'Which vendor is riskiest?', 'What if delivery weight is 50%?', 'Who can absorb Steel volume?', 'Summarise savings in one line'];
    const box = $('#cp-suggest'); box.innerHTML = '';
    sugg.forEach((s) => box.append(h('button', { type: 'button', onclick: () => ask(s) }, s)));
  }
  $('#cp-input').focus();
  $('#copilot-fab').hidden = true; // hide only after focus has moved, so focus never lands on a hidden control
}
function closeCopilot() {
  const panel = $('#copilot');
  if (panel.hidden) return;
  panel.hidden = true; $('#copilot-fab').setAttribute('aria-expanded', 'false');
  $('#copilot-fab').hidden = !(S.route === 'insights' && S.result);
  $('#copilot-fab').focus();
}
async function ask(text) {
  const log = $('#cp-log');
  log.append(h('li', { class: 'u' }, text));
  $('#cp-input').value = '';
  const typing = h('li', { class: 'a typing', 'aria-label': 'Copilot is thinking' }, h('i'), h('i'), h('i'));
  log.append(typing); log.scrollTop = log.scrollHeight;
  try {
    const r = await api(`/api/runs/${S.result.run_id}/chat`, { method: 'POST', body: { message: text, history: CP.history } });
    typing.remove();
    const via = [r.models?.length ? `via ${r.models.join(', ')}` : '', r.tools?.length ? `tools: ${[...new Set(r.tools)].join(', ')}` : '',
      r.corrections?.length ? `data check corrected ${r.corrections.length} claim${r.corrections.length > 1 ? 's' : ''}` : '', `${r.seconds}s`].filter(Boolean).join(' · ');
    const li = h('li', { class: 'a md', html: miniMd(r.answer) });
    li.append(h('span', { class: 'via' }, via));
    log.append(li);
    CP.history.push({ role: 'user', content: text }, { role: 'assistant', content: r.answer });
  } catch (e) {
    typing.remove();
    log.append(h('li', { class: 'a' }, `I could not answer that: ${e.message}`));
  }
  log.scrollTop = log.scrollHeight;
}

/* ================================================================== boot */
function initHero() {
  const hr = new Date().getHours();
  const shift = hr < 5 ? 'Night watch' : hr < 12 ? 'Morning shift' : hr < 17 ? 'Afternoon shift' : hr < 21 ? 'Evening shift' : 'Night watch';
  $('#kicker').innerHTML = `${shift}<span class="kx"> · procurement observatory</span> · ${new Date().toLocaleTimeString('en-IN', { hour: '2-digit', minute: '2-digit' })}`;
  // Decorative crew ring: the ten agents waiting around the shield.
  const ring = $('#crew-ring');
  const ids = Object.keys(NODES);
  const pts = ids.map((id, i) => { const a = (i / ids.length) * Math.PI * 2 - Math.PI / 2; return [Math.cos(a) * 92, Math.sin(a) * 92]; });
  ring.innerHTML = `<defs><radialGradient id="cr-g"><stop offset="0" stop-color="var(--brand)" stop-opacity=".35"/><stop offset="1" stop-color="var(--brand)" stop-opacity="0"/></radialGradient></defs>
    <circle r="116" fill="url(#cr-g)"/><circle r="92" class="cr-orbit"/><circle r="60" class="cr-orbit in"/>
    <g class="cr-spin">${pts.map(([x, y], i) => `<line x1="0" y1="0" x2="${x.toFixed(1)}" y2="${y.toFixed(1)}" class="cr-link" style="--i:${i}"/>`).join('')}
    ${pts.map(([x, y], i) => `<g class="cr-node" style="--i:${i}" transform="translate(${x.toFixed(1)} ${y.toFixed(1)})"><circle r="13"/><text y="4" text-anchor="middle">${i + 1}</text></g>`).join('')}</g>
    <path class="cr-shield" d="M0 -30 22 -22v17c0 14-9 24-22 28-13-4-22-14-22-28v-17Z"/><path class="cr-tick" d="m-9 0 6 6 12-13"/>`;
}

function commands() {
  const haveRun = !!(S.result || S.health?.saved_run);
  const tabs = TAB_LIST.map(([id, label]) => ({ group: 'Insights', label, icon: 'chart', hidden: !haveRun, run: () => { selectTab(id); if (S.route !== 'insights') location.hash = '#/insights'; } }));
  const vend = (S.result?.vendors || []).map((v) => ({ group: 'Vendors', label: short(v.vendor), sub: `${v.decision?.decision || 'out of scope'} · ${v.category} · risk ${Math.round(v.combined_risk)}`, icon: 'flag',
    run: () => { if (S.route !== 'insights') { location.hash = '#/insights'; setTimeout(() => openVendor(v.vendor), 500); } else openVendor(v.vendor); } }));
  return [
    { group: 'Go to', label: 'Configure', sub: 'Data, question, weights', icon: 'sliders', run: () => { location.hash = '#/configure'; } },
    { group: 'Go to', label: 'Agents', sub: 'Live workflow graph', icon: 'network', hidden: $('[data-nav="run"]').classList.contains('locked'), run: () => { location.hash = '#/run'; } },
    { group: 'Go to', label: 'Insights', sub: 'Brief and nine views', icon: 'chart', hidden: $('[data-nav="insights"]').classList.contains('locked'), run: () => { location.hash = '#/insights'; } },
    { group: 'Actions', label: 'Run the 10-agent analysis', icon: 'play', hidden: $('#btn-run').disabled, run: () => startRun() },
    { group: 'Actions', label: 'Replay the saved run at 6×', icon: 'history', hidden: !haveRun, run: async () => { if (!S.result) await loadSaved(true); else location.hash = '#/run'; setTimeout(replay, 500); } },
    { group: 'Actions', label: 'Ask the copilot', icon: 'spark', hidden: !S.result, run: () => { if (S.route !== 'insights') { location.hash = '#/insights'; setTimeout(openCopilot, 500); } else openCopilot(); } },
    { group: 'Actions', label: 'Generate a new sample dataset', icon: 'refresh', run: () => { location.hash = '#/configure'; generate(); } },
    { group: 'Actions', label: 'Switch colour theme', icon: 'moon', run: () => $('#theme-toggle').click() },
    { group: 'Actions', label: 'Download executive report', icon: 'download', hidden: !S.result, run: () => { location.href = `/api/runs/${S.result.run_id}/report`; } },
    ...tabs, ...vend,
  ];
}

async function boot() {
  initFx();
  initHero();
  initPalette(commands);
  addEventListener('resize', debounce(moveNavIndicator, 100));
  document.fonts?.ready.then(moveNavIndicator);
  document.addEventListener('sp-egg', () => toast('Observatory mode: every decision colour at once.'));
  initTheme();
  RUN.graph = new FlowGraph($('#flow-graph'), (id) => inspect(id));
  RUN.graph.buildList($('#flow-list'));
  initConfigure();
  initCopilot();
  window.addEventListener('hashchange', route);
  renderGenCats(); // default categories first, so the panel does not grow when health arrives
  $('.cfg-grid').classList.add('ready');
  await loadHealth();
  renderGenCats();
  await generate();
  route();
  S.booted = true;
  renderLanes();
}

boot();
onEcharts(() => renderMiniChart());
export { RUN };

Writing safeprocure/web/app.js


In [25]:
%%writefile safeprocure/web/insights.js
// Insights: nine views over one finished run.
import { $, $$, h, icon, esc, inr, pct, num, short, css, riskColor, sparkline, api, debounce, setRangeFill, DECISIONS, decisionVar, store, tierChip, onEcharts } from './ui.js';
import { decisionOrbit } from './orbit.js';

const TABS = [
  ['brief', 'Brief'], ['vendors', 'Vendors'], ['risk', 'Risk map'], ['spend', 'Spend'], ['forecast', 'Forecast'],
  ['market', 'Market intel'], ['plan', 'Mitigation'], ['whatif', 'What-if'], ['audit', 'Audit trail'],
];
export const TAB_LIST = TABS;
// Tab changes slide in the direction of travel (View Transitions; instant under reduced motion).
export function selectTab(id) {
  if (!R) { tab = id; store.set('sp-tab', id); return; }
  const from = TABS.findIndex(([t]) => t === tab), to = TABS.findIndex(([t]) => t === id);
  if (from === to) return;
  if (!document.startViewTransition || matchMedia('(prefers-reduced-motion: reduce)').matches) { select(id); return; }
  document.documentElement.dataset.vt = 'tab';
  document.documentElement.style.setProperty('--tdir', to > from ? '1' : '-1');
  const t = document.startViewTransition(() => select(id));
  t.ready.catch(() => {});
  t.finished.catch(() => {}).finally(() => { delete document.documentElement.dataset.vt; });
}
const PILLAR_LABEL = { delivery: 'Delivery', quality: 'Quality', cost: 'Cost', responsiveness: 'Responsiveness', compliance: 'Compliance' };
const CAT_SHORT = { 'Steel & Metals': 'Steel', 'Packaging': 'Packaging', 'Electronic Components': 'Electronics', 'Logistics & Freight': 'Logistics', 'Chemicals & Plastics': 'Chemicals' };

let R = null;          // current result
let tab = store.get('sp-tab', 'brief');
let charts = [];       // [{node, build}]
let vSort = { key: 'combined_risk', dir: -1 };
let vFilter = { q: '', cat: '', dec: '' };

/* ------------------------------------------------------------- charts */
function theme() {
  const c = (n) => css(n);
  return {
    ink: c('--ink'), ink2: c('--ink-2'), ink3: c('--ink-3'), line: c('--line'), line2: c('--line-2'), surface: c('--surface'), surface3: c('--surface-3'),
    brand: c('--brand'), keep: c('--keep'), watch: c('--watch'), develop: c('--develop'), replace: c('--replace'),
    risk: [c('--risk-0'), c('--risk-1'), c('--risk-2'), c('--risk-3'), c('--risk-4')],
  };
}
const base = (t) => ({
  textStyle: { fontFamily: 'Geist, system-ui, sans-serif', color: t.ink2 },
  animationDuration: 700, animationEasing: 'cubicOut',
  tooltip: { backgroundColor: t.surface, borderColor: t.line2, borderWidth: 1, textStyle: { color: t.ink, fontSize: 12.5 }, extraCssText: 'box-shadow:0 8px 28px rgba(0,0,0,.12);border-radius:10px;' },
});
const axis = (t, extra = {}) => ({ axisLine: { lineStyle: { color: t.line2 } }, axisTick: { show: false }, axisLabel: { color: t.ink3, fontSize: 11, fontFamily: 'Geist Mono' }, splitLine: { lineStyle: { color: t.line, type: [3, 4] } }, nameTextStyle: { color: t.ink3, fontSize: 11 }, ...extra });

function chart(node, build) {
  if (!node) return;
  if (!window.echarts) { onEcharts(() => { if (document.body.contains(node)) chart(node, build); }); return; }
  const inst = echarts.getInstanceByDom(node) || echarts.init(node, null, { renderer: 'canvas' });
  inst.setOption(build(theme()), true);
  charts.push({ node, build });
  return inst;
}
export function rerenderCharts() {
  charts = charts.filter((c) => document.body.contains(c.node));
  charts.forEach((c) => echarts.getInstanceByDom(c.node)?.setOption(c.build(theme()), true));
}
window.addEventListener('resize', debounce(() => charts.forEach((c) => echarts.getInstanceByDom(c.node)?.resize()), 120));

/* ---------------------------------------------------------- utilities */
const dec = (v) => v.decision?.decision || '—';
const decChip = (d, src) => h('span', { class: `decision ${d}` }, d, src === 'rule' ? h('span', { class: 'src-tag' }, 'rule') : null);
const scope = () => R.vendors.filter((v) => v.in_scope);
function counts() {
  const c = { Keep: 0, Watch: 0, Develop: 0, Replace: 0 };
  scope().forEach((v) => { if (c[dec(v)] != null) c[dec(v)] += 1; });
  return c;
}
function panel(title, meta, ...body) {
  return h('section', { class: 'panel' }, h('div', { class: 'panel-head' }, h('h2', {}, title), meta ? h('span', { class: 'meta' }, meta) : null), ...body);
}

/* ============================================================== render */
export function renderInsights(result) {
  if (!result) return;
  const fresh = R !== result;
  R = result;
  const tabs = $('#ins-tabs');
  if (fresh || !tabs.children.length) {
    tabs.innerHTML = '';
    const c = counts();
    const badge = { vendors: scope().length, plan: (R.mitigation?.plans || []).length, market: (R.market?.signals || []).length };
    TABS.forEach(([id, label], i) => {
      const b = h('button', { role: 'tab', id: `tab-${id}`, 'aria-controls': 'ins-body', 'aria-selected': id === tab, tabindex: id === tab ? 0 : -1 }, label, badge[id] != null ? h('span', { class: 'n' }, badge[id]) : null);
      b.addEventListener('click', () => selectTab(id));
      b.addEventListener('keydown', (e) => {
        if (!['ArrowRight', 'ArrowLeft', 'Home', 'End'].includes(e.key)) return;
        e.preventDefault();
        const n = e.key === 'Home' ? 0 : e.key === 'End' ? TABS.length - 1 : (i + (e.key === 'ArrowRight' ? 1 : TABS.length - 1)) % TABS.length;
        selectTab(TABS[n][0]); $(`#tab-${TABS[n][0]}`).focus();
      });
      tabs.append(b);
    });
    void c;
    $('#dl-report').href = `/api/runs/${R.run_id}/report`;
    $('#dl-csv').href = `/api/runs/${R.run_id}/csv`;
  }
  select(tab, true);
}

// Narrow screens scroll the tab strip: fade the clipped edge(s) and keep the active tab in view.
function syncTabFade() {
  const t = $('#ins-tabs');
  t.classList.toggle('more-r', t.scrollLeft + t.clientWidth < t.scrollWidth - 2);
  t.classList.toggle('more-l', t.scrollLeft > 2);
}
function revealTab(id) {
  const t = $('#ins-tabs'), a = $(`#tab-${id}`);
  if (!t.dataset.fade) { t.addEventListener('scroll', syncTabFade, { passive: true }); addEventListener('resize', syncTabFade); t.dataset.fade = '1'; }
  if (a) {
    const tr = t.getBoundingClientRect(), ar = a.getBoundingClientRect();
    if (ar.left < tr.left + 24) t.scrollLeft -= tr.left - ar.left + 32;
    else if (ar.right > tr.right - 24) t.scrollLeft += ar.right - tr.right + 32;
  }
  syncTabFade();
}

function select(id, initial = false) {
  tab = id; store.set('sp-tab', id);
  $$('#ins-tabs [role="tab"]').forEach((b) => { const on = b.id === `tab-${id}`; b.setAttribute('aria-selected', on); b.tabIndex = on ? 0 : -1; });
  revealTab(id);
  const body = $('#ins-body');
  body.setAttribute('role', 'tabpanel'); body.setAttribute('aria-labelledby', `tab-${id}`);
  charts.forEach((c) => echarts.getInstanceByDom(c.node)?.dispose());
  charts = [];
  body.innerHTML = '';
  const wrap = h('div', { class: 'tab-body' });
  body.append(wrap);
  ({ brief, vendors, risk, spend, forecast, market, plan, whatif, audit })[id](wrap);
  if (!initial) window.scrollTo({ top: 0 });
}

/* ================================================================ BRIEF */
function brief(root) {
  const b = R.brief || {}, qa = R.qa || {}, c = counts();
  const total = Object.values(c).reduce((a, x) => a + x, 0) || 1;
  const main = h('article', { class: 'panel brief-main' },
    h('p', { class: 'brief-q' }, 'Asked', h('span', {}, `“${R.question || R.plan?.objective || 'Full vendor portfolio review'}”`)),
    h('h2', { class: 'headline' }, b.headline || 'Vendor portfolio review'),
    b.answer ? h('p', { class: 'answer' }, b.answer) : null,
    (b.summary || []).map((p) => h('p', { class: 'para' }, p)),
    h('dl', { class: 'facts' }, (b.key_numbers || []).slice(0, 6).map((k) => h('div', {}, h('dt', {}, k.label), h('dd', { class: String(k.value).length > 12 ? 'long' : '' }, k.value)))));

  const cnt = (st) => (qa.checks || []).filter((x) => x.status === st).length;
  const verified = cnt('verified'), corrected = cnt('corrected'), unsupported = cnt('unsupported');
  const seal = h('div', { class: `qa-seal ${qa.verdict === 'revise' ? 'revise' : ''}` }, icon('shield'),
    h('div', {}, h('b', {}, qa.verdict === 'approve' || !qa.verdict ? 'Approved by independent QA' : 'Published after QA revision'),
      h('span', {}, `${(qa.checks || []).length} claims re-checked against the data: ${verified} verified, ${corrected} corrected, ${unsupported} unsupported · ${R.revisions ? `${R.revisions} revision round` : 'no revision needed'}`)));

  const dist = h('div', { class: 'dist', role: 'img', 'aria-label': DECISIONS.map((d) => `${d}: ${c[d]}`).join(', ') },
    DECISIONS.filter((d) => c[d]).map((d) => h('span', { class: `bg-${d.toLowerCase()}`, style: { flexGrow: c[d] }, title: `${d}: ${c[d]}` }, c[d])));
  const legend = h('div', { class: 'legend' }, DECISIONS.map((d) => h('span', {}, h('i', { class: `bg-${d.toLowerCase()}` }), `${d} ${c[d]}`)));

  const acts = h('ol', { class: 'actions-list' }, (b.top_actions || []).map((a, i) => h('li', {}, h('span', { class: 'n' }, String(i + 1).padStart(2, '0')),
    h('div', {}, h('b', {}, a.action), h('div', { class: 'sub' }, h('span', {}, a.owner), h('span', {}, a.timeline), h('span', {}, a.impact))))));

  const watchlist = scope().filter((v) => ['Replace', 'Watch'].includes(dec(v))).sort((a, b2) => b2.combined_risk - a.combined_risk).slice(0, 6);
  const wl = h('ul', { class: 'checks' }, watchlist.map((v) => h('li', { class: `${dec(v) === 'Replace' ? 'unsupported' : 'corrected'} as-btn` },
    h('button', { type: 'button', class: 'row-btn', onclick: () => openVendor(v.vendor), 'aria-label': `${v.vendor}, ${dec(v)}. Open dossier` },
      icon(dec(v) === 'Replace' ? 'alert' : 'flag'), h('div', {}, h('b', {}, short(v.vendor)), ' ', decChip(dec(v), v.decision?.source), h('span', { class: 'note' }, v.decision?.rationale || ''))))));

  root.append(h('div', { class: 'brief' },
    h('div', { class: 'stack' }, main,
      panel('Needs attention', 'Select a vendor for the full dossier', wl.children.length ? wl : h('p', { class: 'meta' }, 'No vendors flagged.'))),
    h('div', { class: 'stack' },
      seal,
      panel('Decisions', `${total} vendors · drag the orbit, select a star`, decisionOrbit(R, scope(), openVendor), dist, legend),
      panel('Priority actions', null, acts.children.length ? acts : h('p', { class: 'meta' }, 'No actions returned.')))));
}

/* ============================================================== VENDORS */
function vendors(root) {
  const cats = [...new Set(R.vendors.map((v) => v.category))].sort();
  const search = h('input', { type: 'search', placeholder: 'Search vendors', value: vFilter.q, 'aria-label': 'Search vendors' });
  const catSel = h('select', { 'aria-label': 'Filter by category' }, h('option', { value: '' }, 'All categories'), cats.map((c) => h('option', { value: c, selected: vFilter.cat === c }, c)));
  const decSel = h('select', { 'aria-label': 'Filter by decision' }, h('option', { value: '' }, 'All decisions'), DECISIONS.map((d) => h('option', { value: d, selected: vFilter.dec === d }, d)));
  const count = h('span', { class: 'meta' });
  const tbody = h('tbody');
  const cols = [
    ['rank', 'Rank'], ['vendor', 'Vendor'], ['composite', 'Score'], ['combined_risk', 'Risk'], ['otd', 'On-time 12m'],
    ['defect', 'Defect %'], ['pv', 'Price var.'], ['spend_total', 'Spend'], ['decision', 'Decision'],
  ];
  const thead = h('thead', {}, h('tr', {}, cols.map(([k, l]) => {
    const th = h('th', { scope: 'col', class: `c-${k}`, 'aria-sort': vSort.key === k ? (vSort.dir > 0 ? 'ascending' : 'descending') : null }, h('button', { type: 'button' }, l, vSort.key === k ? icon(vSort.dir > 0 ? 'arrow-up' : 'arrow-down', 'sm') : null));
    th.querySelector('button').addEventListener('click', () => { vSort = { key: k, dir: vSort.key === k ? -vSort.dir : (k === 'vendor' || k === 'rank' ? 1 : -1) }; root.innerHTML = ''; vendors(root); });
    return th;
  })));
  const val = (v, k) => ({ rank: v.rank, vendor: v.vendor, composite: v.composite, combined_risk: v.combined_risk, otd: v.kpis.on_time_pct, defect: v.kpis.defect_rate_pct, pv: v.kpis.price_variance_pct, spend_total: v.spend_total, decision: DECISIONS.indexOf(dec(v)) }[k]);
  const draw = () => {
    const q = vFilter.q.toLowerCase();
    const rows = R.vendors.filter((v) => (!q || v.vendor.toLowerCase().includes(q) || v.category.toLowerCase().includes(q)) && (!vFilter.cat || v.category === vFilter.cat) && (!vFilter.dec || dec(v) === vFilter.dec))
      .sort((a, b) => { const x = val(a, vSort.key), y = val(b, vSort.key); return (x > y ? 1 : x < y ? -1 : 0) * vSort.dir; });
    tbody.innerHTML = '';
    count.textContent = `${rows.length} of ${R.vendors.length} vendors`;
    if (!rows.length) { tbody.append(h('tr', {}, h('td', { colspan: cols.length }, h('div', { class: 'empty' }, h('b', {}, 'No vendors match'), 'Clear a filter to see more.')))); return; }
    rows.forEach((v) => {
      const tr = h('tr', { tabindex: 0, 'aria-label': `${v.vendor}, ${dec(v)}` },
        h('td', { class: 'n c-rank' }, v.rank),
        h('td', { class: 'vn c-vendor' }, h('b', {}, short(v.vendor)), h('span', {}, `${v.category} · ${v.region.split(',')[0]}`)),
        h('td', { class: 'c-composite', 'data-label': 'Score' }, h('div', { class: 'scorebar' }, h('span', { class: 'num' }, v.composite.toFixed(1)), h('span', { class: 'b' }, h('i', { style: { width: `${v.composite}%` } })), tierChip(v.tier))),
        h('td', { class: 'n c-combined_risk', 'data-label': 'Risk' }, h('span', { class: 'riskdot', style: { '--rc': riskColor(v.combined_risk) } }, v.combined_risk.toFixed(0))),
        h('td', { class: 'c-otd', 'data-label': 'On-time', html: `<span class="otd">${sparkline(v.otd_series)}<span class="num">${v.kpis.on_time_pct.toFixed(1)}</span></span>` }),
        h('td', { class: 'n c-defect' }, v.kpis.defect_rate_pct.toFixed(2)),
        h('td', { class: `n c-pv ${v.kpis.price_variance_pct > 2 ? 'delta-down' : v.kpis.price_variance_pct < 0 ? 'delta-up' : ''}` }, `${v.kpis.price_variance_pct > 0 ? '+' : ''}${v.kpis.price_variance_pct.toFixed(1)}%`),
        h('td', { class: 'n c-spend_total' }, inr(v.spend_total)),
        h('td', { class: 'c-decision' }, v.decision ? decChip(dec(v), v.decision.source) : h('span', { class: 'meta' }, 'out of scope')));
      tr.addEventListener('click', () => openVendor(v.vendor));
      // preventDefault: otherwise the same Enter activates the dossier's Close button once focus moves there.
      tr.addEventListener('keydown', (e) => { if (e.key === 'Enter' || e.key === ' ') { e.preventDefault(); openVendor(v.vendor); } });
      tbody.append(tr);
    });
  };
  search.addEventListener('input', debounce(() => { vFilter.q = search.value; draw(); }, 120));
  catSel.addEventListener('change', () => { vFilter.cat = catSel.value; draw(); });
  decSel.addEventListener('change', () => { vFilter.dec = decSel.value; draw(); });
  root.append(h('section', { class: 'panel' },
    h('div', { class: 'toolbar' }, h('label', { class: 'search' }, icon('search'), search), catSel, decSel, count),
    h('div', { class: 'vt-wrap vlist-wrap' }, h('table', { class: 'vt vlist' }, thead, tbody))));
  draw();
}

/* ============================================================ VENDOR DRAWER */
export function openVendor(name) {
  const v = R.vendors.find((x) => x.vendor === name);
  if (!v) return;
  const drawer = $('#drawer'), scrim = $('#scrim'), inner = $('#drawer-inner');
  const last = document.activeElement;
  inner.innerHTML = '';
  const close = () => { drawer.hidden = true; scrim.hidden = true; document.removeEventListener('keydown', onKey); last?.focus?.(); };
  const onKey = (e) => {
    if (e.key === 'Escape') close();
    if (e.key === 'Tab') {
      const f = $$('button, a[href], input, select, [tabindex="0"]', drawer).filter((x) => !x.disabled && x.offsetParent);
      if (!f.length) return;
      if (e.shiftKey && document.activeElement === f[0]) { e.preventDefault(); f[f.length - 1].focus(); }
      else if (!e.shiftKey && document.activeElement === f[f.length - 1]) { e.preventDefault(); f[0].focus(); }
    }
  };
  const closeBtn = h('button', { class: 'icon-btn', type: 'button', 'aria-label': 'Close vendor details', onclick: close }, icon('x'));
  const d = v.decision;
  const plan = (R.mitigation?.plans || []).find((p) => p.vendor === v.vendor);
  const sig = (R.market?.signals || []).find((s) => s.category === v.category);
  inner.append(
    h('div', { class: 'dw-head' }, h('div', {}, h('h2', { id: 'drawer-title' }, short(v.vendor)), h('p', {}, `${v.category} · ${v.region} · rank ${v.rank} of ${R.vendors.length} · ${inr(v.spend_total)} spend`)), closeBtn),
    h('div', { class: 'kv', role: 'list', 'aria-label': 'Vendor summary' }, d ? decChip(d.decision, d.source) : null, tierChip(v.tier), h('span', { class: 'tag' }, `Score ${v.composite.toFixed(1)}`),
      h('span', { class: 'tag' }, `Internal risk ${v.internal_risk.toFixed(0)}`), h('span', { class: 'tag' }, `Market risk ${v.market_risk.toFixed(0)}`), h('span', { class: 'tag brand' }, `Combined ${v.combined_risk.toFixed(0)}`)),
    d ? h('div', { class: 'dw-sec' }, h('h3', {}, `Strategist's call · confidence ${Math.round(d.confidence * 100)}%`), h('p', { class: 'rationale' }, d.rationale), d.actions?.length ? h('ul', {}, d.actions.map((a) => h('li', {}, a))) : null) : null,
    h('div', { class: 'dw-sec' }, h('h3', {}, 'On-time delivery: 12 months and 3-month forecast'), h('div', { class: 'chart short', id: 'dw-chart', role: 'img', 'aria-label': `On-time delivery history and forecast for ${v.vendor}` })),
    h('div', { class: 'dw-sec' }, h('h3', {}, 'Pillar scores'), h('div', { class: 'pillars' }, Object.entries(v.pillars).map(([k, s]) => h('div', { class: 'pillar' }, h('span', {}, PILLAR_LABEL[k]), h('span', { class: 'b' }, h('i', { style: { width: `${s}%`, background: s >= 75 ? 'var(--keep)' : s >= 55 ? 'var(--develop)' : s >= 40 ? 'var(--watch)' : 'var(--replace)' } })), h('span', { class: 'v' }, s.toFixed(0)))))),
    h('div', { class: 'dw-sec' }, h('h3', {}, 'Key performance indicators (last 6 months, spend-weighted)'), h('dl', { class: 'kpis' },
      [['On-time', pct(v.kpis.on_time_pct)], ['Fill rate', pct(v.kpis.fill_rate_pct)], ['Defects', pct(v.kpis.defect_rate_pct, 2)], ['Lead time', `${v.kpis.lead_time_days.toFixed(1)} d`],
        ['Price variance', `${v.kpis.price_variance_pct > 0 ? '+' : ''}${v.kpis.price_variance_pct.toFixed(1)}%`], ['Response', `${v.kpis.response_time_hrs.toFixed(0)} h`],
        ['Capacity used', pct(v.kpis.capacity_utilization_pct, 0)], ['OTD trend', `${v.kpis.otd_trend > 0 ? '+' : ''}${v.kpis.otd_trend.toFixed(2)}/mo`], ['Compliance', `${v.compliance_incidents} incidents`]]
        .map(([k, x]) => h('div', {}, h('dt', {}, k), h('dd', {}, x))))),
    plan ? h('div', { class: 'dw-sec' }, h('h3', {}, 'Mitigation plan'), planCard(plan, true)) : null,
    sig ? h('div', { class: 'dw-sec' }, h('h3', {}, `Market signal: ${v.category}`), signalCard(sig)) : null,
  );
  drawer.hidden = false; scrim.hidden = false;
  scrim.onclick = close;
  document.addEventListener('keydown', onKey);
  $$('.kv[role=list] > *', inner).forEach((c) => c.setAttribute('role', 'listitem'));
  closeBtn.focus();
  const f = v.forecast;
  if (f) {
    chart($('#dw-chart'), (t) => forecastOption(t, f, 'otd', 'On-time %', R.config?.otd_floor));
  }
}

function forecastOption(t, f, key, label, threshold) {
  const months = [...f.history_months, ...f.future_months];
  const hist = [...f[key].history, ...f.future_months.map(() => null)];
  const fc = [...f.history_months.map((_, i) => (i === f.history_months.length - 1 ? f[key].history[i] : null)), ...f[key].forecast];
  const lo = [...f.history_months.map(() => null), ...f[key].lower];
  const band = [...f.history_months.map(() => null), ...f[key].upper.map((u, i) => u - f[key].lower[i])];
  return {
    ...base(t), grid: { left: 40, right: 64, top: 18, bottom: 28 },
    tooltip: { ...base(t).tooltip, trigger: 'axis', valueFormatter: (x) => (x == null ? '—' : Number(x).toFixed(1)) },
    xAxis: { type: 'category', data: months, ...axis(t), axisLabel: { ...axis(t).axisLabel, interval: 2 } },
    yAxis: { type: 'value', scale: true, ...axis(t) },
    series: [
      { name: 'band', type: 'line', data: lo, stack: 'b', lineStyle: { opacity: 0 }, symbol: 'none', tooltip: { show: false } },
      { name: '80% band', type: 'line', data: band, stack: 'b', lineStyle: { opacity: 0 }, symbol: 'none', areaStyle: { color: t.brand, opacity: 0.12 }, tooltip: { show: false } },
      { name: label, type: 'line', data: hist, symbol: 'circle', symbolSize: 5, lineStyle: { color: t.ink, width: 2 }, itemStyle: { color: t.ink } },
      { name: 'Forecast', type: 'line', data: fc, symbol: 'circle', symbolSize: 5, lineStyle: { color: t.brand, width: 2, type: [6, 4] }, itemStyle: { color: t.brand },
        markLine: threshold ? { symbol: 'none', label: { formatter: key === 'otd' ? `floor ${threshold}%` : `ceiling ${threshold}%`, color: t.replace, fontSize: 10 }, lineStyle: { color: t.replace, type: [4, 4] }, data: [{ yAxis: threshold }] } : undefined },
    ],
  };
}

/* ================================================================= RISK */
function risk(root) {
  const th = R.config?.risk_threshold ?? 55;
  const sc = h('div', { class: 'chart tall', role: 'img', 'aria-label': 'Risk matrix: internal risk versus market risk, bubble size is spend' });
  const hm = h('div', { class: 'chart tall', role: 'img', 'aria-label': 'Heatmap of pillar scores by vendor' });
  root.append(h('div', { class: 'grid-2' },
    panel('Risk matrix', `Bubble = spend · dashed line = tolerance ${th}`, sc),
    panel('Pillar heatmap', 'Score 0–100 by pillar, sorted by risk', hm)));
  const vs = scope();
  chart(sc, (t) => ({
    ...base(t), grid: { left: 48, right: 24, top: 24, bottom: 44 },
    tooltip: { ...base(t).tooltip, formatter: (p) => `<b>${esc(short(p.data.v.vendor))}</b><br>${esc(p.data.v.category)}<br>Internal ${p.data.value[0].toFixed(0)} · Market ${p.data.value[1].toFixed(0)}<br>Combined ${p.data.v.combined_risk.toFixed(0)} · ${dec(p.data.v)}<br>${inr(p.data.v.spend_total)}` },
    xAxis: { name: 'Internal risk (performance)', nameLocation: 'middle', nameGap: 28, min: 0, max: 100, ...axis(t) },
    yAxis: { name: 'Market risk (live news)', nameLocation: 'middle', nameGap: 34, min: 0, max: 100, ...axis(t) },
    series: [{
      type: 'scatter', data: vs.map((v) => ({ value: [v.internal_risk, v.market_risk], v, symbolSize: 10 + Math.sqrt(v.spend_total / 1e5) * 2.4, itemStyle: { color: decisionVarColor(t, dec(v)), opacity: 0.82, borderColor: t.surface, borderWidth: 1.5 } })),
      label: { show: true, formatter: (p) => (p.data.v.combined_risk >= th - 5 ? short(p.data.v.vendor).split(' ')[0] : ''), position: 'right', color: t.ink2, fontSize: 11 },
      markLine: { silent: true, symbol: 'none', lineStyle: { color: t.replace, type: [5, 5] }, label: { color: t.replace, fontSize: 10, formatter: 'tolerance' },
        data: [[{ coord: [Math.min(100, th / 0.65), 0] }, { coord: [Math.max(0, (th - 35) / 0.65), 100] }]] },
    }],
  }));
  const sorted = [...vs].sort((a, b) => b.combined_risk - a.combined_risk);
  const pillars = Object.keys(PILLAR_LABEL);
  chart(hm, (t) => ({
    ...base(t), grid: { left: 150, right: 16, top: 8, bottom: 60 },
    tooltip: { ...base(t).tooltip, formatter: (p) => `${esc(short(sorted[p.value[1]].vendor))}<br>${PILLAR_LABEL[pillars[p.value[0]]]}: <b>${p.value[2].toFixed(0)}</b>` },
    xAxis: { type: 'category', data: pillars.map((p) => PILLAR_LABEL[p]), ...axis(t), axisLabel: { color: t.ink3, fontSize: 11, rotate: 0 }, splitLine: { show: false } },
    yAxis: { type: 'category', data: sorted.map((v) => short(v.vendor)), inverse: true, ...axis(t), axisLabel: { color: t.ink2, fontSize: 11, fontFamily: 'Geist' }, splitLine: { show: false } },
    visualMap: { min: 0, max: 100, show: true, orient: 'horizontal', left: 'center', bottom: 0, itemHeight: 140, itemWidth: 10, text: ['100', '0'], textStyle: { color: t.ink3, fontSize: 10 }, inRange: { color: [t.risk[4], t.risk[2], t.risk[0]] } },
    series: [{ type: 'heatmap', data: sorted.flatMap((v, y) => pillars.map((p, x) => [x, y, v.pillars[p]])), itemStyle: { borderColor: t.surface, borderWidth: 2, borderRadius: 4 }, label: { show: true, color: '#fff', fontSize: 10, fontFamily: 'Geist Mono', formatter: (p) => p.value[2].toFixed(0) } }],
  }));
}
function decisionVarColor(t, d) { return { Keep: t.keep, Watch: t.watch, Develop: t.develop, Replace: t.replace }[d] || t.ink3; }

/* ================================================================ SPEND */
function spend(root) {
  const sp = R.spend;
  const par = h('div', { class: 'chart', role: 'img', 'aria-label': 'Pareto chart of spend by vendor' });
  const conc = h('div', { class: 'chart', role: 'img', 'aria-label': 'Category concentration (HHI)' });
  const trend = h('div', { class: 'chart short', role: 'img', 'aria-label': 'Monthly spend trend' });
  const levers = (R.cost?.savings_levers || []);
  const risks = (R.cost?.concentration_risks || []);
  root.append(
    h('dl', { class: 'facts panel', style: { marginTop: 0, borderTop: '1px solid var(--line)', padding: '6px 20px 14px' } },
      [['Total spend', inr(sp.total_spend)], ['Vendors holding 80% of spend', `${sp.pareto_vendors_80pct} of ${sp.vendor_count}`], ['Overpaid vs contract (6 mo)', inr(sp.overpay_recent_total)],
        ['Annual savings opportunity', inr(sp.savings_opportunity_total)], ['Concentrated categories', String(sp.categories.filter((c) => c.concentration_flag).length)]].map(([k, x]) => h('div', {}, h('dt', {}, k), h('dd', {}, x)))),
    h('div', { class: 'grid-2', style: { marginTop: '16px' } },
      panel('Spend Pareto', 'Bars: spend · line: cumulative share', par),
      panel('Concentration by category', 'HHI above 2,500 = highly concentrated', conc),
      panel('Savings levers', 'Quantified by the Cost & Spend Analyst', levers.length ? h('ol', { class: 'actions-list' }, levers.map((l, i) => h('li', {}, h('span', { class: 'n' }, String(i + 1).padStart(2, '0')),
        h('div', {}, h('b', {}, l.lever), h('div', { class: 'sub' }, h('span', { class: 'num' }, `${inr(l.estimated_annual_value_inr)} / yr`), h('span', {}, (l.categories || []).map((c) => CAT_SHORT[c] || c).join(', '))), h('p', { class: 'meta', style: { marginTop: '4px' } }, l.rationale))))) : h('p', { class: 'meta' }, 'No levers returned.')),
      panel('Concentration risks', null, risks.length ? h('ul', { class: 'checks' }, risks.map((r) => h('li', { class: 'corrected' }, icon('alert'), h('div', {}, h('b', {}, r.headline), h('span', { class: 'note' }, `${short(r.vendor)} · ${r.evidence}`))))) : h('p', { class: 'meta' }, 'None flagged.')),
      h('div', { class: 'span-2' }, panel('Monthly spend', null, trend))));
  const vs = sp.vendors.slice(0, 20);
  chart(par, (t) => ({
    ...base(t), grid: { left: 54, right: 44, top: 20, bottom: 86 },
    tooltip: { ...base(t).tooltip, trigger: 'axis' },
    xAxis: { type: 'category', data: vs.map((v) => short(v.vendor_name)), ...axis(t), axisLabel: { color: t.ink3, fontSize: 10, rotate: 45, fontFamily: 'Geist' } },
    yAxis: [{ type: 'value', ...axis(t), axisLabel: { ...axis(t).axisLabel, formatter: (x) => inr(x).replace('₹', '') } }, { type: 'value', max: 100, ...axis(t), splitLine: { show: false }, axisLabel: { ...axis(t).axisLabel, formatter: '{value}%' } }],
    series: [
      { type: 'bar', name: 'Spend', data: vs.map((v) => v.spend_total), itemStyle: { color: t.brand, borderRadius: [4, 4, 0, 0] }, tooltip: { valueFormatter: inr } },
      { type: 'line', name: 'Cumulative', yAxisIndex: 1, data: vs.map((v) => +(v.cum_share * 100).toFixed(1)), symbol: 'circle', symbolSize: 5, lineStyle: { color: t.watch, width: 2 }, itemStyle: { color: t.watch }, tooltip: { valueFormatter: (x) => `${x}%` },
        markLine: { symbol: 'none', silent: true, lineStyle: { color: t.ink3, type: [4, 4] }, label: { formatter: '80%', color: t.ink3, fontSize: 10 }, data: [{ yAxis: 80 }] } },
    ],
  }));
  chart(conc, (t) => ({
    ...base(t), grid: { left: 110, right: 30, top: 10, bottom: 30 },
    tooltip: { ...base(t).tooltip, formatter: (p) => { const c = sp.categories[p.dataIndex]; return `<b>${esc(c.category)}</b><br>HHI ${c.hhi}<br>Top: ${esc(short(c.top_vendor))} (${Math.round(c.top_share * 100)}%)`; } },
    xAxis: { type: 'value', ...axis(t), max: 10000 },
    yAxis: { type: 'category', data: sp.categories.map((c) => CAT_SHORT[c.category] || c.category), inverse: true, ...axis(t), axisLabel: { color: t.ink2, fontSize: 12, fontFamily: 'Geist' } },
    series: [{ type: 'bar', data: sp.categories.map((c) => ({ value: c.hhi, itemStyle: { color: c.hhi > 2500 ? t.replace : t.develop, borderRadius: [0, 4, 4, 0] } })), barWidth: 18,
      label: { show: true, position: 'right', color: t.ink2, fontFamily: 'Geist Mono', fontSize: 11 },
      markLine: { symbol: 'none', silent: true, lineStyle: { color: t.ink3, type: [4, 4] }, label: { formatter: '2,500', color: t.ink3, fontSize: 10 }, data: [{ xAxis: 2500 }] } }],
  }));
  chart(trend, (t) => ({
    ...base(t), grid: { left: 54, right: 16, top: 16, bottom: 28 },
    tooltip: { ...base(t).tooltip, trigger: 'axis', valueFormatter: inr },
    xAxis: { type: 'category', data: sp.monthly.months, ...axis(t) },
    yAxis: { type: 'value', ...axis(t), axisLabel: { ...axis(t).axisLabel, formatter: (x) => inr(x).replace('₹', '') } },
    series: [{ type: 'line', data: sp.monthly.spend, smooth: 0.3, symbol: 'none', lineStyle: { color: t.brand, width: 2 }, areaStyle: { color: t.brand, opacity: 0.08 } }],
  }));
}

/* ============================================================= FORECAST */
function forecast(root) {
  const fcs = R.vendors.filter((v) => v.forecast && v.in_scope);
  const warn = R.forecast?.early_warnings || [];
  const ranked = [...fcs].sort((a, b) => (b.forecast.otd_breach - a.forecast.otd_breach) || (a.forecast.otd.slope_per_month - b.forecast.otd.slope_per_month));
  let current = ranked[0]?.vendor;
  let metric = 'otd';
  const sel = h('select', { 'aria-label': 'Vendor to forecast' }, ranked.map((v) => h('option', { value: v.vendor }, `${short(v.vendor)}${v.forecast.otd_breach || v.forecast.defect_breach ? ' · breach' : ''}`)));
  const seg = h('div', { class: 'seg', role: 'tablist', 'aria-label': 'Metric' },
    h('button', { role: 'tab', 'aria-selected': true, 'data-m': 'otd' }, 'On-time %'), h('button', { role: 'tab', 'aria-selected': false, 'data-m': 'defect' }, 'Defect %'));
  const node = h('div', { class: 'chart tall', role: 'img', 'aria-label': 'Forecast chart' });
  const draw = () => {
    const v = R.vendors.find((x) => x.vendor === current);
    if (!v) return;
    const inst = window.echarts?.getInstanceByDom(node);
    const build = (t) => forecastOption(t, v.forecast, metric, metric === 'otd' ? 'On-time %' : 'Defect %', metric === 'otd' ? R.config?.otd_floor : R.config?.defect_ceiling);
    if (inst) { inst.setOption(build(theme()), true); charts = charts.filter((c) => c.node !== node); charts.push({ node, build }); }
    else chart(node, build);
  };
  sel.addEventListener('change', () => { current = sel.value; draw(); });
  $$('button', seg).forEach((b) => b.addEventListener('click', () => { metric = b.dataset.m; $$('button', seg).forEach((x) => x.setAttribute('aria-selected', x === b)); draw(); }));
  const outlook = { deteriorating: [], improving: [], stable: [] };
  fcs.forEach((v) => outlook[v.forecast.outlook]?.push(v));
  root.append(h('div', { class: 'whatif' },
    h('div', { class: 'stack' },
      panel('Early warnings', null, warn.length ? h('ul', { class: 'checks' }, warn.map((w) => h('li', { class: `${w.confidence === 'high' ? 'unsupported' : 'corrected'} as-btn` },
        h('button', { type: 'button', class: 'row-btn', onclick: () => { current = R.vendors.find((x) => x.vendor.startsWith(short(w.vendor)))?.vendor || current; sel.value = current; draw(); } },
          icon('alert'), h('div', {}, h('b', {}, short(w.vendor)), h('span', { class: 'note' }, `${w.projection} · ${w.confidence} confidence`), h('span', { class: 'note' }, `Trigger: ${w.recommended_trigger}`)))))) : h('p', { class: 'meta' }, 'No projected breaches.')),
      panel('Outlook', null, h('div', { class: 'legend', style: { display: 'grid', gap: '8px' } },
        h('span', {}, h('i', { class: 'bg-replace' }), `Deteriorating: ${outlook.deteriorating.map((v) => short(v.vendor)).join(', ') || 'none'}`),
        h('span', {}, h('i', { class: 'bg-keep' }), `Improving: ${outlook.improving.map((v) => short(v.vendor)).join(', ') || 'none'}`),
        h('span', {}, h('i', { style: { background: 'var(--ink-3)' } }), `Stable: ${outlook.stable.length} vendors`)))),
    h('section', { class: 'panel' }, h('div', { class: 'panel-head' }, h('h2', {}, 'Projection with 80% band'), h('div', { class: 'toolbar', style: { margin: 0 } }, sel, seg)), node,
      R.forecast?.summary ? h('p', { class: 'meta', style: { marginTop: '8px' } }, R.forecast.summary) : null)));
  draw();
}

/* =============================================================== MARKET */
function signalCard(s) {
  return h('article', { class: 'signal' },
    h('div', { class: 'score' }, h('span', { class: 'sdot', style: { background: riskColor(s.risk_score) } }), Math.round(s.risk_score), h('small', {}, 'risk')),
    h('div', {},
      h('h3', {}, s.category === 'All' ? s.region : s.category, s.region && s.category !== 'All' && s.region !== 'All India' ? h('span', { class: 'meta' }, s.region) : null, h('span', { class: `dir ${s.direction}` }, s.direction)),
      h('p', { class: 'hl' }, s.headline),
      h('p', { class: 'im' }, s.implication),
      h('p', { class: 'src' }, s.source ? h('span', {}, s.source) : null, s.date ? h('span', { class: 'num' }, s.date) : null,
        s.url ? h('a', { href: s.url, target: '_blank', rel: 'noopener noreferrer' }, 'Open source', icon('external')) : null)));
}
function sourceList(all) {
  const ul = h('ul', { class: 'sources' });
  const item = (s) => h('li', {}, h('a', { href: s.url, target: '_blank', rel: 'noopener noreferrer' }, s.title), h('span', {}, [s.source, s.date, `query “${s.query}”`].filter(Boolean).join(' · ')));
  all.slice(0, 8).forEach((s) => ul.append(item(s)));
  if (all.length <= 8) return ul;
  const btn = h('button', { class: 'btn ghost sm more', type: 'button', 'aria-expanded': 'false' }, `Show all ${all.length} sources`);
  btn.addEventListener('click', () => { all.slice(8).forEach((s) => ul.append(item(s))); btn.remove(); });
  return h('div', {}, ul, btn);
}
function market(root) {
  const m = R.market || {};
  const sigs = [...(m.signals || [])].sort((a, b) => b.risk_score - a.risk_score);
  const node = h('div', { class: 'chart short', role: 'img', 'aria-label': 'External risk score by category and region' });
  root.append(h('div', { class: 'whatif' },
    h('div', { class: 'stack' },
      panel('External risk', 'Scored by the Market Risk Researcher from live news', node, m.summary ? h('p', { class: 'meta', style: { marginTop: '8px' } }, m.summary) : null),
      panel('Sources', `${(m.sources || []).length} articles retrieved this run`, (m.sources || []).length ? sourceList(m.sources) : h('p', { class: 'meta' }, 'No live sources (search unavailable during this run).'))),
    panel('Signals', null, sigs.length ? h('div', { class: 'signals' }, sigs.map(signalCard)) : h('p', { class: 'meta' }, 'No signals returned.'))));
  chart(node, (t) => ({
    ...base(t), grid: { left: 120, right: 30, top: 6, bottom: 24 },
    xAxis: { type: 'value', min: 0, max: 100, ...axis(t) },
    yAxis: { type: 'category', inverse: true, data: sigs.map((s) => (s.category === 'All' ? s.region.split(',')[0] : CAT_SHORT[s.category] || s.category)), ...axis(t), axisLabel: { color: t.ink2, fontFamily: 'Geist', fontSize: 12 } },
    series: [{ type: 'bar', barWidth: 14, data: sigs.map((s) => ({ value: s.risk_score, itemStyle: { color: t.risk[Math.min(4, Math.floor(s.risk_score / 20))], borderRadius: [0, 4, 4, 0] } })), label: { show: true, position: 'right', color: t.ink2, fontFamily: 'Geist Mono', fontSize: 11 },
      markLine: { symbol: 'none', silent: true, lineStyle: { color: t.ink3, type: [4, 4] }, label: { formatter: 'baseline', color: t.ink3, fontSize: 10 }, data: [{ xAxis: 35 }] } }],
  }));
}

/* ================================================================= PLAN */
function planCard(p, compact = false) {
  return h('article', { class: compact ? '' : 'panel plan' },
    compact ? null : h('div', { class: 'plan-head' }, h('div', {}, h('h2', {}, short(p.vendor)), h('p', {}, /^[\d.\s]+$/.test(String(p.risk)) ? `Combined risk ${p.risk}` : p.risk)), h('button', { class: 'btn ghost sm', type: 'button', onclick: () => openVendor(p.vendor) }, 'Vendor dossier', icon('arrow-right'))),
    h('div', { class: 'shift' }, h('span', { class: 'from' }, h('b', {}, short(p.vendor))), icon('arrow-right'),
      (p.alternatives || []).length ? p.alternatives.map((a) => h('span', { class: 'to' }, h('b', {}, short(a.to_vendor)), h('span', { class: 'pct' }, `${Math.round(a.share_of_volume_pct)}%`))) : h('span', { class: 'meta' }, 'No qualified alternative in this category')),
    h('div', { class: 'phases' }, [['30 days', p.days_30], ['60 days', p.days_60], ['90 days', p.days_90]].map(([l, items]) => h('div', { class: 'phase' }, h(compact ? 'h4' : 'h3', {}, l), h('ul', {}, (items || []).map((x) => h('li', {}, x)))))),
    p.expected_impact ? h('p', { class: 'impact' }, h('b', {}, 'Expected impact: '), p.expected_impact) : null);
}
function plan(root) {
  const plans = R.mitigation?.plans || [];
  if (!plans.length) {
    root.append(h('section', { class: 'panel' }, h('div', { class: 'empty' }, h('b', {}, 'No mitigation needed'), 'The risk gate found no vendor above tolerance, so the Mitigation Planner was skipped for this run.')));
    return;
  }
  if (R.mitigation.summary) root.append(h('p', { class: 'lede', style: { margin: '0 0 16px' } }, R.mitigation.summary));
  plans.forEach((p) => root.append(planCard(p)));
}

/* ============================================================== WHAT-IF */
function whatif(root) {
  const weights = { ...(R.config?.weights || {}) };
  const tbody = h('tbody');
  const note = h('p', { class: 'meta', style: { marginTop: '10px' } });
  const baseRank = Object.fromEntries(R.vendors.map((v) => [v.vendor, v.rank]));
  const baseDec = Object.fromEntries(R.vendors.map((v) => [v.vendor, dec(v)]));
  const th = R.config?.risk_threshold ?? 55;
  const kpiById = Object.fromEntries(R.vendors.map((v) => [v.vendor, v]));
  const ws = h('div', { class: 'weights' });
  const compute = () => {
    // Same formula as the server's scorer: pillar scores are weight-independent, so re-weight locally (instant).
    const tot = Object.values(weights).reduce((a, x) => a + Math.max(0, x), 0) || 1;
    const rows = R.vendors.map((v) => ({ v, s: Object.entries(v.pillars).reduce((a, [k, x]) => a + x * (Math.max(0, weights[k] || 0) / tot), 0) }))
      .sort((a, b) => b.s - a.s).map((r, i) => ({ ...r, rank: i + 1 }));
    tbody.innerHTML = '';
    let flips = 0;
    rows.forEach((r) => {
      const mv = baseRank[r.v.vendor] - r.rank;
      const momentum = r.v.internal_risk - (100 - r.v.composite);
      const newInternal = Math.max(0, Math.min(100, 100 - r.s + momentum));
      const newComb = 0.65 * newInternal + 0.35 * r.v.market_risk;
      const crosses = (r.v.combined_risk >= th) !== (newComb >= th);
      if (crosses && r.v.in_scope) flips += 1;
      tbody.append(h('tr', {},
        h('td', { class: 'n' }, r.rank), h('td', { class: 'vn' }, h('b', {}, short(r.v.vendor)), h('span', {}, CAT_SHORT[r.v.category] || r.v.category)),
        h('td', { class: 'n' }, r.s.toFixed(1)),
        h('td', { class: `mv ${mv > 0 ? 'delta-up' : mv < 0 ? 'delta-down' : ''}` }, mv ? `${mv > 0 ? '▲' : '▼'} ${Math.abs(mv)}` : '—'),
        h('td', { class: 'n' }, h('span', { class: 'riskdot', style: { '--rc': riskColor(newComb) } }, newComb.toFixed(0))),
        h('td', {}, baseDec[r.v.vendor] !== '—' ? decChip(baseDec[r.v.vendor]) : null, crosses ? h('span', { class: 'flip' }, ' crosses tolerance') : null)));
    });
    note.textContent = flips ? `${flips} vendor(s) would cross the risk tolerance under these priorities. Re-run the agents to get updated decisions.` : 'No vendor crosses the risk tolerance under these priorities: the agents\' decisions are robust to this change.';
  };
  Object.keys(PILLAR_LABEL).forEach((k) => {
    const id = `wf-${k}`;
    const input = h('input', { type: 'range', id, min: 0, max: 60, value: weights[k] ?? 20 });
    const out = h('output', { for: id }, '');
    const upd = () => { weights[k] = Number(input.value); setRangeFill(input); const tot = Object.values(weights).reduce((a, x) => a + x, 0) || 1; $$('output', ws).forEach((o) => { const kk = o.getAttribute('for').replace('wf-', ''); o.textContent = `${Math.round(weights[kk] / tot * 100)}%`; }); compute(); };
    input.addEventListener('input', upd);
    ws.append(h('div', { class: 'weight' }, h('label', { for: id }, PILLAR_LABEL[k]), input, out));
  });
  const reset = h('button', { class: 'btn ghost sm', type: 'button' }, icon('refresh'), 'Reset to run weights');
  reset.addEventListener('click', () => { root.innerHTML = ''; whatif(root); });
  root.append(h('div', { class: 'whatif' },
    panel('Change priorities', 'Instant: no AI call needed', ws, h('div', { style: { marginTop: '16px' } }, reset), note),
    h('section', { class: 'panel' }, h('div', { class: 'panel-head' }, h('h2', {}, 'Re-ranked portfolio'), h('span', { class: 'meta' }, 'Movement vs the run\'s ranking')),
      h('div', { class: 'vt-wrap' }, h('table', { class: 'vt wf-table', style: { minWidth: '560px' } },
        h('thead', {}, h('tr', {}, ['Rank', 'Vendor', 'Score', 'Move', 'Risk', 'Agent decision'].map((x) => h('th', { scope: 'col' }, x)))), tbody)))));
  $$('input[type=range]', ws).forEach((i) => i.dispatchEvent(new Event('input')));
  void kpiById;
}

/* ================================================================ AUDIT */
function audit(root) {
  const qa = R.qa || {}, q = R.quality, u = R.usage;
  const stageNames = { intake: 'Intake Router', audit: 'Data Quality Auditor', performance: 'Performance Analyst', cost: 'Cost & Spend Analyst', market: 'Market Risk Researcher', forecast: 'Forecast Agent', strategy: 'Procurement Strategist', mitigation: 'Mitigation Planner', qa: 'QA Reviewer', brief: 'Executive Writer' };
  const gantt = h('div', { class: 'chart short', role: 'img', 'aria-label': 'Timeline of agent stages' });
  const st = R.events.filter((e) => e.type === 'stage_start' || e.type === 'stage_end');
  const spans = [];
  st.filter((e) => e.type === 'stage_start').forEach((s) => { const e = st.find((x) => x.type === 'stage_end' && x.stage === s.stage && x.seq > s.seq); if (e) spans.push({ stage: s.stage, a: s.t, b: e.t, status: e.status }); });
  root.append(h('div', { class: 'grid-2' },
    panel('QA review', `Verdict: ${qa.verdict || 'n/a'} · confidence ${Math.round((qa.confidence || 0) * 100)}%`,
      (qa.checks || []).length ? h('ul', { class: 'checks' }, qa.checks.map((c) => h('li', { class: c.status }, icon(c.status === 'verified' ? 'check' : 'alert'), h('div', {}, h('b', {}, c.claim), h('span', { class: 'note' }, `${c.status}: ${c.note}`))))) : h('p', { class: 'meta' }, 'No checks recorded.'),
      (qa.guardrail_flags || []).length ? h('div', { style: { marginTop: '14px' } }, h('h3', { class: 'meta', style: { marginBottom: '8px' } }, 'Automated guardrail flags'), h('ul', { class: 'checks' }, qa.guardrail_flags.map((f) => h('li', { class: 'corrected' }, icon('flag'), h('span', {}, f))))) : null),
    panel('Data quality', `Score ${q.quality_score}/100 · auditor verdict: ${(R.audit?.verdict || '').replaceAll('_', ' ')}`,
      h('ul', { class: 'checks' }, (q.issues || []).map((i) => h('li', { class: 'corrected' }, icon('alert'), h('div', {}, h('b', {}, `${i.count} ${i.type.replace('_', ' ')} in ${i.column}`), h('span', { class: 'note' }, `${i.action}. ${(i.examples || []).slice(0, 2).join('; ')}`))))),
      (R.audit?.caveats || []).length ? h('ul', { class: 'notes', style: { marginTop: '12px' } }, R.audit.caveats.map((c) => h('li', {}, c))) : null),
    panel('Grounding guard', 'Facts checked in code after every agent', (R.grounding || []).length
      ? h('ul', { class: 'checks' }, R.grounding.map((g) => h('li', { class: 'corrected' }, icon('shield'), h('span', {}, g))))
      : h('ul', { class: 'checks' }, h('li', { class: 'verified' }, icon('check'), h('span', {}, 'Every vendor named by the agents exists in the dataset and every savings figure is within the data-supported maximum.')))),
    panel('Routing', `${(R.path || []).length} stage executions · ${R.revisions || 0} QA revision round(s)`,
      h('p', { class: 'rationale' }, (R.path || []).map((p) => stageNames[p] || p).join(' → '))),
    h('div', { class: 'span-2' }, panel('Stage timeline', `Path: ${R.path.map((p) => stageNames[p] || p).join(' → ')}`, gantt)),
    panel('Model usage', `${num(u.llm_calls)} calls · ${num(u.total_tokens)} tokens · ${num(u.failovers)} failovers · ${u.elapsed}s`,
      h('div', { class: 'vt-wrap' }, h('table', { class: 'usage' }, h('thead', {}, h('tr', {}, ['Model', 'Calls', 'Tokens in', 'Tokens out', 'Avg s', 'Failovers in'].map((x) => h('th', { scope: 'col' }, x)))),
        h('tbody', {}, Object.entries(u.per_model).sort((a, b) => b[1].calls - a[1].calls).map(([k, m]) => h('tr', {}, h('td', {}, k.split(':').slice(1).join(':')), h('td', {}, m.calls), h('td', {}, num(m.prompt_tokens)), h('td', {}, num(m.completion_tokens)), h('td', {}, (m.seconds / Math.max(1, m.calls)).toFixed(1)), h('td', {}, m.failovers))))))),
    panel('Intake plan', R.plan?.question_type?.replaceAll('_', ' '),
      h('p', { class: 'rationale' }, R.plan?.objective || ''),
      h('ul', { class: 'notes', style: { marginTop: '10px', color: 'var(--ink-2)' } }, (R.plan?.success_criteria || []).map((c) => h('li', {}, c))),
      R.plan?.instructions_for_team ? h('p', { class: 'meta' }, R.plan.instructions_for_team) : null)));
  chart(gantt, (t) => ({
    ...base(t), grid: { left: 170, right: 20, top: 8, bottom: 28 },
    tooltip: { ...base(t).tooltip, formatter: (p) => `${esc(stageNames[p.data.stage])}<br>${p.data.a.toFixed(1)}s → ${p.data.b.toFixed(1)}s (${(p.data.b - p.data.a).toFixed(1)}s)` },
    xAxis: { type: 'value', name: 'seconds', ...axis(t) },
    yAxis: { type: 'category', inverse: true, data: Object.keys(stageNames).map((k) => stageNames[k]), ...axis(t), axisLabel: { color: t.ink2, fontFamily: 'Geist', fontSize: 11 } },
    series: [{ type: 'custom', renderItem: (params, api2) => {
      const y = api2.value(2), s = api2.coord([api2.value(0), y]), e = api2.coord([api2.value(1), y]);
      const hgt = api2.size([0, 1])[1] * 0.55;
      return { type: 'rect', shape: { x: s[0], y: s[1] - hgt / 2, width: Math.max(3, e[0] - s[0]), height: hgt, r: 4 }, style: { fill: api2.value(3) ? t.brand : t.watch } };
    }, encode: { x: [0, 1], y: 2 }, data: spans.map((sp) => ({ value: [sp.a, sp.b, stageNames[sp.stage], sp.status === 'ok' ? 1 : 0], stage: sp.stage, a: sp.a, b: sp.b })) }],
  }));
}

Writing safeprocure/web/insights.js

## 4 · Dynamic input
Ask anything about your supplier base and set what matters. Change the form and re-run the cells below; the Intake
agent re-plans the analysis for every question. To use your own data, tick **Upload my own CSV / Excel**: columns such
as *Supplier, Month, OTD %, Rejection %, PPV, Spend* are mapped automatically.

In [26]:
import importlib, json, time
import pandas as pd
import safeprocure
from safeprocure import data as D, analytics as A
from safeprocure.config import DEFAULT_WEIGHTS
try:
    import ipywidgets as W
    from IPython.display import display, HTML
    HAVE_W = True
except ImportError:
    HAVE_W = False

form = {}
if HAVE_W:
    style = {"description_width": "150px"}
    lay = W.Layout(width="520px")
    form["question"] = W.Textarea(value="Which vendors put our Q4 production at risk, and what should we do about each one?",
                                  description="Your question", style=style, layout=W.Layout(width="760px", height="64px"))
    form["weights"] = {k: W.IntSlider(value=v, min=0, max=60, description=f"Weight: {k}", style=style, layout=lay) for k, v in DEFAULT_WEIGHTS.items()}
    form["threshold"] = W.IntSlider(value=55, min=35, max=75, description="Risk tolerance", style=style, layout=lay)
    form["otd"] = W.IntSlider(value=85, min=75, max=95, description="On-time floor %", style=style, layout=lay)
    form["defect"] = W.FloatSlider(value=3.0, min=1, max=6, step=0.5, description="Defect ceiling %", style=style, layout=lay)
    form["cats"] = W.SelectMultiple(options=list(D.CATEGORIES), value=list(D.CATEGORIES), description="Categories", style=style, layout=W.Layout(width="520px", height="110px"))
    form["n"] = W.IntSlider(value=20, min=5, max=25, description="Sample vendors", style=style, layout=lay)
    form["months"] = W.IntSlider(value=18, min=6, max=36, description="Months of history", style=style, layout=lay)
    form["seed"] = W.IntText(value=42, description="Random seed", style=style)
    form["upload"] = W.Checkbox(value=False, description="Upload my own CSV / Excel instead")
    display(W.VBox([form["question"], W.HTML("<b>What matters most</b>"), *form["weights"].values(),
                    W.HTML("<b>Scope and tolerance</b>"), form["threshold"], form["otd"], form["defect"], form["cats"],
                    W.HTML("<b>Data</b>"), form["n"], form["months"], form["seed"], form["upload"]]))
else:
    print("ipywidgets not available: defaults will be used (edit CONFIG in the next cell).")

Read the form, load the data and preview the **instant deterministic scorecard** (before any AI is involved).

In [27]:
def read_form():
    if not HAVE_W:
        return {"question": "Which vendors put our Q4 production at risk, and what should we do about each one?",
                "weights": dict(DEFAULT_WEIGHTS), "threshold": 55, "otd": 85, "defect": 3.0,
                "cats": list(D.CATEGORIES), "n": 20, "months": 18, "seed": 42, "upload": False}
    return {"question": form["question"].value, "weights": {k: s.value for k, s in form["weights"].items()},
            "threshold": form["threshold"].value, "otd": form["otd"].value, "defect": form["defect"].value,
            "cats": list(form["cats"].value), "n": form["n"].value, "months": form["months"].value,
            "seed": form["seed"].value, "upload": form["upload"].value}

CONFIG = read_form()
notes = []
if CONFIG["upload"] and IN_COLAB:
    from google.colab import files
    up = files.upload()
    name, raw = next(iter(up.items()))
    df_raw, notes = D.normalise_upload(raw, name)
else:
    df_raw = D.generate(D.GenSpec(n_vendors=CONFIG["n"], months=CONFIG["months"], seed=CONFIG["seed"],
                                  categories=tuple(CONFIG["cats"]) or tuple(D.CATEGORIES)))
    notes = ["Synthetic dataset: all vendor names are fictional."]

clean_df, quality = A.clean(df_raw)
preview = A.score(A.vendor_kpis(clean_df), CONFIG["weights"])
print(f"{quality['vendors']} vendors · {quality['n_months']} months · {A.inr(quality['total_spend'])} spend · "
      f"data quality {quality['quality_score']}/100 · {len(quality['issues'])} issue types auto-cleaned")
for n in notes:
    print("note:", n)
display(preview[["vendor_name", "category", "composite", "tier", "internal_risk", "on_time_pct", "defect_rate_pct",
                 "price_variance_pct", "otd_trend"]].round(2).head(20))

20 vendors · 18 months · ₹27.54 Cr spend · data quality 93.4/100 · 4 issue types auto-cleaned
note: Synthetic dataset: all vendor names are fictional.


,vendor_name,category,composite,tier,internal_risk,on_time_pct,defect_rate_pct,price_variance_pct,otd_trend
0,Coastal Cargo Movers Pvt Ltd,Logistics & Freight,92.08,A,10.96,97.93,0.42,0.36,-0.09
1,Kaveri Alloys Pvt Ltd,Steel & Metals,90.52,A,9.86,96.56,0.48,0.54,-0.09
2,Shree Corrugators Pvt Ltd,Packaging,90.32,A,12.75,95.77,0.49,0.24,-0.29
3,Rasayan Chemicals Pvt Ltd,Chemicals & Plastics,87.72,A,13.17,94.99,0.67,0.79,-0.22
4,GreenWrap Packaging Pvt Ltd,Packaging,80.16,B,22.55,97.08,0.37,8.00,-0.68
5,Quantix Electronics Pvt Ltd,Electronic Components,79.57,B,23.06,91.00,1.22,0.55,-0.19
6,Circuitra Systems Pvt Ltd,Electronic Components,78.77,B,22.54,91.86,1.30,1.37,-0.33
7,SwiftHaul Logistics Pvt Ltd,Logistics & Freight,77.15,B,22.85,90.38,1.09,1.62,0.24
8,PolyShield Films Pvt Ltd,Packaging,76.30,B,26.36,91.62,1.26,2.46,-0.18
9,AquaResin Industries Pvt Ltd,Chemicals & Plastics,75.95,B,25.09,91.36,1.59,1.52,-0.26


## 5 · Run the 10-agent crew
The trace below is the live event stream: which agent is working, which tool it calls, which model answers, and when the
router fails over. A full run takes about 1.5 to 3 minutes on free-tier models.

In [28]:
from safeprocure.flow import prepare_context, run_analysis
from safeprocure.telemetry import RunTelemetry

def printer(ev):
    t, kind = ev["t"], ev["type"]
    a = ev.get("agent", "")
    if kind == "stage_start":
        print(f"{t:6.1f}s  ▶ {a}")
    elif kind == "tool_start":
        args = ", ".join(f"{k}={str(v)[:50]}" for k, v in (ev.get("args") or {}).items())
        print(f"{t:6.1f}s      tool  {ev['tool']}({args})")
    elif kind == "llm_end":
        print(f"{t:6.1f}s      model {ev['model']}  {ev['prompt_tokens']}→{ev['completion_tokens']} tokens  {ev['seconds']}s")
    elif kind == "failover":
        print(f"{t:6.1f}s      router: {a} → {ev['to_model']} ({ev['reason']})")
    elif kind == "stage_end":
        out = ev.get("output") or {}
        summary = out.get("headline") or out.get("summary") or out.get("portfolio_summary") or out.get("objective") or out.get("verdict", "")
        print(f"{t:6.1f}s  ✓ {a} [{ev['status']}, {ev['seconds']}s]  {str(summary)[:150]}")
    elif kind == "route":
        print(f"{t:6.1f}s  ◇ {ev['stage']} → {ev['route']}")
    elif kind == "grounding":
        print(f"{t:6.1f}s  ⚑ grounding guard removed: {ev.get('dropped')}")

ctx = prepare_context(df_raw, weights=CONFIG["weights"], risk_threshold=CONFIG["threshold"], question=CONFIG["question"],
                      otd_floor=CONFIG["otd"], defect_ceiling=CONFIG["defect"], telemetry=RunTelemetry("colab", printer))
RESULT = run_analysis(ctx)
u = RESULT["usage"]
print(f"\nDone in {u['elapsed']} s · {u['llm_calls']} model calls · {u['total_tokens']:,} tokens · {u['failovers']} failovers · "
      f"degraded stages: {RESULT['degraded'] or 'none'}")
os.makedirs("runs", exist_ok=True)
json.dump(RESULT, open("runs/latest.json", "w"), default=str)

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: SafeProcureFlow                                                                                          │
│  ID: b096bc7c-aeb3-43b9-8566-61a8319e17cf                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: SafeProcureFlow                                                                                          │
│  ID: b096bc7c-aeb3-43b9-8566-61a8319e17cf                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: intake                                                                                                 │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

   2.3s  ▶ Intake Router


   4.5s      model GPT-OSS 20B  1399→441 tokens  1.73s
   4.5s  ✓ Intake Router [ok, 2.2s]  Identify vendors that pose a risk to Q4 production and recommend mitigation actions for each
   4.5s  ▶ Data Quality Auditor


...

  78.2s  ▶ Executive Writer
  78.2s      router: Executive Writer → Gemini 3.1 Flash-Lite (budget)


  93.8s      model Gemini 3.1 Flash-Lite  2047→554 tokens  15.31s
  93.8s  ✓ Executive Writer [ok, 15.7s]  Q4 2026 Supply Chain Risk: Apex Metalworks Replacement and Targeted Vendor Monitoring Required


╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: brief                                                                                                  │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: SafeProcureFlow                                                                                          │
│  ID: b096bc7c-aeb3-43b9-8566-61a8319e17cf                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Done in 93.9 s · 26 model calls · 73,046 tokens · 13 failovers · degraded stages: none


## 6 · Executive brief

In [29]:
b = RESULT["brief"]
from collections import Counter
dec = Counter((v.get("decision") or {}).get("decision") for v in RESULT["vendors"] if v["in_scope"])
html_out = f'''
<div style="font-family:system-ui;max-width:900px;border:1px solid #ddd;border-radius:14px;padding:24px 28px">
<div style="color:#777;font-style:italic">“{RESULT['question']}”</div>
<h2 style="font-family:Georgia,serif;font-weight:400;font-size:30px;margin:8px 0">{b.get('headline','')}</h2>
<p style="font-size:16px">{b.get('answer','')}</p>
{''.join(f'<p style="color:#444">{p}</p>' for p in b.get('summary', []))}
<p><b>Key numbers:</b> {' · '.join(f"{k['label']}: <b>{k['value']}</b>" for k in b.get('key_numbers', []))}</p>
<p><b>Decisions:</b> {' · '.join(f'{k} {dec.get(k, 0)}' for k in ('Keep','Watch','Develop','Replace'))}</p>
<ol>{''.join(f"<li><b>{a['action']}</b> · {a['owner']} · {a['timeline']} · {a['impact']}</li>" for a in b.get('top_actions', []))}</ol>
<p style="color:#777">QA verdict: {RESULT['qa'].get('verdict')} · revisions: {RESULT['revisions']} · grounding fixes: {len(RESULT.get('grounding', []))}</p>
</div>'''
display(HTML(html_out))

## 7 · Inline dashboard

In [30]:
import plotly.express as px, plotly.graph_objects as go
from plotly.subplots import make_subplots
vd = pd.DataFrame([{**{k: v[k] for k in ("vendor", "category", "composite", "internal_risk", "market_risk", "combined_risk", "spend_total")},
                    "decision": (v.get("decision") or {}).get("decision", "n/a"), **v["pillars"]} for v in RESULT["vendors"]])
colors = {"Keep": "#0B7A4E", "Watch": "#A85A06", "Develop": "#0A6B9E", "Replace": "#BE2A21", "n/a": "#999"}
th = RESULT["config"]["risk_threshold"]
fig = px.scatter(vd, x="internal_risk", y="market_risk", size="spend_total", color="decision", color_discrete_map=colors,
                 hover_name="vendor", size_max=46, range_x=[0, 100], range_y=[0, 100],
                 title="Risk matrix: internal performance risk vs live market risk (bubble = spend)")
fig.add_shape(type="line", x0=th / 0.65, y0=0, x1=(th - 35) / 0.65, y1=100, line=dict(dash="dash", color="#BE2A21"))
fig.update_layout(template="plotly_white", height=480)
fig.show()

hm = vd.sort_values("combined_risk", ascending=False)
fig = go.Figure(go.Heatmap(z=hm[["delivery", "quality", "cost", "responsiveness", "compliance"]].values, x=["Delivery", "Quality", "Cost", "Responsiveness", "Compliance"],
                           y=hm["vendor"].str.replace(" Pvt Ltd", ""), colorscale="RdYlGn", zmin=0, zmax=100, text=hm[["delivery", "quality", "cost", "responsiveness", "compliance"]].round(0).values, texttemplate="%{text}"))
fig.update_layout(title="Pillar scores (0-100) sorted by combined risk", template="plotly_white", height=620, yaxis_autorange="reversed")
fig.show()

sp = RESULT["spend"]; sv = pd.DataFrame(sp["vendors"])
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_bar(x=sv.vendor_name.str.replace(" Pvt Ltd", ""), y=sv.spend_total, name="Spend (INR)", marker_color="#3E3AE0")
fig.add_scatter(x=sv.vendor_name.str.replace(" Pvt Ltd", ""), y=sv.cum_share * 100, name="Cumulative %", secondary_y=True, line_color="#A85A06")
fig.update_layout(title=f"Spend Pareto: {sp['pareto_vendors_80pct']} of {sp['vendor_count']} vendors hold 80% of spend", template="plotly_white", height=440)
fig.show()

worst = sorted([v for v in RESULT["vendors"] if v.get("forecast")], key=lambda v: v["forecast"]["otd"]["slope_per_month"])[:3]
fig = go.Figure()
for v in worst:
    f = v["forecast"]; name = v["vendor"].replace(" Pvt Ltd", "")
    fig.add_scatter(x=f["history_months"], y=f["otd"]["history"], name=f"{name} (actual)")
    fig.add_scatter(x=f["future_months"], y=f["otd"]["forecast"], name=f"{name} (forecast)", line_dash="dash")
fig.add_hline(y=RESULT["config"]["otd_floor"], line_dash="dot", line_color="#BE2A21", annotation_text="on-time floor")
fig.update_layout(title="On-time delivery: 12-month history and 3-month forecast (worst trends)", template="plotly_white", height=420)
fig.show()

### Decisions, mitigation and market intelligence

In [31]:
table = pd.DataFrame([{"vendor": v["vendor"].replace(" Pvt Ltd", ""), "category": v["category"], "score": v["composite"], "tier": v["tier"],
                       "combined_risk": v["combined_risk"], "decision": (v.get("decision") or {}).get("decision"),
                       "rationale": (v.get("decision") or {}).get("rationale")} for v in RESULT["vendors"]]).sort_values("combined_risk", ascending=False)
def _risk_bg(v):
    c = "#E2F3EA" if v < 40 else "#FBF0DF" if v < RESULT["config"]["risk_threshold"] else "#FBE6E3"
    return f"background-color:{c}"
display(table.style.map(_risk_bg, subset=["combined_risk"]).format({"score": "{:.1f}", "combined_risk": "{:.0f}"}).hide(axis="index"))
for p in RESULT["mitigation"].get("plans", []):
    print(f"\nMitigation for {p['vendor']}: shift → " + ", ".join(f"{a['to_vendor']} ({a['share_of_volume_pct']:.0f}%)" for a in p["alternatives"]))
    for k in ("days_30", "days_60", "days_90"):
        print(f"  {k.replace('_', ' ')}: " + "; ".join(p.get(k, [])))
print("\nLive market signals:")
for s in sorted(RESULT["market"].get("signals", []), key=lambda s: -s["risk_score"]):
    print(f"  {s['risk_score']:>3.0f}  {s['category'] if s['category'] != 'All' else s['region']:<24} {s['headline'][:90]}  ({s.get('source','')} {s.get('date','')})")
print("\nGrounding guard:", RESULT.get("grounding") or "no ungrounded claims found")

vendor,category,score,tier,combined_risk,decision,rationale
Apex Metalworks,Steel & Metals,58.6,C,71,Replace,"Rule-based: composite 59, combined risk 71 vs threshold 55."
Pinnacle Semicon Traders,Electronic Components,56.1,C,54,Watch,"Rule-based: composite 56, combined risk 54 vs threshold 55."
Bharat Freight Lines,Logistics & Freight,55.9,C,49,Watch,"Rule-based: composite 56, combined risk 49 vs threshold 55."
Indus Castings,Steel & Metals,58.2,C,46,Watch,"Rule-based: composite 58, combined risk 46 vs threshold 55."
Vidyut Components,Electronic Components,60.1,C,46,Develop,"Rule-based: composite 60, combined risk 46 vs threshold 55."
Sahyadri Steel Forge,Steel & Metals,61.3,C,44,Watch,"Rule-based: composite 61, combined risk 44 vs threshold 55."
Zenith Specialty Chem,Chemicals & Plastics,59.6,C,43,Develop,"Rule-based: composite 60, combined risk 43 vs threshold 55."
NovaPack Solutions,Packaging,66.9,C,37,Develop,"Rule-based: composite 67, combined risk 37 vs threshold 55."
TransEast Carriers,Logistics & Freight,74.3,B,36,Keep,"Rule-based: composite 74, combined risk 35 vs threshold 55."
Quantix Electronics,Electronic Components,79.6,B,34,Keep,OTD 91.0% and 1.2% defect rate are within acceptable limits; risk score 34.1.



Mitigation for Apex Metalworks Pvt Ltd: shift → Kaveri Alloys Pvt Ltd (10%), Indus Castings Pvt Ltd (10%)
  days 30: Initiate qualification audit with Kaveri Alloys and Indus Castings to confirm capacity, quality control, and compliance records.; Issue trial purchase orders (5% of total volume) to each alternative to validate lead times and defect rates.; Negotiate contract clauses for minimum order quantity, price variance caps, and penalty for OTD < 85%.
  days 60: Analyze trial PO results; adjust volume share if necessary to stay within headroom limits.; Implement safety stock buffer of 15 days for each alternative to cushion any initial ramp‑up delays.; Set up joint quality review meetings with each vendor to align on defect thresholds and corrective action plans.
  days 90: Finalize full dual‑sourcing agreement with both vendors, locking in 20% of Apex’s volume split 10/10.; Integrate alternative vendors into the ERP procurement workflow and update supplier master data.; Conduct 

### Downloadable executive report

In [32]:
from safeprocure.report import render_report
open("safe_procure_report.html", "w", encoding="utf-8").write(render_report(RESULT))
print("Saved safe_procure_report.html")
if IN_COLAB:
    from google.colab import files
    files.download("safe_procure_report.html")

Saved safe_procure_report.html


## 8 · Ask the Procurement Copilot
A tool-using agent that re-checks numbers, runs what-if scenarios and finds alternatives. Edit the question and run again.

In [33]:
from safeprocure.copilot import answer
q = "If delivery mattered twice as much as cost, would any vendor's decision change?"
r = answer(RESULT, q, [], ctx)
print("Q:", q, "\n\nA:", r["answer"], f"\n\n(tools used: {r['tools'] or 'none'} · models: {r['models']} · {r['seconds']} s)")

Q: If delivery mattered twice as much as cost, would any vendor's decision change? 

A: If delivery were weighted twice as heavily as cost, no vendor's decision status (Keep, Watch, Develop, or Replace) would change.

While the scenario analysis shows that five vendors—GreenWrap Packaging Pvt Ltd, AquaResin Industries Pvt Ltd, SwiftHaul Logistics Pvt Ltd, Bharat Freight Lines Pvt Ltd, and Indus Castings Pvt Ltd—would experience shifts in their rankings or scores, these adjustments are not significant enough to alter their current risk classification or strategic designation. The existing procurement strategy remains robust even under this alternative prioritization. 

(tools used: ['scenario_rescore'] · models: ['Gemini 3.1 Flash-Lite'] · 3.8 s)


## 9 · Launch the full web app (dashboard + live agent mission control)
Starts the FastAPI server inside this runtime and opens it in a Colab window. Everything is interactive: change data and
priorities, launch new runs and watch all 10 agents work live, explore 9 insight tabs, replay this run, and chat with the
copilot.

In [34]:
import threading, socket
import uvicorn
PORT = 8899
def _serve():
    uvicorn.run("safeprocure.server:app", host="0.0.0.0", port=PORT, log_level="warning")
if not any(t.name == "safe-procure" for t in threading.enumerate()):
    threading.Thread(target=_serve, name="safe-procure", daemon=True).start()
for _ in range(40):
    try:
        socket.create_connection(("127.0.0.1", PORT), timeout=0.5).close(); break
    except OSError:
        time.sleep(0.25)
if IN_COLAB:
    from google.colab import output
    output.serve_kernel_port_as_window(PORT, anchor_text="Open Safe Procure in a new tab")
    output.serve_kernel_port_as_iframe(PORT, height=900)
else:
    print(f"Safe Procure is running at http://127.0.0.1:{PORT}")

Safe Procure is running at http://127.0.0.1:8899


## 10 · Business value and observations

* **From score to decision.** The output is a defended **Keep / Watch / Develop / Replace** call for every vendor, with
  owners, dates and a mitigation plan, not just a ranked list.
* **Internal and external risk in one view.** Live news shifts vendor risk: a strong supplier in a disrupted category or
  region is flagged before its own KPIs slip.
* **Early warning.** Forecasts flag vendors projected to breach the on-time floor or defect ceiling next quarter.
* **Savings with evidence.** Price-variance overpayment and concentration (HHI) analysis quantify negotiable savings,
  capped at what the data supports.
* **Trustworthy AI.** Numbers come from code; a grounding guard removes invented vendors and figures; an independent QA
  agent can force a revision; every run leaves a full audit trail of agents, tools, models and timings.
* **Resilient on free infrastructure.** Multi-provider routing keeps a 10-agent workflow inside free-tier limits
  (about 1.5 to 3 minutes per run in testing).

**Limitations.** The sample data is synthetic; external risk is scored by an LLM from news headlines (indicative, with
sources shown); forecasts are short-horizon damped linear trends with uncertainty bands, not causal models.